In [31]:
# =========================
# Phase 1 — Dataset Intelligence
# Validation Analysis
# Vehicle Behaviour Analysis
# Temporal Analysis
# Violation Analysis
# =========================

import ast
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# -------------------------
# Config
# -------------------------
INPUT_CSV = "jan to may police violation_anonymized791b166.csv"
OUT_DIR = Path("content/phase1_outputs_1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_N = 25
EPS = 1e-9

# -------------------------
# Helpers
# -------------------------
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def parse_listlike(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    s = str(value).strip()
    if not s:
        return []
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple)):
            return list(parsed)
        return [parsed]
    except Exception:
        if s.startswith("[") and s.endswith("]"):
            s = s[1:-1]
        parts = [p.strip().strip("'").strip('"') for p in s.split(",")]
        return [p for p in parts if p]

def week_start_monday(series):
    dt = pd.to_datetime(series, errors="coerce", utc=True).dt.tz_convert("Asia/Kolkata")
    week_start = dt.dt.normalize() - pd.to_timedelta(dt.dt.weekday, unit="D")
    return week_start.dt.tz_localize(None)

def safe_div(a, b):
    return a / (b + EPS)

def ensure_required_columns(df, cols):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

def save_plot(path):
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()

def severity_from_tags(tags):
    """
    Maps violation tags to a 1–5 severity score.
    Uses the highest applicable severity if multiple tags exist.
    """
    severity_map = {
        5: {
            "DOUBLE PARKING",
            "NEAR ROAD CROSSING",
            "NEAR TRAFFIC LIGHT",
            "NEAR TRAFFIC LIGHT / ZEBRA CROSSING",
            "NEAR ZEBRA CROSSING",
        },
        4: {
            "PARKING IN MAIN ROAD",
            "NEAR BUS STOP",
            "NEAR SCHOOL",
            "NEAR HOSPITAL",
            "OPPOSITE ANOTHER VEHICLE",
        },
        3: {
            "PARKING ON FOOTPATH",
        },
        2: {
            "WRONG PARKING",
            "PARKING OTHER THAN BUS STOP",
        },
        1: {
            "NO PARKING",
        },
    }

    if not tags:
        return 1

    normalized = [clean_text(t).upper() for t in tags]
    score = 1
    for s, vocab in severity_map.items():
        if any(tag in vocab for tag in normalized):
            score = max(score, s)
    return score

def dominant_tag(tags):
    if not tags:
        return ""
    return clean_text(tags[0]).upper()

# -------------------------
# Load
# -------------------------
df = pd.read_csv(INPUT_CSV, low_memory=False)

required_cols = [
    "id", "latitude", "longitude", "vehicle_number", "vehicle_type",
    "violation_type", "created_datetime", "police_station",
    "junction_name", "validation_status"
]
ensure_required_columns(df, required_cols)

for col in ["validation_status", "police_station", "junction_name", "vehicle_number", "vehicle_type"]:
    df[col] = df[col].astype("string")

# -------------------------
# Core parsing
# -------------------------
df["created_datetime_parsed"] = pd.to_datetime(df["created_datetime"], errors="coerce", utc=True)
df["created_datetime_ist"] = df["created_datetime_parsed"].dt.tz_convert("Asia/Kolkata")
df["week_start"] = week_start_monday(df["created_datetime"])
df["hour_ist"] = df["created_datetime_ist"].dt.hour
df["day_of_week"] = df["created_datetime_ist"].dt.day_name()
df["month"] = df["created_datetime_ist"].dt.month
df["is_weekend"] = df["day_of_week"].isin(["Saturday", "Sunday"]).astype(int)

df["violation_tags"] = df["violation_type"].apply(parse_listlike)
df["tag_count"] = df["violation_tags"].apply(len)
df["severity_score"] = df["violation_tags"].apply(severity_from_tags)
df["dominant_violation_tag"] = df["violation_tags"].apply(dominant_tag)

junction_clean = df["junction_name"].fillna("").astype(str).str.strip()
has_junction = junction_clean.ne("") & junction_clean.str.upper().ne("NO JUNCTION")
police_clean = df["police_station"].fillna("UNKNOWN").astype(str).str.strip()
police_clean = police_clean.where(police_clean.ne(""), "UNKNOWN")

df["hotspot_unit"] = np.where(
    has_junction,
    "JUNCTION::" + junction_clean,
    "POLICE_STATION::" + police_clean
)

validation_clean = df["validation_status"].fillna("").astype(str).str.lower()
df["validation_status_clean"] = validation_clean

# A simple numeric flag useful for later phases
status_score_map = {
    "approved": 1.0,
    "rejected": 0.0,
    "processing": 0.5,
    "duplicate": 0.25,
    "created1": 0.5,
}
df["validation_score"] = df["validation_status_clean"].map(status_score_map).fillna(0.5)

# Keep only rows with valid time and coordinates for analysis tables that need them
df_valid = df.dropna(subset=["created_datetime_parsed", "latitude", "longitude"]).copy()

# -------------------------
# 1) Validation Analysis
# -------------------------
validation_summary = (
    df["validation_status_clean"]
    .replace("", "missing")
    .value_counts(dropna=False)
    .rename_axis("validation_status")
    .reset_index(name="count")
)
validation_summary["percent"] = 100 * validation_summary["count"] / len(df)

approval_rate = safe_div(
    int((df["validation_status_clean"] == "approved").sum()),
    len(df)
)

validation_by_vehicle_type = (
    df.groupby("vehicle_type", dropna=False)
    .agg(
        total_records=("id", "size"),
        approved_records=("validation_status_clean", lambda s: (s == "approved").sum()),
        rejected_records=("validation_status_clean", lambda s: (s == "rejected").sum()),
        approved_rate=("validation_status_clean", lambda s: (s == "approved").mean()),
    )
    .reset_index()
    .sort_values("total_records", ascending=False)
)

validation_by_police_station = (
    df.groupby("police_station", dropna=False)
    .agg(
        total_records=("id", "size"),
        approved_rate=("validation_status_clean", lambda s: (s == "approved").mean()),
        rejected_rate=("validation_status_clean", lambda s: (s == "rejected").mean()),
    )
    .reset_index()
    .sort_values(["total_records", "approved_rate"], ascending=[False, False])
)

validation_by_hotspot = (
    df.groupby("hotspot_unit", dropna=False)
    .agg(
        total_records=("id", "size"),
        approved_rate=("validation_status_clean", lambda s: (s == "approved").mean()),
        rejected_rate=("validation_status_clean", lambda s: (s == "rejected").mean()),
        processed_rate=("validation_status_clean", lambda s: (s == "processing").mean()),
    )
    .reset_index()
    .sort_values("total_records", ascending=False)
)

# Validation uncertainty score: high when approval rate is low or mixed
validation_by_hotspot["validation_uncertainty"] = 1.0 - validation_by_hotspot["approved_rate"]
validation_by_vehicle_type["validation_uncertainty"] = 1.0 - validation_by_vehicle_type["approved_rate"]

# -------------------------
# 2) Vehicle Behaviour Analysis
# -------------------------
vehicle_summary = (
    df.groupby("vehicle_number", dropna=False)
    .agg(
        total_violations=("id", "size"),
        unique_hotspots=("hotspot_unit", "nunique"),
        unique_junctions=("junction_name", lambda s: s.fillna("").astype(str).replace("No Junction", "").nunique()),
        first_seen=("created_datetime_parsed", "min"),
        last_seen=("created_datetime_parsed", "max"),
        vehicle_type_mode=("vehicle_type", lambda s: s.mode().iloc[0] if not s.mode().empty else ""),
        approved_violations=("validation_status_clean", lambda s: (s == "approved").sum()),
    )
    .reset_index()
)

vehicle_summary["active_days"] = (
    df_valid.groupby("vehicle_number")["created_datetime_ist"]
    .apply(lambda s: s.dt.date.nunique())
    .reindex(vehicle_summary["vehicle_number"])
    .values
)

vehicle_summary["repeat_violation_flag"] = (vehicle_summary["total_violations"] >= 2).astype(int)
vehicle_summary["chronic_offender_flag"] = (vehicle_summary["total_violations"] >= 5).astype(int)

vehicle_summary["approval_rate"] = safe_div(
    vehicle_summary["approved_violations"],
    vehicle_summary["total_violations"]
)

vehicle_summary = vehicle_summary.sort_values(
    ["total_violations", "approved_violations", "unique_hotspots"],
    ascending=[False, False, False]
).reset_index(drop=True)

top_vehicle_offenders = vehicle_summary.head(TOP_N).copy()
chronic_offenders = vehicle_summary[vehicle_summary["chronic_offender_flag"] == 1].copy()

# Per vehicle-type behaviour
vehicle_type_summary = (
    df.groupby("vehicle_type", dropna=False)
    .agg(
        records=("id", "size"),
        unique_vehicles=("vehicle_number", "nunique"),
        mean_severity=("severity_score", "mean"),
        median_severity=("severity_score", "median"),
        approval_rate=("validation_status_clean", lambda s: (s == "approved").mean()),
    )
    .reset_index()
    .sort_values("records", ascending=False)
)

# -------------------------
# 3) Temporal Analysis
# -------------------------
hourly_distribution = (
    df_valid.groupby("hour_ist")
    .size()
    .reset_index(name="count")
    .sort_values("hour_ist")
)
daily_distribution = (
    df_valid.groupby("day_of_week")
    .size()
    .reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])
    .reset_index(name="count")
)
weekly_distribution = (
    df_valid.groupby("week_start")
    .size()
    .reset_index(name="count")
    .sort_values("week_start")
)
monthly_distribution = (
    df_valid.groupby("month")
    .size()
    .reset_index(name="count")
    .sort_values("month")
)

temporal_by_hour_vehicle_type = (
    df_valid.groupby(["hour_ist", "vehicle_type"])
    .size()
    .reset_index(name="count")
)

temporal_by_day_validation = (
    df_valid.groupby(["day_of_week", "validation_status_clean"])
    .size()
    .reset_index(name="count")
)

peak_hour = int(hourly_distribution.loc[hourly_distribution["count"].idxmax(), "hour_ist"]) if len(hourly_distribution) else None
peak_day = daily_distribution.loc[daily_distribution["count"].idxmax(), "day_of_week"] if len(daily_distribution) else None

# Trend by week: useful for later emerging hotspot stage
weekly_trend_by_hotspot = (
    df_valid.groupby(["hotspot_unit", "week_start"])
    .size()
    .reset_index(name="weekly_count")
    .sort_values(["hotspot_unit", "week_start"])
)

# -------------------------
# 4) Violation Analysis
# -------------------------
# Tag-level analysis
tag_rows = []
for _, row in df.iterrows():
    tags = row["violation_tags"]
    if not tags:
        tag_rows.append({"violation_tag": "UNKNOWN", "id": row["id"], "vehicle_number": row["vehicle_number"], "hotspot_unit": row["hotspot_unit"]})
    else:
        for t in tags:
            tag_rows.append({
                "violation_tag": clean_text(t).upper(),
                "id": row["id"],
                "vehicle_number": row["vehicle_number"],
                "hotspot_unit": row["hotspot_unit"],
            })

tag_df = pd.DataFrame(tag_rows)

violation_tag_summary = (
    tag_df.groupby("violation_tag")
    .agg(
        tag_frequency=("id", "size"),
        unique_vehicles=("vehicle_number", "nunique"),
        unique_hotspots=("hotspot_unit", "nunique"),
    )
    .reset_index()
    .sort_values("tag_frequency", ascending=False)
)

severity_distribution = (
    df.groupby("severity_score")
    .size()
    .reset_index(name="count")
    .sort_values("severity_score")
)

violation_by_vehicle_type = (
    df.groupby("vehicle_type")
    .agg(
        records=("id", "size"),
        mean_severity=("severity_score", "mean"),
        approval_rate=("validation_status_clean", lambda s: (s == "approved").mean()),
    )
    .reset_index()
    .sort_values("records", ascending=False)
)

violation_by_hotspot = (
    df.groupby("hotspot_unit")
    .agg(
        records=("id", "size"),
        mean_severity=("severity_score", "mean"),
        approval_rate=("validation_status_clean", lambda s: (s == "approved").mean()),
        unique_vehicles=("vehicle_number", "nunique"),
    )
    .reset_index()
    .sort_values(["records", "mean_severity"], ascending=[False, False])
)

# Tag co-occurrence patterns
combo_counts = (
    df["violation_tags"]
    .apply(lambda x: tuple(sorted([clean_text(t).upper() for t in x])) if x else ("UNKNOWN",))
    .value_counts()
    .reset_index()
)
combo_counts.columns = ["violation_combo", "frequency"]
combo_counts["violation_combo"] = combo_counts["violation_combo"].apply(lambda x: " | ".join(x))

# -------------------------
# Consolidated Phase 1 feature table
# -------------------------
phase1_features = df.copy()

# Useful derived columns for later stages
phase1_features["validation_is_approved"] = (phase1_features["validation_status_clean"] == "approved").astype(int)
phase1_features["validation_is_rejected"] = (phase1_features["validation_status_clean"] == "rejected").astype(int)
phase1_features["vehicle_total_violations"] = phase1_features.groupby("vehicle_number")["id"].transform("count")
phase1_features["vehicle_chronic_flag"] = (phase1_features["vehicle_total_violations"] >= 5).astype(int)
phase1_features["vehicle_repeat_flag"] = (phase1_features["vehicle_total_violations"] >= 2).astype(int)

phase1_features["hotspot_total_records"] = phase1_features.groupby("hotspot_unit")["id"].transform("count")
phase1_features["hotspot_approval_rate"] = phase1_features.groupby("hotspot_unit")["validation_is_approved"].transform("mean")
phase1_features["hotspot_uncertainty"] = 1.0 - phase1_features["hotspot_approval_rate"]

phase1_features["vehicle_type_total_records"] = phase1_features.groupby("vehicle_type")["id"].transform("count")
phase1_features["vehicle_type_mean_severity"] = phase1_features.groupby("vehicle_type")["severity_score"].transform("mean")

phase1_features["tag_frequency_record"] = phase1_features["dominant_violation_tag"].map(
    violation_tag_summary.set_index("violation_tag")["tag_frequency"]
).fillna(0).astype(int)

# -------------------------
# Save outputs
# -------------------------
validation_summary.to_csv(OUT_DIR / "validation_summary.csv", index=False)
validation_by_vehicle_type.to_csv(OUT_DIR / "validation_by_vehicle_type.csv", index=False)
validation_by_police_station.to_csv(OUT_DIR / "validation_by_police_station.csv", index=False)
validation_by_hotspot.to_csv(OUT_DIR / "validation_by_hotspot.csv", index=False)

vehicle_summary.to_csv(OUT_DIR / "vehicle_summary.csv", index=False)
top_vehicle_offenders.to_csv(OUT_DIR / "top_vehicle_offenders.csv", index=False)
chronic_offenders.to_csv(OUT_DIR / "chronic_offenders.csv", index=False)
vehicle_type_summary.to_csv(OUT_DIR / "vehicle_type_summary.csv", index=False)

hourly_distribution.to_csv(OUT_DIR / "hourly_distribution.csv", index=False)
daily_distribution.to_csv(OUT_DIR / "daily_distribution.csv", index=False)
weekly_distribution.to_csv(OUT_DIR / "weekly_distribution.csv", index=False)
monthly_distribution.to_csv(OUT_DIR / "monthly_distribution.csv", index=False)
temporal_by_hour_vehicle_type.to_csv(OUT_DIR / "temporal_by_hour_vehicle_type.csv", index=False)
temporal_by_day_validation.to_csv(OUT_DIR / "temporal_by_day_validation.csv", index=False)
weekly_trend_by_hotspot.to_csv(OUT_DIR / "weekly_trend_by_hotspot.csv", index=False)

violation_tag_summary.to_csv(OUT_DIR / "violation_tag_summary.csv", index=False)
severity_distribution.to_csv(OUT_DIR / "severity_distribution.csv", index=False)
violation_by_vehicle_type.to_csv(OUT_DIR / "violation_by_vehicle_type.csv", index=False)
violation_by_hotspot.to_csv(OUT_DIR / "violation_by_hotspot.csv", index=False)
combo_counts.to_csv(OUT_DIR / "violation_combo_summary.csv", index=False)

phase1_features.to_csv(OUT_DIR / "phase1_enriched_dataset.csv", index=False)

# -------------------------
# Print summary
# -------------------------
print("Phase 1 complete")
print("Total rows:", len(df))
print("Approved rows:", int((df["validation_status_clean"] == "approved").sum()))
print("Approval rate:", round(approval_rate * 100, 2), "%")
print("Unique vehicles:", df["vehicle_number"].nunique())
print("Unique hotspots:", df["hotspot_unit"].nunique())
print("Peak hour:", peak_hour)
print("Peak day:", peak_day)
print("Chronic offenders (>=5):", len(chronic_offenders))
print("Outputs saved to:", OUT_DIR.resolve())

# -------------------------
# Plots
# -------------------------
plt.figure(figsize=(10, 5))
plt.bar(validation_summary["validation_status"].astype(str), validation_summary["count"])
plt.title("Validation Status Distribution")
plt.xlabel("Validation Status")
plt.ylabel("Count")
save_plot(OUT_DIR / "validation_status_distribution.png")

plt.figure(figsize=(12, 5))
plt.bar(hourly_distribution["hour_ist"].astype(int), hourly_distribution["count"])
plt.title("Hourly Violation Distribution (IST)")
plt.xlabel("Hour")
plt.ylabel("Count")
save_plot(OUT_DIR / "hourly_distribution.png")

plt.figure(figsize=(12, 5))
plt.bar(daily_distribution["day_of_week"].astype(str), daily_distribution["count"])
plt.title("Day-of-Week Violation Distribution")
plt.xlabel("Day")
plt.ylabel("Count")
plt.xticks(rotation=30)
save_plot(OUT_DIR / "daily_distribution.png")

plt.figure(figsize=(12, 6))
top_tags = violation_tag_summary.head(TOP_N)
plt.barh(top_tags["violation_tag"][::-1], top_tags["tag_frequency"][::-1])
plt.title("Top Violation Tags")
plt.xlabel("Frequency")
plt.ylabel("Violation Tag")
save_plot(OUT_DIR / "top_violation_tags.png")

plt.figure(figsize=(12, 6))
top_vehicles = top_vehicle_offenders.head(TOP_N).sort_values("total_violations")
plt.barh(top_vehicles["vehicle_number"], top_vehicles["total_violations"])
plt.title("Top Vehicle Offenders")
plt.xlabel("Total Violations")
plt.ylabel("Vehicle Number")
save_plot(OUT_DIR / "top_vehicle_offenders.png")

Phase 1 complete
Total rows: 298450
Approved rows: 115400
Approval rate: 38.67 %
Unique vehicles: 231890
Unique hotspots: 222
Peak hour: 10
Peak day: Sunday
Chronic offenders (>=5): 3489
Outputs saved to: D:\Flipkart_Hackathon\approach1\content\phase1_outputs_1


In [ ]:
import ast
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

INPUT_CSV = "jan to may police violation_anonymized791b166.csv"
PHASE1_PATH = Path("content/phase1_outputs_1/phase1_enriched_dataset.csv")
OUT_DIR = Path("content/phase2_outputs_1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EPS = 1e-9
TOP_N = 25

SEVERITY_RULES = {
    5: [
        "DOUBLE PARKING",
        "NEAR ROAD CROSSING",
        "NEAR TRAFFIC LIGHT",
        "NEAR ZEBRA CROSSING",
        "NEAR TRAFFIC LIGHT / ZEBRA CROSSING",
        "NEAR TRAFFIC LIGHT/ZEBRA CROSSING",
    ],
    4: [
        "PARKING IN MAIN ROAD",
        "NEAR BUS STOP",
        "NEAR SCHOOL",
        "NEAR HOSPITAL",
        "OPPOSITE ANOTHER VEHICLE",
    ],
    3: [
        "PARKING ON FOOTPATH",
    ],
    2: [
        "WRONG PARKING",
        "PARKING OTHER THAN BUS STOP",
    ],
    1: [
        "NO PARKING",
        "NO PARKING (GENERIC)",
    ],
}

def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def parse_listlike(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    s = str(value).strip()
    if not s:
        return []
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple)):
            return list(parsed)
        return [parsed]
    except Exception:
        if s.startswith("[") and s.endswith("]"):
            s = s[1:-1]
        parts = [p.strip().strip("'").strip('"') for p in s.split(",")]
        return [p for p in parts if p]

def normalize_tag(tag):
    s = clean_text(tag).upper()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\s*/\s*", " / ", s)
    s = s.replace("&", "AND")
    return s.strip()

def make_hotspot_unit(row):
    junction = clean_text(row.get("junction_name", ""))
    if junction and junction.upper() != "NO JUNCTION":
        return f"JUNCTION::{junction}"
    station = clean_text(row.get("police_station", "UNKNOWN"))
    if not station:
        station = "UNKNOWN"
    return f"POLICE_STATION::{station}"

def severity_for_tags(tags):
    if not tags:
        return 1, [], "NO PARKING"

    normalized = [normalize_tag(t) for t in tags]
    hit_tags = []
    best = 1

    for tag in normalized:
        tag_best = 1
        matched = False
        for sev in sorted(SEVERITY_RULES.keys(), reverse=True):
            for pattern in SEVERITY_RULES[sev]:
                p = normalize_tag(pattern)
                if p == tag or p in tag:
                    tag_best = sev
                    matched = True
                    break
            if matched:
                break
        best = max(best, tag_best)
        if matched:
            hit_tags.append(tag)

    dominant = hit_tags[0] if hit_tags else normalized[0]
    return best, hit_tags, dominant

def safe_ratio(num, den):
    return num / (den + EPS)

def save_plot(path):
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()

def main():
    source_path = PHASE1_PATH if PHASE1_PATH.exists() else Path(INPUT_CSV)
    df = pd.read_csv(source_path, low_memory=False)

    if "violation_type" not in df.columns:
        raise ValueError("Missing required column: violation_type")

    if "hotspot_unit" not in df.columns:
        if {"junction_name", "police_station"}.issubset(df.columns):
            df["hotspot_unit"] = df.apply(make_hotspot_unit, axis=1)
        else:
            df["hotspot_unit"] = "UNKNOWN"

    if "validation_status" in df.columns:
        df["validation_status_clean"] = df["validation_status"].fillna("").astype(str).str.lower()
    else:
        df["validation_status_clean"] = "approved"

    df["violation_tags"] = df["violation_type"].apply(parse_listlike)
    df["violation_tag_count"] = df["violation_tags"].apply(len)

    sev_res = df["violation_tags"].apply(severity_for_tags)
    df["severity_score"] = sev_res.apply(lambda x: int(x[0]))
    df["matched_severity_tags"] = sev_res.apply(lambda x: x[1])
    df["dominant_violation_tag"] = sev_res.apply(lambda x: x[2])
    df["severity_normalized"] = df["severity_score"] / 5.0

    df["severity_is_very_high"] = (df["severity_score"] == 5).astype(int)
    df["severity_is_high"] = (df["severity_score"] == 4).astype(int)
    df["severity_is_moderate"] = (df["severity_score"] == 3).astype(int)
    df["severity_is_low"] = (df["severity_score"] == 2).astype(int)
    df["severity_is_baseline"] = (df["severity_score"] == 1).astype(int)

    approved_mask = df["validation_status_clean"].eq("approved")
    approved = df.loc[approved_mask].copy()

    severity_record_cols = [
        c for c in [
            "id", "created_datetime", "created_datetime_parsed", "created_datetime_ist",
            "latitude", "longitude", "vehicle_number", "vehicle_type",
            "police_station", "junction_name", "hotspot_unit",
            "validation_status", "validation_status_clean",
            "violation_type", "violation_tags", "violation_tag_count",
            "severity_score", "severity_normalized", "matched_severity_tags",
            "dominant_violation_tag"
        ] if c in df.columns
    ]

    hotspot_summary = (
        approved.groupby("hotspot_unit", dropna=False)
        .agg(
            approved_records=("severity_score", "size"),
            mean_severity=("severity_score", "mean"),
            median_severity=("severity_score", "median"),
            severity_sum=("severity_score", "sum"),
            very_high_count=("severity_is_very_high", "sum"),
            high_count=("severity_is_high", "sum"),
            moderate_count=("severity_is_moderate", "sum"),
            low_count=("severity_is_low", "sum"),
            baseline_count=("severity_is_baseline", "sum"),
            unique_vehicles=("vehicle_number", "nunique") if "vehicle_number" in approved.columns else ("severity_score", "size"),
        )
        .reset_index()
    )

    hotspot_summary["severity_share"] = safe_ratio(hotspot_summary["severity_sum"], hotspot_summary["severity_sum"].sum())
    hotspot_summary["severity_priority_score"] = (
        0.50 * (hotspot_summary["severity_sum"] / (hotspot_summary["severity_sum"].max() + EPS)) +
        0.30 * (hotspot_summary["mean_severity"] / 5.0) +
        0.20 * safe_ratio(hotspot_summary["approved_records"], hotspot_summary["approved_records"].max())
    ) * 100.0

    hotspot_summary = hotspot_summary.sort_values(
        ["severity_priority_score", "severity_sum", "approved_records"],
        ascending=[False, False, False],
    ).reset_index(drop=True)
    hotspot_summary["severity_rank"] = np.arange(1, len(hotspot_summary) + 1)

    vehicle_type_summary = (
        approved.groupby("vehicle_type", dropna=False)
        .agg(
            approved_records=("severity_score", "size"),
            mean_severity=("severity_score", "mean"),
            median_severity=("severity_score", "median"),
            severity_sum=("severity_score", "sum"),
            unique_vehicles=("vehicle_number", "nunique") if "vehicle_number" in approved.columns else ("severity_score", "size"),
        )
        .reset_index()
        .sort_values(["severity_sum", "approved_records"], ascending=[False, False])
    )

    tag_rows = []
    for _, row in approved.iterrows():
        tags = row["violation_tags"]
        if not tags:
            tag_rows.append({"violation_tag": "NO TAG", "severity_score": row["severity_score"], "hotspot_unit": row["hotspot_unit"]})
        else:
            for t in tags:
                tag_rows.append({
                    "violation_tag": normalize_tag(t),
                    "severity_score": row["severity_score"],
                    "hotspot_unit": row["hotspot_unit"],
                })

    tag_df = pd.DataFrame(tag_rows)
    if tag_df.empty:
        tag_summary = pd.DataFrame(columns=["violation_tag", "frequency", "avg_record_severity", "hotspot_count"])
    else:
        tag_summary = (
            tag_df.groupby("violation_tag", dropna=False)
            .agg(
                frequency=("violation_tag", "size"),
                avg_record_severity=("severity_score", "mean"),
                hotspot_count=("hotspot_unit", "nunique"),
            )
            .reset_index()
            .sort_values(["frequency", "avg_record_severity"], ascending=[False, False])
        )

    validation_severity = (
        df.groupby("validation_status_clean", dropna=False)
        .agg(
            records=("severity_score", "size"),
            mean_severity=("severity_score", "mean"),
            severity_sum=("severity_score", "sum"),
        )
        .reset_index()
        .sort_values("records", ascending=False)
    )

    df[severity_record_cols].to_csv(OUT_DIR / "phase2_enriched_dataset.csv", index=False)
    approved[severity_record_cols].to_csv(OUT_DIR / "phase2_approved_severity_dataset.csv", index=False)
    hotspot_summary.to_csv(OUT_DIR / "phase2_hotspot_severity_scores.csv", index=False)
    vehicle_type_summary.to_csv(OUT_DIR / "phase2_vehicle_type_severity.csv", index=False)
    tag_summary.to_csv(OUT_DIR / "phase2_violation_tag_severity.csv", index=False)
    validation_severity.to_csv(OUT_DIR / "phase2_validation_severity.csv", index=False)

    print("Phase 2 complete")
    print("Input source:", source_path)
    print("Total rows:", len(df))
    print("Approved rows:", len(approved))
    print("Hotspots scored:", len(hotspot_summary))
    print("Mean severity (approved):", round(float(approved["severity_score"].mean()) if len(approved) else 0.0, 4))
    print("Top hotspot:", hotspot_summary.iloc[0]["hotspot_unit"] if len(hotspot_summary) else "N/A")
    print("Outputs saved to:", OUT_DIR.resolve())

    if len(hotspot_summary):
        plt.figure(figsize=(12, 6))
        top_hotspots = hotspot_summary.head(TOP_N).sort_values("severity_priority_score", ascending=True)
        plt.barh(top_hotspots["hotspot_unit"], top_hotspots["severity_priority_score"])
        plt.title("Top Hotspots by Severity Priority Score")
        plt.xlabel("Severity Priority Score")
        plt.ylabel("Hotspot")
        save_plot(OUT_DIR / "top_hotspots_by_severity_priority.png")

    if len(tag_summary):
        plt.figure(figsize=(12, 6))
        top_tags = tag_summary.head(TOP_N).sort_values("frequency", ascending=True)
        plt.barh(top_tags["violation_tag"], top_tags["frequency"])
        plt.title("Top Violation Tags (Approved Records)")
        plt.xlabel("Frequency")
        plt.ylabel("Violation Tag")
        save_plot(OUT_DIR / "top_violation_tags_phase2.png")

    if len(validation_severity):
        plt.figure(figsize=(10, 5))
        plt.bar(validation_severity["validation_status_clean"].astype(str), validation_severity["records"])
        plt.title("Records by Validation Status")
        plt.xlabel("Validation Status")
        plt.ylabel("Records")
        plt.xticks(rotation=30)
        save_plot(OUT_DIR / "validation_status_phase2.png")

if __name__ == "__main__":
    main()

Phase 2 complete
Input source: content\phase1_outputs_1\phase1_enriched_dataset.csv
Total rows: 298450
Approved rows: 115400
Hotspots scored: 220
Mean severity (approved): 1.5732
Top hotspot: POLICE_STATION::HAL Old Airport
Outputs saved to: D:\Flipkart_Hackathon\content\phase2_outputs_1


In [ ]:
# =========================
# Phase 3 — ST-DBSCAN Spatial + Temporal Hotspot Discovery
# Input  : Phase 2 approved, severity-scored dataset (preferred)
# Output : Clustered records, cluster summary, noise points, weekly cluster counts
# =========================

import ast
import warnings
from collections import deque
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import BallTree

warnings.filterwarnings("ignore")

# -------------------------
# Config
# -------------------------
RAW_CSV = "jan to may police violation_anonymized791b166.csv"
PHASE2_APPROVED_PATH = Path("content/phase2_outputs_1/phase2_approved_severity_dataset.csv")
PHASE2_ENRICHED_PATH = Path("content/phase2_outputs_1/phase2_enriched_dataset.csv")

OUT_DIR = Path("content/phase3_outputs_1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ST-DBSCAN parameters from the concept note
EPS_SPATIAL_METERS = 150.0
EPS_TEMPORAL_DAYS = 3.0
MIN_PTS = 15

EPS = 1e-9
TOP_N = 25

# -------------------------
# Helpers
# -------------------------
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def parse_listlike(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    s = str(value).strip()
    if not s:
        return []
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple)):
            return list(parsed)
        return [parsed]
    except Exception:
        if s.startswith("[") and s.endswith("]"):
            s = s[1:-1]
        parts = [p.strip().strip("'").strip('"') for p in s.split(",")]
        return [p for p in parts if p]

def normalize_tag(tag):
    return clean_text(tag).upper().replace("&", "AND").strip()

def severity_from_tags(tags):
    """
    Fallback severity mapping if Phase 2 output is unavailable.
    """
    severity_map = {
        5: {
            "DOUBLE PARKING",
            "NEAR ROAD CROSSING",
            "NEAR TRAFFIC LIGHT",
            "NEAR ZEBRA CROSSING",
            "NEAR TRAFFIC LIGHT / ZEBRA CROSSING",
            "NEAR TRAFFIC LIGHT/ZEBRA CROSSING",
        },
        4: {
            "PARKING IN MAIN ROAD",
            "NEAR BUS STOP",
            "NEAR SCHOOL",
            "NEAR HOSPITAL",
            "OPPOSITE ANOTHER VEHICLE",
        },
        3: {"PARKING ON FOOTPATH"},
        2: {"WRONG PARKING", "PARKING OTHER THAN BUS STOP"},
        1: {"NO PARKING", "NO PARKING (GENERIC)"},
    }
    if not tags:
        return 1
    normalized = [normalize_tag(t) for t in tags]
    score = 1
    for sev in sorted(severity_map.keys(), reverse=True):
        vocab = severity_map[sev]
        if any(any(v == tag or v in tag for v in vocab) for tag in normalized):
            score = sev
            break
    return score

def week_start_monday(series):
    dt = pd.to_datetime(series, errors="coerce", utc=True).dt.tz_convert("Asia/Kolkata")
    week_start = dt.dt.normalize() - pd.to_timedelta(dt.dt.weekday, unit="D")
    return week_start.dt.tz_localize(None)

def mode_or_default(s, default=""):
    s = pd.Series(s).dropna().astype(str)
    if s.empty:
        return default
    m = s.mode()
    if m.empty:
        return default
    return m.iloc[0]

def make_hotspot_unit(row):
    j = clean_text(row.get("junction_name", ""))
    if j and j.upper() != "NO JUNCTION":
        return f"JUNCTION::{j}"
    ps = clean_text(row.get("police_station", "UNKNOWN"))
    if not ps:
        ps = "UNKNOWN"
    return f"POLICE_STATION::{ps}"

def safe_ratio(a, b):
    return a / (b + EPS)

def save_plot(path):
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()

# -------------------------
# Load input
# -------------------------
def load_input():
    if PHASE2_APPROVED_PATH.exists():
        df = pd.read_csv(PHASE2_APPROVED_PATH, low_memory=False)
    elif PHASE2_ENRICHED_PATH.exists():
        df = pd.read_csv(PHASE2_ENRICHED_PATH, low_memory=False)
    else:
        df = pd.read_csv(RAW_CSV, low_memory=False)
    return df

# -------------------------
# ST-DBSCAN core
# -------------------------
def build_spatial_coordinates(lat, lon):
    """
    Local tangent-plane approximation in meters.
    Good enough for a city-scale epsilon of 150 m.
    """
    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)

    lat0 = np.deg2rad(np.nanmean(lat))
    lon0 = np.deg2rad(np.nanmean(lon))
    R = 6371000.0

    lat_rad = np.deg2rad(lat)
    lon_rad = np.deg2rad(lon)

    x = R * (lon_rad - lon0) * np.cos(lat0)
    y = R * (lat_rad - lat0)
    return np.c_[x, y]

def st_dbscan(coords_xy, time_days, eps_spatial_m, eps_temporal_days, min_pts):
    """
    ST-DBSCAN implementation using:
    - spatial radius search via BallTree in meters
    - temporal filtering in days
    - BFS expansion exactly like DBSCAN-style density reachability
    """
    n = len(coords_xy)
    tree = BallTree(coords_xy, metric="euclidean")

    labels = np.full(n, -99, dtype=int)   # -99 = unassigned, -1 = noise
    visited = np.zeros(n, dtype=bool)
    core_mask = np.zeros(n, dtype=bool)
    neighbor_cache = {}

    def region_query(i):
        if i in neighbor_cache:
            return neighbor_cache[i]
        idx = tree.query_radius(coords_xy[i:i+1], r=eps_spatial_m, return_distance=False)[0]
        if idx.size == 0:
            neighbor_cache[i] = idx
            return idx
        temporal_ok = np.abs(time_days[idx] - time_days[i]) <= eps_temporal_days
        neigh = idx[temporal_ok]
        neighbor_cache[i] = neigh
        return neigh

    cluster_id = 0

    for i in range(n):
        if visited[i]:
            continue

        visited[i] = True
        neighbors = region_query(i)

        if len(neighbors) < min_pts:
            labels[i] = -1
            continue

        cluster_id += 1
        labels[i] = cluster_id
        core_mask[i] = True

        seed_queue = deque()
        queued = set()

        for j in neighbors:
            if j != i and j not in queued:
                seed_queue.append(j)
                queued.add(j)

        while seed_queue:
            j = seed_queue.popleft()

            if not visited[j]:
                visited[j] = True
                j_neighbors = region_query(j)
                if len(j_neighbors) >= min_pts:
                    core_mask[j] = True
                    for k in j_neighbors:
                        if k not in queued:
                            seed_queue.append(k)
                            queued.add(k)

            if labels[j] in (-99, -1):
                labels[j] = cluster_id

    return labels, core_mask, neighbor_cache

# -------------------------
# Main
# -------------------------
def main():
    df = load_input()

    required = {
        "id", "latitude", "longitude", "vehicle_number", "vehicle_type",
        "violation_type", "created_datetime", "police_station", "junction_name"
    }
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    # Ensure time columns
    if "created_datetime_ist" not in df.columns:
        df["created_datetime_parsed"] = pd.to_datetime(df["created_datetime"], errors="coerce", utc=True)
        df["created_datetime_ist"] = df["created_datetime_parsed"].dt.tz_convert("Asia/Kolkata")
    else:
        df["created_datetime_ist"] = pd.to_datetime(df["created_datetime_ist"], errors="coerce")
        if getattr(df["created_datetime_ist"].dt, "tz", None) is None:
            # If the column is naive, treat it as IST
            df["created_datetime_ist"] = df["created_datetime_ist"].dt.tz_localize("Asia/Kolkata")

    df["created_datetime_ist_naive"] = df["created_datetime_ist"].dt.tz_localize(None)
    df["week_start"] = week_start_monday(df["created_datetime"])

    # If Phase 2 output is not available, create the severity score here
    if "severity_score" not in df.columns:
        df["violation_tags"] = df["violation_type"].apply(parse_listlike)
        df["severity_score"] = df["violation_tags"].apply(severity_from_tags)

    if "validation_status_clean" not in df.columns:
        if "validation_status" in df.columns:
            df["validation_status_clean"] = df["validation_status"].fillna("").astype(str).str.lower()
        else:
            df["validation_status_clean"] = "approved"

    # Stage 3 uses the approved evidence base
    approved = df.loc[df["validation_status_clean"].eq("approved")].copy()
    approved = approved.dropna(subset=["latitude", "longitude", "created_datetime_ist_naive"]).copy()
    approved = approved.reset_index(drop=True)

    if approved.empty:
        raise ValueError("No approved rows with valid latitude/longitude/timestamp were found.")

    # Hotspot unit fallback if not present
    if "hotspot_unit" not in approved.columns:
        approved["hotspot_unit"] = approved.apply(make_hotspot_unit, axis=1)

    # Spatial + temporal numeric arrays
    coords_xy = build_spatial_coordinates(approved["latitude"].values, approved["longitude"].values)
    time_days = (
        approved["created_datetime_ist_naive"].astype("int64").values.astype(np.float64)
        / 86_400_000_000_000.0
    )

    print("Approved rows:", len(approved))
    print("Unique hotspots (pre-clustering):", approved["hotspot_unit"].nunique())
    print("Running ST-DBSCAN with:")
    print(f"  spatial epsilon = {EPS_SPATIAL_METERS:.1f} meters")
    print(f"  temporal epsilon = {EPS_TEMPORAL_DAYS:.1f} days")
    print(f"  MinPts          = {MIN_PTS}")

    labels, core_mask, neighbor_cache = st_dbscan(
        coords_xy=coords_xy,
        time_days=time_days,
        eps_spatial_m=EPS_SPATIAL_METERS,
        eps_temporal_days=EPS_TEMPORAL_DAYS,
        min_pts=MIN_PTS,
    )

    approved["st_dbscan_cluster_id"] = labels
    approved["is_core_point"] = core_mask.astype(int)
    approved["is_noise"] = (approved["st_dbscan_cluster_id"] == -1).astype(int)

    # Human-readable cluster labels
    def cluster_label_for_group(g):
        named_junctions = g["junction_name"].fillna("").astype(str)
        named_junctions = named_junctions[named_junctions.str.upper().ne("NO JUNCTION")]
        named_junctions = named_junctions[named_junctions.str.strip().ne("")]
        if len(named_junctions) > 0:
            return mode_or_default(named_junctions, default="")
        return ""

    cluster_summary = []
    cluster_ids = sorted([c for c in approved["st_dbscan_cluster_id"].unique() if c != -1])

    for cid in cluster_ids:
        g = approved[approved["st_dbscan_cluster_id"] == cid].copy()

        x = coords_xy[g.index.values, 0]
        y = coords_xy[g.index.values, 1]

        # Approximate spread and centroid
        centroid_x = float(np.mean(x))
        centroid_y = float(np.mean(y))
        spatial_radius_m = float(np.max(np.sqrt((x - centroid_x) ** 2 + (y - centroid_y) ** 2))) if len(g) else 0.0

        cluster_name = cluster_label_for_group(g)
        if not cluster_name:
            # Fall back to police station, then coordinates
            station_name = mode_or_default(g["police_station"], default="")
            if station_name and station_name.upper() != "NO JUNCTION":
                cluster_name = f"{station_name}"
            else:
                cluster_name = f"Cluster {cid}"

        cluster_summary.append({
            "st_dbscan_cluster_id": cid,
            "cluster_label": cluster_name,
            "records": len(g),
            "core_points": int(g["is_core_point"].sum()),
            "noise_points": int(g["is_noise"].sum()),
            "start_time_ist": g["created_datetime_ist"].min(),
            "end_time_ist": g["created_datetime_ist"].max(),
            "duration_days": float((g["created_datetime_ist"].max() - g["created_datetime_ist"].min()).total_seconds() / 86400.0) if len(g) else 0.0,
            "latitude_mean": float(g["latitude"].mean()),
            "longitude_mean": float(g["longitude"].mean()),
            "severity_mean": float(g["severity_score"].mean()) if "severity_score" in g.columns else np.nan,
            "severity_sum": float(g["severity_score"].sum()) if "severity_score" in g.columns else np.nan,
            "unique_vehicles": int(g["vehicle_number"].nunique()) if "vehicle_number" in g.columns else 0,
            "dominant_vehicle_type": mode_or_default(g["vehicle_type"], default=""),
            "dominant_police_station": mode_or_default(g["police_station"], default=""),
            "dominant_junction_name": mode_or_default(g["junction_name"], default=""),
            "dominant_violation_tag": mode_or_default(
                g["violation_type"].apply(lambda x: normalize_tag(parse_listlike(x)[0]) if parse_listlike(x) else "NO TAG"),
                default="NO TAG",
            ),
            "approx_spatial_radius_m": spatial_radius_m,
        })

    cluster_summary_df = pd.DataFrame(cluster_summary).sort_values(
        ["records", "severity_sum"],
        ascending=[False, False]
    ).reset_index(drop=True)

    # Trend-friendly weekly cluster counts for later phase
    approved["cluster_week_start"] = approved["created_datetime_ist"].dt.to_period("W-MON").dt.start_time
    weekly_cluster_counts = (
        approved[approved["st_dbscan_cluster_id"] != -1]
        .groupby(["st_dbscan_cluster_id", "cluster_week_start"])
        .size()
        .reset_index(name="weekly_count")
        .sort_values(["st_dbscan_cluster_id", "cluster_week_start"])
    )

    # Merge cluster label back to records
    approved = approved.merge(
        cluster_summary_df[["st_dbscan_cluster_id", "cluster_label"]],
        on="st_dbscan_cluster_id",
        how="left"
    )
    approved["cluster_label"] = approved["cluster_label"].fillna("Noise")

    # Noise rows
    noise_df = approved[approved["st_dbscan_cluster_id"] == -1].copy()

    # Output files
    approved.to_csv(OUT_DIR / "phase3_clustered_dataset.csv", index=False)
    cluster_summary_df.to_csv(OUT_DIR / "phase3_cluster_summary.csv", index=False)
    weekly_cluster_counts.to_csv(OUT_DIR / "phase3_cluster_weekly_counts.csv", index=False)
    noise_df.to_csv(OUT_DIR / "phase3_noise_points.csv", index=False)

    # Helpful compact dispatch-style table for later stages
    dispatch_view = cluster_summary_df[[
        "st_dbscan_cluster_id", "cluster_label", "records", "severity_sum", "severity_mean",
        "unique_vehicles", "dominant_vehicle_type", "dominant_police_station",
        "dominant_junction_name", "approx_spatial_radius_m", "start_time_ist", "end_time_ist"
    ]].copy()
    dispatch_view.to_csv(OUT_DIR / "phase3_cluster_dispatch_view.csv", index=False)

    # Summary printout
    n_clusters = len(cluster_summary_df)
    n_noise = int((approved["st_dbscan_cluster_id"] == -1).sum())
    print("\nPhase 3 complete")
    print("Clusters found:", n_clusters)
    print("Noise points   :", n_noise)
    print("Clustered rows  :", len(approved) - n_noise)
    print("Output folder   :", OUT_DIR.resolve())

    if n_clusters > 0:
        print("\nTop 10 clusters:")
        print(
            cluster_summary_df[[
                "st_dbscan_cluster_id", "cluster_label", "records",
                "severity_sum", "severity_mean", "unique_vehicles"
            ]].head(10).to_string(index=False)
        )

    # Plots
    if len(cluster_summary_df):
        plt.figure(figsize=(12, 6))
        top_clusters = cluster_summary_df.head(TOP_N).sort_values("records", ascending=True)
        plt.barh(top_clusters["cluster_label"], top_clusters["records"])
        plt.title("Top ST-DBSCAN Clusters by Record Count")
        plt.xlabel("Records")
        plt.ylabel("Cluster")
        save_plot(OUT_DIR / "top_clusters_by_records.png")

        plt.figure(figsize=(12, 6))
        top_sev = cluster_summary_df.head(TOP_N).sort_values("severity_sum", ascending=True)
        plt.barh(top_sev["cluster_label"], top_sev["severity_sum"])
        plt.title("Top ST-DBSCAN Clusters by Severity Sum")
        plt.xlabel("Severity Sum")
        plt.ylabel("Cluster")
        save_plot(OUT_DIR / "top_clusters_by_severity.png")

        plt.figure(figsize=(10, 5))
        plt.hist(cluster_summary_df["records"], bins=30)
        plt.title("Cluster Size Distribution")
        plt.xlabel("Records per Cluster")
        plt.ylabel("Frequency")
        save_plot(OUT_DIR / "cluster_size_distribution.png")

if __name__ == "__main__":
    main()

Approved rows: 115400
Unique hotspots (pre-clustering): 220
Running ST-DBSCAN with:
  spatial epsilon = 150.0 meters
  temporal epsilon = 3.0 days
  MinPts          = 15

Phase 3 complete
Clusters found: 798
Noise points   : 23068
Clustered rows  : 92332
Output folder   : D:\Flipkart_Hackathon\content\phase3_outputs_1

Top 10 clusters:
 st_dbscan_cluster_id                          cluster_label  records  severity_sum  severity_mean  unique_vehicles
                    2                BTP040 - Elite Junction    21733       34509.0       1.587862            18627
                    3         BTP051 - Safina Plaza Junction     9287       15028.0       1.618176             7622
                   32          BTP027 - Modi Bridge Junction     3168        5271.0       1.663826             2725
                   19 BTP001 - 10th Cross, Dr. Rajkumar Road     2934        4795.0       1.634288             2620
                   35       BTP020 - Hosahalli Metro Station     1650        2485.

In [ ]:
# =========================
# Phase 4 — Implicit μ Estimation / Dwell-Time Estimation
# Input  : Phase 3 clustered approved records
# Output : Gap-level dwell events, cluster-level μ summary, vehicle-type μ summary,
#          merged dataset for Stage 5
# =========================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# -------------------------
# Config
# -------------------------
PHASE3_PATHS = [
    Path("content/phase3_outputs_1/phase3_clustered_dataset.csv"),
    Path("content/phase3_outputs_1/phase3_cluster_dispatch_view.csv"),
]

OUT_DIR = Path("content/phase4_outputs_1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EPS = 1e-9
TOP_N = 25

# -------------------------
# Helpers
# -------------------------
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def safe_ratio(a, b):
    return a / (b + EPS)

def parse_datetime_to_ist(df: pd.DataFrame) -> pd.Series:
    """
    Prefer created_datetime_ist if present.
    Otherwise derive it from created_datetime, which is stored in UTC in the source file.
    Returns a timezone-aware IST series when possible.
    """
    if "created_datetime_ist" in df.columns:
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce", utc=True)
        if ts.notna().any():
            return ts.dt.tz_convert("Asia/Kolkata")
        # fallback for naive strings
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce")
        return ts.dt.tz_localize("Asia/Kolkata", nonexistent="NaT", ambiguous="NaT")

    if "created_datetime_parsed" in df.columns:
        ts = pd.to_datetime(df["created_datetime_parsed"], errors="coerce", utc=True)
        return ts.dt.tz_convert("Asia/Kolkata")

    if "created_datetime" not in df.columns:
        raise ValueError("Missing created_datetime / created_datetime_ist column.")

    ts = pd.to_datetime(df["created_datetime"], errors="coerce", utc=True)
    return ts.dt.tz_convert("Asia/Kolkata")

def get_cluster_col(df: pd.DataFrame) -> str:
    for c in ["st_dbscan_cluster_id", "cluster_id", "dbscan_cluster_id"]:
        if c in df.columns:
            return c
    raise ValueError("No cluster id column found. Expected st_dbscan_cluster_id from Phase 3.")

def get_vehicle_col(df: pd.DataFrame) -> str:
    """
    Document says vehicle_number. If updated_vehicle_number exists and has values,
    we use it as a canonical fallback because it is a corrected version from review.
    """
    if "updated_vehicle_number" in df.columns:
        # use updated_vehicle_number where present, otherwise vehicle_number
        if "vehicle_number" not in df.columns:
            df["vehicle_number"] = ""
        updated = df["updated_vehicle_number"].fillna("").astype(str).str.strip()
        original = df["vehicle_number"].fillna("").astype(str).str.strip()
        df["canonical_vehicle_number"] = np.where(updated.ne(""), updated, original)
        return "canonical_vehicle_number"

    if "vehicle_number" not in df.columns:
        raise ValueError("Missing vehicle_number / updated_vehicle_number column.")
    df["canonical_vehicle_number"] = df["vehicle_number"].fillna("").astype(str).str.strip()
    return "canonical_vehicle_number"

def get_vehicle_type_col(df: pd.DataFrame) -> str:
    if "updated_vehicle_type" in df.columns:
        if "vehicle_type" not in df.columns:
            df["vehicle_type"] = ""
        updated = df["updated_vehicle_type"].fillna("").astype(str).str.strip()
        original = df["vehicle_type"].fillna("").astype(str).str.strip()
        df["canonical_vehicle_type"] = np.where(updated.ne(""), updated, original)
        return "canonical_vehicle_type"

    if "vehicle_type" not in df.columns:
        df["canonical_vehicle_type"] = "UNKNOWN"
        return "canonical_vehicle_type"
    df["canonical_vehicle_type"] = df["vehicle_type"].fillna("UNKNOWN").astype(str).str.strip()
    df.loc[df["canonical_vehicle_type"].eq(""), "canonical_vehicle_type"] = "UNKNOWN"
    return "canonical_vehicle_type"

def make_hotspot_unit(row):
    junction = clean_text(row.get("junction_name", ""))
    if junction and junction.upper() != "NO JUNCTION":
        return f"JUNCTION::{junction}"
    station = clean_text(row.get("police_station", "UNKNOWN"))
    if not station:
        station = "UNKNOWN"
    return f"POLICE_STATION::{station}"

def save_plot(path):
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()

def load_phase3():
    for p in PHASE3_PATHS:
        if p.exists():
            return pd.read_csv(p, low_memory=False), p
    raise FileNotFoundError(
        "Phase 3 output not found. Expected phase3_outputs/phase3_clustered_dataset.csv"
    )

# -------------------------
# Main
# -------------------------
def main():
    df, src = load_phase3()
    print("Loaded:", src)

    # Required fields
    if "latitude" not in df.columns or "longitude" not in df.columns:
        raise ValueError("Phase 3 dataset must include latitude and longitude.")

    cluster_col = get_cluster_col(df)
    vehicle_col = get_vehicle_col(df)
    vehicle_type_col = get_vehicle_type_col(df)

    # Parse timestamps
    df["created_datetime_ist"] = parse_datetime_to_ist(df)
    df["created_datetime_ist_naive"] = df["created_datetime_ist"].dt.tz_localize(None)
    df["service_date"] = df["created_datetime_ist_naive"].dt.date

    # Keep approved clustered records only; exclude noise for dwell estimation
    if "validation_status_clean" in df.columns:
        df["validation_status_clean"] = df["validation_status_clean"].fillna("").astype(str).str.lower()
    elif "validation_status" in df.columns:
        df["validation_status_clean"] = df["validation_status"].fillna("").astype(str).str.lower()
    else:
        df["validation_status_clean"] = "approved"

    approved = df.loc[df["validation_status_clean"].eq("approved")].copy()
    approved = approved.dropna(subset=[cluster_col, vehicle_col, "created_datetime_ist_naive"]).copy()
    approved = approved.loc[approved[cluster_col].ne(-1)].copy()
    approved = approved.loc[approved[vehicle_col].fillna("").astype(str).str.strip().ne("")].copy()
    approved = approved.reset_index(drop=True)

    if approved.empty:
        raise ValueError("No approved clustered rows available for dwell-time estimation.")

    # If missing hotspot_unit, reconstruct it
    if "hotspot_unit" not in approved.columns:
        if {"junction_name", "police_station"}.issubset(approved.columns):
            approved["hotspot_unit"] = approved.apply(make_hotspot_unit, axis=1)
        else:
            approved["hotspot_unit"] = f"CLUSTER::{approved[cluster_col].astype(str)}"

    # Sort so groupby diff works correctly
    approved = approved.sort_values(
        [cluster_col, vehicle_col, "service_date", "created_datetime_ist_naive"]
    ).reset_index(drop=True)

    # Consecutive timestamp gaps within same vehicle, same cluster, same day
    group_cols = [cluster_col, vehicle_col, "service_date"]
    approved["prev_created_datetime_ist"] = approved.groupby(group_cols)["created_datetime_ist_naive"].shift(1)
    approved["gap_minutes"] = (
        approved["created_datetime_ist_naive"] - approved["prev_created_datetime_ist"]
    ).dt.total_seconds() / 60.0

    # Keep only valid positive gaps as lower-bound dwell-time estimates
    gaps = approved.loc[
        approved["gap_minutes"].notna() & (approved["gap_minutes"] > 0)
    ].copy()

    if gaps.empty:
        raise ValueError(
            "No valid inter-record gaps found. Check whether vehicles repeat within the same cluster and day."
        )

    # Output a record-level gap log
    gap_log_cols = [
        c for c in [
            "id", cluster_col, "hotspot_unit", vehicle_col, vehicle_type_col,
            "service_date", "prev_created_datetime_ist", "created_datetime_ist_naive",
            "gap_minutes", "latitude", "longitude", "junction_name", "police_station",
            "severity_score", "dominant_violation_tag"
        ] if c in gaps.columns
    ]
    gaps_out = gaps[gap_log_cols].copy()

    # -------------------------
    # Overall summary
    # -------------------------
    overall_mean_dwell = float(gaps_out["gap_minutes"].mean())
    overall_median_dwell = float(gaps_out["gap_minutes"].median())
    overall_mu = 60.0 / (overall_mean_dwell + EPS)

    overall_summary = pd.DataFrame([{
        "scope": "overall",
        "gap_count": int(len(gaps_out)),
        "unique_clusters_with_gaps": int(gaps_out[cluster_col].nunique()),
        "unique_vehicles_with_gaps": int(gaps_out[vehicle_col].nunique()),
        "mean_dwell_minutes": overall_mean_dwell,
        "median_dwell_minutes": overall_median_dwell,
        "mu_departures_per_hour": overall_mu,
    }])

    # -------------------------
    # Vehicle-type overall summary
    # -------------------------
    vehicle_type_summary = (
        gaps_out.groupby(vehicle_type_col, dropna=False)
        .agg(
            gap_count=("gap_minutes", "size"),
            unique_clusters=(cluster_col, "nunique"),
            unique_vehicles=(vehicle_col, "nunique"),
            mean_dwell_minutes=("gap_minutes", "mean"),
            median_dwell_minutes=("gap_minutes", "median"),
            std_dwell_minutes=("gap_minutes", lambda s: float(s.std(ddof=0)) if len(s) else 0.0),
        )
        .reset_index()
        .rename(columns={vehicle_type_col: "vehicle_type"})
        .sort_values(["gap_count", "mean_dwell_minutes"], ascending=[False, False])
    )
    vehicle_type_summary["mu_departures_per_hour"] = 60.0 / (vehicle_type_summary["mean_dwell_minutes"] + EPS)

    # -------------------------
    # Cluster-level overall summary
    # -------------------------
    cluster_overall_summary = (
        gaps_out.groupby(cluster_col, dropna=False)
        .agg(
            gap_count=("gap_minutes", "size"),
            unique_vehicles=(vehicle_col, "nunique"),
            unique_vehicle_types=(vehicle_type_col, "nunique"),
            mean_dwell_minutes=("gap_minutes", "mean"),
            median_dwell_minutes=("gap_minutes", "median"),
            std_dwell_minutes=("gap_minutes", lambda s: float(s.std(ddof=0)) if len(s) else 0.0),
            first_gap_time=("prev_created_datetime_ist", "min"),
            last_gap_time=("created_datetime_ist_naive", "max"),
        )
        .reset_index()
    )

    if "cluster_label" in gaps_out.columns:
        cluster_label_map = (
            gaps_out[[cluster_col, "cluster_label"]]
            .dropna()
            .drop_duplicates(subset=[cluster_col])
            .set_index(cluster_col)["cluster_label"]
            .to_dict()
        )
        cluster_overall_summary["cluster_label"] = cluster_overall_summary[cluster_col].map(cluster_label_map)
    else:
        cluster_overall_summary["cluster_label"] = cluster_overall_summary[cluster_col].map(
            approved.groupby(cluster_col)["hotspot_unit"].agg(lambda s: s.mode().iloc[0] if not s.mode().empty else str(s.iloc[0])).to_dict()
        )

    cluster_overall_summary["mu_departures_per_hour"] = 60.0 / (cluster_overall_summary["mean_dwell_minutes"] + EPS)

    # -------------------------
    # Cluster + vehicle-type summary
    # -------------------------
    cluster_type_summary = (
        gaps_out.groupby([cluster_col, vehicle_type_col], dropna=False)
        .agg(
            gap_count=("gap_minutes", "size"),
            unique_vehicles=(vehicle_col, "nunique"),
            mean_dwell_minutes=("gap_minutes", "mean"),
            median_dwell_minutes=("gap_minutes", "median"),
            std_dwell_minutes=("gap_minutes", lambda s: float(s.std(ddof=0)) if len(s) else 0.0),
        )
        .reset_index()
        .rename(columns={vehicle_type_col: "vehicle_type"})
        .sort_values([cluster_col, "gap_count", "mean_dwell_minutes"], ascending=[True, False, False])
    )
    cluster_type_summary["mu_departures_per_hour"] = 60.0 / (cluster_type_summary["mean_dwell_minutes"] + EPS)

    # -------------------------
    # Wide per-cluster summary for Stage 5 convenience
    # -------------------------
    type_wide = cluster_type_summary.pivot_table(
        index=cluster_col,
        columns="vehicle_type",
        values="mean_dwell_minutes",
        aggfunc="mean"
    )
    type_wide.columns = [f"mean_dwell_minutes__{str(c).replace(' ', '_').upper()}" for c in type_wide.columns]
    type_wide = type_wide.reset_index()

    type_wide_mu = cluster_type_summary.pivot_table(
        index=cluster_col,
        columns="vehicle_type",
        values="mu_departures_per_hour",
        aggfunc="mean"
    )
    type_wide_mu.columns = [f"mu_departures_per_hour__{str(c).replace(' ', '_').upper()}" for c in type_wide_mu.columns]
    type_wide_mu = type_wide_mu.reset_index()

    cluster_wide_summary = cluster_overall_summary.merge(type_wide, on=cluster_col, how="left").merge(
        type_wide_mu, on=cluster_col, how="left"
    )

    # -------------------------
    # Merge back onto records for later stages
    # -------------------------
    merged = approved.merge(
        cluster_overall_summary[[cluster_col, "gap_count", "mean_dwell_minutes", "median_dwell_minutes", "mu_departures_per_hour"]],
        on=cluster_col,
        how="left"
    )

    # -------------------------
    # Save outputs
    # -------------------------
    gaps_out.to_csv(OUT_DIR / "phase4_gap_event_log.csv", index=False)
    overall_summary.to_csv(OUT_DIR / "phase4_overall_mu_summary.csv", index=False)
    vehicle_type_summary.to_csv(OUT_DIR / "phase4_vehicle_type_mu_summary.csv", index=False)
    cluster_overall_summary.to_csv(OUT_DIR / "phase4_cluster_mu_summary.csv", index=False)
    cluster_type_summary.to_csv(OUT_DIR / "phase4_cluster_vehicle_type_mu_summary.csv", index=False)
    cluster_wide_summary.to_csv(OUT_DIR / "phase4_cluster_mu_summary_wide.csv", index=False)
    merged.to_csv(OUT_DIR / "phase4_merged_with_prior_scores.csv", index=False)

    # For convenience in Stage 5
    cluster_overall_summary[[cluster_col, "cluster_label", "mean_dwell_minutes", "mu_departures_per_hour", "gap_count"]].to_csv(
        OUT_DIR / "phase4_stage5_mu_lookup.csv", index=False
    )

    # -------------------------
    # Console summary
    # -------------------------
    print("Phase 4 complete")
    print("Input source:", src)
    print("Approved rows used:", len(approved))
    print("Valid inter-record gaps:", len(gaps_out))
    print("Overall mean dwell (min):", round(overall_mean_dwell, 3))
    print("Overall μ (departures/hour):", round(overall_mu, 6))
    print("Clusters with dwell estimates:", len(cluster_overall_summary))
    print("Vehicle types with dwell estimates:", len(vehicle_type_summary))
    print("Outputs saved to:", OUT_DIR.resolve())

    print("\nVehicle-type summary:")
    print(
        vehicle_type_summary[[
            "vehicle_type", "gap_count", "mean_dwell_minutes", "mu_departures_per_hour"
        ]].to_string(index=False)
    )

    print("\nTop clusters by gap count:")
    print(
        cluster_overall_summary[[
            cluster_col, "cluster_label", "gap_count", "mean_dwell_minutes", "mu_departures_per_hour"
        ]].sort_values("gap_count", ascending=False).head(10).to_string(index=False)
    )

    # -------------------------
    # Plots
    # -------------------------
    plt.figure(figsize=(12, 6))
    top_clusters = cluster_overall_summary.sort_values("gap_count", ascending=True).tail(TOP_N)
    plt.barh(top_clusters["cluster_label"], top_clusters["mean_dwell_minutes"])
    plt.title("Top Clusters by Mean Dwell Time")
    plt.xlabel("Mean dwell time (minutes)")
    plt.ylabel("Cluster")
    save_plot(OUT_DIR / "top_clusters_mean_dwell.png")

    plt.figure(figsize=(12, 6))
    top_types = vehicle_type_summary.sort_values("gap_count", ascending=True).tail(TOP_N)
    plt.barh(top_types["vehicle_type"], top_types["mean_dwell_minutes"])
    plt.title("Vehicle-Type Mean Dwell Time")
    plt.xlabel("Mean dwell time (minutes)")
    plt.ylabel("Vehicle type")
    save_plot(OUT_DIR / "vehicle_type_mean_dwell.png")

    plt.figure(figsize=(10, 5))
    plt.hist(gaps_out["gap_minutes"], bins=40)
    plt.title("Distribution of Inter-Record Dwell Gaps")
    plt.xlabel("Gap minutes")
    plt.ylabel("Frequency")
    save_plot(OUT_DIR / "gap_distribution.png")

if __name__ == "__main__":
    main()

Loaded: content\phase3_outputs_1\phase3_clustered_dataset.csv
Phase 4 complete
Input source: content\phase3_outputs_1\phase3_clustered_dataset.csv
Approved rows used: 92332
Valid inter-record gaps: 2122
Overall mean dwell (min): 98.041
Overall μ (departures/hour): 0.611989
Clusters with dwell estimates: 239
Vehicle types with dwell estimates: 19
Outputs saved to: D:\Flipkart_Hackathon\content\phase4_outputs_1

Vehicle-type summary:
       vehicle_type  gap_count  mean_dwell_minutes  mu_departures_per_hour
            SCOOTER        799          110.365457                0.543648
                CAR        711           95.087201                0.631000
        MOTOR CYCLE        296           98.131757                0.611423
     PASSENGER AUTO        181           53.933702                1.112477
           MAXI-CAB         52           93.576923                0.641184
                LGV         25           83.160000                0.721501
              MOPED         13         

In [ ]:
!pip install osmnx networkx


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# =========================
# Phase 5 — Queueing, Capacity Loss, and Delay Estimation
# Stage 5a: M/M/∞ Queue Model
# Stage 5b: Zhao Capacity Loss Model
# Stage 5c: Modified BPR Delay Model
# Also prepares growth and criticality inputs for Stage 6
# =========================

import ast
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# -------------------------
# Config
# -------------------------
PHASE4_MERGED_PATH = Path("content/phase4_outputs_1/phase4_merged_with_prior_scores.csv")
PHASE4_CLUSTER_PATH = Path("content/phase4_outputs_1/phase4_cluster_mu_summary.csv")
PHASE4_GAP_PATH = Path("content/phase4_outputs_1/phase4_gap_event_log.csv")

OUT_DIR = Path("content/phase5_outputs_1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EPS = 1e-9
TOP_N = 25

# Doc / paper constants
ALPHA_BPR = 0.56
BETA_BPR = 2.12

# Default geometric assumptions when OSMnx data is unavailable
DEFAULT_LANES = 2
DEFAULT_WIDTH_PER_LANE_M = 3.5
DEFAULT_CARRIAGEWAY_WIDTH_M = DEFAULT_LANES * DEFAULT_WIDTH_PER_LANE_M
DEFAULT_LINK_LENGTH_M = 250.0

ROAD_CLASS_PRIORITY = [
    "motorway", "trunk", "primary", "secondary", "tertiary",
    "unclassified", "residential", "living_street", "service", "road"
]

ROAD_CLASS_SPEED_KMH = {
    "motorway": 60.0,
    "trunk": 55.0,
    "primary": 45.0,
    "secondary": 40.0,
    "tertiary": 35.0,
    "unclassified": 30.0,
    "residential": 30.0,
    "living_street": 20.0,
    "service": 25.0,
    "road": 30.0,
}

ROAD_CLASS_BASE_CAPACITY_PER_LANE = {
    "motorway": 2200.0,
    "trunk": 2100.0,
    "primary": 1900.0,
    "secondary": 1800.0,
    "tertiary": 1700.0,
    "unclassified": 1650.0,
    "residential": 1500.0,
    "living_street": 1200.0,
    "service": 1400.0,
    "road": 1600.0,
}

# Typical blocking widths by vehicle class
VEHICLE_WIDTH_M = {
    "SCOOTER": 0.80,
    "MOTOR CYCLE": 0.90,
    "MOTORCYCLE": 0.90,
    "BICYCLE": 0.60,
    "CYCLE": 0.60,
    "PASSENGER AUTO": 1.60,
    "AUTO": 1.60,
    "CAR": 1.90,
    "SUV": 2.00,
    "JEEP": 2.00,
    "VAN": 2.20,
    "TEMPO": 2.20,
    "BUS": 2.60,
    "TRUCK": 2.60,
    "LORRY": 2.60,
    "TANKER": 2.80,
    "TRACTOR": 2.20,
    "MINI TRUCK": 2.20,
    "AMBULANCE": 2.00,
    "UNKNOWN": 1.90,
}

# -------------------------
# Helpers
# -------------------------
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def safe_ratio(a, b):
    return a / (b + EPS)

def minmax(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce").fillna(0.0).astype(float)
    if s.nunique(dropna=True) <= 1:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.min()) / (s.max() - s.min() + EPS)

def parse_listlike(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    s = str(value).strip()
    if not s:
        return []
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple)):
            return list(parsed)
        return [parsed]
    except Exception:
        if s.startswith("[") and s.endswith("]"):
            s = s[1:-1]
        parts = [p.strip().strip("'").strip('"') for p in s.split(",")]
        return [p for p in parts if p]

def parse_datetime_ist(df: pd.DataFrame) -> pd.Series:
    if "created_datetime_ist" in df.columns:
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce", utc=True)
        if ts.notna().any():
            return ts.dt.tz_convert("Asia/Kolkata")
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce")
        return ts.dt.tz_localize("Asia/Kolkata", nonexistent="NaT", ambiguous="NaT")

    if "created_datetime_parsed" in df.columns:
        ts = pd.to_datetime(df["created_datetime_parsed"], errors="coerce", utc=True)
        return ts.dt.tz_convert("Asia/Kolkata")

    if "created_datetime" not in df.columns:
        raise ValueError("Missing created_datetime / created_datetime_ist column.")

    ts = pd.to_datetime(df["created_datetime"], errors="coerce", utc=True)
    return ts.dt.tz_convert("Asia/Kolkata")

def week_start_monday(series):
    dt = pd.to_datetime(series, errors="coerce", utc=True).dt.tz_convert("Asia/Kolkata")
    week_start = dt.dt.normalize() - pd.to_timedelta(dt.dt.weekday, unit="D")
    return week_start.dt.tz_localize(None)

def make_hotspot_unit(row):
    junction = clean_text(row.get("junction_name", ""))
    if junction and junction.upper() != "NO JUNCTION":
        return f"JUNCTION::{junction}"
    station = clean_text(row.get("police_station", "UNKNOWN"))
    if not station:
        station = "UNKNOWN"
    return f"POLICE_STATION::{station}"

def get_cluster_col(df: pd.DataFrame) -> str:
    for c in ["st_dbscan_cluster_id", "cluster_id", "dbscan_cluster_id"]:
        if c in df.columns:
            return c
    raise ValueError("No cluster id column found.")

def get_vehicle_col(df: pd.DataFrame) -> str:
    if "updated_vehicle_number" in df.columns and "vehicle_number" in df.columns:
        upd = df["updated_vehicle_number"].fillna("").astype(str).str.strip()
        orig = df["vehicle_number"].fillna("").astype(str).str.strip()
        df["canonical_vehicle_number"] = np.where(upd.ne(""), upd, orig)
        return "canonical_vehicle_number"
    if "vehicle_number" not in df.columns:
        raise ValueError("Missing vehicle_number / updated_vehicle_number.")
    df["canonical_vehicle_number"] = df["vehicle_number"].fillna("").astype(str).str.strip()
    return "canonical_vehicle_number"

def get_vehicle_type_col(df: pd.DataFrame) -> str:
    if "updated_vehicle_type" in df.columns and "vehicle_type" in df.columns:
        upd = df["updated_vehicle_type"].fillna("").astype(str).str.strip()
        orig = df["vehicle_type"].fillna("").astype(str).str.strip()
        df["canonical_vehicle_type"] = np.where(upd.ne(""), upd, orig)
        return "canonical_vehicle_type"
    if "vehicle_type" in df.columns:
        df["canonical_vehicle_type"] = df["vehicle_type"].fillna("UNKNOWN").astype(str).str.strip()
        df.loc[df["canonical_vehicle_type"].eq(""), "canonical_vehicle_type"] = "UNKNOWN"
        return "canonical_vehicle_type"
    df["canonical_vehicle_type"] = "UNKNOWN"
    return "canonical_vehicle_type"

def normalize_road_class(highway_value):
    vals = parse_listlike(highway_value)
    if not vals:
        raw = clean_text(highway_value).lower()
        if not raw:
            return "road"
        vals = [raw]

    normalized = []
    for v in vals:
        s = clean_text(v).lower()
        s = s.replace(" ", "_")
        if s.endswith("_link"):
            s = s.replace("_link", "")
        if s in {"motorway", "trunk", "primary", "secondary", "tertiary", "unclassified", "residential", "living_street", "service", "road"}:
            normalized.append(s)
        elif s in {"motorway_link"}:
            normalized.append("motorway")
        elif s in {"trunk_link"}:
            normalized.append("trunk")
        elif s in {"primary_link"}:
            normalized.append("primary")
        elif s in {"secondary_link"}:
            normalized.append("secondary")
        elif s in {"tertiary_link"}:
            normalized.append("tertiary")
        else:
            normalized.append("road")

    for cls in ROAD_CLASS_PRIORITY:
        if cls in normalized:
            return cls
    return normalized[0] if normalized else "road"

def parse_lanes(value):
    s = clean_text(value)
    if not s:
        return None
    s = s.replace(";", ",").replace("|", ",")
    parts = [p.strip() for p in s.split(",") if p.strip()]
    nums = []
    for p in parts:
        try:
            nums.append(float(p))
        except Exception:
            pass
    if nums:
        return max(1, int(round(nums[0])))
    try:
        return max(1, int(round(float(s))))
    except Exception:
        return None

def parse_width(value):
    s = clean_text(value)
    if not s:
        return None
    s = s.lower().replace("m", "").strip()
    s = s.replace(",", ".")
    try:
        return float(s)
    except Exception:
        return None

def road_speed_kmh(road_class):
    return ROAD_CLASS_SPEED_KMH.get(road_class, 30.0)

def base_capacity_per_lane(road_class):
    return ROAD_CLASS_BASE_CAPACITY_PER_LANE.get(road_class, 1600.0)

def vehicle_width_m(vehicle_type):
    vt = clean_text(vehicle_type).upper()
    return VEHICLE_WIDTH_M.get(vt, 1.90)

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    p1 = np.radians(lat1)
    p2 = np.radians(lat2)
    dp = np.radians(lat2 - lat1)
    dl = np.radians(lon2 - lon1)
    a = np.sin(dp / 2.0) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2.0) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

def clamp(x, lo, hi):
    return max(lo, min(hi, x))

def save_plot(path):
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()

# -------------------------
# Load data
# -------------------------
if PHASE4_MERGED_PATH.exists():
    df = pd.read_csv(PHASE4_MERGED_PATH, low_memory=False)
else:
    raise FileNotFoundError(
        "Phase 4 merged output not found. Expected phase4_outputs/phase4_merged_with_prior_scores.csv"
    )

cluster_col = get_cluster_col(df)
vehicle_col = get_vehicle_col(df)
vehicle_type_col = get_vehicle_type_col(df)

# Timestamp and baseline columns
df["created_datetime_ist"] = parse_datetime_ist(df)
df["created_datetime_ist_naive"] = df["created_datetime_ist"].dt.tz_localize(None)
df["week_start"] = week_start_monday(df["created_datetime"])
df["cluster_week_start"] = df["created_datetime_ist"].dt.to_period("W-MON").dt.start_time

if "validation_status_clean" in df.columns:
    df["validation_status_clean"] = df["validation_status_clean"].fillna("").astype(str).str.lower()
elif "validation_status" in df.columns:
    df["validation_status_clean"] = df["validation_status"].fillna("").astype(str).str.lower()
else:
    df["validation_status_clean"] = "approved"

if "severity_score" not in df.columns:
    df["severity_score"] = 1.0

if "hotspot_unit" not in df.columns:
    if {"junction_name", "police_station"}.issubset(df.columns):
        df["hotspot_unit"] = df.apply(make_hotspot_unit, axis=1)
    else:
        df["hotspot_unit"] = "UNKNOWN"

approved = df.loc[df["validation_status_clean"].eq("approved")].copy()
approved = approved.dropna(subset=[cluster_col, "created_datetime_ist_naive", "latitude", "longitude"]).copy()
approved = approved.loc[approved[cluster_col].ne(-1)].copy()
approved = approved.loc[approved[vehicle_col].fillna("").astype(str).str.strip().ne("")].copy()
approved = approved.reset_index(drop=True)

if approved.empty:
    raise ValueError("No approved clustered rows available for Stage 5.")

print("Approved clustered rows:", len(approved))
print("Clusters:", approved[cluster_col].nunique())

# -------------------------
# Stage 5a — Arrival rate λ
# -------------------------
peak_window_mask = approved["created_datetime_ist"].dt.hour.between(8, 12, inclusive="both")
peak_window = approved.loc[peak_window_mask].copy()

cluster_lambda = (
    approved.groupby(cluster_col)
    .agg(
        records_total=("id", "size") if "id" in approved.columns else (vehicle_col, "size"),
        records_peak_window=(vehicle_col, lambda s: int(peak_window_mask.loc[s.index].sum())),
        records_peak_hour=(vehicle_col, lambda s: int(
            approved.loc[s.index]
            .groupby(approved.loc[s.index, "created_datetime_ist"].dt.hour)
            .size()
            .max()
            if len(s.index) > 0 else 0
        )),
        first_seen=("created_datetime_ist_naive", "min"),
        last_seen=("created_datetime_ist_naive", "max"),
        centroid_lat=("latitude", "mean"),
        centroid_lon=("longitude", "mean"),
        severity_sum=("severity_score", "sum"),
        severity_mean=("severity_score", "mean"),
        unique_vehicles=(vehicle_col, "nunique"),
        unique_vehicle_types=(vehicle_type_col, "nunique"),
    )
    .reset_index()
)

cluster_lambda["lambda_hr_peak_window"] = cluster_lambda["records_peak_window"] / 5.0
cluster_lambda["lambda_hr_peak_hour"] = cluster_lambda["records_peak_hour"].astype(float)

# -------------------------
# Stage 4 carry-over — μ / dwell time
# -------------------------
mu_source = None
if PHASE4_CLUSTER_PATH.exists():
    mu_source = pd.read_csv(PHASE4_CLUSTER_PATH, low_memory=False)
    if cluster_col not in mu_source.columns and "st_dbscan_cluster_id" in mu_source.columns:
        mu_source = mu_source.rename(columns={"st_dbscan_cluster_id": cluster_col})

    keep_cols = [cluster_col]
    for c in ["cluster_label", "gap_count", "mean_dwell_minutes", "median_dwell_minutes", "std_dwell_minutes", "mu_departures_per_hour"]:
        if c in mu_source.columns:
            keep_cols.append(c)
    mu_source = mu_source[keep_cols].drop_duplicates(cluster_col)
else:
    # Fallback: derive from the merged data if the dedicated Stage 4 file is missing
    tmp = approved.sort_values([cluster_col, vehicle_col, "cluster_week_start", "created_datetime_ist_naive"]).copy()
    tmp["prev_ts"] = tmp.groupby([cluster_col, vehicle_col, tmp["created_datetime_ist"].dt.date])["created_datetime_ist_naive"].shift(1)
    tmp["gap_minutes"] = (tmp["created_datetime_ist_naive"] - tmp["prev_ts"]).dt.total_seconds() / 60.0
    tmp = tmp.loc[tmp["gap_minutes"].notna() & (tmp["gap_minutes"] > 0)].copy()
    mu_source = (
        tmp.groupby(cluster_col)
        .agg(
            gap_count=("gap_minutes", "size"),
            mean_dwell_minutes=("gap_minutes", "mean"),
            median_dwell_minutes=("gap_minutes", "median"),
            std_dwell_minutes=("gap_minutes", lambda s: float(s.std(ddof=0)) if len(s) else 0.0),
        )
        .reset_index()
    )
    mu_source["mu_departures_per_hour"] = 60.0 / (mu_source["mean_dwell_minutes"] + EPS)

cluster_table = cluster_lambda.merge(mu_source, on=cluster_col, how="left")

if "mu_departures_per_hour" not in cluster_table.columns:
    cluster_table["mu_departures_per_hour"] = 60.0 / (cluster_table["mean_dwell_minutes"] + EPS)

cluster_table["mean_dwell_minutes"] = pd.to_numeric(cluster_table["mean_dwell_minutes"], errors="coerce")
cluster_table["mu_departures_per_hour"] = pd.to_numeric(cluster_table["mu_departures_per_hour"], errors="coerce")
cluster_table["mu_departures_per_hour"] = cluster_table["mu_departures_per_hour"].fillna(
    60.0 / (cluster_table["mean_dwell_minutes"] + EPS)
)

cluster_table["blocking_vehicles_L"] = cluster_table["lambda_hr_peak_window"] / (cluster_table["mu_departures_per_hour"] + EPS)

# -------------------------
# Vehicle mix and blocking width
# -------------------------
vehicle_mix = (
    approved.groupby([cluster_col, vehicle_type_col])
    .size()
    .reset_index(name="count")
    .rename(columns={vehicle_type_col: "vehicle_type"})
)

vehicle_mix["vehicle_type"] = vehicle_mix["vehicle_type"].fillna("UNKNOWN").astype(str).str.strip()
vehicle_mix["vehicle_width_m"] = vehicle_mix["vehicle_type"].apply(vehicle_width_m)

cluster_width = (
    vehicle_mix.groupby(cluster_col)
    .apply(lambda g: safe_ratio((g["count"] * g["vehicle_width_m"]).sum(), g["count"].sum()))
    .reset_index(name="vehicle_width_avg_m")
)

cluster_table = cluster_table.merge(cluster_width, on=cluster_col, how="left")
cluster_table["vehicle_width_avg_m"] = cluster_table["vehicle_width_avg_m"].fillna(1.90)

# -------------------------
# Trend acceleration / growth multiplier
# -------------------------
weekly_counts = (
    approved.groupby([cluster_col, "cluster_week_start"])
    .size()
    .reset_index(name="weekly_count")
    .sort_values([cluster_col, "cluster_week_start"])
)

growth_rows = []
for cid, g in weekly_counts.groupby(cluster_col):
    counts = g.sort_values("cluster_week_start")["weekly_count"].to_numpy(dtype=float)
    if len(counts) < 2:
        first_mean = float(counts.mean()) if len(counts) else 0.0
        second_mean = first_mean
        growth_pct = 0.0
    else:
        mid = max(1, len(counts) // 2)
        first_half = counts[:mid]
        second_half = counts[mid:]
        first_mean = float(first_half.mean()) if len(first_half) else 0.0
        second_mean = float(second_half.mean()) if len(second_half) else 0.0
        if first_mean <= 0:
            growth_pct = second_mean
        else:
            growth_pct = (second_mean - first_mean) / (first_mean + EPS)

    growth_multiplier = max(0.5, 1.0 + max(0.0, growth_pct))
    growth_rows.append({
        cluster_col: cid,
        "growth_first_half_mean": first_mean,
        "growth_second_half_mean": second_mean,
        "growth_pct_change": growth_pct,
        "growth_multiplier": growth_multiplier,
    })

growth_df = pd.DataFrame(growth_rows)

# -------------------------
# Road-network context via OSMnx (optional, graceful fallback)
# -------------------------
road_context_rows = []

try:
    import osmnx as ox
    import networkx as nx

    HAS_OSMNX = True
except Exception:
    HAS_OSMNX = False

def pick_edge_data(edge_data):
    highway = edge_data.get("highway", None)
    road_class = normalize_road_class(highway)

    lanes = parse_lanes(edge_data.get("lanes", None))
    width = parse_width(edge_data.get("width", None))
    length_m = edge_data.get("length", None)

    if lanes is None:
        lanes = DEFAULT_LANES
    if width is None:
        width = lanes * DEFAULT_WIDTH_PER_LANE_M
    if length_m is None:
        length_m = DEFAULT_LINK_LENGTH_M

    return road_class, int(lanes), float(width), float(length_m)

def enrich_cluster_geometry(lat, lon):
    """
    Returns road/network context for a cluster centroid.
    If OSMnx is unavailable or fails, returns sensible fallbacks.
    """
    result = {
        "geometry_source": "fallback",
        "osmnx_node_id": np.nan,
        "osmnx_edge_u": np.nan,
        "osmnx_edge_v": np.nan,
        "osmnx_edge_key": np.nan,
        "road_class": "road",
        "lane_count": DEFAULT_LANES,
        "carriageway_width_m": DEFAULT_CARRIAGEWAY_WIDTH_M,
        "link_length_m": DEFAULT_LINK_LENGTH_M,
        "junction_degree": 4,
        "betweenness_centrality": 0.0,
    }

    if not HAS_OSMNX:
        return result

    try:
        # Small local drive graph around the hotspot centroid
        G = ox.graph_from_point((lat, lon), dist=750, network_type="drive", simplify=True, retain_all=False)

        # nearest node / edge
        nearest_node = ox.distance.nearest_nodes(G, X=lon, Y=lat)
        nearest_edge = ox.distance.nearest_edges(G, X=lon, Y=lat)

        u, v, k = nearest_edge
        edge_data = G.edges[u, v, k]

        road_class, lanes, width, length_m = pick_edge_data(edge_data)

        UG = nx.Graph(G)
        # Centrality in the local graph
        bc = nx.betweenness_centrality(UG, normalized=True, weight="length")
        centrality = float(bc.get(nearest_node, 0.0))
        degree = int(UG.degree[nearest_node])

        result.update({
            "geometry_source": "osmnx",
            "osmnx_node_id": nearest_node,
            "osmnx_edge_u": u,
            "osmnx_edge_v": v,
            "osmnx_edge_key": k,
            "road_class": road_class,
            "lane_count": lanes,
            "carriageway_width_m": width,
            "link_length_m": length_m,
            "junction_degree": degree,
            "betweenness_centrality": centrality,
        })
        return result

    except Exception:
        return result

for _, row in cluster_table.iterrows():
    road_context_rows.append({
        cluster_col: row[cluster_col],
        **enrich_cluster_geometry(float(row["centroid_lat"]), float(row["centroid_lon"]))
    })

road_context_df = pd.DataFrame(road_context_rows)

cluster_table = cluster_table.merge(road_context_df, on=cluster_col, how="left")
cluster_table["lane_count"] = cluster_table["lane_count"].fillna(DEFAULT_LANES).astype(int)
cluster_table["carriageway_width_m"] = cluster_table["carriageway_width_m"].fillna(DEFAULT_CARRIAGEWAY_WIDTH_M).astype(float)
cluster_table["link_length_m"] = cluster_table["link_length_m"].fillna(DEFAULT_LINK_LENGTH_M).astype(float)
cluster_table["junction_degree"] = cluster_table["junction_degree"].fillna(4).astype(float)
cluster_table["betweenness_centrality"] = cluster_table["betweenness_centrality"].fillna(0.0).astype(float)

# Normalize criticality across all clusters
cluster_table["junction_degree_norm"] = minmax(cluster_table["junction_degree"])
cluster_table["betweenness_centrality_norm"] = minmax(cluster_table["betweenness_centrality"])
cluster_table["criticality_factor"] = 1.0 + 0.5 * cluster_table["junction_degree_norm"] + 0.5 * cluster_table["betweenness_centrality_norm"]

# -------------------------
# Stage 5b — Zhao capacity reduction
# -------------------------
cluster_table["base_saturation_per_lane_pcu_hr"] = cluster_table["road_class"].map(base_capacity_per_lane).fillna(1600.0)
cluster_table["free_flow_speed_kmh"] = cluster_table["road_class"].apply(road_speed_kmh)
cluster_table["free_flow_time_min"] = (
    cluster_table["link_length_m"] * 60.0 / (cluster_table["free_flow_speed_kmh"] * 1000.0 + EPS)
)

# Base capacity
cluster_table["width_factor"] = cluster_table["carriageway_width_m"] / (cluster_table["lane_count"] * DEFAULT_WIDTH_PER_LANE_M + EPS)
cluster_table["width_factor"] = cluster_table["width_factor"].clip(0.70, 1.20)

cluster_table["base_capacity_pcu_hr"] = (
    cluster_table["base_saturation_per_lane_pcu_hr"] *
    cluster_table["lane_count"] *
    cluster_table["width_factor"]
)

# Effective blocked-width fraction
cluster_table["blocked_width_fraction_raw"] = (
    cluster_table["blocking_vehicles_L"] * cluster_table["vehicle_width_avg_m"]
) / (cluster_table["carriageway_width_m"] + EPS)

cluster_table["blocked_width_fraction"] = cluster_table["blocked_width_fraction_raw"].clip(0.0, 0.95)

cluster_table["reduced_capacity_pcu_hr"] = (
    cluster_table["base_capacity_pcu_hr"] * (1.0 - cluster_table["blocked_width_fraction"])
).clip(lower=cluster_table["base_capacity_pcu_hr"] * 0.10)

cluster_table["capacity_loss_pct"] = 1.0 - safe_ratio(cluster_table["reduced_capacity_pcu_hr"], cluster_table["base_capacity_pcu_hr"])

# -------------------------
# Stage 5c — Modified BPR delay
# -------------------------
cluster_table["V_over_C0"] = cluster_table["lambda_hr_peak_window"] / (cluster_table["base_capacity_pcu_hr"] + EPS)
cluster_table["V_over_Cp"] = cluster_table["lambda_hr_peak_window"] / (cluster_table["reduced_capacity_pcu_hr"] + EPS)

cluster_table["delay_minutes_per_vehicle"] = (
    cluster_table["free_flow_time_min"] * ALPHA_BPR *
    (np.power(cluster_table["V_over_Cp"], BETA_BPR) - np.power(cluster_table["V_over_C0"], BETA_BPR))
)

cluster_table["delay_minutes_per_vehicle"] = cluster_table["delay_minutes_per_vehicle"].clip(lower=0.0)

# Optional composite inputs for Stage 6
cluster_table = cluster_table.merge(growth_df, on=cluster_col, how="left")
cluster_table["growth_multiplier"] = cluster_table["growth_multiplier"].fillna(1.0)
cluster_table["growth_pct_change"] = cluster_table["growth_pct_change"].fillna(0.0)

# -------------------------
# Chronic offender list (for the architecture branch)
# -------------------------
vehicle_offenders = (
    approved.groupby(vehicle_col)
    .agg(
        total_violations=(vehicle_col, "size"),
        unique_clusters=(cluster_col, "nunique"),
        unique_hotspots=("hotspot_unit", "nunique"),
        first_seen=("created_datetime_ist_naive", "min"),
        last_seen=("created_datetime_ist_naive", "max"),
        dominant_vehicle_type=(vehicle_type_col, lambda s: s.mode().iloc[0] if not s.mode().empty else "UNKNOWN"),
    )
    .reset_index()
    .rename(columns={vehicle_col: "vehicle_number"})
    .sort_values(["total_violations", "unique_clusters"], ascending=[False, False])
)
vehicle_offenders["chronic_offender_flag"] = (vehicle_offenders["total_violations"] >= 5).astype(int)
chronic_offenders = vehicle_offenders.loc[vehicle_offenders["chronic_offender_flag"].eq(1)].copy()

# Cluster-level repeat offender counts
repeat_vehicle_counts = (
    approved.groupby([cluster_col, vehicle_col])
    .size()
    .reset_index(name="vehicle_cluster_count")
)
repeat_vehicle_counts["repeat_flag_2plus"] = (repeat_vehicle_counts["vehicle_cluster_count"] >= 2).astype(int)
repeat_vehicle_counts["chronic_flag_5plus"] = (repeat_vehicle_counts["vehicle_cluster_count"] >= 5).astype(int)

cluster_repeat_summary = (
    repeat_vehicle_counts.groupby(cluster_col)
    .agg(
        repeat_vehicle_count_2plus=("repeat_flag_2plus", "sum"),
        chronic_vehicle_count_5plus=("chronic_flag_5plus", "sum"),
    )
    .reset_index()
)

cluster_table = cluster_table.merge(cluster_repeat_summary, on=cluster_col, how="left")
cluster_table["repeat_vehicle_count_2plus"] = cluster_table["repeat_vehicle_count_2plus"].fillna(0).astype(int)
cluster_table["chronic_vehicle_count_5plus"] = cluster_table["chronic_vehicle_count_5plus"].fillna(0).astype(int)

# -------------------------
# Final Stage 5 priority view
# -------------------------
priority_cols = [
    cluster_col, "cluster_label", "records_total", "records_peak_window", "lambda_hr_peak_window",
    "records_peak_hour", "lambda_hr_peak_hour", "mean_dwell_minutes", "mu_departures_per_hour",
    "blocking_vehicles_L", "road_class", "lane_count", "carriageway_width_m", "link_length_m",
    "base_capacity_pcu_hr", "reduced_capacity_pcu_hr", "capacity_loss_pct", "free_flow_speed_kmh",
    "free_flow_time_min", "delay_minutes_per_vehicle", "junction_degree", "betweenness_centrality",
    "criticality_factor", "growth_pct_change", "growth_multiplier", "vehicle_width_avg_m",
    "dominant_vehicle_type", "unique_vehicles", "unique_vehicle_types",
    "repeat_vehicle_count_2plus", "chronic_vehicle_count_5plus", "geometry_source"
]
for c in priority_cols:
    if c not in cluster_table.columns:
        cluster_table[c] = np.nan

priority_table = cluster_table[priority_cols].copy()
priority_table = priority_table.sort_values(
    ["delay_minutes_per_vehicle", "lambda_hr_peak_window", "blocking_vehicles_L"],
    ascending=[False, False, False]
).reset_index(drop=True)
priority_table["stage5_rank"] = np.arange(1, len(priority_table) + 1)

# Helpful normalized versions for later stages
priority_table["delay_minutes_norm"] = minmax(priority_table["delay_minutes_per_vehicle"])
priority_table["lambda_norm"] = minmax(priority_table["lambda_hr_peak_window"])
priority_table["severity_sum_norm"] = minmax(priority_table["records_total"])

# -------------------------
# Merge back onto records for Stage 6
# -------------------------
stage5_record_view = approved.merge(
    priority_table[[
        cluster_col, "cluster_label", "records_total", "lambda_hr_peak_window", "mean_dwell_minutes",
        "mu_departures_per_hour", "blocking_vehicles_L", "road_class", "lane_count",
        "carriageway_width_m", "link_length_m", "base_capacity_pcu_hr", "reduced_capacity_pcu_hr",
        "capacity_loss_pct", "free_flow_time_min", "delay_minutes_per_vehicle",
        "junction_degree", "betweenness_centrality", "criticality_factor",
        "growth_pct_change", "growth_multiplier", "stage5_rank"
    ]],
    on=cluster_col,
    how="left",
    suffixes=("", "_stage5")
)

# -------------------------
# Save outputs
# -------------------------
cluster_table.to_csv(OUT_DIR / "phase5_cluster_capacity_loss.csv", index=False)
priority_table.to_csv(OUT_DIR / "phase5_priority_table.csv", index=False)
stage5_record_view.to_csv(OUT_DIR / "phase5_enriched_records.csv", index=False)
vehicle_mix.to_csv(OUT_DIR / "phase5_vehicle_mix.csv", index=False)
weekly_counts.to_csv(OUT_DIR / "phase5_weekly_cluster_counts.csv", index=False)
growth_df.to_csv(OUT_DIR / "phase5_growth_summary.csv", index=False)
chronic_offenders.to_csv(OUT_DIR / "phase5_chronic_offenders.csv", index=False)
road_context_df.to_csv(OUT_DIR / "phase5_road_context.csv", index=False)

# Compact Stage 5 handoff tables
priority_table[[
    "stage5_rank", cluster_col, "cluster_label", "delay_minutes_per_vehicle",
    "lambda_hr_peak_window", "mu_departures_per_hour", "blocking_vehicles_L",
    "capacity_loss_pct", "criticality_factor", "growth_multiplier", "road_class",
    "lane_count", "carriageway_width_m", "free_flow_time_min"
]].to_csv(OUT_DIR / "phase5_stage5_handoff.csv", index=False)

# -------------------------
# Console summary
# -------------------------
print("Phase 5 complete")
print("Clusters scored:", len(priority_table))
print("Chronic offenders:", len(chronic_offenders))
print("OSMnx available:", HAS_OSMNX)
print("Outputs saved to:", OUT_DIR.resolve())

print("\nTop 10 clusters by delay:")
print(
    priority_table[[
        "stage5_rank", cluster_col, "cluster_label", "delay_minutes_per_vehicle",
        "lambda_hr_peak_window", "mu_departures_per_hour", "blocking_vehicles_L",
        "capacity_loss_pct", "criticality_factor", "growth_multiplier"
    ]].head(10).to_string(index=False)
)

# -------------------------
# Plots
# -------------------------
plt.figure(figsize=(12, 7))
top_delay = priority_table.head(TOP_N).sort_values("delay_minutes_per_vehicle", ascending=True)
plt.barh(top_delay["cluster_label"], top_delay["delay_minutes_per_vehicle"])
plt.title("Top Clusters by Delay Minutes per Vehicle")
plt.xlabel("Delay minutes per vehicle")
plt.ylabel("Cluster")
save_plot(OUT_DIR / "top_clusters_delay.png")

plt.figure(figsize=(12, 7))
top_capacity = priority_table.head(TOP_N).sort_values("capacity_loss_pct", ascending=True)
plt.barh(top_capacity["cluster_label"], top_capacity["capacity_loss_pct"])
plt.title("Top Clusters by Capacity Loss")
plt.xlabel("Capacity loss fraction")
plt.ylabel("Cluster")
save_plot(OUT_DIR / "top_clusters_capacity_loss.png")

plt.figure(figsize=(12, 5))
plt.hist(priority_table["delay_minutes_per_vehicle"], bins=30)
plt.title("Delay Minutes per Vehicle Distribution")
plt.xlabel("Delay minutes per vehicle")
plt.ylabel("Clusters")
save_plot(OUT_DIR / "delay_distribution.png")

Approved clustered rows: 92332
Clusters: 798


KeyboardInterrupt: 

### It might take more than 2 hours so we will use DBSCAN or caching

In [ ]:
# =========================
# Phase 5 — Optimized Queueing, Capacity Loss, and Delay Estimation
# Fast version:
# - Vectorized cluster aggregation
# - Reuses Phase 4 dwell-time outputs
# - Uses cached / fallback road geometry by default
# - Optional OSMnx enrichment only if explicitly enabled
# =========================

import ast
import os
import warnings
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# -------------------------
# Config
# -------------------------
PHASE4_MERGED_PATH = Path("content/phase4_outputs_1/phase4_merged_with_prior_scores.csv")
PHASE4_CLUSTER_PATH = Path("content/phase4_outputs_1/phase4_cluster_mu_summary.csv")
PHASE5_ROAD_CACHE_PATH = Path("content/phase5_outputs_1/phase5_road_context_cache.csv")  # optional cache reuse

OUT_DIR = Path("content/phase5_outputs_1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EPS = 1e-9
TOP_N = 25

# Doc / paper constants
ALPHA_BPR = 0.56
BETA_BPR = 2.12

# Default geometry assumptions when OSMnx data is unavailable
DEFAULT_LANES = 2
DEFAULT_WIDTH_PER_LANE_M = 3.5
DEFAULT_CARRIAGEWAY_WIDTH_M = DEFAULT_LANES * DEFAULT_WIDTH_PER_LANE_M
DEFAULT_LINK_LENGTH_M = 250.0

# Set to True only if you really want OSMnx enrichment
ENABLE_OSMNX = os.environ.get("ENABLE_OSMNX_PHASE5", "1").strip() == "1"

ROAD_CLASS_SPEED_KMH = {
    "motorway": 60.0,
    "trunk": 55.0,
    "primary": 45.0,
    "secondary": 40.0,
    "tertiary": 35.0,
    "unclassified": 30.0,
    "residential": 30.0,
    "living_street": 20.0,
    "service": 25.0,
    "road": 30.0,
}

ROAD_CLASS_BASE_CAPACITY_PER_LANE = {
    "motorway": 2200.0,
    "trunk": 2100.0,
    "primary": 1900.0,
    "secondary": 1800.0,
    "tertiary": 1700.0,
    "unclassified": 1650.0,
    "residential": 1500.0,
    "living_street": 1200.0,
    "service": 1400.0,
    "road": 1600.0,
}

VEHICLE_WIDTH_M = {
    "SCOOTER": 0.80,
    "MOTOR CYCLE": 0.90,
    "MOTORCYCLE": 0.90,
    "BICYCLE": 0.60,
    "CYCLE": 0.60,
    "PASSENGER AUTO": 1.60,
    "AUTO": 1.60,
    "CAR": 1.90,
    "SUV": 2.00,
    "JEEP": 2.00,
    "VAN": 2.20,
    "TEMPO": 2.20,
    "BUS": 2.60,
    "TRUCK": 2.60,
    "LORRY": 2.60,
    "TANKER": 2.80,
    "TRACTOR": 2.20,
    "MINI TRUCK": 2.20,
    "AMBULANCE": 2.00,
    "UNKNOWN": 1.90,
}

# -------------------------
# Helpers
# -------------------------
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def safe_ratio(a, b):
    return a / (b + EPS)

def minmax(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce").fillna(0.0).astype(float)
    if s.nunique(dropna=True) <= 1:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.min()) / (s.max() - s.min() + EPS)

def parse_listlike(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    s = str(value).strip()
    if not s:
        return []
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple)):
            return list(parsed)
        return [parsed]
    except Exception:
        if s.startswith("[") and s.endswith("]"):
            s = s[1:-1]
        parts = [p.strip().strip("'").strip('"') for p in s.split(",")]
        return [p for p in parts if p]

def parse_datetime_ist(df: pd.DataFrame) -> pd.Series:
    if "created_datetime_ist" in df.columns:
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce", utc=True)
        if ts.notna().any():
            return ts.dt.tz_convert("Asia/Kolkata")
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce")
        return ts.dt.tz_localize("Asia/Kolkata", nonexistent="NaT", ambiguous="NaT")

    if "created_datetime_parsed" in df.columns:
        ts = pd.to_datetime(df["created_datetime_parsed"], errors="coerce", utc=True)
        return ts.dt.tz_convert("Asia/Kolkata")

    if "created_datetime" not in df.columns:
        raise ValueError("Missing created_datetime / created_datetime_ist column.")

    ts = pd.to_datetime(df["created_datetime"], errors="coerce", utc=True)
    return ts.dt.tz_convert("Asia/Kolkata")

def week_start_monday(series):
    dt = pd.to_datetime(series, errors="coerce", utc=True).dt.tz_convert("Asia/Kolkata")
    week_start = dt.dt.normalize() - pd.to_timedelta(dt.dt.weekday, unit="D")
    return week_start.dt.tz_localize(None)

def make_hotspot_unit(row):
    junction = clean_text(row.get("junction_name", ""))
    if junction and junction.upper() != "NO JUNCTION":
        return f"JUNCTION::{junction}"
    station = clean_text(row.get("police_station", "UNKNOWN"))
    if not station:
        station = "UNKNOWN"
    return f"POLICE_STATION::{station}"

def standardize_cluster_col(df: pd.DataFrame) -> str:
    for c in ["st_dbscan_cluster_id", "cluster_id", "dbscan_cluster_id"]:
        if c in df.columns:
            return c
    raise ValueError("No cluster id column found.")

def standardize_vehicle_col(df: pd.DataFrame) -> str:
    if "canonical_vehicle_number" in df.columns:
        return "canonical_vehicle_number"
    if "vehicle_number" in df.columns:
        return "vehicle_number"
    raise ValueError("No vehicle number column found.")

def standardize_vehicle_type_col(df: pd.DataFrame) -> str:
    if "canonical_vehicle_type" in df.columns:
        return "canonical_vehicle_type"
    if "vehicle_type" in df.columns:
        return "vehicle_type"
    df["canonical_vehicle_type"] = "UNKNOWN"
    return "canonical_vehicle_type"

def road_speed_kmh(road_class):
    return ROAD_CLASS_SPEED_KMH.get(road_class, 30.0)

def base_capacity_per_lane(road_class):
    return ROAD_CLASS_BASE_CAPACITY_PER_LANE.get(road_class, 1600.0)

def vehicle_width_m(vehicle_type):
    vt = clean_text(vehicle_type).upper()
    return VEHICLE_WIDTH_M.get(vt, 1.90)

def normalize_road_class(value):
    if pd.isna(value):
        return "road"
    s = str(value).strip().lower()
    if not s:
        return "road"
    if isinstance(value, list):
        s = str(value[0]).strip().lower() if value else "road"

    # If OSM returns multiple tags or strings like "['primary', 'secondary']"
    if s.startswith("["):
        vals = parse_listlike(value)
        for v in vals:
            vv = clean_text(v).lower().replace(" ", "_")
            if vv in ROAD_CLASS_SPEED_KMH:
                return vv
        return "road"

    s = s.replace(" ", "_")
    mapping = {
        "motorway_link": "motorway",
        "trunk_link": "trunk",
        "primary_link": "primary",
        "secondary_link": "secondary",
        "tertiary_link": "tertiary",
    }
    s = mapping.get(s, s)
    return s if s in ROAD_CLASS_SPEED_KMH else "road"

def save_plot(path):
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()

# -------------------------
# Load Phase 4 outputs
# -------------------------
if not PHASE4_MERGED_PATH.exists():
    raise FileNotFoundError(
        "Phase 4 merged output not found. Expected phase4_outputs/phase4_merged_with_prior_scores.csv"
    )

df = pd.read_csv(PHASE4_MERGED_PATH, low_memory=False)
cluster_col = standardize_cluster_col(df)
vehicle_col = standardize_vehicle_col(df)
vehicle_type_col = standardize_vehicle_type_col(df)

df["created_datetime_ist"] = parse_datetime_ist(df)
df["created_datetime_ist_naive"] = df["created_datetime_ist"].dt.tz_localize(None)
df["week_start"] = week_start_monday(df["created_datetime"])

if "validation_status_clean" in df.columns:
    df["validation_status_clean"] = df["validation_status_clean"].fillna("").astype(str).str.lower()
elif "validation_status" in df.columns:
    df["validation_status_clean"] = df["validation_status"].fillna("").astype(str).str.lower()
else:
    df["validation_status_clean"] = "approved"

if "severity_score" not in df.columns:
    df["severity_score"] = 1.0

if "hotspot_unit" not in df.columns:
    if {"junction_name", "police_station"}.issubset(df.columns):
        df["hotspot_unit"] = df.apply(make_hotspot_unit, axis=1)
    else:
        df["hotspot_unit"] = "UNKNOWN"

approved = df.loc[df["validation_status_clean"].eq("approved")].copy()
approved = approved.dropna(subset=[cluster_col, "created_datetime_ist_naive", "latitude", "longitude"]).copy()
approved = approved.loc[approved[cluster_col].ne(-1)].copy()
approved = approved.loc[approved[vehicle_col].fillna("").astype(str).str.strip().ne("")].copy()
approved = approved.reset_index(drop=True)

if approved.empty:
    raise ValueError("No approved clustered rows available for Stage 5.")

print("Approved clustered rows:", len(approved))
print("Clusters:", approved[cluster_col].nunique())

# -------------------------
# Stage 5a — λ (arrival rate) by cluster
# Vectorized weekly/peak-window estimation
# -------------------------
approved["hour_ist"] = approved["created_datetime_ist"].dt.hour
approved["peak_window_flag"] = approved["hour_ist"].between(8, 12, inclusive="both").astype(int)

global_distinct_days = max(1, approved["created_datetime_ist_naive"].dt.date.nunique())

cluster_lambda = (
    approved.groupby(cluster_col)
    .agg(
        records_total=(vehicle_col, "size"),
        records_peak_window=("peak_window_flag", "sum"),
        records_peak_hour=(vehicle_col, lambda s: approved.loc[s.index, "hour_ist"].value_counts().max()),
        first_seen=("created_datetime_ist_naive", "min"),
        last_seen=("created_datetime_ist_naive", "max"),
        centroid_lat=("latitude", "mean"),
        centroid_lon=("longitude", "mean"),
        severity_sum=("severity_score", "sum"),
        severity_mean=("severity_score", "mean"),
        unique_vehicles=(vehicle_col, "nunique"),
        unique_vehicle_types=(vehicle_type_col, "nunique"),
    )
    .reset_index()
)

cluster_lambda["lambda_hr_peak_window"] = cluster_lambda["records_peak_window"] / (global_distinct_days * 5.0)
cluster_lambda["lambda_hr_peak_hour"] = pd.to_numeric(cluster_lambda["records_peak_hour"], errors="coerce").fillna(0.0) / global_distinct_days

# -------------------------
# Stage 4 carry-over — μ / dwell time
# Prefer Phase 4 summary; no recomputation here
# -------------------------
if not PHASE4_CLUSTER_PATH.exists():
    raise FileNotFoundError(
        "Phase 4 cluster summary not found. Expected phase4_outputs/phase4_cluster_mu_summary.csv"
    )

mu_df = pd.read_csv(PHASE4_CLUSTER_PATH, low_memory=False)
if cluster_col not in mu_df.columns and "st_dbscan_cluster_id" in mu_df.columns:
    mu_df = mu_df.rename(columns={"st_dbscan_cluster_id": cluster_col})

keep_cols = [cluster_col]
for c in [
    "cluster_label", "gap_count", "mean_dwell_minutes",
    "median_dwell_minutes", "std_dwell_minutes",
    "mu_departures_per_hour"
]:
    if c in mu_df.columns:
        keep_cols.append(c)

mu_df = mu_df[keep_cols].drop_duplicates(cluster_col)
cluster_table = cluster_lambda.merge(mu_df, on=cluster_col, how="left")

if "mu_departures_per_hour" not in cluster_table.columns:
    cluster_table["mu_departures_per_hour"] = 60.0 / (pd.to_numeric(cluster_table["mean_dwell_minutes"], errors="coerce") + EPS)

cluster_table["mean_dwell_minutes"] = pd.to_numeric(cluster_table["mean_dwell_minutes"], errors="coerce")
cluster_table["mu_departures_per_hour"] = pd.to_numeric(cluster_table["mu_departures_per_hour"], errors="coerce")
cluster_table["mu_departures_per_hour"] = cluster_table["mu_departures_per_hour"].fillna(
    60.0 / (cluster_table["mean_dwell_minutes"] + EPS)
)

cluster_table["blocking_vehicles_L"] = (
    cluster_table["lambda_hr_peak_window"] / (cluster_table["mu_departures_per_hour"] + EPS)
)

# -------------------------
# Vehicle mix / average blocking width
# -------------------------
vehicle_mix = (
    approved.groupby([cluster_col, vehicle_type_col])
    .size()
    .reset_index(name="count")
    .rename(columns={vehicle_type_col: "vehicle_type"})
)
vehicle_mix["vehicle_type"] = vehicle_mix["vehicle_type"].fillna("UNKNOWN").astype(str).str.strip()
vehicle_mix["vehicle_width_m"] = vehicle_mix["vehicle_type"].apply(vehicle_width_m)

cluster_width = (
    vehicle_mix.groupby(cluster_col)
    .apply(lambda g: safe_ratio((g["count"] * g["vehicle_width_m"]).sum(), g["count"].sum()))
    .reset_index(name="vehicle_width_avg_m")
)

cluster_table = cluster_table.merge(cluster_width, on=cluster_col, how="left")
cluster_table["vehicle_width_avg_m"] = cluster_table["vehicle_width_avg_m"].fillna(1.90)

# -------------------------
# Trend acceleration / growth multiplier
# -------------------------
weekly_counts = (
    approved.groupby([cluster_col, approved["created_datetime_ist"].dt.to_period("W-MON").apply(lambda p: p.start_time)])
    .size()
    .reset_index(name="weekly_count")
    .rename(columns={"created_datetime_ist": "cluster_week_start"})
    .sort_values([cluster_col, "cluster_week_start"])
)

growth_rows = []
for cid, g in weekly_counts.groupby(cluster_col, sort=False):
    counts = g["weekly_count"].to_numpy(dtype=float)
    if len(counts) < 2:
        first_mean = float(counts.mean()) if len(counts) else 0.0
        second_mean = first_mean
        growth_pct = 0.0
    else:
        mid = max(1, len(counts) // 2)
        first_half = counts[:mid]
        second_half = counts[mid:]
        first_mean = float(first_half.mean()) if len(first_half) else 0.0
        second_mean = float(second_half.mean()) if len(second_half) else 0.0
        growth_pct = safe_ratio((second_mean - first_mean), first_mean) if first_mean > 0 else second_mean

    growth_multiplier = max(0.5, 1.0 + max(0.0, growth_pct))
    growth_rows.append({
        cluster_col: cid,
        "growth_first_half_mean": first_mean,
        "growth_second_half_mean": second_mean,
        "growth_pct_change": growth_pct,
        "growth_multiplier": growth_multiplier,
    })

growth_df = pd.DataFrame(growth_rows)

# -------------------------
# Road-network context
# FAST strategy:
# 1) Reuse cache if available
# 2) If cache missing + OSMnx enabled: download FULL Bengaluru graph ONCE,
#    then do local nearest-node/edge lookups (no per-cluster API calls)
# 3) If OSMnx disabled or fails: use sensible defaults
# -------------------------
road_context_df = None

if PHASE5_ROAD_CACHE_PATH.exists():
    print("Loading cached road context...")
    road_context_df = pd.read_csv(PHASE5_ROAD_CACHE_PATH, low_memory=False)
    if cluster_col not in road_context_df.columns and "st_dbscan_cluster_id" in road_context_df.columns:
        road_context_df = road_context_df.rename(columns={"st_dbscan_cluster_id": cluster_col})
    print(f"  Loaded {len(road_context_df)} cached rows.")

if road_context_df is None:
    # Build default fallback first
    road_context_df = cluster_table[[cluster_col, "centroid_lat", "centroid_lon"]].copy()
    road_context_df["geometry_source"] = "fallback"
    road_context_df["road_class"] = "road"
    road_context_df["lane_count"] = DEFAULT_LANES
    road_context_df["carriageway_width_m"] = DEFAULT_CARRIAGEWAY_WIDTH_M
    road_context_df["link_length_m"] = DEFAULT_LINK_LENGTH_M
    road_context_df["junction_degree"] = 4
    road_context_df["betweenness_centrality"] = 0.0

    if ENABLE_OSMNX:
        try:
            import osmnx as ox
            import networkx as nx

            print("Downloading full Bengaluru road network (one-time, ~30-60s)...")
            G = ox.graph_from_place(
                "Bengaluru, Karnataka, India",
                network_type="drive",
                simplify=True,
                retain_all=False,
            )
            print(f"  Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

            UG = nx.Graph(G)  # undirected copy for degree

            enrich_rows = []
            total = len(cluster_table)
            for i, (_, row) in enumerate(cluster_table.iterrows()):
                cid = row[cluster_col]
                lat = float(row["centroid_lat"])
                lon = float(row["centroid_lon"])

                result = {
                    cluster_col: cid,
                    "geometry_source": "fallback",
                    "road_class": "road",
                    "lane_count": DEFAULT_LANES,
                    "carriageway_width_m": DEFAULT_CARRIAGEWAY_WIDTH_M,
                    "link_length_m": DEFAULT_LINK_LENGTH_M,
                    "junction_degree": 4,
                    "betweenness_centrality": 0.0,
                }
                try:
                    nearest_node = ox.distance.nearest_nodes(G, X=lon, Y=lat)
                    nearest_edge = ox.distance.nearest_edges(G, X=lon, Y=lat)
                    u, v, k = nearest_edge
                    edge_data = G.edges[u, v, k]

                    highway = edge_data.get("highway", "road")
                    road_class = normalize_road_class(highway)

                    lanes = edge_data.get("lanes", None)
                    if isinstance(lanes, list):
                        lanes = lanes[0] if lanes else None
                    try:
                        lanes = int(round(float(lanes))) if lanes is not None else DEFAULT_LANES
                    except Exception:
                        lanes = DEFAULT_LANES
                    lanes = max(1, lanes)

                    width = edge_data.get("width", None)
                    if isinstance(width, list):
                        width = width[0] if width else None
                    try:
                        width = float(str(width).replace("m", "").strip()) if width is not None else lanes * DEFAULT_WIDTH_PER_LANE_M
                    except Exception:
                        width = lanes * DEFAULT_WIDTH_PER_LANE_M

                    length_m = float(edge_data.get("length", DEFAULT_LINK_LENGTH_M))
                    degree = int(UG.degree[nearest_node])

                    # Local betweenness: use ego_graph (2-hop neighborhood) instead of full graph
                    ego = nx.ego_graph(UG, nearest_node, radius=2)
                    if ego.number_of_nodes() > 2:
                        bc = nx.betweenness_centrality(ego, normalized=True, weight="length")
                        centrality = float(bc.get(nearest_node, 0.0))
                    else:
                        centrality = 0.0

                    result.update({
                        "geometry_source": "osmnx",
                        "road_class": road_class,
                        "lane_count": lanes,
                        "carriageway_width_m": width,
                        "link_length_m": length_m,
                        "junction_degree": degree,
                        "betweenness_centrality": centrality,
                    })
                except Exception:
                    pass

                enrich_rows.append(result)

                if (i + 1) % 100 == 0 or (i + 1) == total:
                    print(f"  Enriched {i+1}/{total} clusters")

            road_context_df = pd.DataFrame(enrich_rows)
            print(f"  OSMnx enrichment complete. {sum(1 for r in enrich_rows if r['geometry_source'] == 'osmnx')}/{total} clusters enriched.")

        except Exception as e:
            print(f"  OSMnx enrichment failed: {e}. Using defaults.")
            road_context_df = cluster_table[[cluster_col, "centroid_lat", "centroid_lon"]].copy()
            road_context_df["geometry_source"] = "fallback"
            road_context_df["road_class"] = "road"
            road_context_df["lane_count"] = DEFAULT_LANES
            road_context_df["carriageway_width_m"] = DEFAULT_CARRIAGEWAY_WIDTH_M
            road_context_df["link_length_m"] = DEFAULT_LINK_LENGTH_M
            road_context_df["junction_degree"] = 4
            road_context_df["betweenness_centrality"] = 0.0

# Merge road context
cluster_table = cluster_table.merge(
    road_context_df[[c for c in road_context_df.columns if c in [
        cluster_col, "geometry_source", "road_class", "lane_count",
        "carriageway_width_m", "link_length_m", "junction_degree",
        "betweenness_centrality"
    ]]],
    on=cluster_col,
    how="left"
)

# Fill geometry defaults
cluster_table["geometry_source"] = cluster_table.get("geometry_source", "fallback").fillna("fallback")
cluster_table["road_class"] = cluster_table.get("road_class", "road").fillna("road").astype(str)
cluster_table["lane_count"] = pd.to_numeric(cluster_table.get("lane_count", DEFAULT_LANES), errors="coerce").fillna(DEFAULT_LANES).astype(int)
cluster_table["carriageway_width_m"] = pd.to_numeric(cluster_table.get("carriageway_width_m", DEFAULT_CARRIAGEWAY_WIDTH_M), errors="coerce").fillna(DEFAULT_CARRIAGEWAY_WIDTH_M)
cluster_table["link_length_m"] = pd.to_numeric(cluster_table.get("link_length_m", DEFAULT_LINK_LENGTH_M), errors="coerce").fillna(DEFAULT_LINK_LENGTH_M)
cluster_table["junction_degree"] = pd.to_numeric(cluster_table.get("junction_degree", 4), errors="coerce").fillna(4.0)
cluster_table["betweenness_centrality"] = pd.to_numeric(cluster_table.get("betweenness_centrality", 0.0), errors="coerce").fillna(0.0)

cluster_table["junction_degree_norm"] = minmax(cluster_table["junction_degree"])
cluster_table["betweenness_centrality_norm"] = minmax(cluster_table["betweenness_centrality"])
cluster_table["criticality_factor"] = 1.0 + 0.5 * cluster_table["junction_degree_norm"] + 0.5 * cluster_table["betweenness_centrality_norm"]

# -------------------------
# Stage 5b — Zhao capacity loss
# -------------------------
cluster_table["base_saturation_per_lane_pcu_hr"] = cluster_table["road_class"].map(base_capacity_per_lane).fillna(1600.0)
cluster_table["free_flow_speed_kmh"] = cluster_table["road_class"].apply(road_speed_kmh)
cluster_table["free_flow_time_min"] = cluster_table["link_length_m"] * 60.0 / (cluster_table["free_flow_speed_kmh"] * 1000.0 + EPS)

cluster_table["width_factor"] = cluster_table["carriageway_width_m"] / (cluster_table["lane_count"] * DEFAULT_WIDTH_PER_LANE_M + EPS)
cluster_table["width_factor"] = cluster_table["width_factor"].clip(0.70, 1.20)

cluster_table["base_capacity_pcu_hr"] = (
    cluster_table["base_saturation_per_lane_pcu_hr"] *
    cluster_table["lane_count"] *
    cluster_table["width_factor"]
)

cluster_table["blocked_width_fraction_raw"] = (
    cluster_table["blocking_vehicles_L"] * cluster_table["vehicle_width_avg_m"]
) / (cluster_table["carriageway_width_m"] + EPS)

cluster_table["blocked_width_fraction"] = cluster_table["blocked_width_fraction_raw"].clip(0.0, 0.95)
cluster_table["reduced_capacity_pcu_hr"] = (
    cluster_table["base_capacity_pcu_hr"] * (1.0 - cluster_table["blocked_width_fraction"])
).clip(lower=cluster_table["base_capacity_pcu_hr"] * 0.10)

cluster_table["capacity_loss_pct"] = 1.0 - safe_ratio(cluster_table["reduced_capacity_pcu_hr"], cluster_table["base_capacity_pcu_hr"])

# -------------------------
# Stage 5c — Modified BPR delay
# -------------------------
cluster_table["V_over_C0"] = cluster_table["lambda_hr_peak_window"] / (cluster_table["base_capacity_pcu_hr"] + EPS)
cluster_table["V_over_Cp"] = cluster_table["lambda_hr_peak_window"] / (cluster_table["reduced_capacity_pcu_hr"] + EPS)

cluster_table["delay_minutes_per_vehicle"] = (
    cluster_table["free_flow_time_min"] * ALPHA_BPR *
    (np.power(cluster_table["V_over_Cp"], BETA_BPR) - np.power(cluster_table["V_over_C0"], BETA_BPR))
).clip(lower=0.0)

# -------------------------
# Growth / repeat offender support for Stage 6
# -------------------------
cluster_table = cluster_table.merge(growth_df, on=cluster_col, how="left")
cluster_table["growth_pct_change"] = pd.to_numeric(cluster_table["growth_pct_change"], errors="coerce").fillna(0.0)
cluster_table["growth_multiplier"] = pd.to_numeric(cluster_table["growth_multiplier"], errors="coerce").fillna(1.0)

repeat_vehicle_counts = (
    approved.groupby([cluster_col, vehicle_col])
    .size()
    .reset_index(name="vehicle_cluster_count")
)
repeat_vehicle_counts["repeat_flag_2plus"] = (repeat_vehicle_counts["vehicle_cluster_count"] >= 2).astype(int)
repeat_vehicle_counts["chronic_flag_5plus"] = (repeat_vehicle_counts["vehicle_cluster_count"] >= 5).astype(int)

cluster_repeat_summary = (
    repeat_vehicle_counts.groupby(cluster_col)
    .agg(
        repeat_vehicle_count_2plus=("repeat_flag_2plus", "sum"),
        chronic_vehicle_count_5plus=("chronic_flag_5plus", "sum"),
    )
    .reset_index()
)
cluster_table = cluster_table.merge(cluster_repeat_summary, on=cluster_col, how="left")
cluster_table["repeat_vehicle_count_2plus"] = cluster_table["repeat_vehicle_count_2plus"].fillna(0).astype(int)
cluster_table["chronic_vehicle_count_5plus"] = cluster_table["chronic_vehicle_count_5plus"].fillna(0).astype(int)

# -------------------------
# Final Stage 5 tables
# -------------------------
priority_table = cluster_table.sort_values(
    ["delay_minutes_per_vehicle", "lambda_hr_peak_window", "blocking_vehicles_L"],
    ascending=[False, False, False]
).reset_index(drop=True)

priority_table["stage5_rank"] = np.arange(1, len(priority_table) + 1)
priority_table["delay_minutes_norm"] = minmax(priority_table["delay_minutes_per_vehicle"])
priority_table["lambda_norm"] = minmax(priority_table["lambda_hr_peak_window"])
priority_table["severity_sum_norm"] = minmax(priority_table["severity_sum"])

# Stage 5 handoff view
handoff_cols = [
    "stage5_rank", cluster_col, "cluster_label", "records_total", "records_peak_window",
    "lambda_hr_peak_window", "mean_dwell_minutes", "mu_departures_per_hour", "blocking_vehicles_L",
    "road_class", "lane_count", "carriageway_width_m", "link_length_m", "base_capacity_pcu_hr",
    "reduced_capacity_pcu_hr", "capacity_loss_pct", "free_flow_time_min", "delay_minutes_per_vehicle",
    "junction_degree", "betweenness_centrality", "criticality_factor", "growth_pct_change",
    "growth_multiplier", "vehicle_width_avg_m", "dominant_vehicle_type", "unique_vehicles",
    "unique_vehicle_types", "repeat_vehicle_count_2plus", "chronic_vehicle_count_5plus",
    "geometry_source", "centroid_lat", "centroid_lon"
]
handoff_cols = [c for c in handoff_cols if c in priority_table.columns]

# Record-level view for Stage 6
stage5_record_view = approved.merge(
    priority_table[[
        cluster_col, "cluster_label", "records_total", "lambda_hr_peak_window", "mean_dwell_minutes",
        "mu_departures_per_hour", "blocking_vehicles_L", "road_class", "lane_count",
        "carriageway_width_m", "link_length_m", "base_capacity_pcu_hr", "reduced_capacity_pcu_hr",
        "capacity_loss_pct", "free_flow_time_min", "delay_minutes_per_vehicle",
        "junction_degree", "betweenness_centrality", "criticality_factor",
        "growth_pct_change", "growth_multiplier", "stage5_rank"
    ]],
    on=cluster_col,
    how="left"
)

# Chronic offenders table
vehicle_offenders = (
    approved.groupby(vehicle_col)
    .agg(
        total_violations=(vehicle_col, "size"),
        unique_clusters=(cluster_col, "nunique"),
        unique_hotspots=("hotspot_unit", "nunique"),
        first_seen=("created_datetime_ist_naive", "min"),
        last_seen=("created_datetime_ist_naive", "max"),
        dominant_vehicle_type=(vehicle_type_col, lambda s: s.mode().iloc[0] if not s.mode().empty else "UNKNOWN"),
    )
    .reset_index()
    .rename(columns={vehicle_col: "vehicle_number"})
    .sort_values(["total_violations", "unique_clusters"], ascending=[False, False])
)
vehicle_offenders["chronic_offender_flag"] = (vehicle_offenders["total_violations"] >= 5).astype(int)
chronic_offenders = vehicle_offenders.loc[vehicle_offenders["chronic_offender_flag"].eq(1)].copy()

# -------------------------
# Save outputs
# -------------------------
cluster_table.to_csv(OUT_DIR / "phase5_cluster_capacity_loss.csv", index=False)
priority_table[handoff_cols].to_csv(OUT_DIR / "phase5_priority_table.csv", index=False)
priority_table.to_csv(OUT_DIR / "phase5_priority_table_full.csv", index=False)
stage5_record_view.to_csv(OUT_DIR / "phase5_enriched_records.csv", index=False)
vehicle_mix.to_csv(OUT_DIR / "phase5_vehicle_mix.csv", index=False)
weekly_counts.to_csv(OUT_DIR / "phase5_weekly_cluster_counts.csv", index=False)
growth_df.to_csv(OUT_DIR / "phase5_growth_summary.csv", index=False)
chronic_offenders.to_csv(OUT_DIR / "phase5_chronic_offenders.csv", index=False)
road_context_df.to_csv(OUT_DIR / "phase5_road_context_cache.csv", index=False)

priority_table[[
    "stage5_rank", cluster_col, "cluster_label", "delay_minutes_per_vehicle",
    "lambda_hr_peak_window", "mu_departures_per_hour", "blocking_vehicles_L",
    "capacity_loss_pct", "criticality_factor", "growth_multiplier", "road_class",
    "lane_count", "carriageway_width_m", "free_flow_time_min"
]].to_csv(OUT_DIR / "phase5_stage5_handoff.csv", index=False)

# -------------------------
# Console summary
# -------------------------
print("Phase 5 complete")
print("Approved clustered rows:", len(approved))
print("Clusters scored:", len(priority_table))
print("Chronic offenders:", len(chronic_offenders))
print("OSMnx enabled:", ENABLE_OSMNX)
print("Outputs saved to:", OUT_DIR.resolve())

print("\nTop 10 clusters by delay:")
print(
    priority_table[[
        "stage5_rank", cluster_col, "cluster_label", "delay_minutes_per_vehicle",
        "lambda_hr_peak_window", "mu_departures_per_hour", "blocking_vehicles_L",
        "capacity_loss_pct", "criticality_factor", "growth_multiplier"
    ]].head(10).to_string(index=False)
)

# -------------------------
# Plots
# -------------------------
plt.figure(figsize=(12, 7))
top_delay = priority_table.head(TOP_N).sort_values("delay_minutes_per_vehicle", ascending=True)
plt.barh(top_delay["cluster_label"] if "cluster_label" in top_delay.columns else top_delay[cluster_col], top_delay["delay_minutes_per_vehicle"])
plt.title("Top Clusters by Delay Minutes per Vehicle")
plt.xlabel("Delay minutes per vehicle")
plt.ylabel("Cluster")
save_plot(OUT_DIR / "top_clusters_delay.png")

plt.figure(figsize=(12, 7))
top_capacity = priority_table.head(TOP_N).sort_values("capacity_loss_pct", ascending=True)
plt.barh(top_capacity["cluster_label"] if "cluster_label" in top_capacity.columns else top_capacity[cluster_col], top_capacity["capacity_loss_pct"])
plt.title("Top Clusters by Capacity Loss")
plt.xlabel("Capacity loss fraction")
plt.ylabel("Cluster")
save_plot(OUT_DIR / "top_clusters_capacity_loss.png")

plt.figure(figsize=(12, 5))
plt.hist(priority_table["delay_minutes_per_vehicle"], bins=30)
plt.title("Delay Minutes per Vehicle Distribution")
plt.xlabel("Delay minutes per vehicle")
plt.ylabel("Clusters")
save_plot(OUT_DIR / "delay_distribution.png")

Approved clustered rows: 92332
Clusters: 798
Loading cached road context...
  Loaded 798 cached rows.
Phase 5 complete
Approved clustered rows: 92332
Clusters scored: 798
Chronic offenders: 638
OSMnx enabled: True
Outputs saved to: D:\Flipkart_Hackathon\content\phase5_outputs_1

Top 10 clusters by delay:
 stage5_rank  st_dbscan_cluster_id                                cluster_label  delay_minutes_per_vehicle  lambda_hr_peak_window  mu_departures_per_hour  blocking_vehicles_L  capacity_loss_pct  criticality_factor  growth_multiplier
           1                     2            JUNCTION::BTP040 - Elite Junction               6.986447e-04              19.007547                0.592115            32.101127           0.900000                 1.0           1.000000
           2                     3     JUNCTION::BTP051 - Safina Plaza Junction               2.493127e-04              11.690566                0.661847            17.663557           0.900000                 1.0           1.00

In [ ]:
# =========================
# Phase 6 — Congestion Cost Score (CCS) Computation
# Corrected version:
# - Fills missing labels and numeric fields
# - Uses smoothed normalization to avoid zero-collapse
# - Produces final CCS ranking, dispatch table, alerts, chronic list
# =========================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# -------------------------
# Config
# -------------------------
PHASE5_CLUSTER_PATH = Path("content/phase5_outputs_1/phase5_cluster_capacity_loss.csv")
PHASE5_PRIORITY_PATH = Path("content/phase5_outputs_1/phase5_priority_table_full.csv")
PHASE5_GROWTH_PATH = Path("content/phase5_outputs_1/phase5_growth_summary.csv")
PHASE5_CHRONIC_PATH = Path("content/phase5_outputs_1/phase5_chronic_offenders.csv")
PHASE5_ENRICHED_RECORDS_PATH = Path("content/phase5_outputs_1/phase5_enriched_records.csv")

PHASE4_MERGED_PATH = Path("content/phase4_outputs_1/phase4_merged_with_prior_scores.csv")
PHASE3_SUMMARY_PATH = Path("content/phase3_outputs_1/phase3_cluster_summary.csv")

OUT_DIR = Path("content/phase6_outputs_1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EPS = 1e-9
TOP_N = 25

# -------------------------
# Helpers
# -------------------------
def minmax(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce").fillna(0.0).astype(float)
    if s.nunique(dropna=True) <= 1:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.min()) / (s.max() - s.min() + EPS)

def smooth_norm(s: pd.Series, floor: float = 0.10) -> pd.Series:
    """
    Prevents collapse to zero when a feature is nearly constant.
    Returns values in [floor, 1.0].
    """
    return floor + (1.0 - floor) * minmax(s)

def load_first_existing(paths):
    for p in paths:
        if p.exists():
            return pd.read_csv(p, low_memory=False), p
    return None, None

def standardize_cluster_col(df: pd.DataFrame) -> str:
    for c in ["st_dbscan_cluster_id", "cluster_id", "dbscan_cluster_id"]:
        if c in df.columns:
            return c
    raise ValueError("No cluster id column found.")

def standardize_vehicle_col(df: pd.DataFrame) -> str:
    if "canonical_vehicle_number" in df.columns:
        return "canonical_vehicle_number"
    if "vehicle_number" in df.columns:
        return "vehicle_number"
    raise ValueError("No vehicle number column found.")

def standardize_vehicle_type_col(df: pd.DataFrame):
    if "canonical_vehicle_type" in df.columns:
        return "canonical_vehicle_type"
    if "vehicle_type" in df.columns:
        return "vehicle_type"
    return None

def derive_cluster_label(df: pd.DataFrame, cluster_col: str) -> pd.Series:
    """
    Backfill cluster labels from cluster_label -> hotspot_unit -> junction_name/police_station -> cluster id.
    """
    label = pd.Series([""] * len(df), index=df.index, dtype="object")

    if "cluster_label" in df.columns:
        label = df["cluster_label"].fillna("").astype(str).str.strip()

    if "hotspot_unit" in df.columns:
        empty = label.eq("")
        label.loc[empty] = df.loc[empty, "hotspot_unit"].fillna("").astype(str).str.strip()

    # Try to improve labels from junction_name / police_station if hotspot_unit is missing or generic
    if {"junction_name", "police_station"}.issubset(df.columns):
        empty = label.eq("") | label.str.lower().eq("nan")
        junction = df.loc[empty, "junction_name"].fillna("").astype(str).str.strip()
        ps = df.loc[empty, "police_station"].fillna("").astype(str).str.strip()

        junction_ok = junction.ne("") & junction.str.upper().ne("NO JUNCTION")
        label.loc[empty] = np.where(
            junction_ok,
            "JUNCTION::" + junction.where(junction_ok, ""),
            "POLICE_STATION::" + ps.where(ps.ne(""), "UNKNOWN"),
        )

    empty = label.eq("") | label.str.lower().eq("nan")
    label.loc[empty] = "CLUSTER::" + df.loc[empty, cluster_col].astype(str)

    return label

def road_speed_kmh(road_class):
    ROAD_CLASS_SPEED_KMH = {
        "motorway": 60.0,
        "trunk": 55.0,
        "primary": 45.0,
        "secondary": 40.0,
        "tertiary": 35.0,
        "unclassified": 30.0,
        "residential": 30.0,
        "living_street": 20.0,
        "service": 25.0,
        "road": 30.0,
    }
    return ROAD_CLASS_SPEED_KMH.get(str(road_class).strip().lower(), 30.0)

def base_capacity_per_lane(road_class):
    ROAD_CLASS_BASE_CAPACITY_PER_LANE = {
        "motorway": 2200.0,
        "trunk": 2100.0,
        "primary": 1900.0,
        "secondary": 1800.0,
        "tertiary": 1700.0,
        "unclassified": 1650.0,
        "residential": 1500.0,
        "living_street": 1200.0,
        "service": 1400.0,
        "road": 1600.0,
    }
    return ROAD_CLASS_BASE_CAPACITY_PER_LANE.get(str(road_class).strip().lower(), 1600.0)

def vehicle_width_m(vehicle_type):
    VEHICLE_WIDTH_M = {
        "SCOOTER": 0.80,
        "MOTOR CYCLE": 0.90,
        "MOTORCYCLE": 0.90,
        "BICYCLE": 0.60,
        "CYCLE": 0.60,
        "PASSENGER AUTO": 1.60,
        "AUTO": 1.60,
        "CAR": 1.90,
        "SUV": 2.00,
        "JEEP": 2.00,
        "VAN": 2.20,
        "TEMPO": 2.20,
        "BUS": 2.60,
        "TRUCK": 2.60,
        "LORRY": 2.60,
        "TANKER": 2.80,
        "TRACTOR": 2.20,
        "MINI TRUCK": 2.20,
        "AMBULANCE": 2.00,
        "UNKNOWN": 1.90,
    }
    return VEHICLE_WIDTH_M.get(str(vehicle_type).strip().upper(), 1.90)

def save_plot(path):
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()

# -------------------------
# Load Phase 5 output
# -------------------------
cluster_df, cluster_src = load_first_existing([PHASE5_CLUSTER_PATH, PHASE5_PRIORITY_PATH])
if cluster_df is None:
    raise FileNotFoundError("Missing Phase 5 output. Expected phase5_outputs/*.csv")

cluster_col = standardize_cluster_col(cluster_df)

# -------------------------
# Backfill labels and key fields early
# -------------------------
cluster_df["cluster_label"] = derive_cluster_label(cluster_df, cluster_col)

# Fill important text cols
for col, default in [
    ("road_class", "road"),
    ("geometry_source", "fallback"),
    ("dominant_vehicle_type", "UNKNOWN"),
]:
    if col not in cluster_df.columns:
        cluster_df[col] = default
    cluster_df[col] = cluster_df[col].fillna(default).astype(str).replace({"nan": default, "": default})

# Fill numeric cols safely
numeric_defaults = {
    "delay_minutes_per_vehicle": 0.0,
    "lambda_hr_peak_window": 0.0,
    "lambda_hr_peak_hour": 0.0,
    "mu_departures_per_hour": 0.0,
    "mean_dwell_minutes": 0.0,
    "severity_sum": 0.0,
    "severity_mean": 0.0,
    "growth_pct_change": 0.0,
    "growth_multiplier": 1.0,
    "criticality_factor": 1.0,
    "capacity_loss_pct": 0.0,
    "blocking_vehicles_L": 0.0,
    "records_total": 0.0,
    "records_peak_window": 0.0,
    "records_peak_hour": 0.0,
    "junction_degree": 4.0,
    "betweenness_centrality": 0.0,
    "lane_count": 2.0,
    "carriageway_width_m": 7.0,
    "link_length_m": 250.0,
    "unique_vehicles": 0.0,
    "unique_vehicle_types": 0.0,
    "vehicle_width_avg_m": 1.9,
    "repeat_vehicle_count_2plus": 0.0,
    "chronic_vehicle_count_5plus": 0.0,
}

for col, default in numeric_defaults.items():
    if col not in cluster_df.columns:
        cluster_df[col] = default
    cluster_df[col] = pd.to_numeric(cluster_df[col], errors="coerce").fillna(default)

# Prefer the more informative lambda column
if "lambda_hr_peak_window" in cluster_df.columns and cluster_df["lambda_hr_peak_window"].abs().sum() > 0:
    lambda_col = "lambda_hr_peak_window"
elif "lambda_hr_peak_hour" in cluster_df.columns:
    lambda_col = "lambda_hr_peak_hour"
else:
    raise ValueError("No lambda column found in Phase 5 output.")

# If phase 5 priority file has growth / criticality already, keep them; otherwise backfill
if "growth_multiplier" not in cluster_df.columns:
    cluster_df["growth_multiplier"] = 1.0
if "growth_pct_change" not in cluster_df.columns:
    cluster_df["growth_pct_change"] = 0.0
if "criticality_factor" not in cluster_df.columns:
    cluster_df["criticality_factor"] = 1.0

# If a growth summary exists, merge it in
growth_df, _ = load_first_existing([PHASE5_GROWTH_PATH])
if growth_df is not None:
    if cluster_col not in growth_df.columns and "st_dbscan_cluster_id" in growth_df.columns:
        growth_df = growth_df.rename(columns={"st_dbscan_cluster_id": cluster_col})
    growth_keep = [cluster_col]
    for c in ["growth_first_half_mean", "growth_second_half_mean", "growth_pct_change", "growth_multiplier"]:
        if c in growth_df.columns:
            growth_keep.append(c)
    growth_df = growth_df[growth_keep].drop_duplicates(cluster_col)
    cluster_df = cluster_df.merge(growth_df, on=cluster_col, how="left", suffixes=("", "_g"))
    for c in ["growth_pct_change", "growth_multiplier"]:
        if f"{c}_g" in cluster_df.columns:
            cluster_df[c] = cluster_df[f"{c}_g"].combine_first(cluster_df[c])
            cluster_df.drop(columns=[f"{c}_g"], inplace=True)

# Refill after merge
for col, default in [("growth_pct_change", 0.0), ("growth_multiplier", 1.0)]:
    cluster_df[col] = pd.to_numeric(cluster_df[col], errors="coerce").fillna(default)

# Merge chronic counts if available
if "repeat_vehicle_count_2plus" not in cluster_df.columns:
    cluster_df["repeat_vehicle_count_2plus"] = 0
if "chronic_vehicle_count_5plus" not in cluster_df.columns:
    cluster_df["chronic_vehicle_count_5plus"] = 0

# Backfill cluster label again after merges
cluster_df["cluster_label"] = derive_cluster_label(cluster_df, cluster_col)

# -------------------------
# Ensure CCS inputs exist
# -------------------------
required_inputs = ["delay_minutes_per_vehicle", lambda_col, "severity_sum", "growth_multiplier", "criticality_factor"]
for c in required_inputs:
    if c not in cluster_df.columns:
        raise ValueError(f"Missing required CCS input: {c}")

cluster_df["delay_minutes_per_vehicle"] = pd.to_numeric(cluster_df["delay_minutes_per_vehicle"], errors="coerce").fillna(0.0).clip(lower=0.0)
cluster_df[lambda_col] = pd.to_numeric(cluster_df[lambda_col], errors="coerce").fillna(0.0).clip(lower=0.0)
cluster_df["severity_sum"] = pd.to_numeric(cluster_df["severity_sum"], errors="coerce").fillna(0.0).clip(lower=0.0)
cluster_df["growth_multiplier"] = pd.to_numeric(cluster_df["growth_multiplier"], errors="coerce").fillna(1.0).clip(lower=0.1)
cluster_df["criticality_factor"] = pd.to_numeric(cluster_df["criticality_factor"], errors="coerce").fillna(1.0).clip(lower=0.1)
cluster_df["growth_pct_change"] = pd.to_numeric(cluster_df["growth_pct_change"], errors="coerce").fillna(0.0)

# -------------------------
# Smoothed normalization for CCS
# -------------------------
cluster_df["delay_norm"] = smooth_norm(cluster_df["delay_minutes_per_vehicle"], floor=0.10)
cluster_df["lambda_norm"] = smooth_norm(cluster_df[lambda_col], floor=0.10)
cluster_df["severity_norm"] = smooth_norm(cluster_df["severity_sum"], floor=0.10)
cluster_df["growth_norm"] = smooth_norm(cluster_df["growth_multiplier"], floor=0.10)
cluster_df["criticality_norm"] = smooth_norm(cluster_df["criticality_factor"], floor=0.10)

# Raw product retained for traceability
cluster_df["ccs_raw_product"] = (
    cluster_df["delay_norm"] *
    cluster_df["lambda_norm"] *
    cluster_df["severity_norm"] *
    cluster_df["growth_norm"] *
    cluster_df["criticality_norm"]
)

# Geometric-mean style score keeps the ranking but avoids collapse to near-zero
cluster_df["ccs_score"] = 100.0 * np.power(cluster_df["ccs_raw_product"], 1.0 / 5.0)
cluster_df["ccs_score"] = cluster_df["ccs_score"].fillna(0.0)

# An additional readable linear blend for auditing
cluster_df["ccs_weighted_score"] = 100.0 * (
    0.25 * cluster_df["delay_norm"] +
    0.20 * cluster_df["lambda_norm"] +
    0.20 * cluster_df["severity_norm"] +
    0.15 * cluster_df["growth_norm"] +
    0.20 * cluster_df["criticality_norm"]
)

# Final ranking
rank_view = cluster_df.sort_values(
    ["ccs_score", "delay_minutes_per_vehicle", lambda_col],
    ascending=[False, False, False]
).reset_index(drop=True)
rank_view["ccs_rank"] = np.arange(1, len(rank_view) + 1)

# Risk banding
q80 = rank_view["ccs_score"].quantile(0.80) if len(rank_view) else 0
q60 = rank_view["ccs_score"].quantile(0.60) if len(rank_view) else 0
q40 = rank_view["ccs_score"].quantile(0.40) if len(rank_view) else 0

def risk_band(x):
    if x >= q80:
        return "Critical"
    if x >= q60:
        return "High"
    if x >= q40:
        return "Moderate"
    return "Watch"

rank_view["risk_band"] = rank_view["ccs_score"].apply(risk_band)

# Fix any remaining blank labels before export
rank_view["cluster_label"] = derive_cluster_label(rank_view, cluster_col)

# -------------------------
# Weekly dispatch priority table
# -------------------------
dispatch_cols = [
    "ccs_rank", cluster_col, "cluster_label", "risk_band",
    "ccs_score", "ccs_weighted_score",
    "delay_minutes_per_vehicle", lambda_col,
    "severity_sum", "severity_mean",
    "growth_pct_change", "growth_multiplier",
    "criticality_factor",
    "capacity_loss_pct", "blocking_vehicles_L",
    "road_class", "lane_count", "carriageway_width_m",
    "dominant_vehicle_type", "records_total", "unique_vehicles",
    "geometry_source",
]
dispatch_cols = [c for c in dispatch_cols if c in rank_view.columns]

weekly_dispatch = rank_view[dispatch_cols].copy()
weekly_dispatch.to_csv(OUT_DIR / "phase6_weekly_dispatch_priority_table.csv", index=False)

# -------------------------
# Hotspot CCS ranking
# -------------------------
hotspot_ranking = rank_view[[
    c for c in [
        "ccs_rank", cluster_col, "cluster_label", "risk_band",
        "ccs_score", "ccs_weighted_score", "delay_minutes_per_vehicle",
        lambda_col, "severity_sum", "growth_pct_change",
        "growth_multiplier", "criticality_factor",
        "capacity_loss_pct", "blocking_vehicles_L",
        "road_class", "lane_count", "carriageway_width_m",
        "dominant_vehicle_type", "records_total", "unique_vehicles",
        "geometry_source"
    ] if c in rank_view.columns
]].copy()

hotspot_ranking.to_csv(OUT_DIR / "phase6_hotspot_ranking_ccs.csv", index=False)

# -------------------------
# Emerging hotspot alerts
# -------------------------
alerts = rank_view.copy()
alerts["is_emerging"] = (
    (alerts["growth_pct_change"] > 0) &
    (alerts["records_total"] >= max(10, int(alerts["records_total"].median()))) &
    (alerts["growth_multiplier"] >= 1.0)
)

emerging_alerts = alerts.loc[alerts["is_emerging"]].copy()
if emerging_alerts.empty:
    emerging_alerts = alerts.sort_values(
        ["growth_pct_change", "growth_multiplier", "records_total"],
        ascending=[False, False, False]
    ).head(TOP_N).copy()
else:
    emerging_alerts = emerging_alerts.sort_values(
        ["growth_pct_change", "growth_multiplier", "records_total"],
        ascending=[False, False, False]
    ).head(TOP_N).copy()

if len(emerging_alerts):
    emerging_alerts["alert_level"] = np.where(
        emerging_alerts["growth_pct_change"] >= emerging_alerts["growth_pct_change"].quantile(0.80),
        "Emerging-Critical",
        np.where(
            emerging_alerts["growth_pct_change"] >= emerging_alerts["growth_pct_change"].quantile(0.60),
            "Emerging-High",
            "Emerging-Watch"
        )
    )
else:
    emerging_alerts["alert_level"] = []

emerging_cols = [
    "alert_level", "ccs_rank", cluster_col, "cluster_label",
    "growth_pct_change", "growth_multiplier",
    "ccs_score", "delay_minutes_per_vehicle", lambda_col,
    "severity_sum", "criticality_factor", "records_total",
    "dominant_vehicle_type", "risk_band"
]
emerging_cols = [c for c in emerging_cols if c in emerging_alerts.columns]
emerging_alerts[emerging_cols].to_csv(OUT_DIR / "phase6_emerging_hotspot_alerts.csv", index=False)

# -------------------------
# Chronic offender list
# -------------------------
if PHASE5_CHRONIC_PATH.exists():
    chronic_offenders = pd.read_csv(PHASE5_CHRONIC_PATH, low_memory=False)
else:
    if PHASE5_ENRICHED_RECORDS_PATH.exists():
        recs = pd.read_csv(PHASE5_ENRICHED_RECORDS_PATH, low_memory=False)
    elif PHASE4_MERGED_PATH.exists():
        recs = pd.read_csv(PHASE4_MERGED_PATH, low_memory=False)
    else:
        recs = None

    if recs is not None:
        rec_cluster_col = standardize_cluster_col(recs) if any(c in recs.columns for c in ["st_dbscan_cluster_id", "cluster_id", "dbscan_cluster_id"]) else None
        rec_vehicle_col = "canonical_vehicle_number" if "canonical_vehicle_number" in recs.columns else ("vehicle_number" if "vehicle_number" in recs.columns else None)

        if rec_cluster_col and rec_vehicle_col:
            chronic_offenders = (
                recs.groupby(rec_vehicle_col)
                .agg(
                    total_violations=(rec_vehicle_col, "size"),
                    unique_clusters=(rec_cluster_col, "nunique"),
                )
                .reset_index()
                .rename(columns={rec_vehicle_col: "vehicle_number"})
                .sort_values(["total_violations", "unique_clusters"], ascending=[False, False])
            )
            chronic_offenders["chronic_offender_flag"] = (chronic_offenders["total_violations"] >= 5).astype(int)
            chronic_offenders = chronic_offenders.loc[chronic_offenders["chronic_offender_flag"].eq(1)].copy()
        else:
            chronic_offenders = pd.DataFrame(columns=["vehicle_number", "total_violations", "chronic_offender_flag"])
    else:
        chronic_offenders = pd.DataFrame(columns=["vehicle_number", "total_violations", "chronic_offender_flag"])

chronic_offenders.to_csv(OUT_DIR / "phase6_chronic_offender_list.csv", index=False)

# -------------------------
# Enforcement recommendations
# -------------------------
recommendations = rank_view.copy()
recommendations["recommended_action"] = np.select(
    [
        recommendations["risk_band"].eq("Critical"),
        recommendations["risk_band"].eq("High"),
        recommendations["risk_band"].eq("Moderate"),
    ],
    [
        "Immediate patrol deployment",
        "Targeted enforcement + towing readiness",
        "Monitor and schedule peak-window checks",
    ],
    default="Routine monitoring",
)

recommendations["priority_message"] = (
    recommendations["cluster_label"].astype(str) + " | " +
    recommendations["risk_band"].astype(str) + " | CCS=" +
    recommendations["ccs_score"].round(3).astype(str)
)

recommendations_cols = [
    "ccs_rank", cluster_col, "cluster_label", "risk_band", "recommended_action",
    "delay_minutes_per_vehicle", lambda_col, "severity_sum",
    "growth_pct_change", "criticality_factor", "ccs_score",
    "dominant_vehicle_type", "road_class", "lane_count", "geometry_source"
]
recommendations_cols = [c for c in recommendations_cols if c in recommendations.columns]
recommendations[recommendations_cols].to_csv(OUT_DIR / "phase6_enforcement_recommendations.csv", index=False)

# -------------------------
# Full scored output and handoff
# -------------------------
rank_view.to_csv(OUT_DIR / "phase6_cluster_ccs_full.csv", index=False)

handoff_cols = [
    "ccs_rank", cluster_col, "cluster_label", "risk_band",
    "ccs_score", "delay_minutes_per_vehicle", lambda_col,
    "severity_sum", "growth_pct_change", "growth_multiplier",
    "criticality_factor", "capacity_loss_pct", "blocking_vehicles_L",
    "road_class", "lane_count", "carriageway_width_m",
    "dominant_vehicle_type", "records_total", "unique_vehicles",
    "geometry_source"
]
handoff_cols = [c for c in handoff_cols if c in rank_view.columns]
rank_view[handoff_cols].to_csv(OUT_DIR / "phase6_stage6_handoff.csv", index=False)

# Also useful for auditing
cluster_df.to_csv(OUT_DIR / "phase6_cluster_ccs_audit_full.csv", index=False)

# -------------------------
# Console summary
# -------------------------
print("Phase 6 complete")
print("Input source:", cluster_src)
print("Clusters scored:", len(rank_view))
print("Top CCS cluster:", rank_view.iloc[0][cluster_col] if len(rank_view) else "N/A")
print("Top CCS score:", round(float(rank_view.iloc[0]["ccs_score"]), 3) if len(rank_view) else "N/A")
print("Outputs saved to:", OUT_DIR.resolve())

print("\nTop 10 CCS clusters:")
print(
    rank_view[[
        "ccs_rank", cluster_col, "cluster_label", "risk_band",
        "ccs_score", "delay_minutes_per_vehicle", lambda_col,
        "severity_sum", "growth_pct_change", "criticality_factor"
    ]].head(10).to_string(index=False)
)

print("\nTop 10 emerging alerts:")
if len(emerging_alerts):
    print(
        emerging_alerts[[
            "alert_level", cluster_col, "cluster_label",
            "growth_pct_change", "growth_multiplier", "ccs_score", "records_total"
        ]].head(10).to_string(index=False)
    )
else:
    print("No emerging alerts found.")

# -------------------------
# Plots
# -------------------------
plt.figure(figsize=(12, 7))
top_ccs = rank_view.head(TOP_N).sort_values("ccs_score", ascending=True)
plt.barh(top_ccs["cluster_label"], top_ccs["ccs_score"])
plt.title("Top CCS Hotspots")
plt.xlabel("CCS Score")
plt.ylabel("Hotspot")
save_plot(OUT_DIR / "top_ccs_hotspots.png")

plt.figure(figsize=(12, 7))
top_delay = rank_view.head(TOP_N).sort_values("delay_minutes_per_vehicle", ascending=True)
plt.barh(top_delay["cluster_label"], top_delay["delay_minutes_per_vehicle"])
plt.title("Top Delay Hotspots")
plt.xlabel("Delay Minutes per Vehicle")
plt.ylabel("Hotspot")
save_plot(OUT_DIR / "top_delay_hotspots.png")

plt.figure(figsize=(12, 5))
plt.hist(rank_view["ccs_score"], bins=30)
plt.title("CCS Score Distribution")
plt.xlabel("CCS Score")
plt.ylabel("Clusters")
save_plot(OUT_DIR / "ccs_distribution.png")

Phase 6 complete
Input source: content\phase5_outputs_1\phase5_cluster_capacity_loss.csv
Clusters scored: 798
Top CCS cluster: 2
Top CCS score: 39.811
Outputs saved to: D:\Flipkart_Hackathon\content\phase6_outputs_1

Top 10 CCS clusters:
 ccs_rank  st_dbscan_cluster_id                                       cluster_label risk_band  ccs_score  delay_minutes_per_vehicle  lambda_hr_peak_window  severity_sum  growth_pct_change  criticality_factor
        1                     2                   JUNCTION::BTP040 - Elite Junction  Critical  39.810707               6.986447e-04              19.007547         34509          -0.141515                 1.0
        2                     3            JUNCTION::BTP051 - Safina Plaza Junction  Critical  26.687222               2.493127e-04              11.690566         15028          -0.184463                 1.0
        3                   128                                        CLUSTER::128  Critical  16.615964               0.000000e+00       

In [ ]:
# import os
# import zipfile
# from google.colab import files

# # Output zip name
# zip_filename = "/content/Traffic_Intelligence_All_Phases.zip"

# # Folders to include
# phase_folders = [
#     "/content/phase1_outputs_1",
#     "/content/phase2_outputs_1",
#     "/content/phase3_outputs_1",
#     "/content/phase4_outputs_1",
#     "/content/phase5_outputs_1",
#     "/content/phase6_outputs_1",
# ]

# # Create zip
# with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:

#     for folder in phase_folders:

#         if os.path.exists(folder):

#             for root, dirs, files_in_dir in os.walk(folder):

#                 for file in files_in_dir:

#                     file_path = os.path.join(root, file)

#                     arcname = os.path.relpath(
#                         file_path,
#                         start="/content"
#                     )

#                     zipf.write(file_path, arcname)

# print(f"ZIP created successfully: {zip_filename}")

# # Download automatically
# files.download(zip_filename)

In [ ]:
# =========================
# Phase 1 — Dataset Intelligence
# Validation Analysis
# Vehicle Behaviour Analysis
# Temporal Analysis
# Violation Analysis
# =========================

import ast
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# -------------------------
# Config
# -------------------------
INPUT_CSV = "jan to may police violation_anonymized791b166.csv"
OUT_DIR = Path("content/phase1_outputs_2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_N = 25
EPS = 1e-9

# -------------------------
# Helpers
# -------------------------
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def parse_listlike(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    s = str(value).strip()
    if not s:
        return []
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple)):
            return list(parsed)
        return [parsed]
    except Exception:
        if s.startswith("[") and s.endswith("]"):
            s = s[1:-1]
        parts = [p.strip().strip("'").strip('"') for p in s.split(",")]
        return [p for p in parts if p]

def week_start_monday(series):
    dt = pd.to_datetime(series, errors="coerce", utc=True).dt.tz_convert("Asia/Kolkata")
    week_start = dt.dt.normalize() - pd.to_timedelta(dt.dt.weekday, unit="D")
    return week_start.dt.tz_localize(None)

def safe_div(a, b):
    return a / (b + EPS)

def ensure_required_columns(df, cols):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

def save_plot(path):
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()

def severity_from_tags(tags):
    """
    Maps violation tags to a 1–5 severity score.
    Uses the highest applicable severity if multiple tags exist.
    """
    severity_map = {
        5: {
            "DOUBLE PARKING",
            "NEAR ROAD CROSSING",
            "NEAR TRAFFIC LIGHT",
            "NEAR TRAFFIC LIGHT / ZEBRA CROSSING",
            "NEAR ZEBRA CROSSING",
        },
        4: {
            "PARKING IN MAIN ROAD",
            "NEAR BUS STOP",
            "NEAR SCHOOL",
            "NEAR HOSPITAL",
            "OPPOSITE ANOTHER VEHICLE",
        },
        3: {
            "PARKING ON FOOTPATH",
        },
        2: {
            "WRONG PARKING",
            "PARKING OTHER THAN BUS STOP",
        },
        1: {
            "NO PARKING",
        },
    }

    if not tags:
        return 1

    normalized = [clean_text(t).upper() for t in tags]
    score = 1
    for s, vocab in severity_map.items():
        if any(tag in vocab for tag in normalized):
            score = max(score, s)
    return score

def dominant_tag(tags):
    if not tags:
        return ""
    return clean_text(tags[0]).upper()

# -------------------------
# Load
# -------------------------
df = pd.read_csv(INPUT_CSV, low_memory=False)

required_cols = [
    "id", "latitude", "longitude", "vehicle_number", "vehicle_type",
    "violation_type", "created_datetime", "police_station",
    "junction_name", "validation_status"
]
ensure_required_columns(df, required_cols)

for col in ["validation_status", "police_station", "junction_name", "vehicle_number", "vehicle_type"]:
    df[col] = df[col].astype("string")

# -------------------------
# Core parsing
# -------------------------
df["created_datetime_parsed"] = pd.to_datetime(df["created_datetime"], errors="coerce", utc=True)
df["created_datetime_ist"] = df["created_datetime_parsed"].dt.tz_convert("Asia/Kolkata")
df["week_start"] = week_start_monday(df["created_datetime"])
df["hour_ist"] = df["created_datetime_ist"].dt.hour
df["day_of_week"] = df["created_datetime_ist"].dt.day_name()
df["month"] = df["created_datetime_ist"].dt.month
df["is_weekend"] = df["day_of_week"].isin(["Saturday", "Sunday"]).astype(int)

df["violation_tags"] = df["violation_type"].apply(parse_listlike)
df["tag_count"] = df["violation_tags"].apply(len)
df["severity_score"] = df["violation_tags"].apply(severity_from_tags)
df["dominant_violation_tag"] = df["violation_tags"].apply(dominant_tag)

junction_clean = df["junction_name"].fillna("").astype(str).str.strip()
has_junction = junction_clean.ne("") & junction_clean.str.upper().ne("NO JUNCTION")
police_clean = df["police_station"].fillna("UNKNOWN").astype(str).str.strip()
police_clean = police_clean.where(police_clean.ne(""), "UNKNOWN")

df["hotspot_unit"] = np.where(
    has_junction,
    "JUNCTION::" + junction_clean,
    "POLICE_STATION::" + police_clean
)

validation_clean = df["validation_status"].fillna("").astype(str).str.lower()
df["validation_status_clean"] = validation_clean

# A simple numeric flag useful for later phases
status_score_map = {
    "approved": 1.0,
    "rejected": 0.0,
    "processing": 0.5,
    "duplicate": 0.25,
    "created1": 0.5,
}
df["validation_score"] = df["validation_status_clean"].map(status_score_map).fillna(0.5)

# Keep only rows with valid time and coordinates for analysis tables that need them
df_valid = df.dropna(subset=["created_datetime_parsed", "latitude", "longitude"]).copy()

# -------------------------
# 1) Validation Analysis
# -------------------------
validation_summary = (
    df["validation_status_clean"]
    .replace("", "missing")
    .value_counts(dropna=False)
    .rename_axis("validation_status")
    .reset_index(name="count")
)
validation_summary["percent"] = 100 * validation_summary["count"] / len(df)

approval_rate = safe_div(
    int((df["validation_status_clean"] == "approved").sum()),
    len(df)
)

validation_by_vehicle_type = (
    df.groupby("vehicle_type", dropna=False)
    .agg(
        total_records=("id", "size"),
        approved_records=("validation_status_clean", lambda s: (s == "approved").sum()),
        rejected_records=("validation_status_clean", lambda s: (s == "rejected").sum()),
        approved_rate=("validation_status_clean", lambda s: (s == "approved").mean()),
    )
    .reset_index()
    .sort_values("total_records", ascending=False)
)

validation_by_police_station = (
    df.groupby("police_station", dropna=False)
    .agg(
        total_records=("id", "size"),
        approved_rate=("validation_status_clean", lambda s: (s == "approved").mean()),
        rejected_rate=("validation_status_clean", lambda s: (s == "rejected").mean()),
    )
    .reset_index()
    .sort_values(["total_records", "approved_rate"], ascending=[False, False])
)

validation_by_hotspot = (
    df.groupby("hotspot_unit", dropna=False)
    .agg(
        total_records=("id", "size"),
        approved_rate=("validation_status_clean", lambda s: (s == "approved").mean()),
        rejected_rate=("validation_status_clean", lambda s: (s == "rejected").mean()),
        processed_rate=("validation_status_clean", lambda s: (s == "processing").mean()),
    )
    .reset_index()
    .sort_values("total_records", ascending=False)
)

# Validation uncertainty score: high when approval rate is low or mixed
validation_by_hotspot["validation_uncertainty"] = 1.0 - validation_by_hotspot["approved_rate"]
validation_by_vehicle_type["validation_uncertainty"] = 1.0 - validation_by_vehicle_type["approved_rate"]

# -------------------------
# 2) Vehicle Behaviour Analysis
# -------------------------
vehicle_summary = (
    df.groupby("vehicle_number", dropna=False)
    .agg(
        total_violations=("id", "size"),
        unique_hotspots=("hotspot_unit", "nunique"),
        unique_junctions=("junction_name", lambda s: s.fillna("").astype(str).replace("No Junction", "").nunique()),
        first_seen=("created_datetime_parsed", "min"),
        last_seen=("created_datetime_parsed", "max"),
        vehicle_type_mode=("vehicle_type", lambda s: s.mode().iloc[0] if not s.mode().empty else ""),
        approved_violations=("validation_status_clean", lambda s: (s == "approved").sum()),
    )
    .reset_index()
)

vehicle_summary["active_days"] = (
    df_valid.groupby("vehicle_number")["created_datetime_ist"]
    .apply(lambda s: s.dt.date.nunique())
    .reindex(vehicle_summary["vehicle_number"])
    .values
)

vehicle_summary["repeat_violation_flag"] = (vehicle_summary["total_violations"] >= 2).astype(int)
vehicle_summary["chronic_offender_flag"] = (vehicle_summary["total_violations"] >= 5).astype(int)

vehicle_summary["approval_rate"] = safe_div(
    vehicle_summary["approved_violations"],
    vehicle_summary["total_violations"]
)

vehicle_summary = vehicle_summary.sort_values(
    ["total_violations", "approved_violations", "unique_hotspots"],
    ascending=[False, False, False]
).reset_index(drop=True)

top_vehicle_offenders = vehicle_summary.head(TOP_N).copy()
chronic_offenders = vehicle_summary[vehicle_summary["chronic_offender_flag"] == 1].copy()

# Per vehicle-type behaviour
vehicle_type_summary = (
    df.groupby("vehicle_type", dropna=False)
    .agg(
        records=("id", "size"),
        unique_vehicles=("vehicle_number", "nunique"),
        mean_severity=("severity_score", "mean"),
        median_severity=("severity_score", "median"),
        approval_rate=("validation_status_clean", lambda s: (s == "approved").mean()),
    )
    .reset_index()
    .sort_values("records", ascending=False)
)

# -------------------------
# 3) Temporal Analysis
# -------------------------
hourly_distribution = (
    df_valid.groupby("hour_ist")
    .size()
    .reset_index(name="count")
    .sort_values("hour_ist")
)
daily_distribution = (
    df_valid.groupby("day_of_week")
    .size()
    .reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])
    .reset_index(name="count")
)
weekly_distribution = (
    df_valid.groupby("week_start")
    .size()
    .reset_index(name="count")
    .sort_values("week_start")
)
monthly_distribution = (
    df_valid.groupby("month")
    .size()
    .reset_index(name="count")
    .sort_values("month")
)

temporal_by_hour_vehicle_type = (
    df_valid.groupby(["hour_ist", "vehicle_type"])
    .size()
    .reset_index(name="count")
)

temporal_by_day_validation = (
    df_valid.groupby(["day_of_week", "validation_status_clean"])
    .size()
    .reset_index(name="count")
)

peak_hour = int(hourly_distribution.loc[hourly_distribution["count"].idxmax(), "hour_ist"]) if len(hourly_distribution) else None
peak_day = daily_distribution.loc[daily_distribution["count"].idxmax(), "day_of_week"] if len(daily_distribution) else None

# Trend by week: useful for later emerging hotspot stage
weekly_trend_by_hotspot = (
    df_valid.groupby(["hotspot_unit", "week_start"])
    .size()
    .reset_index(name="weekly_count")
    .sort_values(["hotspot_unit", "week_start"])
)

# -------------------------
# 4) Violation Analysis
# -------------------------
# Tag-level analysis
tag_rows = []
for _, row in df.iterrows():
    tags = row["violation_tags"]
    if not tags:
        tag_rows.append({"violation_tag": "UNKNOWN", "id": row["id"], "vehicle_number": row["vehicle_number"], "hotspot_unit": row["hotspot_unit"]})
    else:
        for t in tags:
            tag_rows.append({
                "violation_tag": clean_text(t).upper(),
                "id": row["id"],
                "vehicle_number": row["vehicle_number"],
                "hotspot_unit": row["hotspot_unit"],
            })

tag_df = pd.DataFrame(tag_rows)

violation_tag_summary = (
    tag_df.groupby("violation_tag")
    .agg(
        tag_frequency=("id", "size"),
        unique_vehicles=("vehicle_number", "nunique"),
        unique_hotspots=("hotspot_unit", "nunique"),
    )
    .reset_index()
    .sort_values("tag_frequency", ascending=False)
)

severity_distribution = (
    df.groupby("severity_score")
    .size()
    .reset_index(name="count")
    .sort_values("severity_score")
)

violation_by_vehicle_type = (
    df.groupby("vehicle_type")
    .agg(
        records=("id", "size"),
        mean_severity=("severity_score", "mean"),
        approval_rate=("validation_status_clean", lambda s: (s == "approved").mean()),
    )
    .reset_index()
    .sort_values("records", ascending=False)
)

violation_by_hotspot = (
    df.groupby("hotspot_unit")
    .agg(
        records=("id", "size"),
        mean_severity=("severity_score", "mean"),
        approval_rate=("validation_status_clean", lambda s: (s == "approved").mean()),
        unique_vehicles=("vehicle_number", "nunique"),
    )
    .reset_index()
    .sort_values(["records", "mean_severity"], ascending=[False, False])
)

# Tag co-occurrence patterns
combo_counts = (
    df["violation_tags"]
    .apply(lambda x: tuple(sorted([clean_text(t).upper() for t in x])) if x else ("UNKNOWN",))
    .value_counts()
    .reset_index()
)
combo_counts.columns = ["violation_combo", "frequency"]
combo_counts["violation_combo"] = combo_counts["violation_combo"].apply(lambda x: " | ".join(x))

# -------------------------
# Consolidated Phase 1 feature table
# -------------------------
phase1_features = df.copy()

# Useful derived columns for later stages
phase1_features["validation_is_approved"] = (phase1_features["validation_status_clean"] == "approved").astype(int)
phase1_features["validation_is_rejected"] = (phase1_features["validation_status_clean"] == "rejected").astype(int)
phase1_features["vehicle_total_violations"] = phase1_features.groupby("vehicle_number")["id"].transform("count")
phase1_features["vehicle_chronic_flag"] = (phase1_features["vehicle_total_violations"] >= 5).astype(int)
phase1_features["vehicle_repeat_flag"] = (phase1_features["vehicle_total_violations"] >= 2).astype(int)

phase1_features["hotspot_total_records"] = phase1_features.groupby("hotspot_unit")["id"].transform("count")
phase1_features["hotspot_approval_rate"] = phase1_features.groupby("hotspot_unit")["validation_is_approved"].transform("mean")
phase1_features["hotspot_uncertainty"] = 1.0 - phase1_features["hotspot_approval_rate"]

phase1_features["vehicle_type_total_records"] = phase1_features.groupby("vehicle_type")["id"].transform("count")
phase1_features["vehicle_type_mean_severity"] = phase1_features.groupby("vehicle_type")["severity_score"].transform("mean")

phase1_features["tag_frequency_record"] = phase1_features["dominant_violation_tag"].map(
    violation_tag_summary.set_index("violation_tag")["tag_frequency"]
).fillna(0).astype(int)

# -------------------------
# Save outputs
# -------------------------
validation_summary.to_csv(OUT_DIR / "validation_summary.csv", index=False)
validation_by_vehicle_type.to_csv(OUT_DIR / "validation_by_vehicle_type.csv", index=False)
validation_by_police_station.to_csv(OUT_DIR / "validation_by_police_station.csv", index=False)
validation_by_hotspot.to_csv(OUT_DIR / "validation_by_hotspot.csv", index=False)

vehicle_summary.to_csv(OUT_DIR / "vehicle_summary.csv", index=False)
top_vehicle_offenders.to_csv(OUT_DIR / "top_vehicle_offenders.csv", index=False)
chronic_offenders.to_csv(OUT_DIR / "chronic_offenders.csv", index=False)
vehicle_type_summary.to_csv(OUT_DIR / "vehicle_type_summary.csv", index=False)

hourly_distribution.to_csv(OUT_DIR / "hourly_distribution.csv", index=False)
daily_distribution.to_csv(OUT_DIR / "daily_distribution.csv", index=False)
weekly_distribution.to_csv(OUT_DIR / "weekly_distribution.csv", index=False)
monthly_distribution.to_csv(OUT_DIR / "monthly_distribution.csv", index=False)
temporal_by_hour_vehicle_type.to_csv(OUT_DIR / "temporal_by_hour_vehicle_type.csv", index=False)
temporal_by_day_validation.to_csv(OUT_DIR / "temporal_by_day_validation.csv", index=False)
weekly_trend_by_hotspot.to_csv(OUT_DIR / "weekly_trend_by_hotspot.csv", index=False)

violation_tag_summary.to_csv(OUT_DIR / "violation_tag_summary.csv", index=False)
severity_distribution.to_csv(OUT_DIR / "severity_distribution.csv", index=False)
violation_by_vehicle_type.to_csv(OUT_DIR / "violation_by_vehicle_type.csv", index=False)
violation_by_hotspot.to_csv(OUT_DIR / "violation_by_hotspot.csv", index=False)
combo_counts.to_csv(OUT_DIR / "violation_combo_summary.csv", index=False)

phase1_features.to_csv(OUT_DIR / "phase1_enriched_dataset.csv", index=False)

# -------------------------
# Print summary
# -------------------------
print("Phase 1 complete")
print("Total rows:", len(df))
print("Approved rows:", int((df["validation_status_clean"] == "approved").sum()))
print("Approval rate:", round(approval_rate * 100, 2), "%")
print("Unique vehicles:", df["vehicle_number"].nunique())
print("Unique hotspots:", df["hotspot_unit"].nunique())
print("Peak hour:", peak_hour)
print("Peak day:", peak_day)
print("Chronic offenders (>=5):", len(chronic_offenders))
print("Outputs saved to:", OUT_DIR.resolve())

# -------------------------
# Plots
# -------------------------
plt.figure(figsize=(10, 5))
plt.bar(validation_summary["validation_status"].astype(str), validation_summary["count"])
plt.title("Validation Status Distribution")
plt.xlabel("Validation Status")
plt.ylabel("Count")
save_plot(OUT_DIR / "validation_status_distribution.png")

plt.figure(figsize=(12, 5))
plt.bar(hourly_distribution["hour_ist"].astype(int), hourly_distribution["count"])
plt.title("Hourly Violation Distribution (IST)")
plt.xlabel("Hour")
plt.ylabel("Count")
save_plot(OUT_DIR / "hourly_distribution.png")

plt.figure(figsize=(12, 5))
plt.bar(daily_distribution["day_of_week"].astype(str), daily_distribution["count"])
plt.title("Day-of-Week Violation Distribution")
plt.xlabel("Day")
plt.ylabel("Count")
plt.xticks(rotation=30)
save_plot(OUT_DIR / "daily_distribution.png")

plt.figure(figsize=(12, 6))
top_tags = violation_tag_summary.head(TOP_N)
plt.barh(top_tags["violation_tag"][::-1], top_tags["tag_frequency"][::-1])
plt.title("Top Violation Tags")
plt.xlabel("Frequency")
plt.ylabel("Violation Tag")
save_plot(OUT_DIR / "top_violation_tags.png")

plt.figure(figsize=(12, 6))
top_vehicles = top_vehicle_offenders.head(TOP_N).sort_values("total_violations")
plt.barh(top_vehicles["vehicle_number"], top_vehicles["total_violations"])
plt.title("Top Vehicle Offenders")
plt.xlabel("Total Violations")
plt.ylabel("Vehicle Number")
save_plot(OUT_DIR / "top_vehicle_offenders.png")

Phase 1 complete
Total rows: 298450
Approved rows: 115400
Approval rate: 38.67 %
Unique vehicles: 231890
Unique hotspots: 222
Peak hour: 10
Peak day: Sunday
Chronic offenders (>=5): 3489
Outputs saved to: D:\Flipkart_Hackathon\content\phase1_outputs_2


In [ ]:
import ast
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

INPUT_CSV = "jan to may police violation_anonymized791b166.csv"
PHASE1_PATH = Path("content/phase1_outputs_2/phase1_enriched_dataset.csv")
OUT_DIR = Path("content/phase2_outputs_2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EPS = 1e-9
TOP_N = 25

SEVERITY_RULES = {
    5: [
        "DOUBLE PARKING",
        "NEAR ROAD CROSSING",
        "NEAR TRAFFIC LIGHT",
        "NEAR ZEBRA CROSSING",
        "NEAR TRAFFIC LIGHT / ZEBRA CROSSING",
        "NEAR TRAFFIC LIGHT/ZEBRA CROSSING",
    ],
    4: [
        "PARKING IN MAIN ROAD",
        "NEAR BUS STOP",
        "NEAR SCHOOL",
        "NEAR HOSPITAL",
        "OPPOSITE ANOTHER VEHICLE",
    ],
    3: [
        "PARKING ON FOOTPATH",
    ],
    2: [
        "WRONG PARKING",
        "PARKING OTHER THAN BUS STOP",
    ],
    1: [
        "NO PARKING",
        "NO PARKING (GENERIC)",
    ],
}

def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def parse_listlike(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    s = str(value).strip()
    if not s:
        return []
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple)):
            return list(parsed)
        return [parsed]
    except Exception:
        if s.startswith("[") and s.endswith("]"):
            s = s[1:-1]
        parts = [p.strip().strip("'").strip('"') for p in s.split(",")]
        return [p for p in parts if p]

def normalize_tag(tag):
    s = clean_text(tag).upper()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\s*/\s*", " / ", s)
    s = s.replace("&", "AND")
    return s.strip()

def make_hotspot_unit(row):
    junction = clean_text(row.get("junction_name", ""))
    if junction and junction.upper() != "NO JUNCTION":
        return f"JUNCTION::{junction}"
    station = clean_text(row.get("police_station", "UNKNOWN"))
    if not station:
        station = "UNKNOWN"
    return f"POLICE_STATION::{station}"

def severity_for_tags(tags):
    if not tags:
        return 1, [], "NO PARKING"

    normalized = [normalize_tag(t) for t in tags]
    hit_tags = []
    best = 1

    for tag in normalized:
        tag_best = 1
        matched = False
        for sev in sorted(SEVERITY_RULES.keys(), reverse=True):
            for pattern in SEVERITY_RULES[sev]:
                p = normalize_tag(pattern)
                if p == tag or p in tag:
                    tag_best = sev
                    matched = True
                    break
            if matched:
                break
        best = max(best, tag_best)
        if matched:
            hit_tags.append(tag)

    dominant = hit_tags[0] if hit_tags else normalized[0]
    return best, hit_tags, dominant

def safe_ratio(num, den):
    return num / (den + EPS)

def save_plot(path):
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()

def main():
    source_path = PHASE1_PATH if PHASE1_PATH.exists() else Path(INPUT_CSV)
    df = pd.read_csv(source_path, low_memory=False)

    if "violation_type" not in df.columns:
        raise ValueError("Missing required column: violation_type")

    if "hotspot_unit" not in df.columns:
        if {"junction_name", "police_station"}.issubset(df.columns):
            df["hotspot_unit"] = df.apply(make_hotspot_unit, axis=1)
        else:
            df["hotspot_unit"] = "UNKNOWN"

    if "validation_status" in df.columns:
        df["validation_status_clean"] = df["validation_status"].fillna("").astype(str).str.lower()
    else:
        df["validation_status_clean"] = "approved"

    df["violation_tags"] = df["violation_type"].apply(parse_listlike)
    df["violation_tag_count"] = df["violation_tags"].apply(len)

    sev_res = df["violation_tags"].apply(severity_for_tags)
    df["severity_score"] = sev_res.apply(lambda x: int(x[0]))
    df["matched_severity_tags"] = sev_res.apply(lambda x: x[1])
    df["dominant_violation_tag"] = sev_res.apply(lambda x: x[2])
    df["severity_normalized"] = df["severity_score"] / 5.0

    df["severity_is_very_high"] = (df["severity_score"] == 5).astype(int)
    df["severity_is_high"] = (df["severity_score"] == 4).astype(int)
    df["severity_is_moderate"] = (df["severity_score"] == 3).astype(int)
    df["severity_is_low"] = (df["severity_score"] == 2).astype(int)
    df["severity_is_baseline"] = (df["severity_score"] == 1).astype(int)

    approved_mask = df["validation_status_clean"].eq("approved")
    approved = df.loc[approved_mask].copy()

    severity_record_cols = [
        c for c in [
            "id", "created_datetime", "created_datetime_parsed", "created_datetime_ist",
            "latitude", "longitude", "vehicle_number", "vehicle_type",
            "police_station", "junction_name", "hotspot_unit",
            "validation_status", "validation_status_clean",
            "violation_type", "violation_tags", "violation_tag_count",
            "severity_score", "severity_normalized", "matched_severity_tags",
            "dominant_violation_tag"
        ] if c in df.columns
    ]

    hotspot_summary = (
        approved.groupby("hotspot_unit", dropna=False)
        .agg(
            approved_records=("severity_score", "size"),
            mean_severity=("severity_score", "mean"),
            median_severity=("severity_score", "median"),
            severity_sum=("severity_score", "sum"),
            very_high_count=("severity_is_very_high", "sum"),
            high_count=("severity_is_high", "sum"),
            moderate_count=("severity_is_moderate", "sum"),
            low_count=("severity_is_low", "sum"),
            baseline_count=("severity_is_baseline", "sum"),
            unique_vehicles=("vehicle_number", "nunique") if "vehicle_number" in approved.columns else ("severity_score", "size"),
        )
        .reset_index()
    )

    hotspot_summary["severity_share"] = safe_ratio(hotspot_summary["severity_sum"], hotspot_summary["severity_sum"].sum())
    hotspot_summary["severity_priority_score"] = (
        0.50 * (hotspot_summary["severity_sum"] / (hotspot_summary["severity_sum"].max() + EPS)) +
        0.30 * (hotspot_summary["mean_severity"] / 5.0) +
        0.20 * safe_ratio(hotspot_summary["approved_records"], hotspot_summary["approved_records"].max())
    ) * 100.0

    hotspot_summary = hotspot_summary.sort_values(
        ["severity_priority_score", "severity_sum", "approved_records"],
        ascending=[False, False, False],
    ).reset_index(drop=True)
    hotspot_summary["severity_rank"] = np.arange(1, len(hotspot_summary) + 1)

    vehicle_type_summary = (
        approved.groupby("vehicle_type", dropna=False)
        .agg(
            approved_records=("severity_score", "size"),
            mean_severity=("severity_score", "mean"),
            median_severity=("severity_score", "median"),
            severity_sum=("severity_score", "sum"),
            unique_vehicles=("vehicle_number", "nunique") if "vehicle_number" in approved.columns else ("severity_score", "size"),
        )
        .reset_index()
        .sort_values(["severity_sum", "approved_records"], ascending=[False, False])
    )

    tag_rows = []
    for _, row in approved.iterrows():
        tags = row["violation_tags"]
        if not tags:
            tag_rows.append({"violation_tag": "NO TAG", "severity_score": row["severity_score"], "hotspot_unit": row["hotspot_unit"]})
        else:
            for t in tags:
                tag_rows.append({
                    "violation_tag": normalize_tag(t),
                    "severity_score": row["severity_score"],
                    "hotspot_unit": row["hotspot_unit"],
                })

    tag_df = pd.DataFrame(tag_rows)
    if tag_df.empty:
        tag_summary = pd.DataFrame(columns=["violation_tag", "frequency", "avg_record_severity", "hotspot_count"])
    else:
        tag_summary = (
            tag_df.groupby("violation_tag", dropna=False)
            .agg(
                frequency=("violation_tag", "size"),
                avg_record_severity=("severity_score", "mean"),
                hotspot_count=("hotspot_unit", "nunique"),
            )
            .reset_index()
            .sort_values(["frequency", "avg_record_severity"], ascending=[False, False])
        )

    validation_severity = (
        df.groupby("validation_status_clean", dropna=False)
        .agg(
            records=("severity_score", "size"),
            mean_severity=("severity_score", "mean"),
            severity_sum=("severity_score", "sum"),
        )
        .reset_index()
        .sort_values("records", ascending=False)
    )

    df[severity_record_cols].to_csv(OUT_DIR / "phase2_enriched_dataset.csv", index=False)
    approved[severity_record_cols].to_csv(OUT_DIR / "phase2_approved_severity_dataset.csv", index=False)
    hotspot_summary.to_csv(OUT_DIR / "phase2_hotspot_severity_scores.csv", index=False)
    vehicle_type_summary.to_csv(OUT_DIR / "phase2_vehicle_type_severity.csv", index=False)
    tag_summary.to_csv(OUT_DIR / "phase2_violation_tag_severity.csv", index=False)
    validation_severity.to_csv(OUT_DIR / "phase2_validation_severity.csv", index=False)

    print("Phase 2 complete")
    print("Input source:", source_path)
    print("Total rows:", len(df))
    print("Approved rows:", len(approved))
    print("Hotspots scored:", len(hotspot_summary))
    print("Mean severity (approved):", round(float(approved["severity_score"].mean()) if len(approved) else 0.0, 4))
    print("Top hotspot:", hotspot_summary.iloc[0]["hotspot_unit"] if len(hotspot_summary) else "N/A")
    print("Outputs saved to:", OUT_DIR.resolve())

    if len(hotspot_summary):
        plt.figure(figsize=(12, 6))
        top_hotspots = hotspot_summary.head(TOP_N).sort_values("severity_priority_score", ascending=True)
        plt.barh(top_hotspots["hotspot_unit"], top_hotspots["severity_priority_score"])
        plt.title("Top Hotspots by Severity Priority Score")
        plt.xlabel("Severity Priority Score")
        plt.ylabel("Hotspot")
        save_plot(OUT_DIR / "top_hotspots_by_severity_priority.png")

    if len(tag_summary):
        plt.figure(figsize=(12, 6))
        top_tags = tag_summary.head(TOP_N).sort_values("frequency", ascending=True)
        plt.barh(top_tags["violation_tag"], top_tags["frequency"])
        plt.title("Top Violation Tags (Approved Records)")
        plt.xlabel("Frequency")
        plt.ylabel("Violation Tag")
        save_plot(OUT_DIR / "top_violation_tags_phase2.png")

    if len(validation_severity):
        plt.figure(figsize=(10, 5))
        plt.bar(validation_severity["validation_status_clean"].astype(str), validation_severity["records"])
        plt.title("Records by Validation Status")
        plt.xlabel("Validation Status")
        plt.ylabel("Records")
        plt.xticks(rotation=30)
        save_plot(OUT_DIR / "validation_status_phase2.png")

if __name__ == "__main__":
    main()

Phase 2 complete
Input source: content\phase1_outputs_2\phase1_enriched_dataset.csv
Total rows: 298450
Approved rows: 115400
Hotspots scored: 220
Mean severity (approved): 1.5732
Top hotspot: POLICE_STATION::HAL Old Airport
Outputs saved to: D:\Flipkart_Hackathon\content\phase2_outputs_2


In [ ]:
# Stage 3 — ST-DBSCAN Spatial + Temporal Clustering
# Input : approved records with latitude, longitude, created_datetime, violation_type, vehicle_number
# Output: clustered records, cluster summary, noise points, weekly cluster counts

import ast
import warnings
from collections import deque
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

warnings.filterwarnings("ignore")

# =========================
# Config
# =========================
RAW_CSV = "jan to may police violation_anonymized791b166.csv"
PHASE2_APPROVED_PATH = Path("content/phase2_outputs_2/phase2_approved_severity_dataset.csv")

OUT_DIR = Path("content/phase3_outputs_2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EPS_SPATIAL_METERS = 150.0
EPS_TEMPORAL_DAYS = 3.0
MIN_PTS = 15
EARTH_RADIUS_M = 6371000.0

# =========================
# Helpers
# =========================
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def parse_listlike(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    s = str(value).strip()
    if not s:
        return []
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple)):
            return list(parsed)
        return [parsed]
    except Exception:
        if s.startswith("[") and s.endswith("]"):
            s = s[1:-1]
        parts = [p.strip().strip("'").strip('"') for p in s.split(",")]
        return [p for p in parts if p]

def parse_created_datetime(df: pd.DataFrame) -> pd.Series:
    if "created_datetime_ist" in df.columns:
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce", utc=True)
        if ts.notna().any():
            return ts.dt.tz_convert("Asia/Kolkata")
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce")
        return ts.dt.tz_localize("Asia/Kolkata", nonexistent="NaT", ambiguous="NaT")

    if "created_datetime_parsed" in df.columns:
        ts = pd.to_datetime(df["created_datetime_parsed"], errors="coerce", utc=True)
        return ts.dt.tz_convert("Asia/Kolkata")

    if "created_datetime" not in df.columns:
        raise ValueError("Missing created_datetime / created_datetime_ist column.")

    ts = pd.to_datetime(df["created_datetime"], errors="coerce", utc=True)
    return ts.dt.tz_convert("Asia/Kolkata")

def ensure_hotspot_unit(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "hotspot_unit" in df.columns:
        df["hotspot_unit"] = df["hotspot_unit"].fillna("").astype(str).str.strip()
        df.loc[df["hotspot_unit"].eq(""), "hotspot_unit"] = np.nan
        return df

    if {"junction_name", "police_station"}.issubset(df.columns):
        def make_hotspot_unit(row):
            junction = clean_text(row.get("junction_name", ""))
            if junction and junction.upper() != "NO JUNCTION":
                return f"JUNCTION::{junction}"
            station = clean_text(row.get("police_station", "UNKNOWN"))
            if not station:
                station = "UNKNOWN"
            return f"POLICE_STATION::{station}"
        df["hotspot_unit"] = df.apply(make_hotspot_unit, axis=1)
        return df

    df["hotspot_unit"] = "UNKNOWN"
    return df

def ensure_severity(df: pd.DataFrame) -> pd.DataFrame:
    if "severity_score" in df.columns:
        df["severity_score"] = pd.to_numeric(df["severity_score"], errors="coerce").fillna(1).astype(int)
        return df

    severity_map = {
        5: {
            "DOUBLE PARKING",
            "NEAR ROAD CROSSING",
            "NEAR TRAFFIC LIGHT",
            "NEAR ZEBRA CROSSING",
            "NEAR TRAFFIC LIGHT / ZEBRA CROSSING",
            "NEAR TRAFFIC LIGHT/ZEBRA CROSSING",
        },
        4: {
            "PARKING IN MAIN ROAD",
            "NEAR BUS STOP",
            "NEAR SCHOOL",
            "NEAR HOSPITAL",
            "OPPOSITE ANOTHER VEHICLE",
        },
        3: {"PARKING ON FOOTPATH"},
        2: {"WRONG PARKING", "PARKING OTHER THAN BUS STOP"},
        1: {"NO PARKING", "NO PARKING (GENERIC)"},
    }

    def severity_from_tags(tags):
        if not tags:
            return 1
        normalized = [clean_text(t).upper().replace("&", "AND").strip() for t in tags]
        score = 1
        for sev in sorted(severity_map.keys(), reverse=True):
            vocab = severity_map[sev]
            if any(any(v == tag or v in tag for v in vocab) for tag in normalized):
                score = sev
                break
        return score

    if "violation_type" in df.columns:
        df["violation_tags"] = df["violation_type"].apply(parse_listlike)
        df["severity_score"] = df["violation_tags"].apply(severity_from_tags)
    else:
        df["severity_score"] = 1

    return df

def project_latlon_to_xy(lat, lon):
    """
    Project lat/lon to a local meter-based plane using an equirectangular approximation.
    Works well for city-scale clustering.
    """
    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)

    lat0 = np.deg2rad(np.nanmean(lat))
    lon0 = np.deg2rad(np.nanmean(lon))

    lat_rad = np.deg2rad(lat)
    lon_rad = np.deg2rad(lon)

    x = EARTH_RADIUS_M * (lon_rad - lon0) * np.cos(lat0)
    y = EARTH_RADIUS_M * (lat_rad - lat0)
    return np.c_[x, y]

def st_dbscan(coords_xy, time_days, eps_spatial_m, eps_temporal_days, min_pts):
    """
    ST-DBSCAN using:
    - spatial radius search on projected XY coordinates (meters)
    - temporal proximity filter in days
    - DBSCAN-style BFS expansion
    """
    n = len(coords_xy)
    tree = BallTree(coords_xy, metric="euclidean")

    labels = np.full(n, -99, dtype=int)  # -99 = unassigned, -1 = noise
    visited = np.zeros(n, dtype=bool)
    core_mask = np.zeros(n, dtype=bool)
    neighbor_cache = {}

    def region_query(i):
        if i in neighbor_cache:
            return neighbor_cache[i]

        idx = tree.query_radius(coords_xy[i:i + 1], r=eps_spatial_m, return_distance=False)[0]
        if idx.size == 0:
            neighbor_cache[i] = idx
            return idx

        temporal_ok = np.abs(time_days[idx] - time_days[i]) <= eps_temporal_days
        neigh = idx[temporal_ok]
        neighbor_cache[i] = neigh
        return neigh

    cluster_id = 0

    for i in range(n):
        if visited[i]:
            continue

        visited[i] = True
        neighbors = region_query(i)

        if len(neighbors) < min_pts:
            labels[i] = -1
            continue

        cluster_id += 1
        labels[i] = cluster_id
        core_mask[i] = True

        seed_queue = deque()
        queued = set()

        for j in neighbors:
            if j != i and j not in queued:
                seed_queue.append(j)
                queued.add(j)

        while seed_queue:
            j = seed_queue.popleft()

            if not visited[j]:
                visited[j] = True
                j_neighbors = region_query(j)
                if len(j_neighbors) >= min_pts:
                    core_mask[j] = True
                    for k in j_neighbors:
                        if k not in queued:
                            seed_queue.append(k)
                            queued.add(k)

            if labels[j] in (-99, -1):
                labels[j] = cluster_id

    return labels, core_mask

def cluster_label_for_group(g: pd.DataFrame) -> str:
    if "junction_name" in g.columns:
        junctions = g["junction_name"].fillna("").astype(str)
        junctions = junctions[junctions.str.strip().ne("")]
        junctions = junctions[junctions.str.upper().ne("NO JUNCTION")]
        if len(junctions):
            mode = junctions.mode()
            if not mode.empty:
                return mode.iloc[0]

    if "hotspot_unit" in g.columns:
        units = g["hotspot_unit"].fillna("").astype(str)
        units = units[units.str.strip().ne("")]
        if len(units):
            mode = units.mode()
            if not mode.empty:
                return mode.iloc[0]

    if "police_station" in g.columns:
        stations = g["police_station"].fillna("").astype(str)
        stations = stations[stations.str.strip().ne("")]
        if len(stations):
            mode = stations.mode()
            if not mode.empty:
                return f"POLICE_STATION::{mode.iloc[0]}"

    return f"Cluster {g.name}"

def week_start_monday(series):
    dt = pd.to_datetime(series, errors="coerce", utc=True).dt.tz_convert("Asia/Kolkata")
    week_start = dt.dt.normalize() - pd.to_timedelta(dt.dt.weekday, unit="D")
    return week_start.dt.tz_localize(None)

# =========================
# Load
# =========================
def load_input():
    if PHASE2_APPROVED_PATH.exists():
        return pd.read_csv(PHASE2_APPROVED_PATH, low_memory=False), PHASE2_APPROVED_PATH
    return pd.read_csv(RAW_CSV, low_memory=False), RAW_CSV

def main():
    df, source = load_input()

    required_cols = {"latitude", "longitude", "created_datetime"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    # Stage 1 carry-through: approved records only
    if "validation_status" in df.columns:
        df["validation_status_clean"] = df["validation_status"].fillna("").astype(str).str.lower()
        df = df[df["validation_status_clean"].eq("approved")].copy()

    df = df.dropna(subset=["latitude", "longitude"]).copy()
    df["created_datetime_ist"] = parse_created_datetime(df)
    df = df.dropna(subset=["created_datetime_ist"]).copy()

    df = ensure_hotspot_unit(df)
    df = ensure_severity(df)

    df["created_datetime_ist_naive"] = df["created_datetime_ist"].dt.tz_localize(None)
    df["cluster_week_start"] = week_start_monday(df["created_datetime_ist"])

    # Project coordinates to meters for spatial radius search
    coords_xy = project_latlon_to_xy(df["latitude"].values, df["longitude"].values)

    # Convert timestamps to days since epoch for temporal radius search
    time_days = (
        df["created_datetime_ist_naive"].astype("int64").values.astype(np.float64)
        / 86_400_000_000_000.0
    )

    print("Input source:", source)
    print("Approved rows used:", len(df))
    print("Running ST-DBSCAN with:")
    print(f"  spatial epsilon = {EPS_SPATIAL_METERS} meters")
    print(f"  temporal epsilon = {EPS_TEMPORAL_DAYS} days")
    print(f"  min_pts         = {MIN_PTS}")

    labels, core_mask = st_dbscan(
        coords_xy=coords_xy,
        time_days=time_days,
        eps_spatial_m=EPS_SPATIAL_METERS,
        eps_temporal_days=EPS_TEMPORAL_DAYS,
        min_pts=MIN_PTS,
    )

    df["st_dbscan_cluster_id"] = labels
    df["is_core_point"] = core_mask.astype(int)
    df["is_noise"] = (df["st_dbscan_cluster_id"] == -1).astype(int)

    # Human-readable labels
    cluster_ids = sorted([c for c in df["st_dbscan_cluster_id"].unique() if c != -1])
    cluster_summary_rows = []

    for cid in cluster_ids:
        g = df[df["st_dbscan_cluster_id"] == cid].copy()
        if g.empty:
            continue

        label = cluster_label_for_group(g)

        cluster_summary_rows.append({
            "st_dbscan_cluster_id": cid,
            "cluster_label": label,
            "records": len(g),
            "core_points": int(g["is_core_point"].sum()),
            "noise_points": int(g["is_noise"].sum()),
            "start_time_ist": g["created_datetime_ist"].min(),
            "end_time_ist": g["created_datetime_ist"].max(),
            "duration_days": float((g["created_datetime_ist"].max() - g["created_datetime_ist"].min()).total_seconds() / 86400.0) if len(g) else 0.0,
            "latitude_mean": float(g["latitude"].mean()),
            "longitude_mean": float(g["longitude"].mean()),
            "unique_vehicles": int(g["vehicle_number"].nunique()) if "vehicle_number" in g.columns else 0,
            "dominant_vehicle_type": (
                g["vehicle_type"].mode().iloc[0] if "vehicle_type" in g.columns and not g["vehicle_type"].mode().empty else "UNKNOWN"
            ),
            "dominant_police_station": (
                g["police_station"].mode().iloc[0] if "police_station" in g.columns and not g["police_station"].mode().empty else ""
            ),
            "dominant_junction_name": (
                g["junction_name"].mode().iloc[0] if "junction_name" in g.columns and not g["junction_name"].mode().empty else ""
            ),
            "severity_sum": float(g["severity_score"].sum()),
            "severity_mean": float(g["severity_score"].mean()),
        })

    cluster_summary = pd.DataFrame(cluster_summary_rows).sort_values(
        ["records", "severity_sum"],
        ascending=[False, False]
    ).reset_index(drop=True)

    # Weekly counts per cluster (for later trend modules)
    weekly_cluster_counts = (
        df[df["st_dbscan_cluster_id"] != -1]
        .groupby(["st_dbscan_cluster_id", "cluster_week_start"])
        .size()
        .reset_index(name="weekly_count")
        .sort_values(["st_dbscan_cluster_id", "cluster_week_start"])
    )

    # Noise points
    noise_df = df[df["st_dbscan_cluster_id"] == -1].copy()

    # Save outputs
    df.to_csv(OUT_DIR / "phase3_clustered_dataset.csv", index=False)
    cluster_summary.to_csv(OUT_DIR / "phase3_cluster_summary.csv", index=False)
    weekly_cluster_counts.to_csv(OUT_DIR / "phase3_cluster_weekly_counts.csv", index=False)
    noise_df.to_csv(OUT_DIR / "phase3_noise_points.csv", index=False)

    # Compact dispatch view
    dispatch_view_cols = [
        "st_dbscan_cluster_id", "cluster_label", "records", "severity_sum",
        "severity_mean", "unique_vehicles", "dominant_vehicle_type",
        "dominant_police_station", "dominant_junction_name",
        "latitude_mean", "longitude_mean", "start_time_ist", "end_time_ist"
    ]
    dispatch_view = cluster_summary[[c for c in dispatch_view_cols if c in cluster_summary.columns]].copy()
    dispatch_view.to_csv(OUT_DIR / "phase3_cluster_dispatch_view.csv", index=False)

    # Summary
    print("\nStage 3 complete")
    print("Clusters found:", len(cluster_summary))
    print("Noise points:", len(noise_df))
    print("Clustered rows:", len(df) - len(noise_df))
    print("Outputs saved to:", OUT_DIR.resolve())

    if len(cluster_summary):
        print("\nTop 10 clusters:")
        print(
            cluster_summary[[
                "st_dbscan_cluster_id", "cluster_label", "records",
                "severity_sum", "severity_mean", "unique_vehicles"
            ]].head(10).to_string(index=False)
        )

if __name__ == "__main__":
    main()

Input source: content\phase2_outputs_2\phase2_approved_severity_dataset.csv
Approved rows used: 115400
Running ST-DBSCAN with:
  spatial epsilon = 150.0 meters
  temporal epsilon = 3.0 days
  min_pts         = 15

Stage 3 complete
Clusters found: 798
Noise points: 23068
Clustered rows: 92332
Outputs saved to: D:\Flipkart_Hackathon\content\phase3_outputs_2

Top 10 clusters:
 st_dbscan_cluster_id                          cluster_label  records  severity_sum  severity_mean  unique_vehicles
                    2                BTP040 - Elite Junction    21733       34509.0       1.587862            18627
                    3         BTP051 - Safina Plaza Junction     9287       15028.0       1.618176             7622
                   32          BTP027 - Modi Bridge Junction     3168        5271.0       1.663826             2725
                   19 BTP001 - 10th Cross, Dr. Rajkumar Road     2934        4795.0       1.634288             2620
                   35       BTP020 - Hosahal

In [ ]:
# Stage 4 — Implicit μ Estimation / Dwell-Time Estimation
# Input : Phase 3 clustered approved records
# Output: gap event log, cluster-level μ summary, cluster×vehicle-type μ summary,
#         vehicle-type summary, and a merged dataset for Stage 5

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# =========================
# Config
# =========================
PHASE3_PATHS = [
    Path("content/phase3_outputs_2/phase3_clustered_dataset.csv"),
    Path("content/phase3_outputs_2/phase3_cluster_dispatch_view.csv"),
]

OUT_DIR = Path("content/phase4_outputs_2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EPS = 1e-9

# =========================
# Helpers
# =========================
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def parse_datetime_ist(df: pd.DataFrame) -> pd.Series:
    if "created_datetime_ist" in df.columns:
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce", utc=True)
        if ts.notna().any():
            return ts.dt.tz_convert("Asia/Kolkata")
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce")
        return ts.dt.tz_localize("Asia/Kolkata", nonexistent="NaT", ambiguous="NaT")

    if "created_datetime_parsed" in df.columns:
        ts = pd.to_datetime(df["created_datetime_parsed"], errors="coerce", utc=True)
        return ts.dt.tz_convert("Asia/Kolkata")

    if "created_datetime" not in df.columns:
        raise ValueError("Missing created_datetime / created_datetime_ist column.")

    ts = pd.to_datetime(df["created_datetime"], errors="coerce", utc=True)
    return ts.dt.tz_convert("Asia/Kolkata")

def standardize_cluster_col(df: pd.DataFrame) -> str:
    for c in ["st_dbscan_cluster_id", "cluster_id", "dbscan_cluster_id"]:
        if c in df.columns:
            return c
    raise ValueError("No cluster id column found.")

def standardize_vehicle_col(df: pd.DataFrame) -> str:
    if "updated_vehicle_number" in df.columns and "vehicle_number" in df.columns:
        updated = df["updated_vehicle_number"].fillna("").astype(str).str.strip()
        original = df["vehicle_number"].fillna("").astype(str).str.strip()
        df["canonical_vehicle_number"] = np.where(updated.ne(""), updated, original)
        return "canonical_vehicle_number"

    if "vehicle_number" in df.columns:
        df["canonical_vehicle_number"] = df["vehicle_number"].fillna("").astype(str).str.strip()
        return "canonical_vehicle_number"

    raise ValueError("No vehicle number column found.")

def standardize_vehicle_type_col(df: pd.DataFrame) -> str:
    if "updated_vehicle_type" in df.columns and "vehicle_type" in df.columns:
        updated = df["updated_vehicle_type"].fillna("").astype(str).str.strip()
        original = df["vehicle_type"].fillna("").astype(str).str.strip()
        df["canonical_vehicle_type"] = np.where(updated.ne(""), updated, original)
        return "canonical_vehicle_type"

    if "vehicle_type" in df.columns:
        df["canonical_vehicle_type"] = df["vehicle_type"].fillna("UNKNOWN").astype(str).str.strip()
        df.loc[df["canonical_vehicle_type"].eq(""), "canonical_vehicle_type"] = "UNKNOWN"
        return "canonical_vehicle_type"

    df["canonical_vehicle_type"] = "UNKNOWN"
    return "canonical_vehicle_type"

def ensure_hotspot_unit(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "hotspot_unit" in df.columns:
        df["hotspot_unit"] = df["hotspot_unit"].fillna("").astype(str).str.strip()
        df.loc[df["hotspot_unit"].eq(""), "hotspot_unit"] = np.nan
        return df

    if {"junction_name", "police_station"}.issubset(df.columns):
        def make_hotspot_unit(row):
            junction = clean_text(row.get("junction_name", ""))
            if junction and junction.upper() != "NO JUNCTION":
                return f"JUNCTION::{junction}"
            station = clean_text(row.get("police_station", "UNKNOWN"))
            if not station:
                station = "UNKNOWN"
            return f"POLICE_STATION::{station}"

        df["hotspot_unit"] = df.apply(make_hotspot_unit, axis=1)
        return df

    df["hotspot_unit"] = "UNKNOWN"
    return df

def mode_non_empty(series, exclude=None, default=""):
    s = pd.Series(series).dropna().astype(str).str.strip()
    s = s[s.ne("")]

    #  remove excluded placeholders like "No Junction"
    if exclude:
        excl = {clean_text(x).lower() for x in exclude}
        s = s[~s.str.lower().isin(excl)]

    if s.empty:
        return default

    m = s.mode()
    if not m.empty:
        return m.iloc[0]
    return s.iloc[0]

def build_cluster_label_for_group(g: pd.DataFrame) -> str:
    # Prefer a real named junction
    if "junction_name" in g.columns:
        junction = mode_non_empty(g["junction_name"], exclude={"No Junction", "NO JUNCTION"}, default="")
        if junction:
            return junction

    # Then a stable hotspot_unit
    if "hotspot_unit" in g.columns:
        unit = mode_non_empty(g["hotspot_unit"], default="")
        if unit:
            return unit

    # Then police station
    if "police_station" in g.columns:
        station = mode_non_empty(g["police_station"], default="")
        if station:
            return f"POLICE_STATION::{station}"

    return f"Cluster {g.name}"

def week_start_monday(series):
    dt = pd.to_datetime(series, errors="coerce", utc=True).dt.tz_convert("Asia/Kolkata")
    week_start = dt.dt.normalize() - pd.to_timedelta(dt.dt.weekday, unit="D")
    return week_start.dt.tz_localize(None)

# =========================
# Load
# =========================
def load_input():
    for p in PHASE3_PATHS:
        if p.exists():
            return pd.read_csv(p, low_memory=False), p
    raise FileNotFoundError("Could not find Phase 3 output file.")

def main():
    df, source = load_input()

    if "latitude" not in df.columns or "longitude" not in df.columns:
        raise ValueError("Phase 3 dataset must include latitude and longitude.")

    cluster_col = standardize_cluster_col(df)
    vehicle_col = standardize_vehicle_col(df)
    vehicle_type_col = standardize_vehicle_type_col(df)

    # Keep approved records only if the column exists
    if "validation_status_clean" in df.columns:
        df["validation_status_clean"] = df["validation_status_clean"].fillna("").astype(str).str.lower()
        df = df[df["validation_status_clean"].eq("approved")].copy()
    elif "validation_status" in df.columns:
        df["validation_status_clean"] = df["validation_status"].fillna("").astype(str).str.lower()
        df = df[df["validation_status_clean"].eq("approved")].copy()

    df = df.dropna(subset=["latitude", "longitude"]).copy()
    df = ensure_hotspot_unit(df)
    df["created_datetime_ist"] = parse_datetime_ist(df)
    df = df.dropna(subset=["created_datetime_ist"]).copy()

    # Exclude noise
    df[cluster_col] = pd.to_numeric(df[cluster_col], errors="coerce")
    df = df[df[cluster_col].ne(-1)].copy()

    df = df.sort_values([cluster_col, vehicle_col, "created_datetime_ist"]).reset_index(drop=True)
    df["created_datetime_ist_naive"] = df["created_datetime_ist"].dt.tz_localize(None)
    df["service_date"] = df["created_datetime_ist_naive"].dt.date
    df["cluster_week_start"] = week_start_monday(df["created_datetime_ist"])

    # Same vehicle + same cluster + same day consecutive gaps
    group_keys = [cluster_col, vehicle_col, "service_date"]
    df["prev_created_datetime_ist"] = df.groupby(group_keys)["created_datetime_ist_naive"].shift(1)
    df["gap_minutes"] = (
        (df["created_datetime_ist_naive"] - df["prev_created_datetime_ist"])
        .dt.total_seconds() / 60.0
    )
    df["gap_hours"] = df["gap_minutes"] / 60.0

    gap_log = df[df["gap_minutes"].notna() & (df["gap_minutes"] > 0)].copy()

    # If no gaps are found, still save empty outputs and stop cleanly
    if gap_log.empty:
        empty_cols = [
            cluster_col, "cluster_label", "gap_count", "unique_vehicles",
            "unique_vehicle_types", "mean_dwell_minutes", "median_dwell_minutes",
            "std_dwell_minutes", "mu_departures_per_hour"
        ]
        empty_df = pd.DataFrame(columns=empty_cols)

        empty_df.to_csv(OUT_DIR / "phase4_cluster_mu_summary.csv", index=False)
        empty_df.to_csv(OUT_DIR / "phase4_cluster_vehicle_type_mu_summary.csv", index=False)
        empty_df.to_csv(OUT_DIR / "phase4_vehicle_type_mu_summary.csv", index=False)
        df.to_csv(OUT_DIR / "phase4_merged_with_prior_scores.csv", index=False)

        print("Stage 4 complete")
        print("Input source:", source)
        print("No valid inter-record gaps found.")
        print("Empty outputs saved to:", OUT_DIR.resolve())
        return

    # ---------- Cluster label map ----------
    cluster_label_map = (
        df.groupby(cluster_col, sort=False)
        .apply(build_cluster_label_for_group)
        .reset_index(name="cluster_label")
    )

    # ---------- Overall vehicle-type summary ----------
    vehicle_type_summary = (
        gap_log.groupby(vehicle_type_col, dropna=False)
        .agg(
            gap_count=("gap_minutes", "size"),
            unique_clusters=(cluster_col, "nunique"),
            unique_vehicles=(vehicle_col, "nunique"),
            mean_dwell_minutes=("gap_minutes", "mean"),
            median_dwell_minutes=("gap_minutes", "median"),
            std_dwell_minutes=("gap_minutes", lambda s: float(s.std(ddof=0)) if len(s) else 0.0),
            min_gap_minutes=("gap_minutes", "min"),
            max_gap_minutes=("gap_minutes", "max"),
        )
        .reset_index()
        .rename(columns={vehicle_type_col: "vehicle_type"})
        .sort_values(["gap_count", "mean_dwell_minutes"], ascending=[False, False])
    )
    vehicle_type_summary["mu_departures_per_hour"] = 60.0 / (vehicle_type_summary["mean_dwell_minutes"] + EPS)

    # ---------- Cluster-level summary ----------
    cluster_summary = (
        gap_log.groupby(cluster_col, dropna=False)
        .agg(
            gap_count=("gap_minutes", "size"),
            unique_vehicles=(vehicle_col, "nunique"),
            unique_vehicle_types=(vehicle_type_col, "nunique"),
            mean_dwell_minutes=("gap_minutes", "mean"),
            median_dwell_minutes=("gap_minutes", "median"),
            std_dwell_minutes=("gap_minutes", lambda s: float(s.std(ddof=0)) if len(s) else 0.0),
            min_gap_minutes=("gap_minutes", "min"),
            max_gap_minutes=("gap_minutes", "max"),
            first_gap_time=("prev_created_datetime_ist", "min"),
            last_gap_time=("created_datetime_ist_naive", "max"),
        )
        .reset_index()
    )

    cluster_summary = cluster_summary.merge(cluster_label_map, on=cluster_col, how="left")
    cluster_summary["mu_departures_per_hour"] = 60.0 / (cluster_summary["mean_dwell_minutes"] + EPS)

    # ---------- Cluster × vehicle-type summary ----------
    cluster_vehicle_type_summary = (
        gap_log.groupby([cluster_col, vehicle_type_col], dropna=False)
        .agg(
            gap_count=("gap_minutes", "size"),
            unique_vehicles=(vehicle_col, "nunique"),
            mean_dwell_minutes=("gap_minutes", "mean"),
            median_dwell_minutes=("gap_minutes", "median"),
            std_dwell_minutes=("gap_minutes", lambda s: float(s.std(ddof=0)) if len(s) else 0.0),
            min_gap_minutes=("gap_minutes", "min"),
            max_gap_minutes=("gap_minutes", "max"),
        )
        .reset_index()
        .rename(columns={vehicle_type_col: "vehicle_type"})
        .sort_values([cluster_col, "gap_count", "mean_dwell_minutes"], ascending=[True, False, False])
    )
    cluster_vehicle_type_summary["mu_departures_per_hour"] = 60.0 / (
        cluster_vehicle_type_summary["mean_dwell_minutes"] + EPS
    )

    # ---------- Useful merged record-level output for Stage 5 ----------
    merged = df.merge(
        cluster_summary[[
            cluster_col, "cluster_label", "gap_count", "mean_dwell_minutes",
            "median_dwell_minutes", "std_dwell_minutes", "mu_departures_per_hour"
        ]],
        on=cluster_col,
        how="left"
    )

    # ---------- Stage 5 handoff ----------
    mu_lookup = cluster_summary[[
        cluster_col, "cluster_label", "gap_count", "mean_dwell_minutes",
        "median_dwell_minutes", "std_dwell_minutes", "mu_departures_per_hour"
    ]].copy()

    # ---------- Save outputs ----------
    gap_cols = [
        c for c in [
            "id", cluster_col, "cluster_label", vehicle_col, vehicle_type_col,
            "service_date", "created_datetime_ist", "created_datetime_ist_naive",
            "prev_created_datetime_ist", "gap_minutes", "gap_hours",
            "latitude", "longitude", "junction_name", "police_station",
            "severity_score", "hotspot_unit"
        ] if c in gap_log.columns
    ]
    gap_log[gap_cols].to_csv(OUT_DIR / "phase4_gap_event_log.csv", index=False)

    cluster_summary.to_csv(OUT_DIR / "phase4_cluster_mu_summary.csv", index=False)
    cluster_vehicle_type_summary.to_csv(OUT_DIR / "phase4_cluster_vehicle_type_mu_summary.csv", index=False)
    vehicle_type_summary.to_csv(OUT_DIR / "phase4_vehicle_type_mu_summary.csv", index=False)
    merged.to_csv(OUT_DIR / "phase4_merged_with_prior_scores.csv", index=False)
    mu_lookup.to_csv(OUT_DIR / "phase4_stage5_mu_lookup.csv", index=False)

    # ---------- Console summary ----------
    overall_mean = float(gap_log["gap_minutes"].mean())
    overall_mu = 60.0 / (overall_mean + EPS)

    print("Stage 4 complete")
    print("Input source:", source)
    print("Approved clustered rows used:", len(df))
    print("Valid inter-record gaps:", len(gap_log))
    print("Overall mean dwell (min):", round(overall_mean, 3))
    print("Overall μ (departures/hour):", round(overall_mu, 6))
    print("Clusters with μ estimates:", len(cluster_summary))
    print("Vehicle types with μ estimates:", len(vehicle_type_summary))
    print("Outputs saved to:", OUT_DIR.resolve())

    print("\nTop 10 clusters by gap count:")
    if len(cluster_summary):
        print(
            cluster_summary[[
                cluster_col, "cluster_label", "gap_count",
                "mean_dwell_minutes", "mu_departures_per_hour"
            ]].sort_values("gap_count", ascending=False).head(10).to_string(index=False)
        )

if __name__ == "__main__":
    main()

Stage 4 complete
Input source: content\phase3_outputs_2\phase3_clustered_dataset.csv
Approved clustered rows used: 92332
Valid inter-record gaps: 2122
Overall mean dwell (min): 98.041
Overall μ (departures/hour): 0.611989
Clusters with μ estimates: 239
Vehicle types with μ estimates: 19
Outputs saved to: D:\Flipkart_Hackathon\content\phase4_outputs_2

Top 10 clusters by gap count:
 st_dbscan_cluster_id                          cluster_label  gap_count  mean_dwell_minutes  mu_departures_per_hour
                    2                BTP040 - Elite Junction        624          101.331731                0.592115
                    3         BTP051 - Safina Plaza Junction        386           90.655440                0.661847
                   32          BTP027 - Modi Bridge Junction         80          110.900000                0.541028
                   19 BTP001 - 10th Cross, Dr. Rajkumar Road         68           58.867647                1.019236
                  133        POLICE_

In [ ]:
import ast
import math
import os
import re
import time
import warnings
from pathlib import Path
from typing import Optional, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

try:
    import requests
except Exception:
    requests = None

try:
    import networkx as nx
except Exception:
    nx = None

try:
    import osmnx as ox
except Exception:
    ox = None


# ============================================================
# Config
# ============================================================
PHASE4_MERGED_PATHS = [
    Path("content/phase4_outputs_2/phase4_merged_with_prior_scores.csv"),
    Path("/content/phase4_outputs_2/phase4_merged_with_prior_scores.csv"),
    Path("phase4_outputs_2/phase4_merged_with_prior_scores.csv"),
]

PHASE3_PATHS = [
    Path("content/phase3_outputs_2/phase3_clustered_dataset.csv"),
    Path("/content/phase3_outputs_2/phase3_clustered_dataset.csv"),
    Path("phase3_outputs_2/phase3_clustered_dataset.csv"),
]

PHASE4_MU_PATHS = [
    Path("content/phase4_outputs_2/phase4_cluster_mu_summary.csv"),
    Path("/content/phase4_outputs_2/phase4_cluster_mu_summary.csv"),
    Path("phase4_outputs_2/phase4_cluster_mu_summary.csv"),
]

OUT_DIR = Path("content/phase5_outputs_2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Persistent on-disk GraphML cache so the expensive OSMnx download is only
# done once across runs.  Delete the file manually to force a fresh download.
GRAPH_CACHE_PATH = OUT_DIR / "osmnx_graph_cache.graphml"

EPS = 1e-9
ALPHA_BPR = 0.56
BETA_BPR = 2.12
DEFAULT_MEAN_DWELL_MINUTES = 94.4

DEFAULT_LANES = 2
DEFAULT_WIDTH_PER_LANE_M = 3.5
DEFAULT_CARRIAGEWAY_WIDTH_M = DEFAULT_LANES * DEFAULT_WIDTH_PER_LANE_M
DEFAULT_LINK_LENGTH_M = 250.0
DEFAULT_ROAD_CLASS = "road"

PEAK_HOURS = {8, 9, 10, 11, 12}  # 8 AM to 1 PM
WEEK_SPLIT_HOURS = 24

ENABLE_OSMNX = os.environ.get("ENABLE_OSMNX_PHASE5", "1").strip() == "1"
print(ENABLE_OSMNX)
ENABLE_MAPPLS = os.environ.get("ENABLE_MAPPLS_PHASE5", "1").strip() == "1"
print(ENABLE_MAPPLS)
MAPPLS_ADDRESS_TOP_N = int(os.environ.get("MAPPLS_ADDRESS_TOP_N", "10"))
MAPPLS_ACCESS_TOKEN = os.environ.get("MAPPLS_ACCESS_TOKEN", "").strip()
MAPPLS_REGION = os.environ.get("MAPPLS_REGION", "IND").strip().upper()

ROAD_CLASS_SPEED_KMH = {
    "motorway": 60.0,
    "trunk": 55.0,
    "primary": 45.0,
    "secondary": 40.0,
    "tertiary": 35.0,
    "unclassified": 30.0,
    "residential": 30.0,
    "living_street": 20.0,
    "service": 25.0,
    "road": 30.0,
}

ROAD_CLASS_BASE_CAPACITY_PER_LANE = {
    "motorway": 2200.0,
    "trunk": 2100.0,
    "primary": 1900.0,
    "secondary": 1800.0,
    "tertiary": 1700.0,
    "unclassified": 1650.0,
    "residential": 1500.0,
    "living_street": 1200.0,
    "service": 1400.0,
    "road": 1600.0,
}

VEHICLE_WIDTH_M = {
    "SCOOTER": 0.80,
    "MOTOR CYCLE": 0.90,
    "MOTORCYCLE": 0.90,
    "BICYCLE": 0.60,
    "CYCLE": 0.60,
    "PASSENGER AUTO": 1.60,
    "AUTO": 1.60,
    "CAR": 1.90,
    "SUV": 2.00,
    "JEEP": 2.00,
    "VAN": 2.20,
    "TEMPO": 2.20,
    "BUS": 2.60,
    "TRUCK": 2.60,
    "LORRY": 2.60,
    "TANKER": 2.80,
    "TRACTOR": 2.20,
    "MINI TRUCK": 2.20,
    "AMBULANCE": 2.00,
    "UNKNOWN": 1.90,
}

SEVERITY_RULES = {
    5: {
        "DOUBLE PARKING",
        "NEAR ROAD CROSSING",
        "NEAR TRAFFIC LIGHT",
        "NEAR ZEBRA CROSSING",
        "NEAR TRAFFIC LIGHT / ZEBRA CROSSING",
        "NEAR TRAFFIC LIGHT/ZEBRA CROSSING",
    },
    4: {
        "PARKING IN MAIN ROAD",
        "NEAR BUS STOP",
        "NEAR SCHOOL",
        "NEAR HOSPITAL",
        "OPPOSITE ANOTHER VEHICLE",
    },
    3: {"PARKING ON FOOTPATH"},
    2: {"WRONG PARKING", "PARKING OTHER THAN BUS STOP"},
    1: {"NO PARKING", "NO PARKING (GENERIC)"},
}


# ============================================================
# Helpers
# ============================================================
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def normalize_tag(tag):
    return clean_text(tag).upper().replace("&", "AND").strip()


def parse_listlike(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    s = str(value).strip()
    if not s:
        return []
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple)):
            return list(parsed)
        return [parsed]
    except Exception:
        if s.startswith("[") and s.endswith("]"):
            s = s[1:-1]
        parts = [p.strip().strip("'").strip('"') for p in s.split(",")]
        return [p for p in parts if p]


def safe_float(x, default=np.nan):
    try:
        if pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default


def valid_coords(lat, lon):
    try:
        return (
            pd.notna(lat)
            and pd.notna(lon)
            and -90 <= float(lat) <= 90
            and -180 <= float(lon) <= 180
        )
    except Exception:
        return False


def minmax(s):
    s = pd.to_numeric(s, errors="coerce").fillna(0.0).astype(float)
    if len(s) == 0:
        return pd.Series(dtype=float)
    if s.nunique(dropna=True) <= 1:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.min()) / (s.max() - s.min() + EPS)


def smooth_norm(s, floor=0.10):
    return floor + (1.0 - floor) * minmax(s)


def safe_ratio(a, b):
    return a / (b + EPS)


def safe_metric(val, digits=2):
    try:
        if pd.isna(val):
            return "0"
        val = float(val)
        if abs(val) >= 1000:
            return f"{val:,.0f}"
        return f"{val:,.{digits}f}"
    except Exception:
        return str(val)


def dominant_label_from_series(series, exclude=None, default=""):
    s = pd.Series(series).dropna().astype(str).str.strip()
    s = s[s.ne("")]
    if exclude:
        excl = {clean_text(x).lower() for x in exclude}
        s = s[~s.str.lower().isin(excl)]
    if s.empty:
        return default
    m = s.mode()
    if not m.empty:
        return m.iloc[0]
    return s.iloc[0]


def standardize_cluster_col(df):
    for c in ["st_dbscan_cluster_id", "cluster_id", "dbscan_cluster_id"]:
        if c in df.columns:
            return c
    raise ValueError("No cluster id column found.")


def standardize_vehicle_col(df):
    for c in ["canonical_vehicle_number", "vehicle_number", "updated_vehicle_number"]:
        if c in df.columns:
            return c
    raise ValueError("No vehicle number column found.")


def standardize_vehicle_type_col(df):
    for c in ["canonical_vehicle_type", "vehicle_type", "updated_vehicle_type"]:
        if c in df.columns:
            return c
    return None


def load_first_existing(paths):
    for p in paths:
        if p.exists():
            return pd.read_csv(p, low_memory=False), p
    return None, None


def load_input():
    df, src = load_first_existing(PHASE4_MERGED_PATHS)
    if df is not None:
        return df, src
    df, src = load_first_existing(PHASE3_PATHS)
    if df is not None:
        return df, src
    raise FileNotFoundError("Could not find phase 4 merged output or phase 3 clustered dataset.")


def load_phase4_mu():
    df, _ = load_first_existing(PHASE4_MU_PATHS)
    return df


def ensure_hotspot_unit(df):
    df = df.copy()
    if "hotspot_unit" in df.columns:
        df["hotspot_unit"] = df["hotspot_unit"].fillna("").astype(str).str.strip()
        df.loc[df["hotspot_unit"].eq(""), "hotspot_unit"] = np.nan
        return df

    def make_hotspot_unit(row):
        junction = clean_text(row.get("junction_name", ""))
        if junction and junction.upper() != "NO JUNCTION":
            return f"JUNCTION::{junction}"
        station = clean_text(row.get("police_station", "UNKNOWN"))
        if not station:
            station = "UNKNOWN"
        return f"POLICE_STATION::{station}"

    if "junction_name" in df.columns or "police_station" in df.columns:
        df["hotspot_unit"] = df.apply(make_hotspot_unit, axis=1)
    else:
        df["hotspot_unit"] = "UNKNOWN"
    return df


def ensure_label_column(df):
    df = df.copy()
    if "cluster_label" in df.columns:
        df["cluster_label"] = df["cluster_label"].fillna("").astype(str).str.strip()
        df.loc[df["cluster_label"].eq(""), "cluster_label"] = np.nan
        return df
    if "hotspot_unit" in df.columns:
        df["cluster_label"] = df["hotspot_unit"].fillna("").astype(str).str.strip()
        df.loc[df["cluster_label"].eq(""), "cluster_label"] = np.nan
        return df
    if "dominant_junction_name" in df.columns:
        df["cluster_label"] = df["dominant_junction_name"].fillna("").astype(str).str.strip()
        df.loc[df["cluster_label"].eq(""), "cluster_label"] = np.nan
        return df
    if "st_dbscan_cluster_id" in df.columns:
        df["cluster_label"] = "CLUSTER::" + df["st_dbscan_cluster_id"].astype(str)
        return df
    df["cluster_label"] = "UNKNOWN"
    return df


def derive_coords(df):
    df = df.copy()
    if {"lat", "lon"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
        df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    elif {"centroid_lat", "centroid_lon"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["centroid_lat"], errors="coerce")
        df["lon"] = pd.to_numeric(df["centroid_lon"], errors="coerce")
    elif {"latitude_mean", "longitude_mean"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["latitude_mean"], errors="coerce")
        df["lon"] = pd.to_numeric(df["longitude_mean"], errors="coerce")
    elif {"latitude", "longitude"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["latitude"], errors="coerce")
        df["lon"] = pd.to_numeric(df["longitude"], errors="coerce")
    else:
        df["lat"] = np.nan
        df["lon"] = np.nan
    return df


def ensure_severity(df):
    df = df.copy()
    if "severity_score" in df.columns:
        df["severity_score"] = pd.to_numeric(df["severity_score"], errors="coerce").fillna(1).clip(lower=1, upper=5).astype(int)
        return df

    def severity_from_tags(tags):
        if not tags:
            return 1
        normalized = [normalize_tag(t) for t in tags]
        score = 1
        for sev in sorted(SEVERITY_RULES.keys(), reverse=True):
            vocab = SEVERITY_RULES[sev]
            if any(any(v == tag or v in tag for v in vocab) for tag in normalized):
                score = sev
                break
        return score

    if "violation_type" in df.columns:
        df["violation_tags"] = df["violation_type"].apply(parse_listlike)
        df["severity_score"] = df["violation_tags"].apply(severity_from_tags)
    else:
        df["severity_score"] = 1
        df["violation_tags"] = [[] for _ in range(len(df))]
    return df


def parse_datetime_ist(df):
    if "created_datetime_ist" in df.columns:
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce", utc=True)
        if ts.notna().any():
            return ts.dt.tz_convert("Asia/Kolkata")
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce")
        return ts.dt.tz_localize("Asia/Kolkata", nonexistent="NaT", ambiguous="NaT")

    if "created_datetime_parsed" in df.columns:
        ts = pd.to_datetime(df["created_datetime_parsed"], errors="coerce", utc=True)
        return ts.dt.tz_convert("Asia/Kolkata")

    if "created_datetime" not in df.columns:
        raise ValueError("Missing created_datetime / created_datetime_ist column.")

    ts = pd.to_datetime(df["created_datetime"], errors="coerce", utc=True)
    return ts.dt.tz_convert("Asia/Kolkata")


def week_start_monday(series):
    dt = pd.to_datetime(series, errors="coerce", utc=True).dt.tz_convert("Asia/Kolkata")
    week_start = dt.dt.normalize() - pd.to_timedelta(dt.dt.weekday, unit="D")
    return week_start.dt.tz_localize(None)


def parse_lane_count(lanes_value):
    if pd.isna(lanes_value):
        return None
    if isinstance(lanes_value, list):
        lanes_value = lanes_value[0] if lanes_value else None
    s = clean_text(lanes_value)
    if not s:
        return None
    m = re.search(r"(\d+(?:\.\d+)?)", s.replace(";", ",").replace("|", ","))
    if m:
        try:
            return max(1, int(round(float(m.group(1)))))
        except Exception:
            pass
    return None


def parse_width_m(width_value):
    if pd.isna(width_value):
        return None
    if isinstance(width_value, list):
        width_value = width_value[0] if width_value else None
    s = clean_text(width_value).lower().replace(",", ".")
    m = re.search(r"(\d+(?:\.\d+)?)", s)
    if m:
        try:
            return float(m.group(1))
        except Exception:
            return None
    return None


def road_class_from_highway(highway_value):
    if pd.isna(highway_value):
        return DEFAULT_ROAD_CLASS

    vals = highway_value if isinstance(highway_value, list) else parse_listlike(highway_value)
    if not vals:
        raw = clean_text(highway_value).lower()
        vals = [raw] if raw else []

    normalized = []
    for v in vals:
        s = clean_text(v).lower().replace(" ", "_")
        s = {
            "motorway_link": "motorway",
            "trunk_link": "trunk",
            "primary_link": "primary",
            "secondary_link": "secondary",
            "tertiary_link": "tertiary",
        }.get(s, s)
        if s in ROAD_CLASS_SPEED_KMH:
            normalized.append(s)
        else:
            normalized.append("road")

    priority = [
        "motorway", "trunk", "primary", "secondary", "tertiary",
        "unclassified", "residential", "living_street", "service", "road"
    ]
    for p in priority:
        if p in normalized:
            return p
    return normalized[0] if normalized else DEFAULT_ROAD_CLASS


def road_speed_kmh(road_class):
    return ROAD_CLASS_SPEED_KMH.get(str(road_class).strip().lower(), 30.0)


def base_capacity_per_lane(road_class):
    return ROAD_CLASS_BASE_CAPACITY_PER_LANE.get(str(road_class).strip().lower(), 1600.0)


def vehicle_width_m(vehicle_type):
    vt = clean_text(vehicle_type).upper()
    return VEHICLE_WIDTH_M.get(vt, VEHICLE_WIDTH_M["UNKNOWN"])


def reverse_geocode_mappls(lat, lon, session):
    out = {"ok": False, "address": "", "error": "", "raw": None, "source": "reverse_geocode"}
    if not valid_coords(lat, lon):
        out["error"] = "invalid_coordinates"
        return out
    if not MAPPLS_ACCESS_TOKEN:
        out["error"] = "missing_MAPPLS_ACCESS_TOKEN"
        return out
    if requests is None:
        out["error"] = "requests_not_available"
        return out
    try:
        resp = session.get(
            "https://search.mappls.com/search/address/rev-geocode",
            params={
                "lat": float(lat),
                "lng": float(lon),
                "region": MAPPLS_REGION,
                "access_token": MAPPLS_ACCESS_TOKEN,
            },
            timeout=25,
        )
        resp.raise_for_status()
        data = resp.json()
        out["raw"] = data
        results = data.get("results") or []
        if results and isinstance(results, list) and isinstance(results[0], dict):
            first = results[0]
            address = first.get("formatted_address") or first.get("address") or ""
            if not address:
                parts = []
                for k in [
                    "houseNumber", "houseName", "poi", "street", "subSubLocality",
                    "subLocality", "locality", "subDistrict", "district", "city",
                    "state", "pincode"
                ]:
                    v = clean_text(first.get(k, ""))
                    if v:
                        parts.append(v)
                address = ", ".join(dict.fromkeys(parts))
            out["address"] = clean_text(address)
            out["ok"] = bool(out["address"])
            if not out["address"]:
                out["error"] = "empty_address"
        return out
    except Exception as e:
        out["error"] = str(e)
        return out


def build_osmnx_graph(unique_coords):
    if not (ENABLE_OSMNX and ox is not None and nx is not None):
        return None, None

    if unique_coords is None or len(unique_coords) == 0:
        return None, None

    try:
        if hasattr(ox, "settings"):
            try:
                ox.settings.use_cache = True
            except Exception:
                pass
            try:
                ox.settings.log_console = False  # reduce noise; errors logged below
            except Exception:
                pass
    except Exception:
        pass

    # ---------- PATCH: persistent on-disk GraphML cache ----------
    # If a cached graph exists from a previous run, load it directly
    # instead of re-downloading from the OSMnx tile server.
    G = None
    if GRAPH_CACHE_PATH.exists():
        try:
            G = ox.load_graphml(str(GRAPH_CACHE_PATH))
            print(f"[OSMnx] Loaded graph from cache: {GRAPH_CACHE_PATH}")
        except Exception as e:
            print(f"[OSMnx] Cache load failed ({e}), re-downloading...")
            G = None

    if G is None:
        lat_min = float(unique_coords["lat_key"].min())
        lat_max = float(unique_coords["lat_key"].max())
        lon_min = float(unique_coords["lon_key"].min())
        lon_max = float(unique_coords["lon_key"].max())

        pad = 0.02
        north = lat_max + pad
        south = lat_min - pad
        east = lon_max + pad
        west = lon_min - pad

        try:
            try:
                G = ox.graph_from_bbox(
                    north=north,
                    south=south,
                    east=east,
                    west=west,
                    network_type="drive",
                    simplify=True,
                    retain_all=False,
                )
                print("[OSMnx] Downloaded graph via bbox (keyword args)")
            except TypeError:
                G = ox.graph_from_bbox(
                    north, south, east, west,
                    network_type="drive",
                    simplify=True,
                    retain_all=False,
                )
                print("[OSMnx] Downloaded graph via bbox (positional args)")
        except Exception as e:
            print(f"[OSMnx] bbox download failed ({e}), trying city-level fallback...")
            try:
                G = ox.graph_from_place(
                    "Bengaluru, Karnataka, India",
                    network_type="drive",
                    simplify=True,
                    retain_all=False,
                )
                print("[OSMnx] Downloaded graph via place name")
            except Exception as e2:
                print(f"[OSMnx] place download also failed ({e2}). Road enrichment will use defaults.")
                G = None

        # Save successfully downloaded graph so the next run is instant.
        if G is not None:
            try:
                ox.save_graphml(G, str(GRAPH_CACHE_PATH))
                print(f"[OSMnx] Graph saved to cache: {GRAPH_CACHE_PATH}")
            except Exception as e:
                print(f"[OSMnx] Could not save graph cache ({e}); continuing anyway.")
    # ---------- end cache patch ----------

    if G is None:
        return None, None

    try:
        UG = nx.Graph(G)
    except Exception:
        UG = G.to_undirected() if hasattr(G, "to_undirected") else None

    bc = {}
    if UG is not None and len(UG.nodes) > 0:
        n_nodes = len(UG.nodes)
        print(f"[OSMnx] Graph has {n_nodes:,} nodes")
        try:
            if n_nodes > 5000:
                k = min(200, max(50, n_nodes // 200))
                print(f"[OSMnx] Approximate betweenness centrality (k={k})...")
                bc = nx.betweenness_centrality(UG, k=k, normalized=True, weight="length", seed=42)
            else:
                print(f"[OSMnx] Exact betweenness centrality on {n_nodes:,} nodes...")
                bc = nx.betweenness_centrality(UG, normalized=True, weight="length")
        except Exception as e:
            print(f"[OSMnx] Betweenness centrality failed ({e}); bc will be 0.0 for all nodes.")
            bc = {}

    return G, bc


def road_context_for_points(unique_coords):
    """
    Returns a DataFrame with road_class, lane_count, width, degree, betweenness,
    using one graph download + cached nearest-node lookups.
    """
    defaults = []
    if unique_coords is None or len(unique_coords) == 0:
        return pd.DataFrame(defaults)

    G, bc = build_osmnx_graph(unique_coords)
    if G is None:
        for _, row in unique_coords.iterrows():
            defaults.append({
                "lat_key": row["lat_key"],
                "lon_key": row["lon_key"],
                "geometry_source": "fallback",
                "road_class": DEFAULT_ROAD_CLASS,
                "lane_count": DEFAULT_LANES,
                "carriageway_width_m": DEFAULT_CARRIAGEWAY_WIDTH_M,
                "link_length_m": DEFAULT_LINK_LENGTH_M,
                "junction_degree": 4,
                "betweenness_centrality": 0.0,
            })
        return pd.DataFrame(defaults)

    total = len(unique_coords)
    print(f"Road context enrichment for {total:,} unique coordinates...")
    for i, row in enumerate(unique_coords.itertuples(index=False), start=1):
        lat = float(row.lat_key)
        lon = float(row.lon_key)
        if i == 1 or i % 25 == 0 or i == total:
            print(f"  road context {i:,}/{total:,}")

        try:
            node = ox.distance.nearest_nodes(G, X=lon, Y=lat)
            edge = ox.distance.nearest_edges(G, X=lon, Y=lat)
            u, v, k = edge
            edge_data = G.edges[u, v, k]

            road_class = road_class_from_highway(edge_data.get("highway", DEFAULT_ROAD_CLASS))
            lane_count = parse_lane_count(edge_data.get("lanes", None))
            if lane_count is None:
                lane_count = DEFAULT_LANES

            width_m = parse_width_m(edge_data.get("width", None))
            if width_m is None:
                width_m = lane_count * DEFAULT_WIDTH_PER_LANE_M

            link_length_m = safe_float(edge_data.get("length", DEFAULT_LINK_LENGTH_M), DEFAULT_LINK_LENGTH_M)

            if hasattr(G, "degree"):
                degree = int(G.degree[node]) if node in G else 4
            else:
                degree = 4

            centrality = float(bc.get(node, 0.0)) if bc else 0.0

            defaults.append({
                "lat_key": lat,
                "lon_key": lon,
                "geometry_source": "osmnx",
                "road_class": road_class,
                "lane_count": lane_count,
                "carriageway_width_m": width_m,
                "link_length_m": link_length_m,
                "junction_degree": degree,
                "betweenness_centrality": centrality,
            })
        except Exception as _exc:
            # PATCH: log the failure reason so we can diagnose whether
            # nearest_nodes/nearest_edges is the source (projection issue,
            # empty graph section, etc.) rather than silently falling back.
            if i <= 3 or i % 50 == 0:
                print(f"  [road_context fallback] coord ({lat:.4f},{lon:.4f}): {_exc}")
            defaults.append({
                "lat_key": lat,
                "lon_key": lon,
                "geometry_source": "fallback",
                "road_class": DEFAULT_ROAD_CLASS,
                "lane_count": DEFAULT_LANES,
                "carriageway_width_m": DEFAULT_CARRIAGEWAY_WIDTH_M,
                "link_length_m": DEFAULT_LINK_LENGTH_M,
                "junction_degree": 4,
                "betweenness_centrality": 0.0,
            })

    return pd.DataFrame(defaults)


def compute_dwell_gaps(records_df, cluster_col, vehicle_col, vehicle_type_col):
    """
    Stage 4: implicit μ estimator.
    Same vehicle, same cluster, same calendar day.
    """
    r = records_df.copy()
    r = r.dropna(subset=[cluster_col]).copy()
    r[cluster_col] = pd.to_numeric(r[cluster_col], errors="coerce")
    r = r[r[cluster_col].ne(-1)].copy()

    if vehicle_col not in r.columns:
        raise ValueError("vehicle number column is required for Stage 4/5.")

    r[vehicle_col] = r[vehicle_col].fillna("").astype(str).str.strip()
    r = r[r[vehicle_col].ne("")]

    # Parse datetime
    r["created_datetime_ist"] = parse_datetime_ist(r)
    r = r.dropna(subset=["created_datetime_ist"]).copy()
    r["created_datetime_ist_naive"] = r["created_datetime_ist"].dt.tz_localize(None)
    r["service_date"] = r["created_datetime_ist_naive"].dt.date
    r["cluster_week_start"] = week_start_monday(r["created_datetime_ist"])
    r["hour_ist"] = r["created_datetime_ist"].dt.hour
    r["is_peak_window"] = r["hour_ist"].isin(PEAK_HOURS).astype(int)

    sort_cols = [cluster_col, vehicle_col, "service_date", "created_datetime_ist_naive"]
    sort_cols = [c for c in sort_cols if c in r.columns]
    r = r.sort_values(sort_cols).copy()

    r["prev_created_datetime_ist_naive"] = (
        r.groupby([cluster_col, vehicle_col, "service_date"])["created_datetime_ist_naive"].shift(1)
    )
    r["gap_minutes"] = (
        r["created_datetime_ist_naive"] - r["prev_created_datetime_ist_naive"]
    ).dt.total_seconds() / 60.0

    gaps = r[r["gap_minutes"].notna() & (r["gap_minutes"] > 0)].copy()

    # per-cluster dwell summary
    if len(gaps):
        dwell_summary = (
            gaps.groupby(cluster_col)
            .agg(
                gap_count=("gap_minutes", "size"),
                mean_dwell_minutes=("gap_minutes", "mean"),
                median_dwell_minutes=("gap_minutes", "median"),
                std_dwell_minutes=("gap_minutes", lambda s: float(s.std(ddof=0)) if len(s) else 0.0),
            )
            .reset_index()
        )
        dwell_summary["mu_departures_per_hour"] = 60.0 / (dwell_summary["mean_dwell_minutes"] + EPS)
    else:
        dwell_summary = pd.DataFrame(columns=[
            cluster_col, "gap_count", "mean_dwell_minutes", "median_dwell_minutes",
            "std_dwell_minutes", "mu_departures_per_hour"
        ])

    # per-cluster, per-vehicle-type dwell summary
    dwell_by_type = pd.DataFrame()
    if vehicle_type_col and vehicle_type_col in gaps.columns and len(gaps):
        dwell_by_type = (
            gaps.groupby([cluster_col, vehicle_type_col])
            .agg(
                gap_count=("gap_minutes", "size"),
                mean_dwell_minutes=("gap_minutes", "mean"),
                median_dwell_minutes=("gap_minutes", "median"),
                std_dwell_minutes=("gap_minutes", lambda s: float(s.std(ddof=0)) if len(s) else 0.0),
            )
            .reset_index()
        )

    weekly_counts = (
        r.groupby([cluster_col, "cluster_week_start"])
        .size()
        .reset_index(name="weekly_count")
        .sort_values([cluster_col, "cluster_week_start"])
    )

    # growth acceleration per cluster
    growth_rows = []
    for cid, g in weekly_counts.groupby(cluster_col):
        counts = g["weekly_count"].to_numpy(dtype=float)
        if len(counts) < 2:
            first_half = float(counts.mean()) if len(counts) else 0.0
            second_half = first_half
            growth_pct = 0.0
        else:
            mid = max(1, len(counts) // 2)
            first_half = float(counts[:mid].mean()) if len(counts[:mid]) else 0.0
            second_half = float(counts[mid:].mean()) if len(counts[mid:]) else 0.0
            growth_pct = safe_ratio(second_half - first_half, first_half) if first_half > 0 else second_half

        growth_multiplier = max(0.5, 1.0 + max(0.0, growth_pct))
        growth_rows.append({
            cluster_col: cid,
            "growth_first_half_mean": first_half,
            "growth_second_half_mean": second_half,
            "growth_pct_change": growth_pct,
            "growth_multiplier": growth_multiplier,
        })

    growth_df = pd.DataFrame(growth_rows)
    return r, gaps, dwell_summary, dwell_by_type, weekly_counts, growth_df


def compute_cluster_table(records_df, cluster_col, vehicle_col, vehicle_type_col, dwell_summary, growth_df):
    """
    Cluster-level aggregate table built from approved records.
    """
    agg = {
        "records_total": (cluster_col, "size"),
        "peak_window_records": ("is_peak_window", "sum"),
        "distinct_days": ("service_date", "nunique"),
        "severity_sum": ("severity_score", "sum"),
        "severity_mean": ("severity_score", "mean"),
        "unique_vehicles": (vehicle_col, "nunique"),
        "centroid_lat": ("lat", "mean"),
        "centroid_lon": ("lon", "mean"),
        "dominant_police_station": ("police_station", lambda s: dominant_label_from_series(s, default="")) if "police_station" in records_df.columns else (cluster_col, lambda s: ""),
        "dominant_junction_name": ("junction_name", lambda s: dominant_label_from_series(s, exclude={"No Junction", "NO JUNCTION"}, default="")) if "junction_name" in records_df.columns else (cluster_col, lambda s: ""),
    }

    if vehicle_type_col and vehicle_type_col in records_df.columns:
        agg["unique_vehicle_types"] = (vehicle_type_col, "nunique")
        agg["dominant_vehicle_type"] = (vehicle_type_col, lambda s: dominant_label_from_series(s, default="UNKNOWN"))
    else:
        agg["unique_vehicle_types"] = (cluster_col, lambda s: 1)
        agg["dominant_vehicle_type"] = (cluster_col, lambda s: "UNKNOWN")

    cluster_table = records_df.groupby(cluster_col).agg(**agg).reset_index()

    # Peak-hour maximum count within peak window
    peak_hour_counts = (
        records_df[records_df["is_peak_window"].eq(1)]
        .groupby([cluster_col, "hour_ist"])
        .size()
        .reset_index(name="hourly_count")
    )
    records_peak_hour = (
        peak_hour_counts.groupby(cluster_col)["hourly_count"]
        .max()
        .rename("records_peak_hour")
        .reset_index()
    )
    cluster_table = cluster_table.merge(records_peak_hour, on=cluster_col, how="left")

    # Merge dwell summary
    if dwell_summary is not None and len(dwell_summary):
        cluster_table = cluster_table.merge(dwell_summary, on=cluster_col, how="left")

    # Merge growth
    if growth_df is not None and len(growth_df):
        cluster_table = cluster_table.merge(growth_df, on=cluster_col, how="left")

    # defaults / cleanup
    for c in ["records_total", "peak_window_records", "distinct_days", "records_peak_hour", "unique_vehicles", "unique_vehicle_types"]:
        if c not in cluster_table.columns:
            cluster_table[c] = 0
        cluster_table[c] = pd.to_numeric(cluster_table[c], errors="coerce").fillna(0)

    cluster_table["distinct_days"] = cluster_table["distinct_days"].clip(lower=1)

    if "mean_dwell_minutes" not in cluster_table.columns:
        cluster_table["mean_dwell_minutes"] = np.nan
    if "mu_departures_per_hour" not in cluster_table.columns:
        cluster_table["mu_departures_per_hour"] = np.nan
    if "growth_first_half_mean" not in cluster_table.columns:
        cluster_table["growth_first_half_mean"] = 0.0
    if "growth_second_half_mean" not in cluster_table.columns:
        cluster_table["growth_second_half_mean"] = 0.0
    if "growth_pct_change" not in cluster_table.columns:
        cluster_table["growth_pct_change"] = 0.0
    if "growth_multiplier" not in cluster_table.columns:
        cluster_table["growth_multiplier"] = 1.0

    # lambda normalization: divide peak-window record count by (distinct active days x peak-hour span)
    GLOBAL_OBSERVATION_DAYS = records_df["service_date"].nunique()

    cluster_table["lambda_hr_peak_window"] = (
        cluster_table["peak_window_records"] /
        (GLOBAL_OBSERVATION_DAYS * 5.0)
    )
    cluster_table["lambda_hr_peak_hour"] = cluster_table["records_peak_hour"] / cluster_table["distinct_days"].clip(lower=1)

    # mu fallback if gap summary missing
    cluster_table["mean_dwell_minutes"] = pd.to_numeric(cluster_table["mean_dwell_minutes"], errors="coerce")
    cluster_table["mu_departures_per_hour"] = pd.to_numeric(cluster_table["mu_departures_per_hour"], errors="coerce")
    cluster_table["mean_dwell_minutes"] = cluster_table["mean_dwell_minutes"].fillna(DEFAULT_MEAN_DWELL_MINUTES)
    cluster_table["mu_departures_per_hour"] = cluster_table["mu_departures_per_hour"].fillna(60.0 / cluster_table["mean_dwell_minutes"].clip(lower=EPS))

    # One cluster label field
    cluster_table["cluster_label"] = np.nan
    cluster_table.loc[cluster_table["dominant_junction_name"].astype(str).str.strip().ne(""), "cluster_label"] = cluster_table["dominant_junction_name"]
    cluster_table.loc[cluster_table["cluster_label"].isna() | cluster_table["cluster_label"].astype(str).str.strip().eq(""), "cluster_label"] = cluster_table["dominant_police_station"].apply(lambda x: f"POLICE_STATION::{x}" if clean_text(x) else np.nan)
    cluster_table.loc[cluster_table["cluster_label"].isna() | cluster_table["cluster_label"].astype(str).str.strip().eq(""), "cluster_label"] = "CLUSTER::" + cluster_table[cluster_col].astype(str)

    # Centroids / coords
    cluster_table["centroid_lat"] = pd.to_numeric(cluster_table["centroid_lat"], errors="coerce")
    cluster_table["centroid_lon"] = pd.to_numeric(cluster_table["centroid_lon"], errors="coerce")
    cluster_table["lat"] = cluster_table["centroid_lat"]
    cluster_table["lon"] = cluster_table["centroid_lon"]

    # vehicle width average
    if vehicle_type_col and vehicle_type_col in records_df.columns:
        records_df["vehicle_width_m"] = records_df[vehicle_type_col].map(vehicle_width_m).fillna(VEHICLE_WIDTH_M["UNKNOWN"])
        avg_width = records_df.groupby(cluster_col)["vehicle_width_m"].mean().rename("vehicle_width_avg_m").reset_index()
        cluster_table = cluster_table.merge(avg_width, on=cluster_col, how="left")
    else:
        cluster_table["vehicle_width_avg_m"] = VEHICLE_WIDTH_M["UNKNOWN"]

    return cluster_table


def compute_repeat_offenders(records_df, cluster_col, vehicle_col, vehicle_type_col):
    r = records_df.copy()
    r[vehicle_col] = r[vehicle_col].fillna("").astype(str).str.strip()
    r = r[r[vehicle_col].ne("")].copy()

    if len(r) == 0:
        return pd.DataFrame(columns=[
            "vehicle_number", "total_violations", "unique_clusters", "unique_hotspots",
            "first_seen", "last_seen", "dominant_vehicle_type", "chronic_offender_flag"
        ])

    if vehicle_type_col and vehicle_type_col in r.columns:
        dom_vtype = lambda s: dominant_label_from_series(s, default="UNKNOWN")
    else:
        dom_vtype = lambda s: "UNKNOWN"

    offender_counts = (
        r.groupby(vehicle_col)
        .agg(
            total_violations=(vehicle_col, "size"),
            unique_clusters=(cluster_col, "nunique"),
            unique_hotspots=("hotspot_unit", "nunique") if "hotspot_unit" in r.columns else (vehicle_col, "size"),
            first_seen=("created_datetime_ist_naive", "min"),
            last_seen=("created_datetime_ist_naive", "max"),
            dominant_vehicle_type=(vehicle_type_col, dom_vtype) if vehicle_type_col and vehicle_type_col in r.columns else (vehicle_col, lambda s: "UNKNOWN"),
        )
        .reset_index()
        .rename(columns={vehicle_col: "vehicle_number"})
        .sort_values(["total_violations", "unique_clusters"], ascending=[False, False])
    )

    offender_counts["chronic_offender_flag"] = (offender_counts["total_violations"] >= 5).astype(int)
    offender_counts = offender_counts[offender_counts["chronic_offender_flag"].eq(1)].copy()
    return offender_counts


def compute_vehicle_mix(records_df, cluster_col, vehicle_col, vehicle_type_col):
    if not vehicle_type_col or vehicle_type_col not in records_df.columns:
        return pd.DataFrame(columns=[cluster_col, "vehicle_type", "count", "share"])

    vehicle_mix = (
        records_df.groupby([cluster_col, vehicle_type_col])
        .agg(count=(vehicle_col, "size"))
        .reset_index()
        .rename(columns={vehicle_type_col: "vehicle_type"})
    )
    vehicle_mix["share"] = vehicle_mix["count"] / vehicle_mix.groupby(cluster_col)["count"].transform("sum")
    return vehicle_mix


def compute_dominant_violation_tag(records_df, cluster_col):
    if "violation_tags" not in records_df.columns:
        return pd.DataFrame(columns=[cluster_col, "dominant_violation_tag"])

    exploded = records_df[[cluster_col, "violation_tags"]].explode("violation_tags").copy()
    exploded["violation_tag"] = exploded["violation_tags"].map(normalize_tag)
    exploded = exploded[exploded["violation_tag"].fillna("").ne("")].copy()

    if len(exploded) == 0:
        return pd.DataFrame(columns=[cluster_col, "dominant_violation_tag"])

    dom_tag = (
        exploded.groupby(cluster_col)["violation_tag"]
        .apply(lambda s: dominant_label_from_series(s, default=""))
        .reset_index(name="dominant_violation_tag")
    )
    dom_tag["dominant_violation_tag"] = dom_tag["dominant_violation_tag"].fillna("")
    return dom_tag


def apply_osmnx_context(cluster_table):
    """
    Adds road_class, lane_count, carriageway_width_m, link_length_m,
    junction_degree, betweenness_centrality, geometry_source.

    FIX: real OSMnx-derived values must take priority over the hardcoded
    defaults. The previous version pre-populated cluster_table with default
    values for these columns BEFORE merging road_context_df (which carries
    the same column names), then called
        cluster_table[c].combine_first(cluster_table[ctx_col])
    combine_first only fills NaNs in the LEFT series -- since the left
    column (`c`) was already fully populated with non-null defaults, the
    freshly-computed `_ctx` values were always discarded, regardless of
    whether OSMnx actually returned real geometry. Every cluster silently
    kept defaults (road_class="road", carriageway_width_m=7.0,
    junction_degree=4, betweenness_centrality=0.0) even when ENABLE_OSMNX
    was on and the graph download succeeded.

    The fix: do not pre-populate defaults before the merge. Merge first,
    then fill only genuine NaNs (i.e. points whose coordinates had no
    match in road_context_df, or rows where OSMnx itself fell back) with
    defaults -- so the real OSMnx value wins whenever it exists.
    """
    cluster_table = cluster_table.copy()

    # Round coordinates for cache / de-dup -- done BEFORE any default columns exist.
    cluster_table["lat_key"] = pd.to_numeric(cluster_table["centroid_lat"], errors="coerce").round(4)
    cluster_table["lon_key"] = pd.to_numeric(cluster_table["centroid_lon"], errors="coerce").round(4)

    unique_coords = (
        cluster_table[["lat_key", "lon_key"]]
        .dropna(subset=["lat_key", "lon_key"])
        .drop_duplicates()
        .copy()
    )

    road_context_df = pd.DataFrame()
    if ENABLE_OSMNX and ox is not None and nx is not None and len(unique_coords) > 0:
        road_context_df = road_context_for_points(unique_coords)

    if len(road_context_df) > 0:
        # Merge FIRST. cluster_table has no road_class/lane_count/etc. columns
        # yet at this point, so there is no name collision and no suffixing --
        # the real OSMnx values land directly under their plain column names.
        cluster_table = cluster_table.merge(road_context_df, on=["lat_key", "lon_key"], how="left")

    # Apply defaults AFTER the merge, filling only genuine NaNs:
    # rows with no OSMnx match at all (OSMnx disabled, graph download failed,
    # or this particular coordinate had no match in road_context_df).
    for c, default in [
        ("geometry_source", "fallback"),
        ("road_class", DEFAULT_ROAD_CLASS),
        ("lane_count", DEFAULT_LANES),
        ("carriageway_width_m", DEFAULT_CARRIAGEWAY_WIDTH_M),
        ("link_length_m", DEFAULT_LINK_LENGTH_M),
        ("junction_degree", 4),
        ("betweenness_centrality", 0.0),
    ]:
        if c not in cluster_table.columns:
            cluster_table[c] = default
        else:
            cluster_table[c] = cluster_table[c].fillna(default)

    cluster_table["road_class"] = cluster_table["road_class"].astype(str)
    cluster_table["lane_count"] = pd.to_numeric(cluster_table["lane_count"], errors="coerce").fillna(DEFAULT_LANES).astype(int)
    cluster_table["carriageway_width_m"] = pd.to_numeric(cluster_table["carriageway_width_m"], errors="coerce").fillna(DEFAULT_CARRIAGEWAY_WIDTH_M)
    cluster_table["link_length_m"] = pd.to_numeric(cluster_table["link_length_m"], errors="coerce").fillna(DEFAULT_LINK_LENGTH_M)
    cluster_table["junction_degree"] = pd.to_numeric(cluster_table["junction_degree"], errors="coerce").fillna(4.0)
    cluster_table["betweenness_centrality"] = pd.to_numeric(cluster_table["betweenness_centrality"], errors="coerce").fillna(0.0)
    cluster_table["geometry_source"] = cluster_table["geometry_source"].astype(str)
    return cluster_table


def enrich_mappls_address(cluster_table):
    """
    Uses Mappls reverse geocode on the top N clusters by delay. Produces a
    human-readable address label only -- by design this does not feed back
    into road_class / lane_count / capacity, since Mappls reverse-geocode
    returns an address string, not road-geometry attributes. Road geometry
    enrichment is OSMnx's job (apply_osmnx_context); Mappls' contribution
    here is display/traceability, not a scoring input.
    """
    cluster_table = cluster_table.copy()
    cluster_table["mappls_address"] = ""

    if not ENABLE_MAPPLS or requests is None or not MAPPLS_ACCESS_TOKEN or len(cluster_table) == 0:
        return cluster_table

    session = requests.Session()
    top_idx = cluster_table.sort_values("delay_minutes_per_vehicle", ascending=False).head(MAPPLS_ADDRESS_TOP_N).index.tolist()

    for n, idx in enumerate(top_idx, start=1):
        lat = safe_float(cluster_table.at[idx, "centroid_lat"])
        lon = safe_float(cluster_table.at[idx, "centroid_lon"])
        if not valid_coords(lat, lon):
            continue
        if n == 1 or n % 3 == 0:
            print(f"Mappls reverse-geocode {n}/{len(top_idx)} ...")
        out = reverse_geocode_mappls(lat, lon, session)
        cluster_table.at[idx, "mappls_address"] = out["address"] if out["ok"] else ""
    return cluster_table


# ============================================================
# Main
# ============================================================
def main():
    t0 = time.perf_counter()
    print("Loading input...")
    records_df, source = load_input()
    print(f"Input source: {source}")
    print(f"Rows loaded: {len(records_df):,}")

    cluster_col = standardize_cluster_col(records_df)
    vehicle_col = standardize_vehicle_col(records_df)
    vehicle_type_col = standardize_vehicle_type_col(records_df)

    records_df = records_df.copy()

    # Keep approved evidence if validation exists; otherwise keep all rows.
    if "validation_status_clean" in records_df.columns:
        records_df["validation_status_clean"] = records_df["validation_status_clean"].fillna("").astype(str).str.lower()
        records_df = records_df[records_df["validation_status_clean"].eq("approved")].copy()
    elif "validation_status" in records_df.columns:
        records_df["validation_status_clean"] = records_df["validation_status"].fillna("").astype(str).str.lower()
        records_df = records_df[records_df["validation_status_clean"].eq("approved")].copy()

    print(f"Approved rows: {len(records_df):,}")

    # Normalize / enrich record-level fields
    records_df = ensure_hotspot_unit(records_df)
    records_df = ensure_severity(records_df)
    records_df = derive_coords(records_df)

    if "latitude" in records_df.columns and "longitude" in records_df.columns:
        records_df["lat"] = pd.to_numeric(records_df["latitude"], errors="coerce")
        records_df["lon"] = pd.to_numeric(records_df["longitude"], errors="coerce")
    else:
        records_df = derive_coords(records_df)

    records_df["created_datetime_ist"] = parse_datetime_ist(records_df)
    records_df = records_df.dropna(subset=["created_datetime_ist"]).copy()
    records_df["created_datetime_ist_naive"] = records_df["created_datetime_ist"].dt.tz_localize(None)
    records_df["service_date"] = records_df["created_datetime_ist_naive"].dt.date
    records_df["cluster_week_start"] = week_start_monday(records_df["created_datetime_ist"])
    records_df["hour_ist"] = records_df["created_datetime_ist"].dt.hour
    records_df["is_peak_window"] = records_df["hour_ist"].isin(PEAK_HOURS).astype(int)

    if vehicle_type_col and vehicle_type_col in records_df.columns:
        records_df["vehicle_width_m"] = records_df[vehicle_type_col].map(vehicle_width_m).fillna(VEHICLE_WIDTH_M["UNKNOWN"])
    else:
        records_df["vehicle_width_m"] = VEHICLE_WIDTH_M["UNKNOWN"]

    records_df[vehicle_col] = records_df[vehicle_col].fillna("").astype(str).str.strip()
    records_df = records_df[records_df[vehicle_col].ne("")].copy()
    records_df = records_df[records_df[cluster_col].notna()].copy()
    records_df[cluster_col] = pd.to_numeric(records_df[cluster_col], errors="coerce")
    records_df = records_df[records_df[cluster_col].ne(-1)].copy()

    print(f"Clusters: {records_df[cluster_col].nunique():,}")

    # --------------------------------------------------------
    # Stage 4: implicit μ estimator
    # --------------------------------------------------------
    print("\nStage 4: estimating dwell time...")
    records_df, gaps_df, dwell_summary, dwell_by_type, weekly_counts, growth_df = compute_dwell_gaps(
        records_df, cluster_col, vehicle_col, vehicle_type_col
    )
    print(f"Valid dwell gaps: {len(gaps_df):,}")
    if len(dwell_summary):
        print(f"Overall mean dwell (minutes): {float(dwell_summary['mean_dwell_minutes'].mean()):.2f}")

    # --------------------------------------------------------
    # Cluster table
    # --------------------------------------------------------
    print("\nAggregating cluster metrics...")
    cluster_table = compute_cluster_table(records_df, cluster_col, vehicle_col, vehicle_type_col, dwell_summary, growth_df)

    # Merge optional stage 4 μ summary if available
    mu_summary = load_phase4_mu()
    if mu_summary is not None and len(mu_summary):
        if cluster_col not in mu_summary.columns and "st_dbscan_cluster_id" in mu_summary.columns:
            mu_summary = mu_summary.rename(columns={"st_dbscan_cluster_id": cluster_col})
        keep_cols = [cluster_col]
        for c in ["cluster_label", "gap_count", "mean_dwell_minutes", "median_dwell_minutes", "std_dwell_minutes", "mu_departures_per_hour"]:
            if c in mu_summary.columns:
                keep_cols.append(c)
        mu_summary = mu_summary[keep_cols].drop_duplicates(cluster_col)
        cluster_table = cluster_table.merge(mu_summary, on=cluster_col, how="left", suffixes=("", "_mu"))

        for c in ["mean_dwell_minutes", "mu_departures_per_hour"]:
            mu_c = f"{c}_mu"
            if mu_c in cluster_table.columns:
                if c in cluster_table.columns:
                    cluster_table[c] = pd.to_numeric(cluster_table[c], errors="coerce").combine_first(pd.to_numeric(cluster_table[mu_c], errors="coerce"))
                else:
                    cluster_table[c] = cluster_table[mu_c]
                cluster_table.drop(columns=[mu_c], inplace=True)

    # Final dwell cleanup
    cluster_table["mean_dwell_minutes"] = pd.to_numeric(cluster_table["mean_dwell_minutes"], errors="coerce").fillna(DEFAULT_MEAN_DWELL_MINUTES)
    cluster_table["mu_departures_per_hour"] = pd.to_numeric(cluster_table["mu_departures_per_hour"], errors="coerce").fillna(60.0 / cluster_table["mean_dwell_minutes"].clip(lower=EPS))

    # --------------------------------------------------------
    # Stage 5a: M/M/∞ queueing model
    # --------------------------------------------------------
    print("\nStage 5a: computing blocking vehicles...")
    cluster_table["blocking_vehicles_L"] = cluster_table["lambda_hr_peak_window"] / (cluster_table["mu_departures_per_hour"] + EPS)

    # --------------------------------------------------------
    # Vehicle mix / repeated offenders
    # --------------------------------------------------------
    print("Building vehicle mix and chronic-offender list...")
    vehicle_mix = compute_vehicle_mix(records_df, cluster_col, vehicle_col, vehicle_type_col)
    chronic_offenders = compute_repeat_offenders(records_df, cluster_col, vehicle_col, vehicle_type_col)

    repeat_summary = (
        records_df.groupby([cluster_col, vehicle_col])
        .size()
        .reset_index(name="vehicle_cluster_count")
    )
    repeat_summary["repeat_flag_2plus"] = (repeat_summary["vehicle_cluster_count"] >= 2).astype(int)
    repeat_summary["chronic_flag_5plus"] = (repeat_summary["vehicle_cluster_count"] >= 5).astype(int)

    repeat_agg = (
        repeat_summary.groupby(cluster_col)
        .agg(
            repeat_vehicle_count_2plus=("repeat_flag_2plus", "sum"),
            chronic_vehicle_count_5plus=("chronic_flag_5plus", "sum"),
        )
        .reset_index()
    )
    cluster_table = cluster_table.merge(repeat_agg, on=cluster_col, how="left")
    cluster_table["repeat_vehicle_count_2plus"] = cluster_table["repeat_vehicle_count_2plus"].fillna(0).astype(int)
    cluster_table["chronic_vehicle_count_5plus"] = cluster_table["chronic_vehicle_count_5plus"].fillna(0).astype(int)

    # Dominant violation tag
    dom_tag = compute_dominant_violation_tag(records_df, cluster_col)
    if len(dom_tag):
        cluster_table = cluster_table.merge(dom_tag, on=cluster_col, how="left")
    else:
        cluster_table["dominant_violation_tag"] = ""
    cluster_table["dominant_violation_tag"] = cluster_table["dominant_violation_tag"].fillna("")

    # --------------------------------------------------------
    # Stage 5b: Zhao capacity reduction model
    # --------------------------------------------------------
    print("\nStage 5b: road-network context + capacity loss...")
    cluster_table = apply_osmnx_context(cluster_table)

    cluster_table["base_saturation_per_lane_pcu_hr"] = cluster_table["road_class"].map(base_capacity_per_lane).fillna(1600.0)
    cluster_table["base_capacity_pcu_hr"] = cluster_table["base_saturation_per_lane_pcu_hr"] * pd.to_numeric(cluster_table["lane_count"], errors="coerce").fillna(DEFAULT_LANES)

    cluster_table["carriageway_width_m"] = pd.to_numeric(cluster_table["carriageway_width_m"], errors="coerce").fillna(DEFAULT_CARRIAGEWAY_WIDTH_M)
    cluster_table["vehicle_width_avg_m"] = pd.to_numeric(cluster_table["vehicle_width_avg_m"], errors="coerce").fillna(VEHICLE_WIDTH_M["UNKNOWN"])
    cluster_table["blocking_vehicles_L"] = pd.to_numeric(cluster_table["blocking_vehicles_L"], errors="coerce").fillna(0.0)

    cluster_table["blocked_width_fraction"] = (
        cluster_table["blocking_vehicles_L"] * cluster_table["vehicle_width_avg_m"]
    ) / (cluster_table["carriageway_width_m"] + EPS)

    cluster_table["blocked_width_fraction"] = cluster_table["blocked_width_fraction"].clip(lower=0.0, upper=0.95)
    cluster_table["reduced_capacity_pcu_hr"] = cluster_table["base_capacity_pcu_hr"] * (1.0 - cluster_table["blocked_width_fraction"])
    cluster_table["reduced_capacity_pcu_hr"] = cluster_table["reduced_capacity_pcu_hr"].clip(lower=cluster_table["base_capacity_pcu_hr"] * 0.10)
    cluster_table["capacity_loss_pct"] = 1.0 - safe_ratio(cluster_table["reduced_capacity_pcu_hr"], cluster_table["base_capacity_pcu_hr"])

    # --------------------------------------------------------
    # Stage 5c: Modified BPR delay
    # --------------------------------------------------------
    print("Stage 5c: BPR delay computation...")
    cluster_table["free_flow_speed_kmh"] = cluster_table["road_class"].apply(road_speed_kmh)
    cluster_table["free_flow_time_min"] = cluster_table["link_length_m"] * 60.0 / (cluster_table["free_flow_speed_kmh"] * 1000.0 + EPS)

    cluster_table["V_over_C0"] = cluster_table["lambda_hr_peak_window"] / (cluster_table["base_capacity_pcu_hr"] + EPS)
    cluster_table["V_over_Cp"] = cluster_table["lambda_hr_peak_window"] / (cluster_table["reduced_capacity_pcu_hr"] + EPS)

    cluster_table["delay_minutes_per_vehicle"] = (
        cluster_table["free_flow_time_min"] * ALPHA_BPR *
        (np.power(cluster_table["V_over_Cp"], BETA_BPR) - np.power(cluster_table["V_over_C0"], BETA_BPR))
    )
    cluster_table["delay_minutes_per_vehicle"] = (
        pd.to_numeric(cluster_table["delay_minutes_per_vehicle"], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .clip(lower=0.0)
    )

    # --------------------------------------------------------
    # Final cleanup and ranking
    # --------------------------------------------------------
    print("\nFinalizing stage 5 outputs...")
    cluster_table["growth_first_half_mean"] = pd.to_numeric(cluster_table["growth_first_half_mean"], errors="coerce").fillna(0.0)
    cluster_table["growth_second_half_mean"] = pd.to_numeric(cluster_table["growth_second_half_mean"], errors="coerce").fillna(0.0)
    cluster_table["growth_pct_change"] = pd.to_numeric(cluster_table["growth_pct_change"], errors="coerce").fillna(0.0)
    cluster_table["growth_multiplier"] = pd.to_numeric(cluster_table["growth_multiplier"], errors="coerce").fillna(1.0).clip(lower=0.1)

    cluster_table["criticality_factor"] = (
        1.0
        + 0.5 * minmax(pd.to_numeric(cluster_table["junction_degree"], errors="coerce").fillna(0))
        + 0.5 * minmax(pd.to_numeric(cluster_table["betweenness_centrality"], errors="coerce").fillna(0))
    )
    cluster_table["criticality_factor"] = pd.to_numeric(cluster_table["criticality_factor"], errors="coerce").fillna(1.0)

    # Stage 5 rank based on delay
    cluster_table = cluster_table.sort_values(
        ["delay_minutes_per_vehicle", "lambda_hr_peak_window", "blocking_vehicles_L"],
        ascending=[False, False, False]
    ).reset_index(drop=True)
    cluster_table["stage5_rank"] = np.arange(1, len(cluster_table) + 1)

    # Cleanup labels and coordinates
    cluster_table["cluster_label"] = cluster_table["cluster_label"].fillna("").astype(str).str.strip()
    cluster_table.loc[cluster_table["cluster_label"].eq(""), "cluster_label"] = "CLUSTER::" + cluster_table[cluster_col].astype(str)

    cluster_table["centroid_lat"] = pd.to_numeric(cluster_table["centroid_lat"], errors="coerce")
    cluster_table["centroid_lon"] = pd.to_numeric(cluster_table["centroid_lon"], errors="coerce")
    cluster_table["lat"] = cluster_table["centroid_lat"]
    cluster_table["lon"] = cluster_table["centroid_lon"]

    # Optional Mappls reverse geocode labels for top hotspots only
    cluster_table = enrich_mappls_address(cluster_table)

    # Merge Mappls address back to records
    if "mappls_address" not in records_df.columns:
        records_df["mappls_address"] = ""
    records_df = records_df.merge(
        cluster_table[[cluster_col, "mappls_address"]],
        on=cluster_col,
        how="left",
        suffixes=("", "_stage5")
    )
    if "mappls_address_stage5" in records_df.columns:
        records_df["mappls_address"] = records_df["mappls_address"].combine_first(records_df["mappls_address_stage5"])
        records_df.drop(columns=["mappls_address_stage5"], inplace=True)

    # --------------------------------------------------------
    # Outputs
    # --------------------------------------------------------
    print("\nSaving outputs...")

    cluster_table.to_csv(OUT_DIR / "phase5_cluster_capacity_loss.csv", index=False)
    cluster_table.to_csv(OUT_DIR / "phase5_priority_table_full.csv", index=False)

    stage5_handoff_cols = [
        cluster_col, "cluster_label", "stage5_rank", "records_total", "distinct_days",
        "peak_window_records", "records_peak_hour", "lambda_hr_peak_window",
        "lambda_hr_peak_hour", "mean_dwell_minutes", "mu_departures_per_hour",
        "blocking_vehicles_L", "road_class", "lane_count", "carriageway_width_m",
        "base_saturation_per_lane_pcu_hr", "base_capacity_pcu_hr",
        "reduced_capacity_pcu_hr", "capacity_loss_pct", "free_flow_time_min",
        "delay_minutes_per_vehicle", "growth_first_half_mean",
        "growth_second_half_mean", "growth_pct_change", "growth_multiplier",
        "criticality_factor", "dominant_vehicle_type", "dominant_violation_tag",
        "geometry_source", "repeat_vehicle_count_2plus", "chronic_vehicle_count_5plus",
        "unique_vehicles", "unique_vehicle_types", "centroid_lat", "centroid_lon",
        "mappls_address"
    ]
    stage5_handoff_cols = [c for c in stage5_handoff_cols if c in cluster_table.columns]
    cluster_table[stage5_handoff_cols].to_csv(OUT_DIR / "phase5_stage5_handoff.csv", index=False)

    records_df.to_csv(OUT_DIR / "phase5_enriched_records.csv", index=False)
    vehicle_mix.to_csv(OUT_DIR / "phase5_vehicle_mix.csv", index=False)
    weekly_counts.to_csv(OUT_DIR / "phase5_weekly_cluster_counts.csv", index=False)
    growth_df.to_csv(OUT_DIR / "phase5_growth_summary.csv", index=False)
    chronic_offenders.to_csv(OUT_DIR / "phase5_chronic_offenders.csv", index=False)

    if len(dwell_summary):
        dwell_summary.to_csv(OUT_DIR / "phase5_cluster_mu_summary.csv", index=False)
    if len(dwell_by_type):
        dwell_by_type.to_csv(OUT_DIR / "phase5_cluster_mu_by_vehicle_type.csv", index=False)

    road_context_export_cols = [
        "geometry_source", "road_class", "lane_count", "carriageway_width_m",
        "link_length_m", "junction_degree", "betweenness_centrality"
    ]
    export_cols = [cluster_col, "cluster_label"] + [c for c in road_context_export_cols if c in cluster_table.columns]
    cluster_table[export_cols].to_csv(OUT_DIR / "phase5_road_context_cache.csv", index=False)

    elapsed = time.perf_counter() - t0
    print("Stage 5 complete")
    print("Clusters scored:", len(cluster_table))
    print("Chronic offenders:", len(chronic_offenders))
    print("Outputs saved to:", OUT_DIR.resolve())
    print(f"Elapsed: {elapsed:.1f} sec")

    print("\nTop 10 clusters by delay:")
    top_show_cols = [
        "stage5_rank", cluster_col, "cluster_label", "delay_minutes_per_vehicle",
        "lambda_hr_peak_window", "mu_departures_per_hour", "blocking_vehicles_L",
        "capacity_loss_pct", "criticality_factor", "growth_multiplier"
    ]
    top_show_cols = [c for c in top_show_cols if c in cluster_table.columns]
    if len(cluster_table):
        print(cluster_table[top_show_cols].head(10).to_string(index=False))

    print("\nRoad-context sanity check (should show real variance, not constants):")
    sanity_cols = ["road_class", "carriageway_width_m", "base_capacity_pcu_hr",
                   "junction_degree", "betweenness_centrality", "criticality_factor", "geometry_source"]
    sanity_cols = [c for c in sanity_cols if c in cluster_table.columns]
    if sanity_cols:
        print(cluster_table[sanity_cols].describe(include="all"))
        if "geometry_source" in cluster_table.columns:
            print("\ngeometry_source counts:")
            print(cluster_table["geometry_source"].value_counts())


if __name__ == "__main__":
    main()

True
True
Loading input...
Input source: content\phase4_outputs_2\phase4_merged_with_prior_scores.csv
Rows loaded: 92,332
Approved rows: 92,332
Clusters: 798

Stage 4: estimating dwell time...
Valid dwell gaps: 2,122
Overall mean dwell (minutes): 84.68

Aggregating cluster metrics...

Stage 5a: computing blocking vehicles...
Building vehicle mix and chronic-offender list...

Stage 5b: road-network context + capacity loss...
[OSMnx] bbox download failed (graph_from_bbox() takes 1 positional argument but 4 positional arguments (and 3 keyword-only arguments) were given), trying city-level fallback...
[OSMnx] Downloaded graph via place name
[OSMnx] Graph saved to cache: content\phase5_outputs_2\osmnx_graph_cache.graphml
[OSMnx] Graph has 155,368 nodes
[OSMnx] Approximate betweenness centrality (k=200)...
Road context enrichment for 766 unique coordinates...
  road context 1/766
  road context 25/766
  road context 50/766
  road context 75/766
  road context 100/766
  road context 125/766
 

In [2]:
import osmnx as ox, networkx as nx, os
print("osmnx import OK:", ox.__version__)
print("networkx import OK:", nx.__version__)
print("ENABLE_OSMNX_PHASE5 =", os.environ.get("ENABLE_OSMNX_PHASE5"))
try:
    G = ox.graph_from_place("Bengaluru, Karnataka, India", network_type="drive", simplify=True, retain_all=False)
    print("Graph download OK:", G.number_of_nodes(), "nodes")
except Exception as e:
    print("Graph download FAILED:", repr(e))

osmnx import OK: 2.1.0
networkx import OK: 3.2.1
ENABLE_OSMNX_PHASE5 = None
Graph download OK: 155368 nodes


In [46]:
cluster_table[
[
'lambda_hr_peak_window',
'mu_departures_per_hour',
'blocking_vehicles_L',

'carriageway_width_m',
'capacity_loss_pct',
'base_capacity_pcu_hr',
'criticality_factor',
'growth_multiplier',
'delay_minutes_per_vehicle'
]
].describe()

,lambda_hr_peak_window,mu_departures_per_hour,blocking_vehicles_L,carriageway_width_m,capacity_loss_pct,base_capacity_pcu_hr,criticality_factor,growth_multiplier,delay_minutes_per_vehicle
count,798.000000,239.000000,239.000000,798.0,2.390000e+02,7.980000e+02,798.0,798.000000,2.390000e+02
mean,0.099109,7.013810,0.429637,7.0,4.879043e-02,3.200000e+03,1.0,3.576545,4.022229e-06
std,0.808595,14.683670,2.417438,0.0,1.322412e-01,9.555684e-12,0.0,10.216310,4.792070e-05
min,0.000000,0.119048,0.000000,7.0,3.125278e-13,3.200000e+03,1.0,1.000000,0.000000e+00
25%,0.009434,0.506338,0.005629,7.0,1.040786e-03,3.200000e+03,1.0,1.000000,1.249346e-14
50%,0.024528,1.313869,0.042788,7.0,7.755840e-03,3.200000e+03,1.0,1.000000,2.797140e-13
75%,0.048585,4.600686,0.181247,7.0,3.203760e-02,3.200000e+03,1.0,1.500000,5.877389e-12
max,19.007547,60.000000,32.101127,7.0,9.000000e-01,3.200000e+03,1.0,183.500000,6.986447e-04


In [24]:
import ast
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from sklearn.ensemble import IsolationForest
    from sklearn.preprocessing import RobustScaler
except Exception as e:
    raise ImportError("scikit-learn is required for Layer A emerging hotspot detection.") from e

warnings.filterwarnings("ignore")

# =========================
# Config
# =========================
INPUT_PATHS = [
    Path("content/phase5_outputs_2/phase5_enriched_records.csv"),
    Path("phase5_outputs_2/phase5_enriched_records.csv"),
    Path("/content/phase5_outputs_2/phase5_enriched_records.csv"),
    Path("content/phase4_outputs_2/phase4_merged_with_prior_scores.csv"),
    Path("phase4_outputs_2/phase4_merged_with_prior_scores.csv"),
    Path("/content/phase4_outputs_2/phase4_merged_with_prior_scores.csv"),
    Path("content/phase3_outputs_2/phase3_clustered_dataset.csv"),
    Path("phase3_outputs_2/phase3_clustered_dataset.csv"),
    Path("/content/phase3_outputs_2/phase3_clustered_dataset.csv"),
]

OUT_DIR = Path("content/layer_a_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EPS = 1e-9
TOP_N = 50
RECENT_WINDOW_WEEKS = 4
ALERT_FRACTIONS = (0.80, 0.60, 0.40)

# =========================
# Helpers
# =========================
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def parse_listlike(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    s = str(value).strip()
    if not s:
        return []
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple)):
            return list(parsed)
        return [parsed]
    except Exception:
        if s.startswith("[") and s.endswith("]"):
            s = s[1:-1]
        parts = [p.strip().strip("'").strip('"') for p in s.split(",")]
        return [p for p in parts if p]


def load_first_existing(paths):
    for p in paths:
        if p.exists():
            return pd.read_csv(p, low_memory=False), p
    return None, None


def standardize_cluster_col(df):
    for c in ["st_dbscan_cluster_id", "cluster_id", "dbscan_cluster_id"]:
        if c in df.columns:
            return c
    raise ValueError("No cluster id column found.")


def standardize_vehicle_col(df):
    for c in ["canonical_vehicle_number", "vehicle_number", "updated_vehicle_number"]:
        if c in df.columns:
            return c
    return None


def standardize_vehicle_type_col(df):
    for c in ["canonical_vehicle_type", "vehicle_type", "updated_vehicle_type"]:
        if c in df.columns:
            return c
    return None


def ensure_label_column(df):
    if df is None or len(df) == 0:
        return df
    df = df.copy()
    if "cluster_label" in df.columns:
        df["cluster_label"] = df["cluster_label"].fillna("").astype(str).str.strip()
        return df
    if "hotspot_unit" in df.columns:
        df["cluster_label"] = df["hotspot_unit"].fillna("").astype(str).str.strip()
        return df
    if "dominant_junction_name" in df.columns:
        df["cluster_label"] = df["dominant_junction_name"].fillna("").astype(str).str.strip()
        return df
    if "st_dbscan_cluster_id" in df.columns:
        df["cluster_label"] = "CLUSTER::" + df["st_dbscan_cluster_id"].astype(str)
        return df
    df["cluster_label"] = "UNKNOWN"
    return df


def derive_coords(df):
    if df is None or len(df) == 0:
        return df
    df = df.copy()
    if "lat" in df.columns and "lon" in df.columns:
        df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
        df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    elif {"centroid_lat", "centroid_lon"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["centroid_lat"], errors="coerce")
        df["lon"] = pd.to_numeric(df["centroid_lon"], errors="coerce")
    elif {"latitude_mean", "longitude_mean"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["latitude_mean"], errors="coerce")
        df["lon"] = pd.to_numeric(df["longitude_mean"], errors="coerce")
    elif {"latitude", "longitude"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["latitude"], errors="coerce")
        df["lon"] = pd.to_numeric(df["longitude"], errors="coerce")
    else:
        df["lat"] = np.nan
        df["lon"] = np.nan
    return df


def parse_datetime_ist(df):
    if "created_datetime_ist" in df.columns:
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce", utc=True)
        if ts.notna().any():
            return ts.dt.tz_convert("Asia/Kolkata")
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce")
        return ts.dt.tz_localize("Asia/Kolkata", nonexistent="NaT", ambiguous="NaT")

    if "created_datetime_parsed" in df.columns:
        ts = pd.to_datetime(df["created_datetime_parsed"], errors="coerce", utc=True)
        return ts.dt.tz_convert("Asia/Kolkata")

    if "created_datetime" not in df.columns:
        return pd.Series(pd.NaT, index=df.index)

    ts = pd.to_datetime(df["created_datetime"], errors="coerce", utc=True)
    return ts.dt.tz_convert("Asia/Kolkata")


def week_start_monday(series):
    dt = pd.to_datetime(series, errors="coerce", utc=True).dt.tz_convert("Asia/Kolkata")
    week_start = dt.dt.normalize() - pd.to_timedelta(dt.dt.weekday, unit="D")
    return week_start.dt.tz_localize(None)


def normalize_tag(tag):
    return clean_text(tag).upper().replace("&", "AND").strip()


def minmax(s):
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    valid = s.dropna()
    if len(valid) == 0 or valid.nunique(dropna=True) <= 1:
        return pd.Series(np.zeros(len(s)), index=s.index, dtype=float)
    mn = valid.min()
    mx = valid.max()
    return (s.fillna(mn) - mn) / (mx - mn + EPS)


def robust_norm(s):
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    valid = s.dropna()
    if len(valid) == 0:
        return pd.Series(np.zeros(len(s)), index=s.index, dtype=float)
    lo = valid.quantile(0.05)
    hi = valid.quantile(0.95)
    if pd.isna(lo) or pd.isna(hi) or abs(float(hi) - float(lo)) < EPS:
        return pd.Series(np.zeros(len(s)), index=s.index, dtype=float)
    return ((s.fillna(lo) - lo) / (hi - lo + EPS)).clip(0.0, 1.0)


def safe_slope(y):
    y = np.asarray(y, dtype=float)
    if len(y) < 2:
        return 0.0
    x = np.arange(len(y), dtype=float)
    try:
        return float(np.polyfit(x, y, 1)[0])
    except Exception:
        return 0.0


def dominant_label(series, default=""):
    s = pd.Series(series).dropna().astype(str).str.strip()
    s = s[s.ne("")]
    if s.empty:
        return default
    m = s.mode()
    if not m.empty:
        return m.iloc[0]
    return s.iloc[0]


def alert_level_from_score(score, q80, q60, q40):
    if score >= q80:
        return "Emerging-Critical"
    if score >= q60:
        return "Emerging-High"
    if score >= q40:
        return "Emerging-Watch"
    return "Stable"


def load_source_data():
    df, path = load_first_existing(INPUT_PATHS)
    if df is None:
        raise FileNotFoundError("No record-level source file found for Layer A.")
    return df, path


# =========================
# Layer A: Emerging Hotspot Detection
# =========================
def build_layer_a_emerging_hotspots():
    raw_df, source_path = load_source_data()
    df = raw_df.copy()

    cluster_col = standardize_cluster_col(df)
    vehicle_col = standardize_vehicle_col(df)
    vehicle_type_col = standardize_vehicle_type_col(df)

    if "validation_status" in df.columns:
        df["validation_status_clean"] = df["validation_status"].fillna("").astype(str).str.lower()
        df = df[df["validation_status_clean"].eq("approved")].copy()

    df = ensure_label_column(df)
    df = derive_coords(df)

    if "violation_tags" not in df.columns and "violation_type" in df.columns:
        df["violation_tags"] = df["violation_type"].apply(parse_listlike)
    elif "violation_tags" in df.columns:
        df["violation_tags"] = df["violation_tags"].apply(parse_listlike)
    else:
        df["violation_tags"] = [[] for _ in range(len(df))]

    df["created_datetime_ist"] = parse_datetime_ist(df)
    df = df.dropna(subset=["created_datetime_ist"]).copy()
    df["created_datetime_ist_naive"] = df["created_datetime_ist"].dt.tz_localize(None)
    df["service_date"] = df["created_datetime_ist_naive"].dt.date
    df["week_start"] = week_start_monday(df["created_datetime_ist"])
    df["week_start"] = pd.to_datetime(df["week_start"], errors="coerce")
    df["is_peak_window"] = df["created_datetime_ist"].dt.hour.between(8, 12, inclusive="both").astype(int)

    if "severity_score" not in df.columns:
        severity_rules = {
            5: {
                "DOUBLE PARKING",
                "NEAR ROAD CROSSING",
                "NEAR TRAFFIC LIGHT",
                "NEAR ZEBRA CROSSING",
                "NEAR TRAFFIC LIGHT / ZEBRA CROSSING",
                "NEAR TRAFFIC LIGHT/ZEBRA CROSSING",
            },
            4: {
                "PARKING IN MAIN ROAD",
                "NEAR BUS STOP",
                "NEAR SCHOOL",
                "NEAR HOSPITAL",
                "OPPOSITE ANOTHER VEHICLE",
            },
            3: {"PARKING ON FOOTPATH"},
            2: {"WRONG PARKING", "PARKING OTHER THAN BUS STOP"},
            1: {"NO PARKING", "NO PARKING (GENERIC)"},
        }

        def severity_from_tags(tags):
            if not tags:
                return 1
            normalized = [normalize_tag(t) for t in tags]
            score = 1
            for sev in sorted(severity_rules.keys(), reverse=True):
                vocab = severity_rules[sev]
                if any(any(v == tag or v in tag for v in vocab) for tag in normalized):
                    score = sev
                    break
            return score

        df["severity_score"] = df["violation_tags"].apply(severity_from_tags)
    else:
        df["severity_score"] = pd.to_numeric(df["severity_score"], errors="coerce").fillna(1.0)

    agg_kwargs = {
        "records_total": (cluster_col, "size"),
        "distinct_days": ("service_date", "nunique"),
        "severity_sum": ("severity_score", "sum"),
        "severity_mean": ("severity_score", "mean"),
        "peak_window_records": ("is_peak_window", "sum"),
        "unique_vehicles": (vehicle_col, "nunique") if vehicle_col and vehicle_col in df.columns else (cluster_col, "size"),
        "unique_vehicle_types": (vehicle_type_col, "nunique") if vehicle_type_col and vehicle_type_col in df.columns else (cluster_col, "size"),
        "lat": ("lat", "mean"),
        "lon": ("lon", "mean"),
        "cluster_label": ("cluster_label", dominant_label),
    }

    cluster = df.groupby(cluster_col).agg(**agg_kwargs).reset_index()
    cluster["distinct_days"] = pd.to_numeric(cluster["distinct_days"], errors="coerce").fillna(1).clip(lower=1)

    weekly = (
        df.groupby([cluster_col, "week_start"])
        .size()
        .reset_index(name="weekly_count")
        .sort_values([cluster_col, "week_start"])
    )

    trend_rows = []
    for cid, g in weekly.groupby(cluster_col):
        counts = g["weekly_count"].to_numpy(dtype=float)
        weeks = len(counts)
        if weeks == 0:
            first_half = second_half = 0.0
            recent_mean = prev_mean = 0.0
            slope = 0.0
            peak_ratio = 0.0
        else:
            mid = max(1, weeks // 2)
            first_half = float(counts[:mid].mean()) if len(counts[:mid]) else 0.0
            second_half = float(counts[mid:].mean()) if len(counts[mid:]) else first_half

            if weeks >= 2 * RECENT_WINDOW_WEEKS:
                recent_mean = float(counts[-RECENT_WINDOW_WEEKS:].mean())
                prev_mean = float(counts[-2 * RECENT_WINDOW_WEEKS:-RECENT_WINDOW_WEEKS].mean())
            elif weeks >= 2:
                recent_mean = second_half
                prev_mean = first_half
            else:
                recent_mean = prev_mean = float(counts.mean())

            slope = safe_slope(counts)
            peak_ratio = float(counts.max() / (counts.mean() + EPS)) if counts.mean() > 0 else float(counts.max())

        growth_pct = (second_half - first_half) / (first_half + EPS) if first_half > 0 else second_half
        recent_vs_prev = (recent_mean + 1.0) / (prev_mean + 1.0)
        resurgence_score = recent_vs_prev * (1.0 + max(0.0, growth_pct))
        growth_multiplier = 1.0 + max(0.0, growth_pct)

        trend_rows.append(
            {
                cluster_col: cid,
                "weekly_first_half_mean": first_half,
                "weekly_second_half_mean": second_half,
                "weekly_recent_mean": recent_mean,
                "weekly_previous_mean": prev_mean,
                "weekly_trend_slope": slope,
                "peak_week_ratio": peak_ratio,
                "growth_pct_change": float(np.clip(growth_pct, -0.8, 3.0)),
                "growth_multiplier": float(np.clip(growth_multiplier, 0.2, 4.0)),
                "recent_vs_prev_ratio": recent_vs_prev,
                "resurgence_score": resurgence_score,
            }
        )

    trend_df = pd.DataFrame(trend_rows)
    cluster = cluster.merge(trend_df, on=cluster_col, how="left")

    for c in [
        "weekly_first_half_mean",
        "weekly_second_half_mean",
        "weekly_recent_mean",
        "weekly_previous_mean",
        "weekly_trend_slope",
        "peak_week_ratio",
        "growth_pct_change",
        "growth_multiplier",
        "recent_vs_prev_ratio",
        "resurgence_score",
        "records_total",
        "distinct_days",
        "severity_sum",
        "severity_mean",
        "peak_window_records",
        "unique_vehicles",
        "unique_vehicle_types",
    ]:
        if c in cluster.columns:
            cluster[c] = pd.to_numeric(cluster[c], errors="coerce")

    cluster["recent_volume_norm"] = robust_norm(np.log1p(cluster["weekly_recent_mean"].fillna(0.0)))
    cluster["growth_norm"] = robust_norm(cluster["growth_pct_change"].fillna(0.0))
    cluster["trend_slope_norm"] = robust_norm(cluster["weekly_trend_slope"].fillna(0.0))
    cluster["resurgence_norm"] = robust_norm(np.log1p(cluster["resurgence_score"].fillna(0.0)))
    cluster["peak_ratio_norm"] = robust_norm(cluster["peak_week_ratio"].fillna(0.0))
    cluster["severity_norm"] = robust_norm(cluster["severity_sum"].fillna(0.0))
    cluster["records_norm"] = robust_norm(np.log1p(cluster["records_total"].fillna(0.0)))

    if len(cluster) >= 5:
        features = cluster[[
            "recent_volume_norm",
            "growth_norm",
            "trend_slope_norm",
            "resurgence_norm",
            "peak_ratio_norm",
            "severity_norm",
            "records_norm",
        ]].fillna(0.0)

        scaler = RobustScaler()
        X = scaler.fit_transform(features)

        iso = IsolationForest(
            n_estimators=300,
            contamination="auto",
            random_state=42,
            bootstrap=False,
        )
        iso.fit(X)
        anomaly_raw = -iso.decision_function(X)
        cluster["anomaly_score_raw"] = anomaly_raw
        cluster["anomaly_score"] = minmax(pd.Series(anomaly_raw, index=cluster.index))
    else:
        cluster["anomaly_score_raw"] = 0.0
        cluster["anomaly_score"] = 0.0

    cluster["emerging_hotspot_score"] = (
        0.32 * cluster["anomaly_score"].fillna(0.0)
        + 0.26 * cluster["growth_norm"].fillna(0.0)
        + 0.18 * cluster["resurgence_norm"].fillna(0.0)
        + 0.12 * cluster["trend_slope_norm"].fillna(0.0)
        + 0.12 * cluster["recent_volume_norm"].fillna(0.0)
    ) * 100.0

    cluster["emerging_hotspot_score"] = cluster["emerging_hotspot_score"].clip(0.0, 100.0)
    cluster = cluster.sort_values(
        ["emerging_hotspot_score", "growth_pct_change", "weekly_recent_mean", "records_total"],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)

    cluster["emerging_rank"] = np.arange(1, len(cluster) + 1)

    q80 = cluster["emerging_hotspot_score"].quantile(ALERT_FRACTIONS[0]) if len(cluster) else 0.0
    q60 = cluster["emerging_hotspot_score"].quantile(ALERT_FRACTIONS[1]) if len(cluster) else 0.0
    q40 = cluster["emerging_hotspot_score"].quantile(ALERT_FRACTIONS[2]) if len(cluster) else 0.0

    cluster["alert_level"] = cluster["emerging_hotspot_score"].apply(
        lambda x: alert_level_from_score(x, q80, q60, q40)
    )
    cluster["is_emerging_hotspot"] = cluster["alert_level"].ne("Stable").astype(int)

    cluster["cluster_label"] = cluster["cluster_label"].fillna("").astype(str).str.strip()
    cluster.loc[cluster["cluster_label"].eq(""), "cluster_label"] = "CLUSTER::" + cluster[cluster_col].astype(str)

    cluster["priority_message"] = (
        cluster["cluster_label"]
        + " | "
        + cluster["alert_level"].astype(str)
        + " | Score="
        + cluster["emerging_hotspot_score"].round(2).astype(str)
    )

    weekly_out = weekly.merge(
        cluster[[cluster_col, "cluster_label", "emerging_hotspot_score", "alert_level", "emerging_rank"]],
        on=cluster_col,
        how="left",
    )

    top_alerts = cluster[cluster["is_emerging_hotspot"].eq(1)].copy()
    top_alerts = top_alerts.sort_values(
        ["alert_level", "emerging_hotspot_score", "growth_pct_change"],
        ascending=[True, False, False],
    ).head(TOP_N).reset_index(drop=True)

    cluster.to_csv(OUT_DIR / "layer_a_emerging_hotspots_full.csv", index=False)
    top_alerts.to_csv(OUT_DIR / "layer_a_emerging_hotspots_top.csv", index=False)
    weekly_out.to_csv(OUT_DIR / "layer_a_weekly_trends.csv", index=False)

    summary = pd.DataFrame(
        [
            {
                "input_source": str(source_path),
                "clusters_analyzed": len(cluster),
                "emerging_hotspots": int(cluster["is_emerging_hotspot"].sum()),
                "top_emerging_score": float(cluster["emerging_hotspot_score"].max()) if len(cluster) else 0.0,
                "mean_emerging_score": float(cluster["emerging_hotspot_score"].mean()) if len(cluster) else 0.0,
                "mean_growth_pct": float(cluster["growth_pct_change"].mean()) if len(cluster) else 0.0,
            }
        ]
    )
    summary.to_csv(OUT_DIR / "layer_a_summary.csv", index=False)

    print("Layer A complete")
    print("Input source:", source_path)
    print("Clusters analyzed:", len(cluster))
    print("Emerging hotspots:", int(cluster["is_emerging_hotspot"].sum()))
    print("Outputs saved to:", OUT_DIR.resolve())
    print("\nTop 10 emerging hotspots:")
    cols = [
        c for c in [
            "emerging_rank",
            cluster_col,
            "cluster_label",
            "alert_level",
            "emerging_hotspot_score",
            "anomaly_score",
            "growth_pct_change",
            "resurgence_score",
            "weekly_recent_mean",
            "records_total",
            "severity_sum",
        ] if c in top_alerts.columns
    ]
    if len(top_alerts):
        print(top_alerts[cols].head(10).to_string(index=False))

    return cluster, top_alerts, weekly_out, summary


# Run
layer_a_full, layer_a_top, layer_a_weekly, layer_a_summary = build_layer_a_emerging_hotspots()

Layer A complete
Input source: content\phase5_outputs_2\phase5_enriched_records.csv
Clusters analyzed: 798
Emerging hotspots: 479
Outputs saved to: D:\Flipkart_Hackathon\approach1\content\layer_a_outputs

Top 10 emerging hotspots:
 emerging_rank  st_dbscan_cluster_id                             cluster_label       alert_level  emerging_hotspot_score  anomaly_score  growth_pct_change  resurgence_score  weekly_recent_mean  records_total  severity_sum
             1                   206 BTP095 - Satellite Bus Stand, Mysore Road Emerging-Critical               95.508506       0.884944                3.0        208.025000                78.5            162           278
             2                   639           POLICE_STATION::HAL Old Airport Emerging-Critical               95.219163       0.957180                3.0         78.979167               110.5            233           464
             3                   290               POLICE_STATION::Kodigehalli Emerging-Critical       

In [25]:
import math
from pathlib import Path

import numpy as np
import pandas as pd

# =========================
# Layer C - Novel Feature Engineering
# ROP, TVS, VDI, Validation Uncertainty, Resurgence Score
# =========================

EPS = 1e-9
OUT_DIR = Path("content/layer_c_outputs_2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PHASE5_CLUSTER_PATHS = [
    Path("content/phase5_outputs_2/phase5_cluster_capacity_loss.csv"),
    Path("content/phase5_outputs_2/phase5_priority_table_full.csv"),
    Path("content/phase5_outputs_2/phase5_stage5_handoff.csv"),
    Path("phase5_outputs_2/phase5_cluster_capacity_loss.csv"),
    Path("phase5_outputs_2/phase5_priority_table_full.csv"),
    Path("phase5_outputs_2/phase5_stage5_handoff.csv"),
    Path("/content/phase5_outputs_2/phase5_cluster_capacity_loss.csv"),
    Path("/content/phase5_outputs_2/phase5_priority_table_full.csv"),
    Path("/content/phase5_outputs_2/phase5_stage5_handoff.csv"),
]

PHASE5_RECORD_PATHS = [
    Path("content/phase5_outputs_2/phase5_enriched_records.csv"),
    Path("phase5_outputs_2/phase5_enriched_records.csv"),
    Path("/content/phase5_outputs_2/phase5_enriched_records.csv"),
]

VALIDATION_SOURCE_PATHS = [
    Path("content/phase4_outputs_2/phase4_merged_with_prior_scores.csv"),
    Path("phase4_outputs_2/phase4_merged_with_prior_scores.csv"),
    Path("/content/phase4_outputs_2/phase4_merged_with_prior_scores.csv"),
    Path("content/phase3_outputs_2/phase3_clustered_dataset.csv"),
    Path("phase3_outputs_2/phase3_clustered_dataset.csv"),
    Path("/content/phase3_outputs_2/phase3_clustered_dataset.csv"),
    Path("content/phase5_outputs_2/phase5_enriched_records.csv"),
    Path("phase5_outputs_2/phase5_enriched_records.csv"),
    Path("/content/phase5_outputs_2/phase5_enriched_records.csv"),
]

SEVERITY_RULES = {
    5: {
        "DOUBLE PARKING",
        "NEAR ROAD CROSSING",
        "NEAR TRAFFIC LIGHT",
        "NEAR ZEBRA CROSSING",
        "NEAR TRAFFIC LIGHT / ZEBRA CROSSING",
        "NEAR TRAFFIC LIGHT/ZEBRA CROSSING",
    },
    4: {
        "PARKING IN MAIN ROAD",
        "NEAR BUS STOP",
        "NEAR SCHOOL",
        "NEAR HOSPITAL",
        "OPPOSITE ANOTHER VEHICLE",
    },
    3: {"PARKING ON FOOTPATH"},
    2: {"WRONG PARKING", "PARKING OTHER THAN BUS STOP"},
    1: {"NO PARKING", "NO PARKING (GENERIC)"},
}


def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def normalize_tag(tag):
    return clean_text(tag).upper().replace("&", "AND").strip()


def parse_listlike(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    s = str(value).strip()
    if not s:
        return []
    try:
        parsed = eval(s, {"__builtins__": {}})  # safe-ish for list-like literals only
        if isinstance(parsed, (list, tuple)):
            return list(parsed)
        return [parsed]
    except Exception:
        if s.startswith("[") and s.endswith("]"):
            s = s[1:-1]
        parts = [p.strip().strip("'").strip('"') for p in s.split(",")]
        return [p for p in parts if p]


def safe_numeric(s):
    return pd.to_numeric(s, errors="coerce")


def standardize_cluster_col(df):
    for c in ["st_dbscan_cluster_id", "cluster_id", "dbscan_cluster_id"]:
        if c in df.columns:
            return c
    raise ValueError("No cluster id column found.")


def standardize_vehicle_col(df):
    for c in ["canonical_vehicle_number", "vehicle_number", "updated_vehicle_number"]:
        if c in df.columns:
            return c
    return None


def standardize_vehicle_type_col(df):
    for c in ["canonical_vehicle_type", "vehicle_type", "updated_vehicle_type"]:
        if c in df.columns:
            return c
    return None


def ensure_label_column(df):
    df = df.copy()
    if "cluster_label" in df.columns:
        df["cluster_label"] = df["cluster_label"].fillna("").astype(str).str.strip()
        return df
    if "hotspot_unit" in df.columns:
        df["cluster_label"] = df["hotspot_unit"].fillna("").astype(str).str.strip()
        return df
    if "dominant_junction_name" in df.columns:
        df["cluster_label"] = df["dominant_junction_name"].fillna("").astype(str).str.strip()
        return df
    if "st_dbscan_cluster_id" in df.columns:
        df["cluster_label"] = "CLUSTER::" + df["st_dbscan_cluster_id"].astype(str)
        return df
    df["cluster_label"] = "UNKNOWN"
    return df


def derive_coords(df):
    df = df.copy()
    if "lat" in df.columns and "lon" in df.columns:
        df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
        df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    elif {"centroid_lat", "centroid_lon"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["centroid_lat"], errors="coerce")
        df["lon"] = pd.to_numeric(df["centroid_lon"], errors="coerce")
    elif {"latitude_mean", "longitude_mean"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["latitude_mean"], errors="coerce")
        df["lon"] = pd.to_numeric(df["longitude_mean"], errors="coerce")
    elif {"latitude", "longitude"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["latitude"], errors="coerce")
        df["lon"] = pd.to_numeric(df["longitude"], errors="coerce")
    else:
        df["lat"] = np.nan
        df["lon"] = np.nan
    return df


def parse_datetime_ist(df):
    if "created_datetime_ist" in df.columns:
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce", utc=True)
        if ts.notna().any():
            return ts.dt.tz_convert("Asia/Kolkata")
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce")
        return ts.dt.tz_localize("Asia/Kolkata", nonexistent="NaT", ambiguous="NaT")

    if "created_datetime_parsed" in df.columns:
        ts = pd.to_datetime(df["created_datetime_parsed"], errors="coerce", utc=True)
        return ts.dt.tz_convert("Asia/Kolkata")

    if "created_datetime" not in df.columns:
        return pd.Series(pd.NaT, index=df.index)

    ts = pd.to_datetime(df["created_datetime"], errors="coerce", utc=True)
    return ts.dt.tz_convert("Asia/Kolkata")


def week_start_monday(series):
    dt = pd.to_datetime(series, errors="coerce", utc=True).dt.tz_convert("Asia/Kolkata")
    week_start = dt.dt.normalize() - pd.to_timedelta(dt.dt.weekday, unit="D")
    return week_start.dt.tz_localize(None)


def severity_from_tags(tags):
    if not tags:
        return 1
    normalized = [normalize_tag(t) for t in tags]
    for sev in sorted(SEVERITY_RULES.keys(), reverse=True):
        vocab = SEVERITY_RULES[sev]
        if any(any(v == tag or v in tag for v in vocab) for tag in normalized):
            return sev
    return 1


def dominant_label(series, default=""):
    s = pd.Series(series).dropna().astype(str).str.strip()
    s = s[s.ne("")]
    if s.empty:
        return default
    m = s.mode()
    return m.iloc[0] if not m.empty else s.iloc[0]


def load_first_existing(paths):
    for p in paths:
        if p.exists():
            return pd.read_csv(p, low_memory=False), p
    return None, None


def minmax(s):
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    valid = s.dropna()
    if valid.empty or valid.nunique() <= 1:
        return pd.Series(np.zeros(len(s)), index=s.index, dtype=float)
    mn = valid.min()
    mx = valid.max()
    return (s.fillna(mn) - mn) / (mx - mn + EPS)


def smooth_norm(s, floor=0.10):
    return floor + (1.0 - floor) * minmax(s)


def safe_metric(val, digits=2):
    try:
        if pd.isna(val):
            return "0"
        val = float(val)
        if abs(val) >= 1000:
            return f"{val:,.0f}"
        return f"{val:,.{digits}f}"
    except Exception:
        return str(val)


def monotone_chain(points):
    pts = sorted(set(map(tuple, points)))
    if len(pts) <= 1:
        return pts

    def cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    lower = []
    for p in pts:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
            lower.pop()
        lower.append(p)

    upper = []
    for p in reversed(pts):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
            upper.pop()
        upper.append(p)

    return lower[:-1] + upper[:-1]


def polygon_area(poly):
    if len(poly) < 3:
        return 0.0
    x = np.array([p[0] for p in poly], dtype=float)
    y = np.array([p[1] for p in poly], dtype=float)
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))


def latlon_to_local_xy(lat, lon, ref_lat):
    r = 6371000.0
    x = np.deg2rad(lon) * r * np.cos(np.deg2rad(ref_lat))
    y = np.deg2rad(lat) * r
    return x, y


def estimate_cluster_area_m2(g):
    pts = g[["lat", "lon"]].dropna().drop_duplicates()
    if len(pts) == 0:
        return np.nan
    if len(pts) == 1:
        return 1000.0

    ref_lat = float(pts["lat"].mean())
    xy = np.array([latlon_to_local_xy(lat, lon, ref_lat) for lat, lon in pts[["lat", "lon"]].to_numpy()])

    if len(xy) >= 3:
        hull = monotone_chain(xy)
        area = polygon_area(hull)
        if area > 0:
            return max(area, 1000.0)

    x_span = float(np.ptp(xy[:, 0]))
    y_span = float(np.ptp(xy[:, 1]))
    padded_area = max((x_span + 50.0) * (y_span + 50.0), 1000.0)
    return padded_area


def compute_tvs(weekly_counts):
    counts = np.asarray(weekly_counts, dtype=float)
    if len(counts) <= 1:
        return 0.0
    mean = float(np.mean(counts))
    std = float(np.std(counts, ddof=0))
    return std / (mean + EPS)


def compute_resurgence_score(weekly_counts):
    counts = np.asarray(weekly_counts, dtype=float)
    if len(counts) == 0:
        return np.nan
    if len(counts) == 1:
        return 1.0

    recent_n = min(2, len(counts))
    recent_avg = float(np.mean(counts[-recent_n:]))
    baseline = counts[:-recent_n]
    if len(baseline) == 0:
        baseline = counts[:-1]
    baseline_avg = float(np.mean(baseline)) if len(baseline) else float(np.mean(counts))

    return float(np.clip((recent_avg + 1.0) / (baseline_avg + 1.0), 0.0, 5.0))


def load_layer_c_sources():
    cluster_df, cluster_src = load_first_existing(PHASE5_CLUSTER_PATHS)
    records_df, records_src = load_first_existing(PHASE5_RECORD_PATHS)
    review_df, review_src = load_first_existing(VALIDATION_SOURCE_PATHS)
    return cluster_df, cluster_src, records_df, records_src, review_df, review_src


# =========================
# Load data
# =========================
cluster_df, cluster_src, records_df, records_src, review_df, review_src = load_layer_c_sources()

if cluster_df is None:
    raise FileNotFoundError("Could not find the Stage 5 cluster table.")

cluster_col = standardize_cluster_col(cluster_df)
cluster_df = ensure_label_column(derive_coords(cluster_df.copy()))
cluster_df = cluster_df.drop_duplicates(subset=[cluster_col]).reset_index(drop=True)

if records_df is None:
    records_df = review_df.copy() if review_df is not None else cluster_df.copy()

# Build the analysis record table
records_df = records_df.copy()
if cluster_col not in records_df.columns:
    rec_cluster_col = standardize_cluster_col(records_df)
    if rec_cluster_col != cluster_col:
        records_df = records_df.rename(columns={rec_cluster_col: cluster_col})

records_df = ensure_label_column(records_df)
records_df = derive_coords(records_df)

if "validation_status" in records_df.columns:
    records_df["validation_status_clean"] = records_df["validation_status"].fillna("").astype(str).str.lower()
    approved = records_df[records_df["validation_status_clean"].eq("approved")].copy()
    if len(approved) == 0:
        approved = records_df.copy()
else:
    approved = records_df.copy()

approved["created_datetime_ist"] = parse_datetime_ist(approved)
approved = approved.dropna(subset=["created_datetime_ist"]).copy()
approved["service_date"] = approved["created_datetime_ist"].dt.tz_localize(None).dt.date
approved["week_start"] = week_start_monday(approved["created_datetime_ist"])

if "violation_tags" not in approved.columns and "violation_type" in approved.columns:
    approved["violation_tags"] = approved["violation_type"].apply(parse_listlike)
elif "violation_tags" in approved.columns:
    approved["violation_tags"] = approved["violation_tags"].apply(parse_listlike)
else:
    approved["violation_tags"] = [[] for _ in range(len(approved))]

approved["severity_score"] = approved["violation_tags"].apply(severity_from_tags)

vehicle_col = standardize_vehicle_col(approved)
vehicle_type_col = standardize_vehicle_type_col(approved)

if "hotspot_unit" not in approved.columns:
    if {"junction_name", "police_station"}.issubset(approved.columns):
        def make_hotspot_unit(row):
            junction = clean_text(row.get("junction_name", ""))
            if junction and junction.upper() != "NO JUNCTION":
                return f"JUNCTION::{junction}"
            station = clean_text(row.get("police_station", "UNKNOWN")) or "UNKNOWN"
            return f"POLICE_STATION::{station}"
        approved["hotspot_unit"] = approved.apply(make_hotspot_unit, axis=1)
    else:
        approved["hotspot_unit"] = "UNKNOWN"

# Global observation window
global_min_date = pd.to_datetime(approved["service_date"], errors="coerce").min()
global_max_date = pd.to_datetime(approved["service_date"], errors="coerce").max()
global_observation_days = max(int((global_max_date - global_min_date).days) + 1 if pd.notna(global_min_date) and pd.notna(global_max_date) else 1, 1)

# =========================
# Cluster-level feature engineering
# =========================
cluster_groups = approved.groupby(cluster_col, dropna=False)

base_metrics = cluster_groups.agg(
    records_total=(cluster_col, "size"),
    distinct_days=("service_date", "nunique"),
    unique_hotspots=("hotspot_unit", "nunique"),
    centroid_lat=("lat", "mean"),
    centroid_lon=("lon", "mean"),
    severity_sum=("severity_score", "sum"),
    severity_mean=("severity_score", "mean"),
).reset_index()

if vehicle_col is not None and vehicle_col in approved.columns:
    vcount = approved.groupby(cluster_col)[vehicle_col].nunique().reset_index(name="unique_vehicles")
    base_metrics = base_metrics.merge(vcount, on=cluster_col, how="left")
else:
    base_metrics["unique_vehicles"] = np.nan

if vehicle_type_col is not None and vehicle_type_col in approved.columns:
    vtype = approved.groupby(cluster_col)[vehicle_type_col].agg(dominant_label).reset_index(name="dominant_vehicle_type")
    base_metrics = base_metrics.merge(vtype, on=cluster_col, how="left")
else:
    base_metrics["dominant_vehicle_type"] = "UNKNOWN"

if "violation_tags" in approved.columns:
    exploded = approved[[cluster_col, "violation_tags"]].explode("violation_tags").copy()
    exploded["violation_tag"] = exploded["violation_tags"].map(normalize_tag)
    exploded = exploded[exploded["violation_tag"].ne("")]
    dom_tag = exploded.groupby(cluster_col)["violation_tag"].agg(dominant_label).reset_index(name="dominant_violation_tag")
    base_metrics = base_metrics.merge(dom_tag, on=cluster_col, how="left")
else:
    base_metrics["dominant_violation_tag"] = ""

# Weekly counts for temporal features
weekly_counts = (
    approved.groupby([cluster_col, "week_start"])
    .size()
    .reset_index(name="weekly_count")
    .sort_values([cluster_col, "week_start"])
)

weekly_lists = weekly_counts.groupby(cluster_col)["weekly_count"].apply(list).reset_index(name="weekly_count_list")
base_metrics = base_metrics.merge(weekly_lists, on=cluster_col, how="left")

# ROP: recurring occupancy persistence
base_metrics["active_days"] = base_metrics["distinct_days"].fillna(0).astype(float)
base_metrics["observation_days"] = global_observation_days
base_metrics["ROP"] = base_metrics["active_days"] / (base_metrics["observation_days"] + EPS)

# TVS: temporal volatility score
base_metrics["TVS"] = base_metrics["weekly_count_list"].apply(lambda x: compute_tvs(x if isinstance(x, list) else []))

# Resurgence score
base_metrics["resurgence_score"] = base_metrics["weekly_count_list"].apply(
    lambda x: compute_resurgence_score(x if isinstance(x, list) else [])
)

# VDI: violation density index
areas = []
for cid, g in approved.groupby(cluster_col):
    area_m2 = estimate_cluster_area_m2(g)
    areas.append((cid, area_m2))
area_df = pd.DataFrame(areas, columns=[cluster_col, "cluster_area_m2"])
base_metrics = base_metrics.merge(area_df, on=cluster_col, how="left")
base_metrics["VDI"] = base_metrics["records_total"] / (base_metrics["cluster_area_m2"].replace(0, np.nan) + EPS)

# Validation uncertainty from broader review source if available
validation_source = review_df.copy() if review_df is not None else None
if validation_source is not None and len(validation_source):
    if cluster_col not in validation_source.columns:
        rv_cluster_col = standardize_cluster_col(validation_source)
        if rv_cluster_col != cluster_col:
            validation_source = validation_source.rename(columns={rv_cluster_col: cluster_col})

    if "validation_status" in validation_source.columns:
        validation_source = validation_source.copy()
        validation_source["validation_status_clean"] = validation_source["validation_status"].fillna("").astype(str).str.lower()
        validation_source = validation_source.dropna(subset=[cluster_col]).copy()

        validation_summary = (
            validation_source.groupby(cluster_col)
            .agg(
                review_total=(cluster_col, "size"),
                review_present=("validation_status_clean", lambda s: int((s != "").sum())),
                approved_count=("validation_status_clean", lambda s: int((s == "approved").sum())),
            )
            .reset_index()
        )

        validation_summary["reviewed_ratio"] = validation_summary["review_present"] / (validation_summary["review_total"] + EPS)
        validation_summary["approval_rate"] = validation_summary["approved_count"] / (validation_summary["review_present"].replace(0, np.nan) + EPS)
        validation_summary["validation_confidence"] = validation_summary["reviewed_ratio"] * validation_summary["approval_rate"].fillna(0.0)
        validation_summary["validation_uncertainty"] = 1.0 - validation_summary["validation_confidence"]
        validation_summary["validation_uncertainty"] = validation_summary["validation_uncertainty"].clip(lower=0.0, upper=1.0)

        base_metrics = base_metrics.merge(
            validation_summary[[cluster_col, "review_total", "review_present", "approved_count", "reviewed_ratio", "approval_rate", "validation_confidence", "validation_uncertainty"]],
            on=cluster_col,
            how="left",
        )
    else:
        base_metrics["review_total"] = np.nan
        base_metrics["review_present"] = np.nan
        base_metrics["approved_count"] = np.nan
        base_metrics["reviewed_ratio"] = np.nan
        base_metrics["approval_rate"] = np.nan
        base_metrics["validation_confidence"] = np.nan
        base_metrics["validation_uncertainty"] = np.nan
else:
    base_metrics["review_total"] = np.nan
    base_metrics["review_present"] = np.nan
    base_metrics["approved_count"] = np.nan
    base_metrics["reviewed_ratio"] = np.nan
    base_metrics["approval_rate"] = np.nan
    base_metrics["validation_confidence"] = np.nan
    base_metrics["validation_uncertainty"] = np.nan

# Fill safe defaults
base_metrics["dominant_vehicle_type"] = base_metrics["dominant_vehicle_type"].fillna("UNKNOWN")
base_metrics["dominant_violation_tag"] = base_metrics["dominant_violation_tag"].fillna("")
base_metrics["validation_uncertainty"] = pd.to_numeric(base_metrics["validation_uncertainty"], errors="coerce")
base_metrics["validation_uncertainty"] = base_metrics["validation_uncertainty"].fillna(base_metrics["validation_uncertainty"].median(skipna=True) if base_metrics["validation_uncertainty"].notna().any() else 0.5)
base_metrics["validation_uncertainty"] = base_metrics["validation_uncertainty"].clip(lower=0.0, upper=1.0)

# Optional compact composite for downstream use
base_metrics["ROP_norm"] = minmax(base_metrics["ROP"])
base_metrics["TVS_norm"] = minmax(base_metrics["TVS"])
base_metrics["VDI_norm"] = minmax(base_metrics["VDI"])
base_metrics["validation_uncertainty_norm"] = minmax(base_metrics["validation_uncertainty"])
base_metrics["resurgence_norm"] = minmax(base_metrics["resurgence_score"])

base_metrics["layer_c_index"] = (
    0.25 * base_metrics["ROP_norm"]
    + 0.20 * base_metrics["TVS_norm"]
    + 0.20 * base_metrics["VDI_norm"]
    + 0.20 * base_metrics["validation_uncertainty_norm"]
    + 0.15 * base_metrics["resurgence_norm"]
)

# Merge back onto Stage 5 cluster table
layer_c_df = cluster_df.merge(base_metrics, on=cluster_col, how="left", suffixes=("", "_layerc"))

# Preserve / clean useful fields
layer_c_df["cluster_label"] = layer_c_df["cluster_label"].fillna("").astype(str).str.strip()
layer_c_df["cluster_label_display"] = (
    layer_c_df["cluster_label"].astype(str) + " (Cluster " + layer_c_df[cluster_col].astype(str) + ")"
)

# Handle duplicate suffix columns if any
for col in list(layer_c_df.columns):
    if col.endswith("_layerc"):
        layer_c_df.drop(columns=[col], inplace=True, errors="ignore")

# Keep one row per cluster
layer_c_df = layer_c_df.drop_duplicates(subset=[cluster_col]).reset_index(drop=True)

# Sorting for presentation
layer_c_df["layer_c_index"] = pd.to_numeric(layer_c_df["layer_c_index"], errors="coerce").fillna(0.0)
layer_c_df = layer_c_df.sort_values(
    ["layer_c_index", "resurgence_score", "VDI", "ROP"],
    ascending=[False, False, False, False],
).reset_index(drop=True)

layer_c_df["layer_c_rank"] = np.arange(1, len(layer_c_df) + 1)

# =========================
# Outputs
# =========================
layer_c_export_cols = [
    "layer_c_rank",
    cluster_col,
    "cluster_label",
    "cluster_label_display",
    "records_total",
    "distinct_days",
    "active_days",
    "observation_days",
    "ROP",
    "TVS",
    "VDI",
    "cluster_area_m2",
    "validation_uncertainty",
    "validation_confidence",
    "review_total",
    "review_present",
    "approved_count",
    "resurgence_score",
    "dominant_vehicle_type",
    "dominant_violation_tag",
    "layer_c_index",
    "lat",
    "lon",
]

layer_c_export = layer_c_df[[c for c in layer_c_export_cols if c in layer_c_df.columns]].copy()

layer_c_export.to_csv(OUT_DIR / "layer_c_novel_features.csv", index=False)
layer_c_df.to_csv(OUT_DIR / "phase5_with_layer_c.csv", index=False)

summary = pd.DataFrame([{
    "cluster_source": str(cluster_src),
    "records_source": str(records_src) if records_df is not None else "",
    "validation_source": str(review_src) if review_df is not None else "",
    "clusters_scored": int(len(layer_c_df)),
    "mean_ROP": float(pd.to_numeric(layer_c_df["ROP"], errors="coerce").mean()),
    "mean_TVS": float(pd.to_numeric(layer_c_df["TVS"], errors="coerce").mean()),
    "mean_VDI": float(pd.to_numeric(layer_c_df["VDI"], errors="coerce").replace([np.inf, -np.inf], np.nan).mean()),
    "mean_validation_uncertainty": float(pd.to_numeric(layer_c_df["validation_uncertainty"], errors="coerce").mean()),
    "mean_resurgence_score": float(pd.to_numeric(layer_c_df["resurgence_score"], errors="coerce").mean()),
}])
summary.to_csv(OUT_DIR / "layer_c_summary.csv", index=False)

weekly_counts.to_csv(OUT_DIR / "layer_c_weekly_counts.csv", index=False)

print("Layer C complete")
print("Cluster source:", cluster_src)
print("Record source:", records_src)
print("Validation source:", review_src)
print("Clusters scored:", len(layer_c_df))
print("Outputs saved to:", OUT_DIR.resolve())
print("\nSummary:")
print(summary.to_string(index=False))

print("\nTop Layer C hotspots:")
show_cols = [
    "layer_c_rank", cluster_col, "cluster_label_display",
    "ROP", "TVS", "VDI", "validation_uncertainty", "resurgence_score", "layer_c_index"
]
print(layer_c_export[[c for c in show_cols if c in layer_c_export.columns]].head(15).to_string(index=False))

Layer C complete
Cluster source: content\phase5_outputs_2\phase5_cluster_capacity_loss.csv
Record source: content\phase5_outputs_2\phase5_enriched_records.csv
Validation source: content\phase4_outputs_2\phase4_merged_with_prior_scores.csv
Clusters scored: 798
Outputs saved to: D:\Flipkart_Hackathon\approach1\content\layer_c_outputs_2

Summary:
                                           cluster_source                                       records_source                                            validation_source  clusters_scored  mean_ROP  mean_TVS  mean_VDI  mean_validation_uncertainty  mean_resurgence_score
content\phase5_outputs_2\phase5_cluster_capacity_loss.csv content\phase5_outputs_2\phase5_enriched_records.csv content\phase4_outputs_2\phase4_merged_with_prior_scores.csv              798  0.051257  0.373597    0.0055                 6.943286e-11               1.583935

Top Layer C hotspots:
 layer_c_rank  st_dbscan_cluster_id                                    cluster_label_disp

In [23]:
import ast
import warnings
from pathlib import Path
from typing import Iterable, Optional, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ============================================================
# Stage 6 - corrected hotspot scoring and output generation
# ============================================================
# Major fixes:
#   1) CCS no longer collapses when one factor is constant.
#   2) delay_minutes_per_vehicle is repaired with a unit-consistent
#      congestion model when the upstream value is tiny / flat.
#   3) criticality_factor is rebuilt from available graph/load proxies.
#   4) growth calculation uses smoothing to avoid repeated 3.0 / 4.0 clips.
#   5) duplicate physical hotspots are deduplicated by label + rounded coords.
#   6) output tables are preserved: full cluster table + distinct hotspot ranking.
# ============================================================

PHASE5_DIRS = [
    Path("phase5_outputs_2"),
    Path("/content/phase5_outputs_2"),
    Path("content/phase5_outputs_2"),
]

PHASE4_DIRS = [
    Path("phase4_outputs_2"),
    Path("/content/phase4_outputs_2"),
    Path("content/phase4_outputs_2"),
]

PHASE3_DIRS = [
    Path("phase3_outputs_2"),
    Path("/content/phase3_outputs_2"),
    Path("content/phase3_outputs_2"),
]

OUT_DIR = Path("content/phase6_outputs_2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EPS = 1e-9

# Use a modest smoothing term so sparse windows do not explode or collapse.
GROWTH_SMOOTHING = 2.0


# =========================
# Generic helpers
# =========================
def clean_text(x) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()


def normalize_tag(tag) -> str:
    return clean_text(tag).upper().replace("&", "AND").strip()


def parse_listlike(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    s = str(value).strip()
    if not s:
        return []
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple)):
            return list(parsed)
        return [parsed]
    except Exception:
        if s.startswith("[") and s.endswith("]"):
            s = s[1:-1]
        parts = [p.strip().strip("'").strip('"') for p in s.split(",")]
        return [p for p in parts if p]


def load_first_existing(base_dirs: Iterable[Path], filenames: Iterable[str]):
    for d in base_dirs:
        for name in filenames:
            p = d / name
            if p.exists():
                return pd.read_csv(p, low_memory=False), p
    return None, None


def standardize_cluster_col(df: pd.DataFrame) -> str:
    for c in ["st_dbscan_cluster_id", "cluster_id", "dbscan_cluster_id"]:
        if c in df.columns:
            return c
    raise ValueError("No cluster id column found.")


def standardize_vehicle_col(df: pd.DataFrame):
    for c in ["canonical_vehicle_number", "vehicle_number", "updated_vehicle_number"]:
        if c in df.columns:
            return c
    return None


def standardize_vehicle_type_col(df: pd.DataFrame):
    for c in ["canonical_vehicle_type", "vehicle_type", "updated_vehicle_type"]:
        if c in df.columns:
            return c
    return None


def ensure_label_column(df: pd.DataFrame):
    if df is None or len(df) == 0:
        return df
    df = df.copy()
    if "cluster_label" in df.columns:
        df["cluster_label"] = df["cluster_label"].fillna("").astype(str).str.strip()
        return df
    if "hotspot_unit" in df.columns:
        df["cluster_label"] = df["hotspot_unit"].fillna("").astype(str).str.strip()
        return df
    if "dominant_junction_name" in df.columns:
        df["cluster_label"] = df["dominant_junction_name"].fillna("").astype(str).str.strip()
        return df
    if "st_dbscan_cluster_id" in df.columns:
        df["cluster_label"] = "CLUSTER::" + df["st_dbscan_cluster_id"].astype(str)
        return df
    df["cluster_label"] = "UNKNOWN"
    return df


def derive_coords(df: pd.DataFrame):
    if df is None or len(df) == 0:
        return df
    df = df.copy()
    if "lat" in df.columns and "lon" in df.columns:
        df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
        df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    elif {"centroid_lat", "centroid_lon"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["centroid_lat"], errors="coerce")
        df["lon"] = pd.to_numeric(df["centroid_lon"], errors="coerce")
    elif {"latitude_mean", "longitude_mean"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["latitude_mean"], errors="coerce")
        df["lon"] = pd.to_numeric(df["longitude_mean"], errors="coerce")
    elif {"latitude", "longitude"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["latitude"], errors="coerce")
        df["lon"] = pd.to_numeric(df["longitude"], errors="coerce")
    else:
        df["lat"] = np.nan
        df["lon"] = np.nan
    return df


def parse_datetime_ist(df: pd.DataFrame) -> pd.Series:
    if "created_datetime_ist" in df.columns:
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce", utc=True)
        if ts.notna().any():
            return ts.dt.tz_convert("Asia/Kolkata")
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce")
        return ts.dt.tz_localize("Asia/Kolkata", nonexistent="NaT", ambiguous="NaT")

    if "created_datetime_parsed" in df.columns:
        ts = pd.to_datetime(df["created_datetime_parsed"], errors="coerce", utc=True)
        return ts.dt.tz_convert("Asia/Kolkata")

    if "created_datetime" not in df.columns:
        raise ValueError("Missing created_datetime / created_datetime_ist column.")

    ts = pd.to_datetime(df["created_datetime"], errors="coerce", utc=True)
    return ts.dt.tz_convert("Asia/Kolkata")


def week_start_monday(series):
    dt = pd.to_datetime(series, errors="coerce", utc=True).dt.tz_convert("Asia/Kolkata")
    week_start = dt.dt.normalize() - pd.to_timedelta(dt.dt.weekday, unit="D")
    return week_start.dt.tz_localize(None)


def safe_metric(val, digits=2):
    try:
        if pd.isna(val):
            return "0"
        val = float(val)
        if abs(val) >= 1000:
            return f"{val:,.0f}"
        return f"{val:,.{digits}f}"
    except Exception:
        return str(val)


def dominant_label(series, exclude=None, default=""):
    s = pd.Series(series).dropna().astype(str).str.strip()
    s = s[s.ne("")]
    if exclude:
        excl = {clean_text(x).lower() for x in exclude}
        s = s[~s.str.lower().isin(excl)]
    if s.empty:
        return default
    m = s.mode()
    if not m.empty:
        return m.iloc[0]
    return s.iloc[0]


# =========================
# Domain helpers
# =========================
def vehicle_width_m(vehicle_type):
    widths = {
        "SCOOTER": 0.80,
        "MOTOR CYCLE": 0.90,
        "MOTORCYCLE": 0.90,
        "BICYCLE": 0.60,
        "CYCLE": 0.60,
        "PASSENGER AUTO": 1.60,
        "AUTO": 1.60,
        "CAR": 1.90,
        "SUV": 2.00,
        "JEEP": 2.00,
        "VAN": 2.20,
        "TEMPO": 2.20,
        "BUS": 2.60,
        "TRUCK": 2.60,
        "LORRY": 2.60,
        "TANKER": 2.80,
        "TRACTOR": 2.20,
        "MINI TRUCK": 2.20,
        "AMBULANCE": 2.00,
        "UNKNOWN": 1.90,
    }
    return widths.get(clean_text(vehicle_type).upper(), 1.90)


def road_speed_kmh(road_class):
    speeds = {
        "motorway": 60.0,
        "trunk": 55.0,
        "primary": 45.0,
        "secondary": 40.0,
        "tertiary": 35.0,
        "unclassified": 30.0,
        "residential": 30.0,
        "living_street": 20.0,
        "service": 25.0,
        "road": 30.0,
    }
    return speeds.get(clean_text(road_class).lower(), 30.0)


def base_capacity_per_lane(road_class):
    cap = {
        "motorway": 2200.0,
        "trunk": 2100.0,
        "primary": 1900.0,
        "secondary": 1800.0,
        "tertiary": 1700.0,
        "unclassified": 1650.0,
        "residential": 1500.0,
        "living_street": 1200.0,
        "service": 1400.0,
        "road": 1600.0,
    }
    return cap.get(clean_text(road_class).lower(), 1600.0)


# =========================
# Normalization / scoring
# =========================
def minmax(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    valid = s.dropna()
    if valid.nunique(dropna=True) <= 1:
        return pd.Series(np.zeros(len(s)), index=s.index, dtype=float)
    mn = valid.min()
    mx = valid.max()
    return (s.fillna(mn) - mn) / (mx - mn + EPS)


def active_minmax(s: pd.Series) -> Tuple[pd.Series, bool]:
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    if s.dropna().nunique(dropna=True) <= 1:
        return pd.Series(np.nan, index=s.index, dtype=float), False
    mn = s.min(skipna=True)
    mx = s.max(skipna=True)
    return (s.fillna(mn) - mn) / (mx - mn + EPS), True


def weighted_component_score(df: pd.DataFrame, components):
    """
    PATCH: restore the concept-note multiplicative CCS formula.

    Original design: CCS ∝ Delay × λ × Severity × Growth × Criticality
    Each factor is min-max normalised to [0,1] and then the *product* is
    taken, NOT a weighted average.  To prevent one zero factor from killing
    the whole score, constant / all-null components are replaced by their
    column median (floored at 0.1) instead of being dropped.

    The weighted-average variant was a safe fallback but loses the
    multiplicative interaction structure that distinguishes high-delay +
    high-growth hotspots from ones that are merely high on one axis.

    components: list of (source_col, weight, output_name)
      weight is used as the *exponent* in the product  — component^weight —
      so higher-weight factors have more influence while the score stays
      bounded in [0, 1].
    """
    out = df.copy()
    active = []
    for source_col, weight, out_col in components:
        if source_col not in out.columns:
            out[out_col] = np.nan
            continue
        norm, is_active = active_minmax(out[source_col])
        if not is_active:
            # Constant column: use a neutral value (0.5) so it does not kill
            # the product but also does not inflate it.
            norm = pd.Series(0.5, index=out.index, dtype=float)
        out[out_col] = norm.fillna(0.5)
        active.append((out_col, weight))

    if not active:
        out["ccs_score"] = 0.0
        out["ccs_raw_product"] = 0.0
        return out

    # Multiplicative combination: ∏ component_i ^ weight_i
    # Then rescale by the total weight so the result sits in [0, 1].
    total_weight = float(sum(w for _, w in active))
    log_sum = np.zeros(len(out), dtype=float)
    for out_col, weight in active:
        vals = out[out_col].to_numpy(dtype=float).clip(1e-6, 1.0)
        log_sum += (weight / max(total_weight, EPS)) * np.log(vals)

    raw = np.exp(log_sum)          # back to linear, in (0, 1]
    out["ccs_raw_product"] = raw
    out["ccs_score"] = 100.0 * np.clip(raw, 0.0, 1.0)
    return out


def risk_band_from_score(df: pd.DataFrame):
    """
    Prefer fixed thresholds on a calibrated 0-100 score. If the distribution is
    too compressed, fall back to quantile buckets so the output remains usable.
    """
    if df is None or len(df) == 0 or "ccs_score" not in df.columns:
        return df

    df = df.copy()
    score = pd.to_numeric(df["ccs_score"], errors="coerce").fillna(0.0)
    spread = float(score.quantile(0.90) - score.quantile(0.10)) if len(score) else 0.0

    if spread < 5.0:
        q80 = score.quantile(0.80)
        q60 = score.quantile(0.60)
        q40 = score.quantile(0.40)

        def band_q(x):
            if x >= q80:
                return "Critical"
            if x >= q60:
                return "High"
            if x >= q40:
                return "Moderate"
            return "Watch"

        df["risk_band"] = score.apply(band_q)
        return df

    def band_fixed(x):
        if x >= 80:
            return "Critical"
        if x >= 60:
            return "High"
        if x >= 40:
            return "Moderate"
        return "Watch"

    df["risk_band"] = score.apply(band_fixed)
    return df


# =========================
# Record-level backfill
# =========================
def load_record_level_source():
    df, path = load_first_existing(
        PHASE5_DIRS + PHASE4_DIRS + PHASE3_DIRS,
        [
            "phase5_enriched_records.csv",
            "phase4_merged_with_prior_scores.csv",
            "phase3_clustered_dataset.csv",
        ],
    )
    return df, path


def fill_missing_from_records(cluster_df, records_df, cluster_col, vehicle_col=None, vehicle_type_col=None):
    """
    Backfill cluster-level metrics from the record-level source.
    The record-level source is also used to derive stable growth metrics.
    """
    if records_df is None or len(records_df) == 0:
        return cluster_df.copy(), pd.DataFrame(), records_df

    r = records_df.copy()
    r = r.dropna(subset=[cluster_col]).copy()
    r[cluster_col] = pd.to_numeric(r[cluster_col], errors="coerce")
    r = r[r[cluster_col].ne(-1)].copy()

    r["created_datetime_ist"] = parse_datetime_ist(r)
    r = r.dropna(subset=["created_datetime_ist"]).copy()
    r["created_datetime_ist_naive"] = r["created_datetime_ist"].dt.tz_localize(None)
    r["service_date"] = r["created_datetime_ist_naive"].dt.date
    r["cluster_week_start"] = week_start_monday(r["created_datetime_ist"])
    r["hour_ist"] = r["created_datetime_ist"].dt.hour
    r["is_peak_window"] = r["hour_ist"].between(8, 12, inclusive="both").astype(int)

    has_vehicle = vehicle_col is not None and vehicle_col in r.columns
    has_vtype = vehicle_type_col is not None and vehicle_type_col in r.columns
    has_lat = "lat" in r.columns
    has_lon = "lon" in r.columns

    metrics = r.groupby(cluster_col).agg(
        records_total=(cluster_col, "size"),
        distinct_days=("service_date", "nunique"),
        peak_window_records=("is_peak_window", "sum"),
        severity_sum=("severity_score", "sum") if "severity_score" in r.columns else (cluster_col, "size"),
        severity_mean=("severity_score", "mean") if "severity_score" in r.columns else (cluster_col, "size"),
        unique_vehicles=(vehicle_col, "nunique") if has_vehicle else (cluster_col, "size"),
        unique_vehicle_types=(vehicle_type_col, "nunique") if has_vtype else (cluster_col, "size"),
        centroid_lat=("lat", "mean") if has_lat else (cluster_col, "size"),
        centroid_lon=("lon", "mean") if has_lon else (cluster_col, "size"),
        dominant_vehicle_type=(vehicle_type_col, dominant_label) if has_vtype else (cluster_col, lambda s: "UNKNOWN"),
        dominant_police_station=("police_station", dominant_label) if "police_station" in r.columns else (cluster_col, lambda s: ""),
        dominant_junction_name=("junction_name", dominant_label) if "junction_name" in r.columns else (cluster_col, lambda s: ""),
    ).reset_index()

    metrics["distinct_days"] = pd.to_numeric(metrics["distinct_days"], errors="coerce").fillna(1).clip(lower=1)
    metrics["lambda_hr_peak_window"] = metrics["peak_window_records"] / (metrics["distinct_days"] * 5.0)

    weekly = (
        r.groupby([cluster_col, "cluster_week_start"])
        .size()
        .reset_index(name="weekly_count")
        .sort_values([cluster_col, "cluster_week_start"])
    )

    growth_rows = []
    for cid, g in weekly.groupby(cluster_col):
        counts = g["weekly_count"].to_numpy(dtype=float)
        if len(counts) < 2:
            first_half = float(counts.mean()) if len(counts) else 0.0
            second_half = first_half
        else:
            mid = max(1, len(counts) // 2)
            first_half = float(counts[:mid].mean()) if len(counts[:mid]) else 0.0
            second_half = float(counts[mid:].mean()) if len(counts[mid:]) else 0.0

        smoothed_first = first_half + GROWTH_SMOOTHING
        smoothed_second = second_half + GROWTH_SMOOTHING
        growth_multiplier = smoothed_second / max(smoothed_first, EPS)
        growth_pct = growth_multiplier - 1.0

        # Keep the scale sane, but do not clip so hard that many clusters become identical.
        growth_pct = float(np.clip(growth_pct, -0.8, 1.5))
        growth_multiplier = float(np.clip(growth_multiplier, 0.5, 2.5))

        growth_rows.append(
            {
                cluster_col: cid,
                "growth_first_half_mean": first_half,
                "growth_second_half_mean": second_half,
                "growth_pct_change": growth_pct,
                "growth_multiplier": growth_multiplier,
            }
        )

    growth_df = pd.DataFrame(growth_rows)
    metrics = metrics.merge(growth_df, on=cluster_col, how="left")

    out = cluster_df.copy()
    if cluster_col not in out.columns:
        raise ValueError(f"Expected cluster column {cluster_col} in cluster_df.")

    out = out.merge(metrics, on=cluster_col, how="left", suffixes=("", "_rec"))

    for col in metrics.columns:
        if col == cluster_col:
            continue
        rec_col = f"{col}_rec"
        if rec_col in out.columns:
            if col in out.columns:
                if pd.api.types.is_numeric_dtype(out[col]) or pd.api.types.is_numeric_dtype(out[rec_col]):
                    out[col] = pd.to_numeric(out[col], errors="coerce").combine_first(
                        pd.to_numeric(out[rec_col], errors="coerce")
                    )
                else:
                    out[col] = out[col].combine_first(out[rec_col])
            else:
                out[col] = out[rec_col]
            out.drop(columns=[rec_col], inplace=True)

    return out, weekly, r


# =========================
# Load Stage 5 table
# =========================
cluster_df, cluster_src = load_first_existing(
    PHASE5_DIRS,
    ["phase5_cluster_capacity_loss.csv", "phase5_priority_table_full.csv", "phase5_stage5_handoff.csv"],
)

if cluster_df is None:
    raise FileNotFoundError(
        "Could not find Stage 5 output. Expected phase5_cluster_capacity_loss.csv or phase5_priority_table_full.csv"
    )

cluster_col = standardize_cluster_col(cluster_df)
vehicle_col = standardize_vehicle_col(cluster_df)
vehicle_type_col = standardize_vehicle_type_col(cluster_df)

cluster_df = cluster_df.copy()
cluster_df = ensure_label_column(cluster_df)
cluster_df = derive_coords(cluster_df)

# Key columns that may be absent in some upstream outputs.
for col, default in [
    ("records_total", np.nan),
    ("distinct_days", np.nan),
    ("peak_window_records", np.nan),
    ("lambda_hr_peak_window", np.nan),
    ("lambda_hr_peak_hour", np.nan),
    ("mean_dwell_minutes", np.nan),
    ("mu_departures_per_hour", np.nan),
    ("blocking_vehicles_L", np.nan),
    ("severity_sum", np.nan),
    ("severity_mean", np.nan),
    ("growth_pct_change", np.nan),
    ("growth_multiplier", np.nan),
    ("criticality_factor", np.nan),
    ("delay_minutes_per_vehicle", np.nan),
    ("capacity_loss_pct", np.nan),
    ("junction_degree", np.nan),
    ("betweenness_centrality", np.nan),
    ("lane_count", np.nan),
    ("carriageway_width_m", np.nan),
    ("link_length_m", np.nan),
    ("base_capacity_pcu_hr", np.nan),
    ("reduced_capacity_pcu_hr", np.nan),
    ("unique_vehicles", np.nan),
    ("unique_vehicle_types", np.nan),
    ("repeat_vehicle_count_2plus", 0),
    ("chronic_vehicle_count_5plus", 0),
]:
    if col not in cluster_df.columns:
        cluster_df[col] = default

for col in [
    "records_total",
    "distinct_days",
    "peak_window_records",
    "lambda_hr_peak_window",
    "lambda_hr_peak_hour",
    "mean_dwell_minutes",
    "mu_departures_per_hour",
    "blocking_vehicles_L",
    "severity_sum",
    "severity_mean",
    "growth_pct_change",
    "growth_multiplier",
    "criticality_factor",
    "delay_minutes_per_vehicle",
    "capacity_loss_pct",
    "junction_degree",
    "betweenness_centrality",
    "lane_count",
    "carriageway_width_m",
    "link_length_m",
    "base_capacity_pcu_hr",
    "reduced_capacity_pcu_hr",
    "unique_vehicles",
    "unique_vehicle_types",
]:
    cluster_df[col] = pd.to_numeric(cluster_df[col], errors="coerce")

cluster_df["effective_capacity_pcu_hr"] = cluster_df["reduced_capacity_pcu_hr"]

# Safe label repair.
cluster_df["cluster_label"] = cluster_df["cluster_label"].fillna("").astype(str).str.strip()
cluster_df.loc[cluster_df["cluster_label"].eq(""), "cluster_label"] = np.nan

if "dominant_junction_name" in cluster_df.columns:
    dominant_junction_series = cluster_df["dominant_junction_name"].fillna("").astype(str)
else:
    dominant_junction_series = pd.Series("", index=cluster_df.index)

if "hotspot_unit" in cluster_df.columns:
    hotspot_unit_series = cluster_df["hotspot_unit"].fillna("").astype(str)
else:
    hotspot_unit_series = pd.Series("", index=cluster_df.index)

cluster_df.loc[cluster_df["cluster_label"].isna(), "cluster_label"] = dominant_junction_series
cluster_df.loc[cluster_df["cluster_label"].astype(str).str.strip().eq(""), "cluster_label"] = hotspot_unit_series
cluster_df.loc[cluster_df["cluster_label"].astype(str).str.strip().eq(""), "cluster_label"] = "CLUSTER::" + cluster_df[cluster_col].astype(str)

# =========================
# Backfill from record-level source
# =========================
records_df, records_src = load_record_level_source()
weekly_df = pd.DataFrame()

if records_df is not None and len(records_df):
    if cluster_col not in records_df.columns:
        rec_cluster_col = standardize_cluster_col(records_df)
        if rec_cluster_col != cluster_col:
            records_df = records_df.rename(columns={rec_cluster_col: cluster_col})
    if vehicle_col is None:
        vehicle_col = standardize_vehicle_col(records_df)
    if vehicle_type_col is None:
        vehicle_type_col = standardize_vehicle_type_col(records_df)

    records_df = records_df.copy()
    records_df = ensure_label_column(records_df)
    records_df = derive_coords(records_df)
    records_df["created_datetime_ist"] = parse_datetime_ist(records_df)

    cluster_df, weekly_df, records_df = fill_missing_from_records(
        cluster_df, records_df, cluster_col, vehicle_col, vehicle_type_col
    )

# Re-coerce after backfill merge.
for col in [
    "records_total",
    "distinct_days",
    "peak_window_records",
    "lambda_hr_peak_window",
    "lambda_hr_peak_hour",
    "mean_dwell_minutes",
    "mu_departures_per_hour",
    "blocking_vehicles_L",
    "severity_sum",
    "severity_mean",
    "growth_pct_change",
    "growth_multiplier",
    "criticality_factor",
    "delay_minutes_per_vehicle",
    "capacity_loss_pct",
    "junction_degree",
    "betweenness_centrality",
    "lane_count",
    "carriageway_width_m",
    "link_length_m",
    "unique_vehicles",
    "unique_vehicle_types",
    "base_capacity_pcu_hr",
    "reduced_capacity_pcu_hr",
]:
    if col in cluster_df.columns:
        cluster_df[col] = pd.to_numeric(cluster_df[col], errors="coerce")

# =========================
# Repair missing / broken row-wise metrics
# =========================
# Severity backfill.
mask = cluster_df["severity_sum"].isna() & cluster_df["severity_mean"].notna()
cluster_df.loc[mask, "severity_sum"] = cluster_df.loc[mask, "severity_mean"] * cluster_df.loc[mask, "records_total"].clip(lower=1)

mask = cluster_df["severity_mean"].isna() & cluster_df["severity_sum"].notna()
cluster_df.loc[mask, "severity_mean"] = cluster_df.loc[mask, "severity_sum"] / cluster_df.loc[mask, "records_total"].clip(lower=1)

# Peak-window arrival rate.
mask = cluster_df["lambda_hr_peak_window"].isna()
cluster_df.loc[mask, "lambda_hr_peak_window"] = (
    cluster_df.loc[mask, "peak_window_records"] / (cluster_df.loc[mask, "distinct_days"].clip(lower=1) * 5.0)
)

# Service rate from dwell time.
mask = cluster_df["mu_departures_per_hour"].isna()
cluster_df.loc[mask, "mu_departures_per_hour"] = 60.0 / (cluster_df.loc[mask, "mean_dwell_minutes"] + EPS)

# Blocking load.
mask = cluster_df["blocking_vehicles_L"].isna()
cluster_df.loc[mask, "blocking_vehicles_L"] = (
    cluster_df.loc[mask, "lambda_hr_peak_window"] / (cluster_df.loc[mask, "mu_departures_per_hour"] + EPS)
)

# Lane count / carriageway width repair.
if "lane_count" in cluster_df.columns:
    mask = cluster_df["lane_count"].isna() & cluster_df["carriageway_width_m"].notna()
    cluster_df.loc[mask, "lane_count"] = np.maximum(
        1,
        np.rint(cluster_df.loc[mask, "carriageway_width_m"] / 3.5),
    )

if "carriageway_width_m" in cluster_df.columns:
    mask = cluster_df["carriageway_width_m"].isna() & cluster_df["lane_count"].notna()
    cluster_df.loc[mask, "carriageway_width_m"] = cluster_df.loc[mask, "lane_count"].clip(lower=1) * 3.5

# Base / reduced capacity repair.
if "base_capacity_pcu_hr" in cluster_df.columns:
    mask = cluster_df["base_capacity_pcu_hr"].isna()
    cluster_df.loc[mask, "base_capacity_pcu_hr"] = (
        cluster_df.loc[mask, "road_class"].apply(base_capacity_per_lane).to_numpy()
        * cluster_df.loc[mask, "lane_count"].fillna(1).clip(lower=1).to_numpy()
    )

if "capacity_loss_pct" in cluster_df.columns:
    # If capacity loss is missing or completely collapsed, derive a usable proxy.
    cap_loss = pd.to_numeric(cluster_df["capacity_loss_pct"], errors="coerce")
    if cap_loss.notna().sum() == 0 or cap_loss.nunique(dropna=True) <= 1:
        util = (
            cluster_df["lambda_hr_peak_window"] / (cluster_df["mu_departures_per_hour"] + EPS)
        ).replace([np.inf, -np.inf], np.nan)
        util = util.fillna(util.median(skipna=True) if util.notna().any() else 0.0)
        util = util.clip(lower=0.0, upper=5.0)

        sev_norm = minmax(cluster_df["severity_sum"].fillna(0.0))
        blocking_norm = minmax(cluster_df["blocking_vehicles_L"].fillna(0.0))
        records_norm = minmax(cluster_df["records_total"].fillna(0.0))

        # Proxy: stronger when the cluster is loaded, blocked, and severe.
        capacity_loss_proxy = 100.0 * (
            0.40 * np.clip(util / 1.5, 0.0, 1.0)
            + 0.25 * blocking_norm
            + 0.20 * sev_norm
            + 0.15 * records_norm
        )
        cluster_df["capacity_loss_pct"] = capacity_loss_proxy.clip(lower=0.0, upper=95.0)
    else:
        cluster_df["capacity_loss_pct"] = cap_loss

# Reduced capacity.
if "reduced_capacity_pcu_hr" in cluster_df.columns:
    mask = cluster_df["reduced_capacity_pcu_hr"].isna() & cluster_df["base_capacity_pcu_hr"].notna()
    cluster_df.loc[mask, "reduced_capacity_pcu_hr"] = cluster_df.loc[mask, "base_capacity_pcu_hr"] * (
        1.0 - cluster_df.loc[mask, "capacity_loss_pct"].fillna(0.0) / 100.0
    )

cluster_df["effective_capacity_pcu_hr"] = cluster_df["reduced_capacity_pcu_hr"]

# Delay repair: if the upstream delay is tiny / flat / missing, rebuild it with
# a unit-consistent congestion proxy.
def rebuild_delay_minutes(df: pd.DataFrame) -> pd.Series:
    mean_dwell = pd.to_numeric(df.get("mean_dwell_minutes"), errors="coerce")
    mean_dwell = mean_dwell.fillna(mean_dwell.median(skipna=True) if mean_dwell.notna().any() else 2.0)
    mean_dwell = mean_dwell.clip(lower=0.5, upper=180.0)

    lam = pd.to_numeric(df.get("lambda_hr_peak_window"), errors="coerce").fillna(0.0).clip(lower=0.0)
    mu = pd.to_numeric(df.get("mu_departures_per_hour"), errors="coerce").replace([np.inf, -np.inf], np.nan)
    mu = mu.fillna(mu.median(skipna=True) if mu.notna().any() else 1.0).clip(lower=0.1)

    util = (lam / (mu + EPS)).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    util = util.clip(lower=0.0, upper=5.0)
    overflow = np.maximum(0.0, util - 1.0)

    sev_norm = minmax(pd.to_numeric(df.get("severity_sum"), errors="coerce").fillna(0.0))
    block_norm = minmax(pd.to_numeric(df.get("blocking_vehicles_L"), errors="coerce").fillna(0.0))

    # This is an extra delay proxy, not total travel time:
    #   - base delay grows with utilization,
    #   - overflow adds a nonlinear penalty,
    #   - severity / blocking increase the delay.
    delay = (
        mean_dwell
        * (1.0 + 0.4 * sev_norm + 0.3 * block_norm)
        * (1.0 + overflow)
    )

    return delay.clip(lower=0.5, upper=60.0)

existing_delay = pd.to_numeric(cluster_df["delay_minutes_per_vehicle"], errors="coerce")
if (
    existing_delay.notna().sum() == 0
    or existing_delay.nunique(dropna=True) <= 2
    or float(existing_delay.median(skipna=True) or 0.0) < 0.05
):
    cluster_df["delay_minutes_per_vehicle"] = rebuild_delay_minutes(cluster_df)
else:
    cluster_df["delay_minutes_per_vehicle"] = existing_delay

# Final safety-net fill for anything still missing.
for col in [
    "lambda_hr_peak_window",
    "lambda_hr_peak_hour",
    "mean_dwell_minutes",
    "mu_departures_per_hour",
    "blocking_vehicles_L",
    "delay_minutes_per_vehicle",
    "severity_sum",
    "severity_mean",
    "capacity_loss_pct",
    "base_capacity_pcu_hr",
    "reduced_capacity_pcu_hr",
]:
    if col in cluster_df.columns:
        cluster_df[col] = pd.to_numeric(cluster_df[col], errors="coerce").fillna(0.0)

# =========================
# Novel Feature B: criticality factor
# =========================
def build_criticality_factor(df: pd.DataFrame) -> pd.Series:
    # Primary source: actual graph proxies.
    deg = pd.to_numeric(df.get("junction_degree"), errors="coerce") if "junction_degree" in df.columns else pd.Series(np.nan, index=df.index)
    bet = pd.to_numeric(df.get("betweenness_centrality"), errors="coerce") if "betweenness_centrality" in df.columns else pd.Series(np.nan, index=df.index)

    deg_active = deg.notna().sum() > 0 and deg.nunique(dropna=True) > 1
    bet_active = bet.notna().sum() > 0 and bet.nunique(dropna=True) > 1

    if deg_active or bet_active:
        deg_norm = minmax(deg.fillna(deg.median(skipna=True) if deg.notna().any() else 0.0)) if deg_active else pd.Series(0.0, index=df.index)
        bet_norm = minmax(bet.fillna(bet.median(skipna=True) if bet.notna().any() else 0.0)) if bet_active else pd.Series(0.0, index=df.index)

        return 1.0 + 0.55 * deg_norm + 0.45 * bet_norm

    # Fallback: use load / severity / roadway proxy so the factor is still informative.
    records_norm = minmax(pd.to_numeric(df.get("records_total"), errors="coerce").fillna(0.0))
    unique_veh_norm = minmax(pd.to_numeric(df.get("unique_vehicles"), errors="coerce").fillna(0.0))
    block_norm = minmax(pd.to_numeric(df.get("blocking_vehicles_L"), errors="coerce").fillna(0.0))
    caploss_norm = minmax(pd.to_numeric(df.get("capacity_loss_pct"), errors="coerce").fillna(0.0))
    sev_norm = minmax(pd.to_numeric(df.get("severity_sum"), errors="coerce").fillna(0.0))

    road_class_factor = pd.Series(
        pd.to_numeric(df.get("road_class"), errors="coerce"), index=df.index
    ) if False else pd.Series(0.0, index=df.index)
    # The road class factor is intentionally omitted unless you map road class
    # categories to numeric order; the load/severity proxy is enough to prevent collapse.

    proxy = (
        0.30 * records_norm
        + 0.20 * unique_veh_norm
        + 0.20 * block_norm
        + 0.15 * caploss_norm
        + 0.15 * sev_norm
        + 0.00 * road_class_factor
    )
    return 1.0 + 0.5 * proxy.clip(0.0, 1.0)

cluster_df["criticality_factor"] = build_criticality_factor(cluster_df)

# If the source already had better non-null values than the proxy, preserve them.
existing_criticality = pd.to_numeric(cluster_df.get("criticality_factor"), errors="coerce")
if existing_criticality.notna().sum() > 0 and existing_criticality.nunique(dropna=True) > 1:
    # Keep the more informative of the two by replacing only missing rows.
    cluster_df["criticality_factor"] = existing_criticality.combine_first(cluster_df["criticality_factor"])

# Growth repair.
mask = cluster_df["growth_pct_change"].isna()
cluster_df.loc[mask, "growth_pct_change"] = 0.0
# PATCH: widen growth clip.  The original upper=3.0 cap compressed genuine
# strong-trend hotspots (e.g. 400% week-over-week growth) to the same value
# as 300%.  Upper=8.0 preserves differentiation while still guarding against
# data artifacts (e.g., a cluster with 1 record one week and 10 the next).
cluster_df["growth_pct_change"] = cluster_df["growth_pct_change"].clip(lower=-0.8, upper=8.0)

mask = cluster_df["growth_multiplier"].isna()
cluster_df.loc[mask, "growth_multiplier"] = 1.0 + cluster_df.loc[mask, "growth_pct_change"].clip(lower=0.0)
# Upper=9.0 matches the widened growth_pct_change ceiling (1 + 8.0).
cluster_df["growth_multiplier"] = cluster_df["growth_multiplier"].fillna(1.0).clip(lower=0.2, upper=9.0)

# =========================
# CCS computation
# =========================
# Weighted average on active components so one flat factor does not zero out the whole score.
cluster_df = weighted_component_score(
    cluster_df,
    components=[
        ("delay_minutes_per_vehicle", 0.25, "delay_norm"),
        ("lambda_hr_peak_window", 0.25, "lambda_norm"),
        ("severity_sum", 0.25, "severity_norm"),
        ("growth_multiplier", 0.10, "growth_norm"),
        ("criticality_factor", 0.15, "criticality_norm"),
    ],
)

# Also expose the underlying criticality proxy for inspection.
cluster_df["criticality_norm"] = pd.to_numeric(cluster_df["criticality_norm"], errors="coerce").fillna(0.0)
cluster_df["delay_norm"] = pd.to_numeric(cluster_df["delay_norm"], errors="coerce").fillna(0.0)
cluster_df["lambda_norm"] = pd.to_numeric(cluster_df["lambda_norm"], errors="coerce").fillna(0.0)
cluster_df["severity_norm"] = pd.to_numeric(cluster_df["severity_norm"], errors="coerce").fillna(0.0)
cluster_df["growth_norm"] = pd.to_numeric(cluster_df["growth_norm"], errors="coerce").fillna(0.0)

cluster_df = risk_band_from_score(cluster_df)

# =========================
# Dedupe physically identical hotspots
# =========================
def build_hotspot_key(df: pd.DataFrame) -> pd.Series:
    label = df.get("cluster_label", pd.Series("", index=df.index)).fillna("").astype(str).map(normalize_tag)
    if "lat" in df.columns and "lon" in df.columns:
        lat = pd.to_numeric(df["lat"], errors="coerce").round(5).astype(str)
        lon = pd.to_numeric(df["lon"], errors="coerce").round(5).astype(str)
    else:
        lat = pd.Series("", index=df.index)
        lon = pd.Series("", index=df.index)
    return label + "|" + lat + "|" + lon

cluster_df["physical_hotspot_key"] = build_hotspot_key(cluster_df)
cluster_df["cluster_label_display"] = (
    cluster_df["cluster_label"].astype(str) + " (Cluster " + cluster_df[cluster_col].astype(str) + ")"
)
cluster_df["priority_message"] = (
    cluster_df["cluster_label_display"] + " | "
    + cluster_df["risk_band"].astype(str)
    + " | CCS="
    + cluster_df["ccs_score"].round(3).astype(str)
)

# Sort by risk, then keep only one row per physical hotspot for the presentation tables.
cluster_df = cluster_df.sort_values(
    ["ccs_score", "delay_minutes_per_vehicle", "lambda_hr_peak_window", "records_total"],
    ascending=[False, False, False, False],
).reset_index(drop=True)

hotspot_df = cluster_df.drop_duplicates(subset=["physical_hotspot_key"], keep="first").copy().reset_index(drop=True)
cluster_df["ccs_rank"] = np.arange(1, len(cluster_df) + 1)
hotspot_df["ccs_rank"] = np.arange(1, len(hotspot_df) + 1)

# =========================
# Recommended actions
# =========================
hotspot_df["recommended_action"] = np.select(
    [
        hotspot_df["risk_band"].eq("Critical"),
        hotspot_df["risk_band"].eq("High"),
        hotspot_df["risk_band"].eq("Moderate"),
    ],
    [
        "Immediate patrol deployment",
        "Targeted enforcement + towing readiness",
        "Monitor and schedule peak-window checks",
    ],
    default="Routine monitoring",
)

# =========================
# Emerging hotspot alerts
# =========================
alerts = hotspot_df.copy()
if len(alerts):
    median_records = alerts["records_total"].median()
    min_record_threshold = max(10, int(round(median_records)) if pd.notna(median_records) else 10)

    # Require meaningful growth, but also let the distribution decide who is "emerging".
    positive_growth = alerts.loc[alerts["growth_pct_change"] > 0, "growth_pct_change"]
    growth_cut = float(positive_growth.quantile(0.70)) if len(positive_growth) else 0.25
    growth_cut = max(0.25, growth_cut)

    alerts["is_emerging"] = (
        (alerts["growth_pct_change"] >= growth_cut)
        & (alerts["records_total"] >= min_record_threshold)
    )
else:
    alerts["is_emerging"] = pd.Series(dtype=bool)

alerts = alerts[alerts["is_emerging"]].copy()

if len(alerts):
    alerts = alerts.sort_values(
        ["growth_pct_change", "growth_multiplier", "ccs_score"],
        ascending=[False, False, False],
    ).head(25).copy()
    q80 = alerts["growth_pct_change"].quantile(0.80)
    q60 = alerts["growth_pct_change"].quantile(0.60)

    def alert_level(x):
        if x >= q80:
            return "Emerging-Critical"
        if x >= q60:
            return "Emerging-High"
        return "Emerging-Watch"

    alerts["alert_level"] = alerts["growth_pct_change"].apply(alert_level)
else:
    alerts["alert_level"] = pd.Series(dtype=str)

# =========================
# Chronic offenders
# =========================
if records_df is not None and len(records_df):
    r = records_df.copy()
    rec_cluster_col = standardize_cluster_col(r)
    if cluster_col not in r.columns and rec_cluster_col:
        r = r.rename(columns={rec_cluster_col: cluster_col})
    if "created_datetime_ist" not in r.columns:
        r["created_datetime_ist"] = parse_datetime_ist(r)
    r["created_datetime_ist"] = pd.to_datetime(r["created_datetime_ist"], errors="coerce")
    r = r.dropna(subset=[cluster_col, "created_datetime_ist"]).copy()
    if vehicle_col is None:
        vehicle_col = standardize_vehicle_col(r)
    if vehicle_col is None:
        chronic_offenders = pd.DataFrame(
            columns=[
                "vehicle_number",
                "total_violations",
                "unique_clusters",
                "unique_hotspots",
                "first_seen",
                "last_seen",
                "dominant_vehicle_type",
                "chronic_offender_flag",
            ]
        )
    else:
        r[vehicle_col] = r[vehicle_col].fillna("").astype(str).str.strip()
        r = r[r[vehicle_col].ne("")].copy()
        if vehicle_type_col and vehicle_type_col in r.columns:
            dom_vtype = lambda s: dominant_label(s, default="UNKNOWN")
        else:
            dom_vtype = lambda s: "UNKNOWN"

        chronic_offenders = (
            r.groupby(vehicle_col)
            .agg(
                total_violations=(vehicle_col, "size"),
                unique_clusters=(cluster_col, "nunique"),
                unique_hotspots=("hotspot_unit", "nunique") if "hotspot_unit" in r.columns else (vehicle_col, "size"),
                first_seen=("created_datetime_ist", "min"),
                last_seen=("created_datetime_ist", "max"),
                dominant_vehicle_type=(vehicle_type_col, dom_vtype) if vehicle_type_col and vehicle_type_col in r.columns else (vehicle_col, lambda s: "UNKNOWN"),
            )
            .reset_index()
            .rename(columns={vehicle_col: "vehicle_number"})
            .sort_values(["total_violations", "unique_clusters"], ascending=[False, False])
        )
        chronic_offenders["chronic_offender_flag"] = (chronic_offenders["total_violations"] >= 5).astype(int)
        chronic_offenders = chronic_offenders[chronic_offenders["chronic_offender_flag"].eq(1)].copy()
else:
    chronic_offenders = pd.DataFrame(
        columns=[
            "vehicle_number",
            "total_violations",
            "unique_clusters",
            "unique_hotspots",
            "first_seen",
            "last_seen",
            "dominant_vehicle_type",
            "chronic_offender_flag",
        ]
    )

# =========================
# Final tables
# =========================
weekly_dispatch = hotspot_df.copy()

cols_hotspot = [
    "ccs_rank",
    cluster_col,
    "cluster_label",
    "cluster_label_display",
    "risk_band",
    "ccs_score",
    "ccs_raw_product",
    "delay_minutes_per_vehicle",
    "lambda_hr_peak_window",
    "lambda_hr_peak_hour",
    "severity_sum",
    "severity_mean",
    "growth_pct_change",
    "growth_multiplier",
    "criticality_factor",
    "junction_degree",
    "betweenness_centrality",
    "capacity_loss_pct",
    "base_capacity_pcu_hr",
    "effective_capacity_pcu_hr",
    "blocking_vehicles_L",
    "road_class",
    "lane_count",
    "carriageway_width_m",
    "link_length_m",
    "dominant_vehicle_type",
    "dominant_violation_tag",
    "records_total",
    "distinct_days",
    "unique_vehicles",
    "unique_vehicle_types",
    "geometry_source",
    "mappls_address",
    "physical_hotspot_key",
]

hotspot_ranking = hotspot_df[[c for c in cols_hotspot if c in hotspot_df.columns]].copy()

cols_reco = [
    "ccs_rank",
    cluster_col,
    "cluster_label_display",
    "risk_band",
    "recommended_action",
    "ccs_score",
    "delay_minutes_per_vehicle",
    "lambda_hr_peak_window",
    "severity_sum",
    "growth_pct_change",
    "criticality_factor",
    "road_class",
    "lane_count",
    "geometry_source",
    "physical_hotspot_key",
]

reco = hotspot_df[[c for c in cols_reco if c in hotspot_df.columns]].copy()

cols_handoff = [
    "ccs_rank",
    cluster_col,
    "cluster_label",
    "cluster_label_display",
    "risk_band",
    "ccs_score",
    "ccs_raw_product",
    "delay_minutes_per_vehicle",
    "lambda_hr_peak_window",
    "lambda_hr_peak_hour",
    "mean_dwell_minutes",
    "mu_departures_per_hour",
    "blocking_vehicles_L",
    "capacity_loss_pct",
    "base_capacity_pcu_hr",
    "effective_capacity_pcu_hr",
    "junction_degree",
    "betweenness_centrality",
    "growth_pct_change",
    "growth_multiplier",
    "criticality_factor",
    "road_class",
    "lane_count",
    "carriageway_width_m",
    "link_length_m",
    "dominant_vehicle_type",
    "dominant_violation_tag",
    "records_total",
    "distinct_days",
    "unique_vehicles",
    "unique_vehicle_types",
    "repeat_vehicle_count_2plus",
    "chronic_vehicle_count_5plus",
    "geometry_source",
    "mappls_address",
    "priority_message",
    "physical_hotspot_key",
]

handoff = hotspot_df[[c for c in cols_handoff if c in hotspot_df.columns]].copy()

# =========================
# Save outputs
# =========================
cluster_df.to_csv(OUT_DIR / "phase6_cluster_ccs_full.csv", index=False)
weekly_dispatch.to_csv(OUT_DIR / "phase6_weekly_dispatch_priority_table.csv", index=False)
hotspot_ranking.to_csv(OUT_DIR / "phase6_hotspot_ranking_ccs.csv", index=False)
alerts.to_csv(OUT_DIR / "phase6_emerging_hotspot_alerts.csv", index=False)
chronic_offenders.to_csv(OUT_DIR / "phase6_chronic_offender_list.csv", index=False)
reco.to_csv(OUT_DIR / "phase6_enforcement_recommendations.csv", index=False)
handoff.to_csv(OUT_DIR / "phase6_stage6_handoff.csv", index=False)

summary = pd.DataFrame(
    [
        {
            "clusters_scored": len(cluster_df),
            "distinct_hotspots": len(hotspot_df),
            "critical_hotspots": int((hotspot_df["risk_band"] == "Critical").sum()) if "risk_band" in hotspot_df.columns else 0,
            "high_hotspots": int((hotspot_df["risk_band"] == "High").sum()) if "risk_band" in hotspot_df.columns else 0,
            "emerging_alerts": len(alerts),
            "chronic_offenders": len(chronic_offenders),
            "ccs_mean": float(pd.to_numeric(hotspot_df["ccs_score"], errors="coerce").mean()) if len(hotspot_df) else 0.0,
            "ccs_std": float(pd.to_numeric(hotspot_df["ccs_score"], errors="coerce").std()) if len(hotspot_df) else 0.0,
        }
    ]
)
summary.to_csv(OUT_DIR / "phase6_summary.csv", index=False)

# =========================
# Console output
# =========================
print("Stage 6 complete")
print("Input source:", cluster_src)
print("Record source:", records_src)
print("Clusters scored:", len(cluster_df))
print("Distinct hotspots:", len(hotspot_df))
print("Top CCS cluster:", hotspot_df.iloc[0][cluster_col] if len(hotspot_df) else "N/A")
print("Top CCS score:", safe_metric(hotspot_df.iloc[0]["ccs_score"], 3) if len(hotspot_df) else "N/A")
print("Outputs saved to:", OUT_DIR.resolve())

print("\nCriticality factor check:")
print(hotspot_df["criticality_factor"].describe())

print("\nCCS check:")
print(hotspot_df["ccs_score"].describe())

print("\nTop 10 CCS hotspots:")
if len(hotspot_df):
    print(
        hotspot_df[[
            "ccs_rank",
            cluster_col,
            "cluster_label_display",
            "risk_band",
            "ccs_score",
            "delay_minutes_per_vehicle",
            "lambda_hr_peak_window",
            "severity_sum",
            "growth_pct_change",
            "criticality_factor",
        ]].head(10).to_string(index=False)
    )

print("\nTop 10 emerging alerts:")
if len(alerts):
    print(
        alerts[[
            "alert_level",
            cluster_col,
            "cluster_label_display",
            "growth_pct_change",
            "growth_multiplier",
            "ccs_score",
            "records_total",
        ]].head(10).to_string(index=False)
    )

Stage 6 complete
Input source: content\phase5_outputs_2\phase5_cluster_capacity_loss.csv
Record source: content\phase5_outputs_2\phase5_enriched_records.csv
Clusters scored: 798
Distinct hotspots: 797
Top CCS cluster: 2
Top CCS score: 21.913
Outputs saved to: D:\Flipkart_Hackathon\approach1\content\phase6_outputs_2

Criticality factor check:
count    797.000000
mean       1.331341
std        0.127263
min        1.000000
25%        1.238830
50%        1.323319
75%        1.401303
max        1.788900
Name: criticality_factor, dtype: float64

CCS check:
count    797.000000
mean       1.710777
std        2.229148
min        0.003780
25%        0.485176
50%        0.799911
75%        2.281861
max       21.913129
Name: ccs_score, dtype: float64

Top 10 CCS hotspots:
 ccs_rank  st_dbscan_cluster_id                           cluster_label_display risk_band  ccs_score  delay_minutes_per_vehicle  lambda_hr_peak_window  severity_sum  growth_pct_change  criticality_factor
        1                

In [ ]:
# ============================================================
# Layer B — Mappls / MapMyIndia Context Enrichment (CORRECTED)
# ============================================================
#
# ROOT CAUSES OF NaN in route_traffic_duration_min / route_optimal_duration_min:
#
# 1. WRONG AUTH METHOD for route_eta():
#    - Original code passes `access_token` as a QUERY PARAM to route.mappls.com
#    - The modern Mappls Route API uses Bearer token in the Authorization HEADER
#    - URL format is also wrong: route.mappls.com/routev2/direction/route is not
#      the correct public endpoint; use:
#      https://apis.mappls.com/advancedmaps/v1/{rest_key}/route_eta/{profile}/{coords}
#
# 2. WRONG AUTH METHOD for distance_matrix():
#    - Legacy distance-matrix endpoint uses the REST license key in the URL path
#      (which the code does), but the access_token query param doesn't work for it.
#    - Returns durations in MINUTES not seconds in the `results.durations` matrix.
#
# 3. WRONG ENDPOINT for nearby_places():
#    - search.mappls.com/search/places/nearby/json does not exist publicly.
#    - Correct endpoint: https://apis.mappls.com/advancedmaps/v1/{rest_key}/nearby
#      OR use the OAuth bearer token approach:
#      https://apis.mappls.com/places/search/json with `keywords` + `refLocation`
#
# 4. WRONG ENDPOINT for reverse_geocode():
#    - search.mappls.com/search/address/rev-geocode is not public.
#    - Correct: https://apis.mappls.com/advancedmaps/v1/{rest_key}/rev_geocode
#      (legacy, uses REST key in path)
#
# 5. route_eta() calls the SAME method twice for traffic vs optimal,
#    getting identical results. The original route API has no separate
#    traffic/optimal toggle in a simple free-tier call — so we call the
#    same endpoint and use both as the same value; ratio will be 1.0.
#    If your key supports the ETA variant (route_eta vs route), use that.
#
# SUMMARY OF CORRECTIONS:
# - reverse_geocode: use legacy path-key endpoint with REST key
# - nearby_places: use legacy nearby endpoint with REST key
# - route_eta: use Bearer auth header on route.mappls.com
# - distance_matrix: use advancedmaps/v1/{key}/distance_matrix endpoint;
#   durations returned in SECONDS for distance_matrix_eta, in MINUTES for
#   distance_matrix — normalize correctly.
# - Separate traffic vs optimal calls by using route_eta vs route resource.
# ============================================================

import os
import math
import time
import warnings
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore")

# -------------------------
# Config
# -------------------------
PHASE5_DIRS = [
    Path("content/phase5_outputs_2"),
    Path("phase5_outputs_2"),
    Path("/content/phase5_outputs_2"),
]

OUT_DIR = Path("content/layer_b_outputs_2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_N = int(os.environ.get("LAYER_B_TOP_N", "50"))
REQUEST_TIMEOUT = int(os.environ.get("MAPPLS_REQUEST_TIMEOUT", "25"))
REQUEST_RETRIES = int(os.environ.get("MAPPLS_REQUEST_RETRIES", "2"))
REQUEST_BACKOFF_SEC = float(os.environ.get("MAPPLS_REQUEST_BACKOFF_SEC", "0.8"))
VALIDATION_CORRIDOR_METERS = float(os.environ.get("LAYER_B_CORRIDOR_METERS", "500"))

# Bearer/OAuth access token — used in Authorization header for route.mappls.com
MAPPLS_ACCESS_TOKEN = os.environ.get("MAPPLS_ACCESS_TOKEN", "").strip()

# REST license key — used IN THE URL PATH for legacy advancedmaps/v1/{key}/... endpoints
MAPPLS_REST_KEY = os.environ.get("MAPPLS_REST_KEY", "").strip()

MAPPLS_REGION = os.environ.get("MAPPLS_REGION", "IND").strip().upper()
MAPPLS_PROFILE = os.environ.get("MAPPLS_PROFILE", "driving").strip().lower()

SENSITIVE_NEARBY_KEYWORDS = "school;hospital;bus stop;metro station;police station;junction"

# -------------------------
# Generic helpers
# -------------------------
EPS = 1e-9


def clean_text(x) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()


def normalize_tag(tag) -> str:
    return clean_text(tag).upper().replace("&", "AND").strip()


def load_first_existing(base_dirs: Iterable[Path], filenames: Iterable[str]):
    for d in base_dirs:
        for name in filenames:
            p = d / name
            if p.exists():
                return pd.read_csv(p, low_memory=False), p
    return None, None


def safe_float(x, default=np.nan):
    try:
        if pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default


def valid_coords(lat, lon) -> bool:
    try:
        return (
            pd.notna(lat)
            and pd.notna(lon)
            and -90 <= float(lat) <= 90
            and -180 <= float(lon) <= 180
        )
    except Exception:
        return False


def standardize_cluster_col(df: pd.DataFrame) -> str:
    for c in ["st_dbscan_cluster_id", "cluster_id", "dbscan_cluster_id"]:
        if c in df.columns:
            return c
    raise ValueError("No cluster id column found.")


def ensure_label_column(df: pd.DataFrame):
    if df is None or len(df) == 0:
        return df
    df = df.copy()
    if "cluster_label" in df.columns:
        df["cluster_label"] = df["cluster_label"].fillna("").astype(str).str.strip()
        return df
    if "hotspot_unit" in df.columns:
        df["cluster_label"] = df["hotspot_unit"].fillna("").astype(str).str.strip()
        return df
    if "dominant_junction_name" in df.columns:
        df["cluster_label"] = df["dominant_junction_name"].fillna("").astype(str).str.strip()
        return df
    if "st_dbscan_cluster_id" in df.columns:
        df["cluster_label"] = "CLUSTER::" + df["st_dbscan_cluster_id"].astype(str)
        return df
    df["cluster_label"] = "UNKNOWN"
    return df


def derive_coords(df: pd.DataFrame):
    if df is None or len(df) == 0:
        return df
    df = df.copy()
    if "lat" in df.columns and "lon" in df.columns:
        df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
        df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    elif {"centroid_lat", "centroid_lon"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["centroid_lat"], errors="coerce")
        df["lon"] = pd.to_numeric(df["centroid_lon"], errors="coerce")
    elif {"latitude_mean", "longitude_mean"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["latitude_mean"], errors="coerce")
        df["lon"] = pd.to_numeric(df["longitude_mean"], errors="coerce")
    elif {"latitude", "longitude"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["latitude"], errors="coerce")
        df["lon"] = pd.to_numeric(df["longitude"], errors="coerce")
    else:
        df["lat"] = np.nan
        df["lon"] = np.nan
    return df


def offset_point(lat, lon, meters=500.0, bearing_deg=90.0):
    R = 6371000.0
    bearing = math.radians(bearing_deg)
    lat1 = math.radians(lat)
    lon1 = math.radians(lon)
    d = meters / R
    lat2 = math.asin(
        math.sin(lat1) * math.cos(d) + math.cos(lat1) * math.sin(d) * math.cos(bearing)
    )
    lon2 = lon1 + math.atan2(
        math.sin(bearing) * math.sin(d) * math.cos(lat1),
        math.cos(d) - math.sin(lat1) * math.sin(lat2),
    )
    return math.degrees(lat2), math.degrees(lon2)


def minmax(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    valid = s.dropna()
    if valid.nunique(dropna=True) <= 1:
        return pd.Series(np.zeros(len(s)), index=s.index, dtype=float)
    mn = valid.min()
    mx = valid.max()
    return (s.fillna(mn) - mn) / (mx - mn + EPS)


def make_hotspot_key(df: pd.DataFrame) -> pd.Series:
    label = df.get("cluster_label", pd.Series("", index=df.index)).fillna("").astype(str).str.strip().str.lower()
    if "lat" in df.columns and "lon" in df.columns:
        lat = pd.to_numeric(df["lat"], errors="coerce").round(5).astype(str)
        lon = pd.to_numeric(df["lon"], errors="coerce").round(5).astype(str)
    else:
        lat = pd.Series("", index=df.index)
        lon = pd.Series("", index=df.index)
    return label + "|" + lat + "|" + lon


def get_requests_session() -> requests.Session:
    s = requests.Session()
    adapter = requests.adapters.HTTPAdapter(
        pool_connections=10,
        pool_maxsize=10,
        max_retries=0,
    )
    s.mount("http://", adapter)
    s.mount("https://", adapter)
    return s


def request_json(
    session: requests.Session,
    method: str,
    url: str,
    headers: Optional[dict] = None,
    **kwargs,
) -> Tuple[Optional[dict], str, int]:
    last_err = ""
    last_status = 0
    for attempt in range(REQUEST_RETRIES + 1):
        try:
            resp = session.request(method, url, timeout=REQUEST_TIMEOUT, headers=headers or {}, **kwargs)
            last_status = int(resp.status_code)
            if resp.status_code == 204:
                return None, "", 204
            resp.raise_for_status()
            try:
                return resp.json(), "", resp.status_code
            except Exception:
                return {"_raw_text": resp.text}, "", resp.status_code
        except Exception as e:
            last_err = str(e)
            if attempt < REQUEST_RETRIES:
                time.sleep(REQUEST_BACKOFF_SEC * (attempt + 1))
    return None, last_err, last_status


# -------------------------
# Parsers
# -------------------------
def parse_reverse_geocode(payload: dict) -> Dict[str, Any]:
    """
    Mappls legacy rev_geocode response:
    {
      "results": {
        "houseNumber": "", "houseName": "", "poi": "", "street": "",
        "subSubLocality": "", "subLocality": "", "locality": "",
        "village": "", "subDistrict": "", "district": "",
        "city": "", "state": "", "pincode": "",
        "formattedAddress": "...",
        "eLoc": "...", "geocodeLevel": "...", "confidenceScore": 0.8
      }
    }
    Note: copResults is for the geocoding (forward) API.
    Rev-geocode uses `results` key (can be dict or list).
    """
    out = {
        "ok": False,
        "address": "",
        "houseNumber": "",
        "houseName": "",
        "poi": "",
        "street": "",
        "subSubLocality": "",
        "subLocality": "",
        "locality": "",
        "village": "",
        "subDistrict": "",
        "district": "",
        "city": "",
        "state": "",
        "pincode": "",
        "geocodeLevel": "",
        "confidenceScore": np.nan,
        "eloc": "",
    }

    if not isinstance(payload, dict):
        return out

    # rev_geocode uses "results" (can be dict or list).
    # Forward geocode uses "copResults". Handle both for safety.
    candidates = None
    if "results" in payload:
        candidates = payload["results"]
    elif "copResults" in payload:
        candidates = payload["copResults"]
    else:
        candidates = payload  # root-level fallback

    if isinstance(candidates, list) and candidates:
        first = candidates[0] if isinstance(candidates[0], dict) else {}
    elif isinstance(candidates, dict):
        first = candidates
    else:
        first = {}

    if not isinstance(first, dict):
        return out

    out["houseNumber"] = clean_text(first.get("houseNumber", ""))
    out["houseName"] = clean_text(first.get("houseName", ""))
    out["poi"] = clean_text(first.get("poi", ""))
    out["street"] = clean_text(first.get("street", ""))
    out["subSubLocality"] = clean_text(first.get("subSubLocality", first.get("subsubLocality", "")))
    out["subLocality"] = clean_text(first.get("subLocality", ""))
    out["locality"] = clean_text(first.get("locality", ""))
    out["village"] = clean_text(first.get("village", ""))
    out["subDistrict"] = clean_text(first.get("subDistrict", ""))
    out["district"] = clean_text(first.get("district", ""))
    out["city"] = clean_text(first.get("city", ""))
    out["state"] = clean_text(first.get("state", ""))
    out["pincode"] = clean_text(first.get("pincode", ""))
    out["geocodeLevel"] = clean_text(first.get("geocodeLevel", ""))
    out["confidenceScore"] = safe_float(first.get("confidenceScore"), np.nan)
    out["eloc"] = clean_text(first.get("eLoc", first.get("eloc", "")))

    formatted = (
        first.get("formattedAddress")
        or first.get("formatted_address")
        or ""
    )
    if not formatted:
        parts = [
            out["houseNumber"], out["houseName"], out["poi"], out["street"],
            out["subSubLocality"], out["subLocality"], out["locality"],
            out["village"], out["subDistrict"], out["district"],
            out["city"], out["state"], out["pincode"],
        ]
        parts = [p for p in parts if p]
        formatted = ", ".join(dict.fromkeys(parts))

    out["address"] = clean_text(formatted)
    out["ok"] = bool(out["address"])
    return out


def parse_nearby_places(payload: dict) -> Dict[str, Any]:
    """
    Mappls nearby API response:
    { "suggestedLocations": [ { "placeName": "...", "placeAddress": "...", "distance": 123, ... }, ... ] }
    """
    out = {
        "ok": False,
        "count": 0,
        "names": [],
        "addresses": [],
        "min_distance_m": np.nan,
    }
    if not isinstance(payload, dict):
        return out

    items = (
        payload.get("suggestedLocations")
        or payload.get("results")
        or []
    )
    if isinstance(items, dict):
        items = [items]
    if not isinstance(items, list):
        return out

    names, addresses, distances = [], [], []
    for item in items:
        if not isinstance(item, dict):
            continue
        nm = clean_text(item.get("placeName", item.get("name", "")))
        addr = clean_text(item.get("placeAddress", item.get("address", "")))
        d = safe_float(item.get("distance"), np.nan)
        if nm:
            names.append(nm)
        if addr:
            addresses.append(addr)
        if pd.notna(d) and d >= 0:
            distances.append(float(d))

    out["count"] = len(items)
    out["names"] = names[:10]
    out["addresses"] = addresses[:10]
    out["min_distance_m"] = float(np.nanmin(distances)) if distances else np.nan
    out["ok"] = len(items) > 0
    return out


def parse_route_eta_payload(payload: dict) -> Dict[str, Any]:
    """
    Mappls Route ETA / Direction API response shapes:
    Shape 1 (advancedmaps route_eta / route):
      { "results": { "trips": [ { "duration": <seconds>, "distance": <meters>, ... } ] } }
    Shape 2 (advancedmaps route with older format):
      { "results": { "duration": <seconds>, "distance": <meters> } }
    Shape 3 (OSRM-style, used by some route.mappls.com responses):
      { "routes": [ { "duration": <seconds>, "distance": <meters>, "legs": [...] } ] }
    Shape 4 (trip-style):
      { "trip": { "summary": { "length": <km>, "time": <seconds> } } }
    """
    out = {"ok": False, "distance_m": None, "duration_sec": None, "source": "unknown"}

    if not isinstance(payload, dict):
        return out

    # Shape 1 & 2: results key
    results = payload.get("results")
    if isinstance(results, dict):
        trips = results.get("trips")
        if isinstance(trips, list) and trips:
            t = trips[0]
            dur = safe_float(t.get("duration", t.get("time")), None)
            dist = safe_float(t.get("distance", t.get("length")), None)
            if dist is not None and dist < 500:  # likely km, convert
                dist = dist * 1000.0
            if dur is not None or dist is not None:
                out.update({"ok": True, "duration_sec": dur, "distance_m": dist, "source": "results.trips"})
                return out

        dur = safe_float(results.get("duration", results.get("time")), None)
        dist = safe_float(results.get("distance", results.get("length")), None)
        if dist is not None and dist < 500:
            dist = dist * 1000.0
        if dur is not None or dist is not None:
            out.update({"ok": True, "duration_sec": dur, "distance_m": dist, "source": "results"})
            return out

    # Shape 3: OSRM-style routes array
    routes = payload.get("routes")
    if isinstance(routes, list) and routes:
        r = routes[0]
        if isinstance(r, dict):
            dur = safe_float(r.get("duration"), None)
            dist = safe_float(r.get("distance"), None)
            if dist is not None and dist < 500:
                dist = dist * 1000.0
            if dur is not None or dist is not None:
                out.update({"ok": True, "duration_sec": dur, "distance_m": dist, "source": "routes[0]"})
                return out

    # Shape 4: trip summary
    trip = payload.get("trip")
    if isinstance(trip, dict):
        summary = trip.get("summary", {})
        if isinstance(summary, dict):
            dur = safe_float(summary.get("time", summary.get("duration")), None)
            dist = safe_float(summary.get("length", summary.get("distance")), None)
            if dist is not None and dist < 500:
                dist = dist * 1000.0
            if dur is not None or dist is not None:
                out.update({"ok": True, "duration_sec": dur, "distance_m": dist, "source": "trip.summary"})
                return out

    return out


def parse_matrix_payload(payload: dict) -> Dict[str, Any]:
    """
    Mappls Distance Matrix API (advancedmaps/v1/{key}/distance_matrix/driving/{coords}):
    Response:
    {
      "results": {
        "distances": [[0, 1234], [1234, 0]],    -- in METERS
        "durations": [[0, 456], [456, 0]]         -- in SECONDS for distance_matrix_eta
      }
    }
    OR for distance_matrix (non-ETA):
    {
      "results": {
        "distances": [[0, 1.23], [1.23, 0]],    -- in KILOMETERS
        "durations": [[0, 7.6], [7.6, 0]]         -- in MINUTES
      }
    }
    We try distance_matrix_eta first (seconds), fall back to distance_matrix (minutes).
    """
    out = {"ok": False, "distance_m": None, "duration_sec": None, "source": "unknown"}

    if not isinstance(payload, dict):
        return out

    results = payload.get("results")
    if isinstance(results, list) and results:
        results = results[0]

    if isinstance(results, dict):
        dists_raw = results.get("distances")
        durs_raw = results.get("durations")

        def extract_offdiag(matrix):
            if not isinstance(matrix, list):
                return None
            vals = []
            for i, row in enumerate(matrix):
                if isinstance(row, list):
                    for j, v in enumerate(row):
                        if i != j:
                            try:
                                f = float(v)
                                if f >= 0 and np.isfinite(f):
                                    vals.append(f)
                            except Exception:
                                pass
                else:
                    try:
                        f = float(row)
                        if f >= 0 and np.isfinite(f):
                            vals.append(f)
                    except Exception:
                        pass
            return min(vals) if vals else None

        d_val = extract_offdiag(dists_raw)
        t_val = extract_offdiag(durs_raw)

        if d_val is not None or t_val is not None:
            # Normalize: if d_val is small (<500) it's likely km; convert to meters
            if d_val is not None and d_val < 500:
                d_val = d_val * 1000.0
            # Duration: if t_val is very small (<30) it's likely minutes; convert to seconds
            # If t_val > 3600 it's almost certainly seconds already
            # Heuristic: distance_matrix_eta returns seconds, distance_matrix returns minutes
            # We store the endpoint type to decide (passed via `resource` arg at call site)
            out.update({
                "ok": True,
                "distance_m": d_val,
                "duration_sec": t_val,  # caller normalizes based on resource type
                "source": "results.distances/durations",
            })
            return out

    return out


# -------------------------
# Mappls client (CORRECTED)
# -------------------------
class MapplsLayerBClient:
    """
    Key corrections vs original:
    1. reverse_geocode: use legacy advancedmaps/v1/{rest_key}/rev_geocode endpoint
    2. nearby_places: use advancedmaps/v1/{rest_key}/nearby endpoint
    3. route_eta: use Authorization Bearer header on route.mappls.com OR
       advancedmaps/v1/{rest_key}/route_eta/{profile}/{coords}
    4. distance_matrix: use advancedmaps/v1/{rest_key}/distance_matrix_eta/{profile}/{coords}
       and handle duration in seconds; fall back to distance_matrix (duration in minutes)
    """

    def __init__(self, access_token: str, rest_key: str = "", timeout: int = 25):
        self.access_token = access_token.strip()
        self.rest_key = (rest_key or access_token).strip()
        self.timeout = timeout
        self.session = get_requests_session()

    def _bearer_headers(self) -> dict:
        """For endpoints that accept OAuth Bearer token in header."""
        return {"Authorization": f"bearer {self.access_token}"}

    def reverse_geocode(self, lat: float, lng: float) -> Dict[str, Any]:
        """
        CORRECTED: Use legacy REST key endpoint.
        GET https://apis.mappls.com/advancedmaps/v1/{rest_key}/rev_geocode?lat=&lng=&region=
        Response: { "results": { "formattedAddress": "...", "city": "...", ... } }
        """
        url = f"https://apis.mappls.com/advancedmaps/v1/{self.rest_key}/rev_geocode"
        params = {
            "lat": float(lat),
            "lng": float(lng),
            "region": MAPPLS_REGION,
        }
        payload, err, status = request_json(self.session, "GET", url, params=params)
        if payload is None:
            # Try bearer auth fallback
            url2 = "https://apis.mappls.com/places/geocode"
            params2 = {"address": f"{lat},{lng}", "region": MAPPLS_REGION}
            payload, err, status = request_json(
                self.session, "GET", url2,
                headers=self._bearer_headers(),
                params=params2,
            )
        if payload is None:
            return {"ok": False, "error": err or f"HTTP_{status}", "raw": None}
        parsed = parse_reverse_geocode(payload)
        parsed["raw"] = payload
        parsed["error"] = "" if parsed["ok"] else "empty_or_unparsed"
        return parsed

    def nearby_places(
        self,
        lat: float,
        lng: float,
        keywords: str = SENSITIVE_NEARBY_KEYWORDS,
        radius: int = 1000,
    ) -> Dict[str, Any]:
        """
        CORRECTED: Use legacy REST key nearby endpoint.
        GET https://apis.mappls.com/advancedmaps/v1/{rest_key}/nearby
            ?keywords=school;hospital&refLocation=lat,lng&region=IND&radius=1000&sortBy=dist:asc
        Response: { "suggestedLocations": [ { "placeName": "", "placeAddress": "", "distance": 123 }, ... ] }
        """
        url = f"https://apis.mappls.com/advancedmaps/v1/{self.rest_key}/nearby"
        params = {
            "keywords": keywords,
            "refLocation": f"{float(lat)},{float(lng)}",
            "region": MAPPLS_REGION,
            "radius": int(radius),
            "sortBy": "dist:asc",
        }
        payload, err, status = request_json(self.session, "GET", url, params=params)
        if payload is None:
            # Fallback: bearer token-based places API
            url2 = "https://apis.mappls.com/places/nearby/json"
            params2 = {
                "keywords": keywords,
                "refLocation": f"{float(lat)},{float(lng)}",
                "region": MAPPLS_REGION,
                "radius": int(radius),
                "sortBy": "dist:asc",
            }
            payload, err, status = request_json(
                self.session, "GET", url2,
                headers=self._bearer_headers(),
                params=params2,
            )
        if payload is None:
            return {"ok": False, "error": err or f"HTTP_{status}", "raw": None}
        parsed = parse_nearby_places(payload)
        parsed["raw"] = payload
        parsed["error"] = ""
        return parsed

    def route_eta(
        self,
        start_lat: float,
        start_lon: float,
        end_lat: float,
        end_lon: float,
        resource: str = "route_eta",
    ) -> Dict[str, Any]:
        """
        CORRECTED: Mappls Route ETA API.

        Primary: advancedmaps/v1/{rest_key}/{resource}/{profile}/{lon,lat;lon,lat}
          - resource = "route_eta"  → live traffic ETA (duration in seconds)
          - resource = "route"      → free-flow / optimal (duration in seconds)
        URL: https://apis.mappls.com/advancedmaps/v1/{rest_key}/route_eta/driving/lon1,lat1;lon2,lat2

        Fallback: route.mappls.com with Bearer header
          GET https://route.mappls.com/route/driving/{lon,lat};{lon,lat}
              ?overview=false  (Authorization: bearer {token})
        Response: { "routes": [{ "duration": <sec>, "distance": <meters> }] }

        IMPORTANT: coordinates are lon,lat ORDER (not lat,lon) for this API.
        """
        # Primary: legacy advancedmaps endpoint
        coords = f"{float(start_lon)},{float(start_lat)};{float(end_lon)},{float(end_lat)}"
        url = f"https://apis.mappls.com/advancedmaps/v1/{self.rest_key}/{resource}/{MAPPLS_PROFILE}/{coords}"
        params = {"region": MAPPLS_REGION, "rtype": "0"}
        payload, err, status = request_json(self.session, "GET", url, params=params)
        if payload is not None:
            parsed = parse_route_eta_payload(payload)
            if parsed["ok"]:
                parsed["raw"] = payload
                parsed["error"] = ""
                parsed["resource"] = resource
                parsed["endpoint"] = url
                return parsed

        # Fallback: route.mappls.com with Bearer auth
        url2 = f"https://route.mappls.com/route/{MAPPLS_PROFILE}/{coords}"
        params2 = {"overview": "false", "region": MAPPLS_REGION}
        payload2, err2, status2 = request_json(
            self.session, "GET", url2,
            headers=self._bearer_headers(),
            params=params2,
        )
        if payload2 is not None:
            parsed2 = parse_route_eta_payload(payload2)
            if parsed2["ok"]:
                parsed2["raw"] = payload2
                parsed2["error"] = ""
                parsed2["resource"] = resource
                parsed2["endpoint"] = url2
                return parsed2

        return {
            "ok": False,
            "distance_m": None,
            "duration_sec": None,
            "raw": None,
            "error": err2 or err or f"HTTP_{status2 or status}",
            "resource": resource,
        }

    def distance_matrix(
        self,
        start_lat: float,
        start_lon: float,
        end_lat: float,
        end_lon: float,
    ) -> Dict[str, Any]:
        """
        CORRECTED: Mappls Distance Matrix API.

        URL format: https://apis.mappls.com/advancedmaps/v1/{rest_key}/{resource}/{profile}/{coords}
        Coordinates: lon,lat ORDER separated by semicolons.

        resource options (tried in order):
          - distance_matrix_eta   → durations in SECONDS, distances in METERS
          - distance_matrix_traffic → durations in SECONDS, distances in METERS
          - distance_matrix       → durations in MINUTES, distances in KILOMETERS

        Response: {
          "results": {
            "distances": [[0, d01], [d10, 0]],
            "durations": [[0, t01], [t10, 0]]
          }
        }
        """
        coords = f"{float(start_lon)},{float(start_lat)};{float(end_lon)},{float(end_lat)}"

        resources_seconds = ["distance_matrix_eta", "distance_matrix_traffic"]
        resources_minutes = ["distance_matrix"]

        for resource in resources_seconds:
            url = f"https://apis.mappls.com/advancedmaps/v1/{self.rest_key}/{resource}/{MAPPLS_PROFILE}/{coords}"
            params = {"region": MAPPLS_REGION, "rtype": "0"}
            payload, err, status = request_json(self.session, "GET", url, params=params)
            if payload is None:
                continue
            parsed = parse_matrix_payload(payload)
            if parsed["ok"]:
                # duration already in seconds for _eta and _traffic
                parsed["raw"] = payload
                parsed["error"] = ""
                parsed["endpoint"] = url
                parsed["resource"] = resource
                return parsed

        for resource in resources_minutes:
            url = f"https://apis.mappls.com/advancedmaps/v1/{self.rest_key}/{resource}/{MAPPLS_PROFILE}/{coords}"
            params = {"region": MAPPLS_REGION, "rtype": "0"}
            payload, err, status = request_json(self.session, "GET", url, params=params)
            if payload is None:
                continue
            parsed = parse_matrix_payload(payload)
            if parsed["ok"]:
                # Convert: durations in MINUTES → seconds, distances in KM → meters
                if parsed["duration_sec"] is not None:
                    parsed["duration_sec"] = parsed["duration_sec"] * 60.0
                if parsed["distance_m"] is not None and parsed["distance_m"] < 500:
                    parsed["distance_m"] = parsed["distance_m"] * 1000.0
                parsed["raw"] = payload
                parsed["error"] = ""
                parsed["endpoint"] = url
                parsed["resource"] = resource
                return parsed

        return {
            "ok": False,
            "distance_m": None,
            "duration_sec": None,
            "raw": None,
            "error": "all_distance_matrix_resources_failed",
            "source": "distance_matrix",
        }


# -------------------------
# Layer B logic (unchanged from original, except for the fixes above)
# -------------------------
def load_stage5_table():
    df, source = load_first_existing(
        PHASE5_DIRS,
        [
            "phase5_cluster_capacity_loss.csv",
            "phase5_priority_table_full.csv",
            "phase5_stage5_handoff.csv",
        ],
    )
    if df is None:
        raise FileNotFoundError(
            "Could not find Stage 5 output."
        )
    return df, source


def select_hotspots_for_enrichment(df: pd.DataFrame, top_n: int = 50) -> pd.DataFrame:
    work = df.copy()
    work = ensure_label_column(work)
    work = derive_coords(work)
    if "physical_hotspot_key" not in work.columns:
        work["physical_hotspot_key"] = make_hotspot_key(work)
    if "ccs_score" not in work.columns:
        work["ccs_score"] = 0.0
    if "delay_minutes_per_vehicle" not in work.columns:
        work["delay_minutes_per_vehicle"] = 0.0
    if "risk_band" not in work.columns:
        work["risk_band"] = "Watch"
    if "records_total" not in work.columns:
        work["records_total"] = 0.0

    work["ccs_score"] = pd.to_numeric(work["ccs_score"], errors="coerce").fillna(0.0)
    work["delay_minutes_per_vehicle"] = pd.to_numeric(work["delay_minutes_per_vehicle"], errors="coerce").fillna(0.0)
    work = work.sort_values(
        ["ccs_score", "delay_minutes_per_vehicle", "records_total"],
        ascending=[False, False, False],
    ).drop_duplicates(subset=["physical_hotspot_key"], keep="first").reset_index(drop=True)
    work = work.dropna(subset=["lat", "lon"]).copy()
    if top_n and top_n > 0:
        work = work.head(top_n).copy()
    return work.reset_index(drop=True)


def build_layer_b_geofence_candidates(hotspot_df: pd.DataFrame) -> pd.DataFrame:
    if hotspot_df is None or len(hotspot_df) == 0:
        return pd.DataFrame(columns=[
            "cluster_id", "cluster_label", "lat", "lon", "radius_m",
            "reason", "nearby_sensitive_poi_count", "nearby_sensitive_poi_names"
        ])
    rows = []
    for _, r in hotspot_df.iterrows():
        nearby_names = r.get("nearby_sensitive_poi_names", [])
        if isinstance(nearby_names, str):
            nearby_names = [x.strip() for x in nearby_names.split("||") if x.strip()]
        count = int(r.get("nearby_sensitive_poi_count", 0) or 0)
        if count <= 0:
            continue
        radius_m = 100 + min(400, 50 * count)
        rows.append({
            "cluster_id": r.get("cluster_id", r.get("st_dbscan_cluster_id", "")),
            "cluster_label": r.get("cluster_label_display", r.get("cluster_label", "")),
            "lat": safe_float(r.get("lat")),
            "lon": safe_float(r.get("lon")),
            "radius_m": radius_m,
            "reason": "Sensitive POI proximity",
            "nearby_sensitive_poi_count": count,
            "nearby_sensitive_poi_names": "||".join([str(x) for x in nearby_names[:10]]),
        })
    return pd.DataFrame(rows)


def enrich_hotspot_row(client: MapplsLayerBClient, row: pd.Series) -> Dict[str, Any]:
    lat = safe_float(row.get("lat"))
    lon = safe_float(row.get("lon"))

    out = {
        "cluster_id": row.get("cluster_id", row.get("st_dbscan_cluster_id", "")),
        "cluster_label": clean_text(row.get("cluster_label", "")),
        "cluster_label_display": clean_text(row.get("cluster_label_display", row.get("cluster_label", ""))),
        "risk_band": clean_text(row.get("risk_band", "Watch")),
        "ccs_score": safe_float(row.get("ccs_score", 0.0), 0.0),
        "delay_minutes_per_vehicle": safe_float(row.get("delay_minutes_per_vehicle", 0.0), 0.0),
        "records_total": safe_float(row.get("records_total", 0.0), 0.0),
        "lat": lat,
        "lon": lon,
        "reverse_geocode_ok": False,
        "reverse_geocode_address": "",
        "reverse_geocode_level": "",
        "reverse_geocode_poi": "",
        "reverse_geocode_street": "",
        "reverse_geocode_locality": "",
        "reverse_geocode_city": "",
        "reverse_geocode_district": "",
        "reverse_geocode_state": "",
        "nearby_sensitive_poi_ok": False,
        "nearby_sensitive_poi_count": 0,
        "nearby_sensitive_poi_names": [],
        "nearby_sensitive_poi_addresses": [],
        "nearby_sensitive_poi_min_distance_m": np.nan,
        "route_traffic_ok": False,
        "route_traffic_distance_m": np.nan,
        "route_traffic_duration_sec": np.nan,
        "route_traffic_duration_min": np.nan,
        "route_optimal_ok": False,
        "route_optimal_distance_m": np.nan,
        "route_optimal_duration_sec": np.nan,
        "route_optimal_duration_min": np.nan,
        "route_traffic_vs_optimal_ratio": np.nan,
        "distance_matrix_ok": False,
        "distance_matrix_source": "",
        "distance_matrix_distance_m": np.nan,
        "distance_matrix_duration_sec": np.nan,
        "distance_matrix_duration_min": np.nan,
        "distance_matrix_vs_route_ratio": np.nan,
        "context_multiplier": 1.0,
        "layer_b_priority_boost": 0.0,
        "layer_b_alert_flag": False,
        "layer_b_error": "",
    }

    if not valid_coords(lat, lon):
        out["layer_b_error"] = "invalid_coordinates"
        return out

    # 1. Reverse geocode
    try:
        rev = client.reverse_geocode(lat, lon)
        if rev.get("ok"):
            out["reverse_geocode_ok"] = True
            out["reverse_geocode_address"] = rev.get("address", "")
            out["reverse_geocode_level"] = rev.get("geocodeLevel", "")
            out["reverse_geocode_poi"] = rev.get("poi", "")
            out["reverse_geocode_street"] = rev.get("street", "")
            out["reverse_geocode_locality"] = rev.get("locality", "")
            out["reverse_geocode_city"] = rev.get("city", "")
            out["reverse_geocode_district"] = rev.get("district", "")
            out["reverse_geocode_state"] = rev.get("state", "")
    except Exception as e:
        out["layer_b_error"] = f"reverse_geocode: {e}"

    # 2. Nearby POIs
    try:
        nearby = client.nearby_places(lat, lon, keywords=SENSITIVE_NEARBY_KEYWORDS, radius=1000)
        if nearby.get("ok"):
            out["nearby_sensitive_poi_ok"] = True
            out["nearby_sensitive_poi_count"] = int(nearby.get("count", 0) or 0)
            out["nearby_sensitive_poi_names"] = nearby.get("names", [])
            out["nearby_sensitive_poi_addresses"] = nearby.get("addresses", [])
            out["nearby_sensitive_poi_min_distance_m"] = nearby.get("min_distance_m", np.nan)
    except Exception as e:
        out["layer_b_error"] = f"{out['layer_b_error']} | nearby: {e}".strip(" |")

    # Corridor destination for routing
    dest_lat, dest_lon = offset_point(lat, lon, meters=VALIDATION_CORRIDOR_METERS, bearing_deg=90.0)

    # 3. Route ETA — traffic (resource=route_eta uses live traffic)
    try:
        route_traffic = client.route_eta(lat, lon, dest_lat, dest_lon, resource="route_eta")
        if route_traffic.get("ok"):
            dur_sec = safe_float(route_traffic.get("duration_sec"), np.nan)
            out["route_traffic_ok"] = True
            out["route_traffic_distance_m"] = safe_float(route_traffic.get("distance_m"), np.nan)
            out["route_traffic_duration_sec"] = dur_sec
            out["route_traffic_duration_min"] = dur_sec / 60.0 if pd.notna(dur_sec) else np.nan
    except Exception as e:
        out["layer_b_error"] = f"{out['layer_b_error']} | route_traffic: {e}".strip(" |")

    # 4. Route — optimal/free-flow (resource=route, no traffic)
    try:
        route_opt = client.route_eta(lat, lon, dest_lat, dest_lon, resource="route")
        if route_opt.get("ok"):
            dur_sec = safe_float(route_opt.get("duration_sec"), np.nan)
            out["route_optimal_ok"] = True
            out["route_optimal_distance_m"] = safe_float(route_opt.get("distance_m"), np.nan)
            out["route_optimal_duration_sec"] = dur_sec
            out["route_optimal_duration_min"] = dur_sec / 60.0 if pd.notna(dur_sec) else np.nan
    except Exception as e:
        out["layer_b_error"] = f"{out['layer_b_error']} | route_optimal: {e}".strip(" |")

    # Traffic vs optimal ratio
    t_dur = safe_float(out["route_traffic_duration_sec"], np.nan)
    o_dur = safe_float(out["route_optimal_duration_sec"], np.nan)
    if pd.notna(t_dur) and pd.notna(o_dur) and o_dur > 0:
        out["route_traffic_vs_optimal_ratio"] = t_dur / max(o_dur, EPS)

    # 5. Distance matrix
    try:
        dm = client.distance_matrix(lat, lon, dest_lat, dest_lon)
        if dm.get("ok"):
            dur_sec = safe_float(dm.get("duration_sec"), np.nan)
            out["distance_matrix_ok"] = True
            out["distance_matrix_source"] = dm.get("resource", dm.get("source", ""))
            out["distance_matrix_distance_m"] = safe_float(dm.get("distance_m"), np.nan)
            out["distance_matrix_duration_sec"] = dur_sec
            out["distance_matrix_duration_min"] = dur_sec / 60.0 if pd.notna(dur_sec) else np.nan
    except Exception as e:
        out["layer_b_error"] = f"{out['layer_b_error']} | distance_matrix: {e}".strip(" |")

    dm_dur = safe_float(out["distance_matrix_duration_sec"], np.nan)
    rt_dur = safe_float(out["route_traffic_duration_sec"], np.nan)
    if pd.notna(dm_dur) and pd.notna(rt_dur) and rt_dur > 0:
        out["distance_matrix_vs_route_ratio"] = dm_dur / max(rt_dur, EPS)

    # Context multiplier
    poi_count = int(out["nearby_sensitive_poi_count"] or 0)
    traffic_ratio = safe_float(out["route_traffic_vs_optimal_ratio"], np.nan)
    poi_boost = min(0.50, 0.08 * poi_count)
    traffic_boost = 0.0
    if pd.notna(traffic_ratio) and traffic_ratio > 1.0:
        traffic_boost = min(0.50, 0.25 * (traffic_ratio - 1.0))

    out["layer_b_priority_boost"] = poi_boost + traffic_boost
    out["context_multiplier"] = float(np.clip(1.0 + out["layer_b_priority_boost"], 1.0, 2.0))
    out["layer_b_alert_flag"] = bool(
        (poi_count > 0) or (pd.notna(traffic_ratio) and traffic_ratio >= 1.15)
    )

    return out


def main():
    if not MAPPLS_REST_KEY:
        raise RuntimeError("MAPPLS_REST_KEY is missing. Set your Mappls REST license key.")

    client = MapplsLayerBClient(
        access_token=MAPPLS_ACCESS_TOKEN,
        rest_key=MAPPLS_REST_KEY,
        timeout=REQUEST_TIMEOUT,
    )

    stage5_df, stage5_src = load_stage5_table()
    stage5_df = stage5_df.copy()
    stage5_df = ensure_label_column(stage5_df)
    stage5_df = derive_coords(stage5_df)
    hotspots = select_hotspots_for_enrichment(stage5_df, top_n=TOP_N)

    print("Stage 5 source:", stage5_src)
    print("Hotspots selected for Layer B:", len(hotspots))

    enriched_rows = []
    for i, row in enumerate(hotspots.itertuples(index=False), start=1):
        row_series = pd.Series(row._asdict())
        enriched = enrich_hotspot_row(client, row_series)
        enriched_rows.append(enriched)
        if i % 5 == 0 or i == len(hotspots):
            print(f"Processed {i}/{len(hotspots)} hotspots")

    layer_b_df = pd.DataFrame(enriched_rows)

    if len(layer_b_df):
        layer_b_df["cluster_label_display"] = (
            layer_b_df["cluster_label"].astype(str)
            + " (Cluster "
            + layer_b_df["cluster_id"].astype(str)
            + ")"
        )
        layer_b_df["nearby_sensitive_poi_names_str"] = layer_b_df["nearby_sensitive_poi_names"].apply(
            lambda xs: " || ".join(xs[:10]) if isinstance(xs, list) else clean_text(xs)
        )
        layer_b_df["nearby_sensitive_poi_addresses_str"] = layer_b_df["nearby_sensitive_poi_addresses"].apply(
            lambda xs: " || ".join(xs[:10]) if isinstance(xs, list) else clean_text(xs)
        )

    geofence_candidates = build_layer_b_geofence_candidates(layer_b_df)

    merge_cols = [
        "cluster_id",
        "reverse_geocode_ok", "reverse_geocode_address", "reverse_geocode_level",
        "reverse_geocode_poi", "reverse_geocode_street", "reverse_geocode_locality",
        "reverse_geocode_city", "reverse_geocode_district", "reverse_geocode_state",
        "nearby_sensitive_poi_ok", "nearby_sensitive_poi_count",
        "nearby_sensitive_poi_names", "nearby_sensitive_poi_addresses",
        "nearby_sensitive_poi_min_distance_m",
        "route_traffic_ok", "route_traffic_distance_m",
        "route_traffic_duration_sec", "route_traffic_duration_min",
        "route_optimal_ok", "route_optimal_distance_m",
        "route_optimal_duration_sec", "route_optimal_duration_min",
        "route_traffic_vs_optimal_ratio",
        "distance_matrix_ok", "distance_matrix_source",
        "distance_matrix_distance_m", "distance_matrix_duration_sec",
        "distance_matrix_duration_min", "distance_matrix_vs_route_ratio",
        "context_multiplier", "layer_b_priority_boost",
        "layer_b_alert_flag", "layer_b_error",
    ]
    merge_cols = [c for c in merge_cols if c in layer_b_df.columns]

    stage5_with_b = stage5_df.merge(
        layer_b_df[merge_cols].rename(columns={"cluster_id": standardize_cluster_col(stage5_df)}),
        on=standardize_cluster_col(stage5_df),
        how="left",
    )

    for col, default in [("context_multiplier", 1.0), ("layer_b_priority_boost", 0.0), ("layer_b_alert_flag", False)]:
        if col not in stage5_with_b.columns:
            stage5_with_b[col] = default

    summary = pd.DataFrame([{
        "input_source": str(stage5_src),
        "hotspots_enriched": int(len(layer_b_df)),
        "reverse_geocoded": int(layer_b_df["reverse_geocode_ok"].sum()) if len(layer_b_df) else 0,
        "nearby_context_hits": int(layer_b_df["nearby_sensitive_poi_ok"].sum()) if len(layer_b_df) else 0,
        "traffic_routes_ok": int(layer_b_df["route_traffic_ok"].sum()) if len(layer_b_df) else 0,
        "distance_matrix_ok": int(layer_b_df["distance_matrix_ok"].sum()) if len(layer_b_df) else 0,
        "mean_route_traffic_min": float(
            pd.to_numeric(layer_b_df["route_traffic_duration_min"], errors="coerce").dropna().mean()
        ) if len(layer_b_df) else np.nan,
        "mean_route_optimal_min": float(
            pd.to_numeric(layer_b_df["route_optimal_duration_min"], errors="coerce").dropna().mean()
        ) if len(layer_b_df) else np.nan,
        "mean_context_multiplier": float(
            pd.to_numeric(layer_b_df["context_multiplier"], errors="coerce").dropna().mean()
        ) if len(layer_b_df) else np.nan,
        "alerts_flagged": int(layer_b_df["layer_b_alert_flag"].sum()) if len(layer_b_df) else 0,
    }])

    layer_b_df.to_csv(OUT_DIR / "layer_b_mappls_enriched_hotspots.csv", index=False)
    geofence_candidates.to_csv(OUT_DIR / "layer_b_geofence_candidates.csv", index=False)
    stage5_with_b.to_csv(OUT_DIR / "phase5_with_layer_b.csv", index=False)
    summary.to_csv(OUT_DIR / "layer_b_summary.csv", index=False)

    print("Layer B complete")
    print("Output directory:", OUT_DIR.resolve())
    print("\nSummary:")
    print(summary.to_string(index=False))

    show_cols = [
        "cluster_id", "cluster_label_display", "reverse_geocode_address",
        "nearby_sensitive_poi_count", "route_traffic_duration_min",
        "route_optimal_duration_min", "distance_matrix_duration_min",
        "route_traffic_vs_optimal_ratio", "context_multiplier",
        "layer_b_priority_boost", "layer_b_alert_flag",
    ]
    show_cols = [c for c in show_cols if c in layer_b_df.columns]
    if len(layer_b_df) and show_cols:
        print("\nTop 10 Layer B rows:")
        print(layer_b_df[show_cols].head(10).to_string(index=False))


if __name__ == "__main__":
    main()

Stage 5 source: content\phase5_outputs_2\phase5_cluster_capacity_loss.csv
Hotspots selected for Layer B: 50
Processed 5/50 hotspots
Processed 10/50 hotspots
Processed 15/50 hotspots
Processed 20/50 hotspots
Processed 25/50 hotspots
Processed 30/50 hotspots
Processed 35/50 hotspots
Processed 40/50 hotspots
Processed 45/50 hotspots
Processed 50/50 hotspots
Layer B complete
Output directory: D:\Flipkart_Hackathon\approach1\content\layer_b_outputs_2

Summary:
                                             input_source  hotspots_enriched  reverse_geocoded  nearby_context_hits  traffic_routes_ok  distance_matrix_ok  mean_route_traffic_min  mean_route_optimal_min  mean_context_multiplier  alerts_flagged
content\phase5_outputs_2\phase5_cluster_capacity_loss.csv                 50                 0                    0                 50                  50                3.445467                     NaN                      1.0               0

Top 10 Layer B rows:
 cluster_id                   

In [27]:
import ast
import math
import os
import warnings
from pathlib import Path
from typing import Any, Dict, Iterable, Optional, Tuple

import networkx as nx
import numpy as np
import pandas as pd
import osmnx as ox

warnings.filterwarnings("ignore")

# ============================================================
# Layer D — Network Spillover / Propagation Analysis
# Optimized with:
#   - OSMnx GraphML caching (download once, reuse later)
#   - nearest_nodes() for hotspot-to-road-node assignment
#   - cached betweenness centrality
#   - pairwise hotspot graph built vectorially
# ============================================================

# -------------------------
# Config
# -------------------------
BASE_DIRS = [
    Path("content/phase5_outputs_2"),
    Path("phase5_outputs_2"),
    Path("/content/phase5_outputs_2"),
]

# If Layer B/C outputs exist, we can merge them; if not, code still runs.
LAYER_B_DIRS = [
    Path("content/layer_b_outputs_2"),
    Path("layer_b_outputs_2"),
    Path("/content/layer_b_outputs_2"),
]

LAYER_C_DIRS = [
    Path("content/layer_c_outputs_2"),
    Path("layer_c_outputs_2"),
    Path("/content/layer_c_outputs_2"),
]

OUT_DIR = Path("content/layer_d_outputs_2")
CACHE_DIR = OUT_DIR / "cache"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

EPS = 1e-9

GRAPH_QUERY = os.environ.get("LAYER_D_GRAPH_QUERY", "Bengaluru, Karnataka, India").strip()
GRAPH_FALLBACK_DIST_M = int(os.environ.get("LAYER_D_GRAPH_FALLBACK_DIST_M", "25000"))
GRAPH_BETWEENNESS_K = int(os.environ.get("LAYER_D_BETWEENNESS_K", "128"))

NEIGHBOR_RADIUS_M = float(os.environ.get("LAYER_D_NEIGHBOR_RADIUS_M", "1200"))
SPILLOVER_DECAY_M = float(os.environ.get("LAYER_D_SPILLOVER_DECAY_M", "600"))
TOP_N = int(os.environ.get("LAYER_D_TOP_N", "0"))  # 0 = all

GRAPHML_CACHE = CACHE_DIR / "bengaluru_drive_graph.graphml"
CENTRALITY_CACHE = CACHE_DIR / "bengaluru_drive_graph_node_betweenness.csv"

# -------------------------
# Generic helpers
# -------------------------
def clean_text(x) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()


def safe_float(x, default=np.nan):
    try:
        if pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default

def safe_series(g, col, default=np.nan):
    if col in g.columns:
        return pd.to_numeric(g[col], errors="coerce")
    return pd.Series([default] * len(g), index=g.index, dtype=float)

def load_first_existing(base_dirs: Iterable[Path], filenames: Iterable[str]):
    for d in base_dirs:
        for name in filenames:
            p = d / name
            if p.exists():
                return pd.read_csv(p, low_memory=False), p
    return None, None


def standardize_cluster_col(df: pd.DataFrame) -> str:
    for c in ["st_dbscan_cluster_id", "cluster_id", "dbscan_cluster_id"]:
        if c in df.columns:
            return c
    raise ValueError("No cluster id column found.")


def ensure_label_column(df: pd.DataFrame):
    if df is None or len(df) == 0:
        return df
    df = df.copy()
    if "cluster_label" in df.columns:
        df["cluster_label"] = df["cluster_label"].fillna("").astype(str).str.strip()
        return df
    if "hotspot_unit" in df.columns:
        df["cluster_label"] = df["hotspot_unit"].fillna("").astype(str).str.strip()
        return df
    if "dominant_junction_name" in df.columns:
        df["cluster_label"] = df["dominant_junction_name"].fillna("").astype(str).str.strip()
        return df
    if "st_dbscan_cluster_id" in df.columns:
        df["cluster_label"] = "CLUSTER::" + df["st_dbscan_cluster_id"].astype(str)
        return df
    df["cluster_label"] = "UNKNOWN"
    return df


def derive_coords(df: pd.DataFrame):
    if df is None or len(df) == 0:
        return df
    df = df.copy()

    if "lat" in df.columns and "lon" in df.columns:
        df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
        df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    elif {"centroid_lat", "centroid_lon"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["centroid_lat"], errors="coerce")
        df["lon"] = pd.to_numeric(df["centroid_lon"], errors="coerce")
    elif {"latitude_mean", "longitude_mean"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["latitude_mean"], errors="coerce")
        df["lon"] = pd.to_numeric(df["longitude_mean"], errors="coerce")
    elif {"latitude", "longitude"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["latitude"], errors="coerce")
        df["lon"] = pd.to_numeric(df["longitude"], errors="coerce")
    else:
        df["lat"] = np.nan
        df["lon"] = np.nan
    return df


def dominant_label(series, default=""):
    s = pd.Series(series).dropna().astype(str).str.strip()
    s = s[s.ne("")]
    if s.empty:
        return default
    m = s.mode()
    if not m.empty:
        return m.iloc[0]
    return s.iloc[0]


def minmax(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    valid = s.dropna()
    if len(valid) == 0 or valid.nunique(dropna=True) <= 1:
        return pd.Series(np.zeros(len(s)), index=s.index, dtype=float)
    mn = valid.min()
    mx = valid.max()
    return (s.fillna(mn) - mn) / (mx - mn + EPS)


def active_minmax(s: pd.Series) -> Tuple[pd.Series, bool]:
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    valid = s.dropna()
    if len(valid) == 0 or valid.nunique(dropna=True) <= 1:
        return pd.Series(np.nan, index=s.index, dtype=float), False
    mn = valid.min()
    mx = valid.max()
    return (s.fillna(mn) - mn) / (mx - mn + EPS), True


def weighted_score(df: pd.DataFrame, specs, prefix: str):
    """
    specs = [(source_col, weight), ...]
    Only active (non-constant, non-empty) components contribute.
    """
    out = df.copy()
    active = []

    for source_col, weight in specs:
        norm_col = f"{prefix}_{source_col}_norm".replace(" ", "_")
        if source_col not in out.columns:
            out[norm_col] = np.nan
            continue

        norm, is_active = active_minmax(out[source_col])
        out[norm_col] = norm
        if is_active:
            active.append((norm_col, weight))

    if not active:
        out[f"{prefix}_raw"] = 0.0
        out[f"{prefix}_score"] = 0.0
        return out

    total_weight = float(sum(w for _, w in active))
    raw = np.zeros(len(out), dtype=float)
    for norm_col, weight in active:
        raw += weight * out[norm_col].fillna(0.0).to_numpy(dtype=float)

    raw = raw / max(total_weight, EPS)
    out[f"{prefix}_raw"] = raw
    out[f"{prefix}_score"] = 100.0 * np.clip(raw, 0.0, 1.0)
    return out


def make_hotspot_key(df: pd.DataFrame) -> pd.Series:
    label = df.get("cluster_label", pd.Series("", index=df.index)).fillna("").astype(str).str.strip().str.lower()
    if "lat" in df.columns and "lon" in df.columns:
        lat = pd.to_numeric(df["lat"], errors="coerce").round(5).astype(str)
        lon = pd.to_numeric(df["lon"], errors="coerce").round(5).astype(str)
    else:
        lat = pd.Series("", index=df.index)
        lon = pd.Series("", index=df.index)
    return label + "|" + lat + "|" + lon


def haversine_matrix(lat_rad: np.ndarray, lon_rad: np.ndarray) -> np.ndarray:
    """
    Vectorized pairwise haversine distance matrix (meters).
    """
    R = 6371000.0
    dlat = lat_rad[:, None] - lat_rad[None, :]
    dlon = lon_rad[:, None] - lon_rad[None, :]
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat_rad[:, None]) * np.cos(lat_rad[None, :]) * np.sin(dlon / 2.0) ** 2
    return 2.0 * R * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))


def safe_merge_extra_columns(base: pd.DataFrame, extra: pd.DataFrame, key: str = "cluster_id") -> pd.DataFrame:
    """
    Merge extra enrichment into base using key, keeping base values first and
    filling only missing values from extra.
    """
    if extra is None or len(extra) == 0 or key not in extra.columns:
        return base

    # Coerce the join key to string on both sides so int64 vs object never
    # causes a ValueError — cluster_id can arrive as int64 from a CSV and as
    # str after astype(str) downstream.
    base = base.copy()
    base[key] = base[key].astype(str)
    extra = extra.copy()
    extra[key] = extra[key].astype(str)
    extra = extra.drop_duplicates(subset=[key], keep="first")
    extra_cols = [c for c in extra.columns if c != key]
    if not extra_cols:
        return base

    merged = base.merge(extra[[key] + extra_cols], on=key, how="left", suffixes=("", "_extra"))

    for c in extra_cols:
        ec = f"{c}_extra"
        if ec not in merged.columns:
            continue

        if c in merged.columns:
            if pd.api.types.is_numeric_dtype(merged[c]) or pd.api.types.is_numeric_dtype(merged[ec]):
                merged[c] = pd.to_numeric(merged[c], errors="coerce").combine_first(
                    pd.to_numeric(merged[ec], errors="coerce")
                )
            else:
                merged[c] = merged[c].combine_first(merged[ec])
        else:
            merged[c] = merged[ec]

        merged.drop(columns=[ec], inplace=True, errors="ignore")

    return merged


def collapse_physical_hotspots(df: pd.DataFrame) -> pd.DataFrame:
    """
    Collapse duplicate physical hotspots using rounded lat/lon + normalized label.
    """
    if df is None or len(df) == 0:
        return df

    work = df.copy()
    work["physical_hotspot_key"] = make_hotspot_key(work)

    rows = []
    for key, g in work.groupby("physical_hotspot_key", sort=False):
        g = g.copy()
        top = g.sort_values(
            ["ccs_score", "delay_minutes_per_vehicle", "records_total"],
            ascending=[False, False, False],
        ).iloc[0]

        row = {
            "physical_hotspot_key": key,
            "merged_cluster_ids": "|".join(sorted({str(x) for x in g["cluster_id"].dropna().tolist()})),
            "cluster_id": top.get("cluster_id"),
            "cluster_label": top.get("cluster_label", ""),
            "cluster_label_display": top.get("cluster_label_display", ""),
            "risk_band": top.get("risk_band", "Watch"),
            "lat": float(pd.to_numeric(g["lat"], errors="coerce").mean()),
            "lon": float(pd.to_numeric(g["lon"], errors="coerce").mean()),
            "ccs_score": float(pd.to_numeric(g["ccs_score"], errors="coerce").max()),
            "delay_minutes_per_vehicle": float(pd.to_numeric(g["delay_minutes_per_vehicle"], errors="coerce").max()),
            "records_total": float(pd.to_numeric(g.get("records_total", 0), errors="coerce").fillna(0).sum()),
            "distinct_days": float(pd.to_numeric(g.get("distinct_days", 0), errors="coerce").fillna(0).max()),
            "severity_sum": float(pd.to_numeric(g.get("severity_sum", 0), errors="coerce").fillna(0).sum()),
            "severity_mean": float(pd.to_numeric(g.get("severity_mean", 0), errors="coerce").fillna(0).mean()),
            "growth_pct_change": float(pd.to_numeric(g.get("growth_pct_change", 0), errors="coerce").fillna(0).max()),
            "growth_multiplier": float(pd.to_numeric(g.get("growth_multiplier", 1), errors="coerce").fillna(1).max()),
            "context_multiplier": float(pd.to_numeric(g.get("context_multiplier", 1), errors="coerce").fillna(1).max()),
            "layer_b_priority_boost": float(pd.to_numeric(g.get("layer_b_priority_boost", 0), errors="coerce").fillna(0).max()),
            "layer_b_alert_flag": bool(g.get("layer_b_alert_flag", False).astype(bool).any()) if "layer_b_alert_flag" in g.columns else False,
            "validation_uncertainty": float(pd.to_numeric(g.get("validation_uncertainty", np.nan), errors="coerce").fillna(np.nan).mean()),
            "resurgence_score": float(pd.to_numeric(g.get("resurgence_score", np.nan), errors="coerce").fillna(np.nan).max()),
            "persistence_score": float(pd.to_numeric(g.get("persistence_score", np.nan), errors="coerce").fillna(np.nan).max()),
            "anomaly_score": float(pd.to_numeric(g.get("anomaly_score", np.nan), errors="coerce").fillna(np.nan).max()),
            "rop": float(pd.to_numeric(g.get("rop", np.nan), errors="coerce").fillna(np.nan).max()),
            "tvs": float(pd.to_numeric(g.get("tvs", np.nan), errors="coerce").fillna(np.nan).max()),
            "vdi": float(pd.to_numeric(g.get("vdi", np.nan), errors="coerce").fillna(np.nan).max()),
            "nearby_sensitive_poi_count": float(pd.to_numeric(g.get("nearby_sensitive_poi_count", 0), errors="coerce").fillna(0).sum()),
            "road_class": clean_text(top.get("road_class", "road")),
            "lane_count": float(pd.to_numeric(g.get("lane_count", np.nan), errors="coerce").fillna(np.nan).mean()),
            "carriageway_width_m": float(pd.to_numeric(g.get("carriageway_width_m", np.nan), errors="coerce").fillna(np.nan).mean()),
            "link_length_m": float(pd.to_numeric(g.get("link_length_m", np.nan), errors="coerce").fillna(np.nan).mean()),
            "junction_degree": float(pd.to_numeric(g.get("junction_degree", np.nan), errors="coerce").fillna(np.nan).mean()),
            "betweenness_centrality": float(pd.to_numeric(g.get("betweenness_centrality", np.nan), errors="coerce").fillna(np.nan).mean()),
            "geometry_source": clean_text(top.get("geometry_source", "fallback")),
            "mappls_address": clean_text(top.get("mappls_address", "")),
            "road_node_id": top.get("road_node_id", np.nan),
            "road_node_distance_m": float(pd.to_numeric(g.get("road_node_distance_m", np.nan), errors="coerce").fillna(np.nan).mean()),
            "road_node_degree": float(pd.to_numeric(g.get("road_node_degree", np.nan), errors="coerce").fillna(np.nan).mean()),
            "road_node_betweenness": float(pd.to_numeric(g.get("road_node_betweenness", np.nan), errors="coerce").fillna(np.nan).mean()),
            "source_pressure_score": float(safe_series(g, "source_pressure_score").mean()),
            "source_pressure_norm": float(safe_series(g, "source_pressure_norm").mean()),
            "spillover_out_score": float(safe_series(g, "spillover_out_score").fillna(0.0).max()),
            "spillover_in_score": float(safe_series(g, "spillover_in_score").fillna(0.0).max()),
            "spillover_total_score": float(safe_series(g, "spillover_total_score").fillna(0.0).max()),
            "propagation_radius_m": float(safe_series(g, "propagation_radius_m").mean()),
            "network_pagerank": float(safe_series(g, "network_pagerank").mean()),
            "network_component_id": int(pd.to_numeric(g.get("network_component_id", 0), errors="coerce").fillna(0).iloc[0]) if "network_component_id" in g.columns else 0,
            "network_component_size": int(pd.to_numeric(g.get("network_component_size", 1), errors="coerce").fillna(1).iloc[0]) if "network_component_size" in g.columns else 1,
            "neighbor_count": int(safe_series(g, "neighbor_count",0).max()),
            "in_neighbor_count": int(safe_series(g, "in_neighbor_count",0).max()),
            "out_neighbor_count": int(safe_series(g, "out_neighbor_count",0).max()),
            "influence_asymmetry": float(safe_series(g, "influence_asymmetry",0).max()),
            "network_vulnerability_score": float(safe_series(g, "network_vulnerability_score").mean()),
            "layer_d_alert_flag": bool(g.get("layer_d_alert_flag", False).astype(bool).any()) if "layer_d_alert_flag" in g.columns else False,
        }

        rows.append(row)

    out = pd.DataFrame(rows)
    out["cluster_label"] = out["cluster_label"].fillna("").astype(str).str.strip()
    out["cluster_label_display"] = out["cluster_label_display"].fillna("").astype(str).str.strip()
    return out


# -------------------------
# Layer B / C optional load
# -------------------------
def load_optional_table(base_dirs: Iterable[Path], filenames: Iterable[str]):
    df, src = load_first_existing(base_dirs, filenames)
    return df, src


# -------------------------
# Road graph caching
# -------------------------
def load_or_build_road_graph(place_query: str, fallback_center: Tuple[float, float]):
    """
    Reuse GraphML if present. Otherwise download once and cache it.
    """
    if GRAPHML_CACHE.exists():
        try:
            G = ox.io.load_graphml(GRAPHML_CACHE)
            return G, f"cache:{GRAPHML_CACHE}"
        except Exception as e:
            print(f"Graph cache load failed, rebuilding: {e}")

    try:
        G = ox.graph_from_place(
            place_query,
            network_type="drive",
            simplify=True,
            retain_all=False,
        )
        ox.io.save_graphml(G, GRAPHML_CACHE)
        return G, f"downloaded_place:{place_query}"
    except Exception as place_err:
        lat0, lon0 = fallback_center
        try:
            G = ox.graph_from_point(
                (lat0, lon0),
                dist=GRAPH_FALLBACK_DIST_M,
                network_type="drive",
                simplify=True,
                retain_all=False,
            )
            ox.io.save_graphml(G, GRAPHML_CACHE)
            return G, f"downloaded_point:{place_err}"
        except Exception as point_err:
            raise RuntimeError(
                f"Failed to build road graph from place and point. place_error={place_err!r}, point_error={point_err!r}"
            )


def load_or_compute_node_betweenness(G):
    """
    Compute betweenness once and cache it. Uses approximate BC with k-sampling
    when the graph is large, because full betweenness can be costly.
    """
    if CENTRALITY_CACHE.exists():
        try:
            cache_df = pd.read_csv(CENTRALITY_CACHE)
            if {"node_id", "betweenness"}.issubset(cache_df.columns):
                return dict(zip(cache_df["node_id"], cache_df["betweenness"])), "cache"
        except Exception:
            pass

    # Simplify parallel edges before BC.
    DG = ox.convert.to_digraph(G, weight="length")
    Gu = DG.to_undirected()

    n_nodes = len(Gu)
    if n_nodes <= 1500:
        bc = nx.betweenness_centrality(Gu, normalized=True, weight="length")
        method = "exact"
    else:
        k = min(GRAPH_BETWEENNESS_K, max(64, int(np.sqrt(n_nodes))))
        bc = nx.betweenness_centrality(Gu, k=k, normalized=True, weight="length", seed=42)
        method = f"approx_k={k}"

    cache_df = pd.DataFrame(
        [{"node_id": node, "betweenness": val} for node, val in bc.items()]
    )
    cache_df.to_csv(CENTRALITY_CACHE, index=False)
    return bc, method


def assign_road_nodes(hotspots: pd.DataFrame, G, bc_map: Dict[Any, float]) -> pd.DataFrame:
    work = hotspots.copy()
    X = pd.to_numeric(work["lon"], errors="coerce").to_numpy(dtype=float)
    Y = pd.to_numeric(work["lat"], errors="coerce").to_numpy(dtype=float)

    print("Assigning nearest road nodes...")
    nearest_nodes, nearest_dists = ox.distance.nearest_nodes(G, X=X, Y=Y, return_dist=True)

    work["road_node_id"] = pd.Series(nearest_nodes, index=work.index)
    work["road_node_distance_m"] = pd.Series(nearest_dists, index=work.index).astype(float)

    # Degree from the road graph.
    road_deg = []
    road_bc = []
    for node in work["road_node_id"].tolist():
        try:
            road_deg.append(int(G.degree[node]))
        except Exception:
            road_deg.append(np.nan)
        road_bc.append(float(bc_map.get(node, np.nan)))

    work["road_node_degree"] = pd.Series(road_deg, index=work.index)
    work["road_node_betweenness"] = pd.Series(road_bc, index=work.index)

    return work


# -------------------------
# Layer D computation
# -------------------------
def ensure_feature_defaults(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create fallback feature columns if layer A/B/C outputs are not present.
    """
    work = df.copy()

    defaults = {
        "context_multiplier": 1.0,
        "layer_b_priority_boost": 0.0,
        "layer_b_alert_flag": False,
        "validation_uncertainty": np.nan,
        "resurgence_score": np.nan,
        "persistence_score": np.nan,
        "anomaly_score": np.nan,
        "rop": np.nan,
        "tvs": np.nan,
        "vdi": np.nan,
    }
    for c, default in defaults.items():
        if c not in work.columns:
            work[c] = default

    # Reasonable derived fallbacks if Layer C is not present.
    if "resurgence_score" not in df.columns or pd.to_numeric(work["resurgence_score"], errors="coerce").notna().sum() == 0:
        if "growth_pct_change" in work.columns:
            work["resurgence_score"] = pd.to_numeric(work["growth_pct_change"], errors="coerce").fillna(0.0).clip(lower=0.0)
        elif "growth_multiplier" in work.columns:
            work["resurgence_score"] = (pd.to_numeric(work["growth_multiplier"], errors="coerce").fillna(1.0) - 1.0).clip(lower=0.0)
        else:
            work["resurgence_score"] = 0.0

    if "persistence_score" not in df.columns or pd.to_numeric(work["persistence_score"], errors="coerce").notna().sum() == 0:
        records_norm = minmax(work.get("records_total", pd.Series(0, index=work.index)).fillna(0))
        distinct_days_norm = minmax(work.get("distinct_days", pd.Series(0, index=work.index)).fillna(0))
        repeat_norm = minmax(work.get("repeat_vehicle_count_2plus", pd.Series(0, index=work.index)).fillna(0))
        chronic_norm = minmax(work.get("chronic_vehicle_count_5plus", pd.Series(0, index=work.index)).fillna(0))
        work["persistence_score"] = 100.0 * (
            0.35 * records_norm +
            0.25 * distinct_days_norm +
            0.25 * repeat_norm +
            0.15 * chronic_norm
        )

    if "validation_uncertainty" not in df.columns or pd.to_numeric(work["validation_uncertainty"], errors="coerce").notna().sum() == 0:
        work["validation_uncertainty"] = 0.0

    if "anomaly_score" not in df.columns or pd.to_numeric(work["anomaly_score"], errors="coerce").notna().sum() == 0:
        work["anomaly_score"] = 0.0

    # Helpful context boost proxy for Layer B (if present, use it; otherwise zero).
    if "context_multiplier" in work.columns:
        work["context_boost"] = pd.to_numeric(work["context_multiplier"], errors="coerce").fillna(1.0) - 1.0
    else:
        work["context_boost"] = 0.0

    return work


def build_source_pressure(df: pd.DataFrame) -> pd.DataFrame:
    """
    Composite "pressure" score used as the source term in spillover propagation.
    """
    work = df.copy()

    # Positive-growth version of resurgence to avoid penalizing declining hotspots.
    if "growth_surge" not in work.columns:
        if "growth_pct_change" in work.columns:
            work["growth_surge"] = pd.to_numeric(work["growth_pct_change"], errors="coerce").fillna(0.0).clip(lower=0.0)
        elif "growth_multiplier" in work.columns:
            work["growth_surge"] = (pd.to_numeric(work["growth_multiplier"], errors="coerce").fillna(1.0) - 1.0).clip(lower=0.0)
        else:
            work["growth_surge"] = 0.0

    # Add a few composite terms before weighting.
    if "context_boost" not in work.columns:
        work["context_boost"] = pd.to_numeric(work.get("context_multiplier", 1.0), errors="coerce").fillna(1.0) - 1.0

    specs = [
        ("ccs_score", 0.25),
        ("delay_minutes_per_vehicle", 0.15),
        ("growth_surge", 0.10),
        ("layer_b_priority_boost", 0.10),
        ("context_boost", 0.08),
        ("persistence_score", 0.10),
        ("resurgence_score", 0.10),
        ("validation_uncertainty", 0.05),
        ("anomaly_score", 0.04),
        ("rop", 0.06),
        ("tvs", 0.04),
        ("vdi", 0.03),
    ]

    work = weighted_score(work, specs=specs, prefix="layer_d_pressure")
    work["source_pressure_score"] = pd.to_numeric(work["layer_d_pressure_score"], errors="coerce").fillna(0.0)

    # Fallback: if weighted_score produced all-zeros (every feature was constant
    # or missing), use a simple minmax of ccs_score so pressure is never flat.
    if work["source_pressure_score"].max() == 0.0:
        fallback = minmax(pd.to_numeric(work.get("ccs_score", pd.Series(0, index=work.index)), errors="coerce").fillna(0.0))
        if fallback.max() > 0:
            print("WARNING: all pressure features were constant — falling back to ccs_score minmax for source pressure.")
            work["source_pressure_score"] = fallback * 100.0
        else:
            # Last resort: uniform pressure so spillover at least runs on proximity alone.
            print("WARNING: ccs_score also constant — using uniform source pressure.")
            work["source_pressure_score"] = 50.0

    work["source_pressure_norm"] = work["source_pressure_score"] / 100.0
    return work


def build_spillover_network(hotspots: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Build directed hotspot spillover graph from pairwise proximity and source pressure.
    """
    work = hotspots.copy().reset_index(drop=True)
    work = work.reset_index(drop=True)

    work["source_idx"] = np.arange(len(work))
    n = len(work)
    if n == 0:
        return pd.DataFrame(), pd.DataFrame()

    lat = pd.to_numeric(work["lat"], errors="coerce").to_numpy(dtype=float)
    lon = pd.to_numeric(work["lon"], errors="coerce").to_numpy(dtype=float)
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)

    print("Computing pairwise hotspot distances...")
    dist_m = haversine_matrix(lat_rad, lon_rad)

    # Candidate pairs within the spillover radius.
    pair_mask = np.triu((dist_m > 0) & (dist_m <= NEIGHBOR_RADIUS_M), k=1)
    pair_idx = np.column_stack(np.where(pair_mask))

    edge_rows = []
    print(f"Building spillover edges for {len(pair_idx)} hotspot pairs...")
    for idx, (i, j) in enumerate(pair_idx, start=1):
        d = float(dist_m[i, j])
        decay = float(np.exp(-d / max(SPILLOVER_DECAY_M, EPS)))

        src_pressure_i = float(work.loc[i, "source_pressure_norm"])
        src_pressure_j = float(work.loc[j, "source_pressure_norm"])

        cent_i = float(work.loc[i, "road_node_betweenness_norm"]) if "road_node_betweenness_norm" in work.columns else 0.0
        cent_j = float(work.loc[j, "road_node_betweenness_norm"]) if "road_node_betweenness_norm" in work.columns else 0.0

        # Directed influences.
        w_ij = src_pressure_i * decay * (1.0 + 0.5 * cent_j)
        w_ji = src_pressure_j * decay * (1.0 + 0.5 * cent_i)

        edge_rows.append(
            {
                "source_idx": i,
                "target_idx": j,
                "source_cluster_id": work.loc[i, "cluster_id"],
                "target_cluster_id": work.loc[j, "cluster_id"],
                "source_label": clean_text(work.loc[i, "cluster_label_display"]),
                "target_label": clean_text(work.loc[j, "cluster_label_display"]),
                "distance_m": d,
                "decay_factor": decay,
                "source_pressure_norm": src_pressure_i,
                "target_pressure_norm": src_pressure_j,
                "source_road_centrality_norm": cent_i,
                "target_road_centrality_norm": cent_j,
                "influence_weight": w_ij,
            }
        )
        edge_rows.append(
            {
                "source_idx": j,
                "target_idx": i,
                "source_cluster_id": work.loc[j, "cluster_id"],
                "target_cluster_id": work.loc[i, "cluster_id"],
                "source_label": clean_text(work.loc[j, "cluster_label_display"]),
                "target_label": clean_text(work.loc[i, "cluster_label_display"]),
                "distance_m": d,
                "decay_factor": decay,
                "source_pressure_norm": src_pressure_j,
                "target_pressure_norm": src_pressure_i,
                "source_road_centrality_norm": cent_j,
                "target_road_centrality_norm": cent_i,
                "influence_weight": w_ji,
            }
        )

        if idx % 1000 == 0:
            print(f"  processed {idx}/{len(pair_idx)} pairs")

    edge_df = pd.DataFrame(edge_rows)

    # Build directed graph for PageRank and an undirected graph for components.
    DG = nx.DiGraph()
    UG = nx.Graph()
    for idx, row in work.iterrows():
        node_key = int(idx)
        DG.add_node(node_key)
        UG.add_node(node_key)

    for r in edge_df.itertuples(index=False):
        w = float(r.influence_weight)
        if w <= 0.0:
            continue  # skip zero-weight edges — they carry no spillover signal
        DG.add_edge(int(r.source_idx), int(r.target_idx), weight=w)
        UG.add_edge(int(r.source_idx), int(r.target_idx), weight=w)

    # PageRank over spillover graph.
    if DG.number_of_edges() > 0:
        pr = nx.pagerank(DG, alpha=0.85, weight="weight", max_iter=200)
    else:
        # No real edges (all weights were zero) — uniform rank.
        uniform = 1.0 / max(len(DG), 1)
        pr = {node: uniform for node in DG.nodes()}

    pagerank_df = pd.DataFrame(
        [{"source_idx": k, "network_pagerank": v} for k, v in pr.items()]
    )

    # Component id / size.
    comp_rows = []
    for comp_id, comp in enumerate(nx.connected_components(UG), start=1):
        size = len(comp)
        for node in comp:
            comp_rows.append(
                {
                    "source_idx": int(node),
                    "network_component_id": comp_id,
                    "network_component_size": size,
                }
            )
    comp_df = pd.DataFrame(comp_rows)

    # ----------------------------------------------------------------
    # Aggregate spillover scores directly onto work via index maps.
    # We deliberately avoid merges here — merges on computed DataFrames
    # have caused silent column-missing bugs across pandas versions.
    # Instead we build plain Python dicts keyed by source_idx (integer
    # row position) and assign column arrays directly.
    # ----------------------------------------------------------------
    n_work = len(work)

    # Initialize all network columns to their neutral defaults first.
    out_score_arr   = np.zeros(n_work, dtype=float)
    in_score_arr    = np.zeros(n_work, dtype=float)
    out_nbr_arr     = np.zeros(n_work, dtype=int)
    in_nbr_arr      = np.zeros(n_work, dtype=int)
    prop_rad_arr    = np.full(n_work, np.nan, dtype=float)
    pr_arr          = np.zeros(n_work, dtype=float)
    comp_id_arr     = np.zeros(n_work, dtype=int)
    comp_sz_arr     = np.ones(n_work,  dtype=int)

    if len(edge_df):
        iw = pd.to_numeric(edge_df["influence_weight"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        dm = pd.to_numeric(edge_df["distance_m"],       errors="coerce").fillna(0.0).to_numpy(dtype=float)
        si = edge_df["source_idx"].to_numpy(dtype=int)
        ti = edge_df["target_idx"].to_numpy(dtype=int)

        # Out-scores: sum influence weights leaving each source.
        np.add.at(out_score_arr, si, iw)

        # In-scores: sum influence weights arriving at each target.
        np.add.at(in_score_arr,  ti, iw)

        # Out-neighbor count: number of distinct targets per source.
        src_neighbors: dict = {}   # source_idx -> set of target_idx
        tgt_sources:   dict = {}   # target_idx -> set of source_idx
        for s_i, t_i in zip(si.tolist(), ti.tolist()):
            src_neighbors.setdefault(s_i, set()).add(t_i)
            tgt_sources.setdefault(t_i, set()).add(s_i)
        for s_i, nbrs in src_neighbors.items():
            out_nbr_arr[s_i] = len(nbrs)
        for t_i, srcs in tgt_sources.items():
            in_nbr_arr[t_i] = len(srcs)

        # Weighted-average propagation radius per source.
        # prop_radius[s] = sum(w*d) / sum(w)
        num_arr = np.zeros(n_work, dtype=float)
        den_arr = np.zeros(n_work, dtype=float)
        np.add.at(num_arr, si, iw * dm)
        np.add.at(den_arr, si, iw)
        with np.errstate(invalid="ignore", divide="ignore"):
            prop_rad_arr = np.where(den_arr > 0, num_arr / den_arr, np.nan)

    # PageRank: assign from dict (node -> value), node == source_idx.
    for node, val in pr.items():
        idx_i = int(node)
        if 0 <= idx_i < n_work:
            pr_arr[idx_i] = float(val)

    # Component id/size: assign from comp_rows list.
    for row in comp_rows:
        idx_i = int(row["source_idx"])
        if 0 <= idx_i < n_work:
            comp_id_arr[idx_i] = int(row["network_component_id"])
            comp_sz_arr[idx_i] = int(row["network_component_size"])

    # Assign all columns directly — no merges, no missing-column risk.
    work = work.copy()
    work["spillover_out_score"]      = out_score_arr
    work["spillover_in_score"]       = in_score_arr
    work["spillover_total_score"]    = out_score_arr + in_score_arr
    work["out_neighbor_count"]       = out_nbr_arr
    work["in_neighbor_count"]        = in_nbr_arr
    work["neighbor_count"]           = out_nbr_arr + in_nbr_arr
    work["propagation_radius_m"]     = prop_rad_arr
    work["network_pagerank"]         = pr_arr
    work["network_component_id"]     = comp_id_arr
    work["network_component_size"]   = comp_sz_arr
    work["influence_asymmetry"]      = out_score_arr - in_score_arr

    # Additional derived flags.
    pr_q80 = work["network_pagerank"].quantile(0.80) if len(work) else 0.0
    spill_q80 = work["spillover_total_score"].quantile(0.80) if len(work) else 0.0
    work["layer_d_alert_flag"] = (
        (work["network_pagerank"] >= pr_q80) |
        (work["spillover_total_score"] >= spill_q80)
    )

    return work, edge_df


def finalize_layer_d(hotspots: pd.DataFrame, edges: pd.DataFrame) -> pd.DataFrame:
    """
    Compute final Layer D scores from source pressure + spillover + road-network metrics.
    """
    work = hotspots.copy()

    # Normalize road-node metrics and spillover metrics.
    if "road_node_betweenness" not in work.columns:
        work["road_node_betweenness"] = np.nan
    if "road_node_degree" not in work.columns:
        work["road_node_degree"] = np.nan
    if "road_node_distance_m" not in work.columns:
        work["road_node_distance_m"] = np.nan
    if "validation_uncertainty" not in work.columns:
        work["validation_uncertainty"] = np.nan
    if "resurgence_score" not in work.columns:
        work["resurgence_score"] = np.nan
    if "persistence_score" not in work.columns:
        work["persistence_score"] = np.nan
    if "anomaly_score" not in work.columns:
        work["anomaly_score"] = np.nan
    if "context_multiplier" not in work.columns:
        work["context_multiplier"] = 1.0
    if "layer_b_priority_boost" not in work.columns:
        work["layer_b_priority_boost"] = 0.0

    # Road-node centrality normalization.
    work["road_node_betweenness_norm"] = minmax(work["road_node_betweenness"].fillna(0.0))
    work["road_node_degree_norm"] = minmax(work["road_node_degree"].fillna(0.0))
    work["road_node_distance_norm"] = minmax(work["road_node_distance_m"].fillna(work["road_node_distance_m"].max() if work["road_node_distance_m"].notna().any() else 0.0))

    # Ensure source pressure exists.
    work = build_source_pressure(work)

    # If any Layer C/novel features are present, retain their normalized footprints.
    if "growth_pct_change" in work.columns:
        work["growth_surge"] = pd.to_numeric(work["growth_pct_change"], errors="coerce").fillna(0.0).clip(lower=0.0)
    elif "growth_multiplier" in work.columns:
        work["growth_surge"] = (pd.to_numeric(work["growth_multiplier"], errors="coerce").fillna(1.0) - 1.0).clip(lower=0.0)
    else:
        work["growth_surge"] = 0.0

    # Final vulnerability score:
    # source pressure + outgoing spillover + incoming spillover + road-network centrality + PageRank.
    work = weighted_score(
        work,
        specs=[
            ("source_pressure_score", 0.35),
            ("spillover_out_score", 0.20),
            ("spillover_in_score", 0.15),
            ("road_node_betweenness", 0.15),
            ("network_pagerank", 0.10),
            ("resurgence_score", 0.05),
            ("validation_uncertainty", 0.05),
        ],
        prefix="layer_d_vulnerability",
    )
    work["network_vulnerability_score"] = pd.to_numeric(work["layer_d_vulnerability_score"], errors="coerce").fillna(0.0)

    # Helpful transparency columns.
    work["spillover_total_score"] = pd.to_numeric(work.get("spillover_total_score", work["spillover_out_score"] + work["spillover_in_score"]), errors="coerce").fillna(0.0)
    work["secondary_risk_score"] = work["spillover_in_score"]
    work["source_pressure_norm"] = pd.to_numeric(work["source_pressure_norm"], errors="coerce").fillna(0.0)

    # Rank + alerting.
    work = work.sort_values(
        ["network_vulnerability_score", "spillover_total_score", "ccs_score"],
        ascending=[False, False, False],
    ).reset_index(drop=True)
    work["network_rank"] = np.arange(1, len(work) + 1)

    vuln_q80 = work["network_vulnerability_score"].quantile(0.80) if len(work) else 0.0
    spill_q80 = work["spillover_total_score"].quantile(0.80) if len(work) else 0.0
    work["layer_d_alert_flag"] = (
        (work["network_vulnerability_score"] >= vuln_q80) |
        (work["spillover_total_score"] >= spill_q80)
    )

    # Propagation class for dashboarding.
    work["propagation_class"] = np.select(
        [
            work["spillover_total_score"] >= work["spillover_total_score"].quantile(0.90),
            work["spillover_total_score"] >= work["spillover_total_score"].quantile(0.70),
            work["spillover_total_score"] >= work["spillover_total_score"].quantile(0.50),
        ],
        [
            "Severe Propagator",
            "Strong Propagator",
            "Moderate Propagator",
        ],
        default="Localized",
    )

    return work


def main():
    # -------------------------
    # Load base Layer B / Phase 5 table
    # -------------------------
    base_df, base_src = load_first_existing(
        BASE_DIRS,
        [
            "phase5_with_layer_b.csv",
            "phase5_cluster_capacity_loss.csv",
            "phase5_priority_table_full.csv",
            "phase5_stage5_handoff.csv",
        ],
    )
    if base_df is None:
        raise FileNotFoundError(
            "Could not find a Layer B / Stage 5 hotspot table. "
            "Expected phase5_with_layer_b.csv or phase5_cluster_capacity_loss.csv."
        )

    base_df = base_df.copy()
    base_cluster_col = standardize_cluster_col(base_df)

    # Standardize to a working cluster_id column, but keep original too.
    if "cluster_id" not in base_df.columns:
        base_df["cluster_id"] = base_df[base_cluster_col]
    else:
        base_df["cluster_id"] = base_df["cluster_id"].fillna(base_df[base_cluster_col])
    # Normalise to string — CSVs may load it as int64, but hotspot_df uses str.
    base_df["cluster_id"] = base_df["cluster_id"].astype(str)

    base_df = ensure_label_column(base_df)
    base_df = derive_coords(base_df)

    # Helpful display label.
    if "cluster_label_display" not in base_df.columns:
        base_df["cluster_label_display"] = (
            base_df["cluster_label"].astype(str) +
            " (Cluster " +
            base_df["cluster_id"].astype(str) +
            ")"
        )

    # -------------------------
    # Optional Layer C enrichment
    # -------------------------
    layer_c_df, layer_c_src = load_optional_table(
        LAYER_C_DIRS,
        [
            "layer_c_enriched_hotspots.csv",
            "layer_c_hotspot_features.csv",
            "layer_c_novel_features.csv",
            "phase5_with_layer_c.csv",
            "layer_c_final.csv",
        ],
    )
    if layer_c_df is not None and len(layer_c_df):
        layer_c_df = layer_c_df.copy()
        try:
            layer_c_cluster_col = standardize_cluster_col(layer_c_df)
            if "cluster_id" not in layer_c_df.columns:
                layer_c_df["cluster_id"] = layer_c_df[layer_c_cluster_col]
            layer_c_df["cluster_id"] = layer_c_df["cluster_id"].astype(str)
            layer_c_df = ensure_label_column(layer_c_df)
            layer_c_df = derive_coords(layer_c_df)
            base_df = safe_merge_extra_columns(base_df, layer_c_df, key="cluster_id")
        except Exception as e:
            print(f"Layer C merge skipped: {e}")

    # -------------------------
    # Ensure required columns exist
    # -------------------------
    required_numeric_defaults = {
        "ccs_score": 0.0,
        "delay_minutes_per_vehicle": 0.0,
        "records_total": 0.0,
        "distinct_days": 1.0,
        "severity_sum": 0.0,
        "severity_mean": 0.0,
        "growth_pct_change": 0.0,
        "growth_multiplier": 1.0,
        "layer_b_priority_boost": 0.0,
        "context_multiplier": 1.0,
        "validation_uncertainty": np.nan,
        "resurgence_score": np.nan,
        "persistence_score": np.nan,
        "anomaly_score": np.nan,
        "rop": np.nan,
        "tvs": np.nan,
        "vdi": np.nan,
        "repeat_vehicle_count_2plus": 0.0,
        "chronic_vehicle_count_5plus": 0.0,
        "nearby_sensitive_poi_count": 0.0,
        "lane_count": np.nan,
        "carriageway_width_m": np.nan,
        "link_length_m": np.nan,
        "junction_degree": np.nan,
        "betweenness_centrality": np.nan,
        "road_node_distance_m": np.nan,
        "road_node_degree": np.nan,
        "road_node_betweenness": np.nan,
        "network_pagerank": np.nan,
        "spillover_out_score": np.nan,
        "spillover_in_score": np.nan,
        "spillover_total_score": np.nan,
        "propagation_radius_m": np.nan,
    }
    for c, default in required_numeric_defaults.items():
        if c not in base_df.columns:
            base_df[c] = default

    # Coerce numerics.
    for c in required_numeric_defaults:
        base_df[c] = pd.to_numeric(base_df[c], errors="coerce")

    # Derive missing/flat Layer C-like features.
    base_df = ensure_feature_defaults(base_df)

    # If label display was not present in optional files, build it.
    if "cluster_label_display" not in base_df.columns:
        base_df["cluster_label_display"] = (
            base_df["cluster_label"].astype(str) +
            " (Cluster " +
            base_df["cluster_id"].astype(str) +
            ")"
        )

    # Use a consistent hotspot key.
    base_df["physical_hotspot_key"] = make_hotspot_key(base_df)

    # Keep only coordinates-bearing rows.
    base_df["lat"] = pd.to_numeric(base_df["lat"], errors="coerce")
    base_df["lon"] = pd.to_numeric(base_df["lon"], errors="coerce")
    base_df = base_df.dropna(subset=["lat", "lon"]).copy()

    # Optional top-N restriction.
    if TOP_N and TOP_N > 0:
        base_df = base_df.sort_values(
            ["ccs_score", "delay_minutes_per_vehicle", "records_total"],
            ascending=[False, False, False],
        ).head(TOP_N).copy()

    # -------------------------
    # Collapse duplicates (same physical location)
    # -------------------------
    print("Collapsing duplicate physical hotspots...")
    hotspot_df = collapse_physical_hotspots(base_df)

    if len(hotspot_df) == 0:
        raise RuntimeError("No hotspot rows found after collapsing duplicates.")

    hotspot_df["cluster_label_display"] = hotspot_df["cluster_label_display"].fillna("").astype(str)
    hotspot_df["cluster_label_display"] = np.where(
        hotspot_df["cluster_label_display"].str.strip().eq(""),
        hotspot_df["cluster_label"].astype(str) + " (Cluster " + hotspot_df["cluster_id"].astype(str) + ")",
        hotspot_df["cluster_label_display"],
    )

    # -------------------------
    # Build / load road graph once
    # -------------------------
    center_lat = float(pd.to_numeric(hotspot_df["lat"], errors="coerce").mean())
    center_lon = float(pd.to_numeric(hotspot_df["lon"], errors="coerce").mean())
    print("Loading road graph...")
    G, graph_source = load_or_build_road_graph(GRAPH_QUERY, fallback_center=(center_lat, center_lon))
    print("Road graph source:", graph_source)

    print("Loading / computing node betweenness...")
    bc_map, bc_method = load_or_compute_node_betweenness(G)
    print("Betweenness method:", bc_method)

    # -------------------------
    # Road-node assignment
    # -------------------------
    hotspot_df = assign_road_nodes(hotspot_df, G, bc_map)

    # Normalize road metrics for later use.
    hotspot_df["road_node_betweenness_norm"] = minmax(hotspot_df["road_node_betweenness"].fillna(0.0))
    hotspot_df["road_node_degree_norm"] = minmax(hotspot_df["road_node_degree"].fillna(0.0))
    hotspot_df["road_node_distance_norm"] = minmax(hotspot_df["road_node_distance_m"].fillna(hotspot_df["road_node_distance_m"].max() if hotspot_df["road_node_distance_m"].notna().any() else 0.0))

    # If layer B priority/context exists, turn it into a boost term.
    if "context_multiplier" in hotspot_df.columns:
        hotspot_df["context_boost"] = pd.to_numeric(hotspot_df["context_multiplier"], errors="coerce").fillna(1.0) - 1.0
    else:
        hotspot_df["context_boost"] = 0.0

    if "layer_b_priority_boost" not in hotspot_df.columns:
        hotspot_df["layer_b_priority_boost"] = 0.0

    # -------------------------
    # Build hotspot spillover graph
    # -------------------------
    hotspot_df = build_source_pressure(hotspot_df)

    # Make sure road centrality is present as a numeric proxy.
    hotspot_df["road_node_betweenness"] = pd.to_numeric(hotspot_df["road_node_betweenness"], errors="coerce").fillna(0.0)
    hotspot_df["road_node_degree"] = pd.to_numeric(hotspot_df["road_node_degree"], errors="coerce").fillna(0.0)

    print("Building hotspot spillover network...")
    network_df, edge_df = build_spillover_network(hotspot_df)

    # network_df IS the fully enriched hotspot frame — build_spillover_network
    # assigns all spillover/pagerank/component columns directly onto the work
    # DataFrame and returns it. There is no merge step needed; using hotspot_df
    # here would silently overwrite computed scores with the stub zeros that
    # collapse_physical_hotspots put in place earlier.
    hotspot_df = network_df.copy()

    # Standardize types on the network columns (guard against any edge case).
    for c in ["spillover_out_score", "spillover_in_score", "spillover_total_score",
              "neighbor_count", "in_neighbor_count", "out_neighbor_count",
              "influence_asymmetry", "propagation_radius_m",
              "network_pagerank", "network_component_id", "network_component_size"]:
        if c not in hotspot_df.columns:
            hotspot_df[c] = np.nan
        hotspot_df[c] = pd.to_numeric(hotspot_df[c], errors="coerce")

    hotspot_df["spillover_out_score"]   = hotspot_df["spillover_out_score"].fillna(0.0)
    hotspot_df["spillover_in_score"]    = hotspot_df["spillover_in_score"].fillna(0.0)
    hotspot_df["spillover_total_score"] = hotspot_df["spillover_total_score"].fillna(
        hotspot_df["spillover_out_score"] + hotspot_df["spillover_in_score"]
    )
    hotspot_df["neighbor_count"]          = hotspot_df["neighbor_count"].fillna(0).astype(int)
    hotspot_df["in_neighbor_count"]       = hotspot_df["in_neighbor_count"].fillna(0).astype(int)
    hotspot_df["out_neighbor_count"]      = hotspot_df["out_neighbor_count"].fillna(0).astype(int)
    hotspot_df["network_component_id"]   = hotspot_df["network_component_id"].fillna(0).astype(int)
    hotspot_df["network_component_size"] = hotspot_df["network_component_size"].fillna(1).astype(int)
    hotspot_df["layer_d_alert_flag"]     = hotspot_df["layer_d_alert_flag"].fillna(False).astype(bool) if "layer_d_alert_flag" in hotspot_df.columns else False

    # -------------------------
    # Final Layer D score
    # -------------------------
    hotspot_df = finalize_layer_d(hotspot_df, edge_df)

    # -------------------------
    # Clean display / output columns
    # -------------------------
    hotspot_df["cluster_id"] = hotspot_df["cluster_id"].astype(str)
    hotspot_df["cluster_label"] = hotspot_df["cluster_label"].fillna("").astype(str)
    hotspot_df["cluster_label_display"] = hotspot_df["cluster_label_display"].fillna("").astype(str)
    hotspot_df["road_class"] = hotspot_df.get("road_class", "road").fillna("road").astype(str)
    hotspot_df["mappls_address"] = hotspot_df.get("mappls_address", "").fillna("").astype(str)

    # Network confidence / quality fields
    hotspot_df["road_node_distance_m"] = pd.to_numeric(hotspot_df["road_node_distance_m"], errors="coerce")
    hotspot_df["road_node_degree"] = pd.to_numeric(hotspot_df["road_node_degree"], errors="coerce")
    hotspot_df["road_node_betweenness"] = pd.to_numeric(hotspot_df["road_node_betweenness"], errors="coerce")

    # -------------------------
    # Output files
    # -------------------------
    full_output = hotspot_df.copy()
    edge_output = edge_df.copy()

    # Ranking / hotspot summary.
    hotspot_summary_cols = [
        "network_rank",
        "cluster_id",
        "cluster_label",
        "cluster_label_display",
        "risk_band",
        "network_vulnerability_score",
        "source_pressure_score",
        "spillover_out_score",
        "spillover_in_score",
        "spillover_total_score",
        "secondary_risk_score",
        "propagation_radius_m",
        "neighbor_count",
        "road_node_betweenness",
        "road_node_degree",
        "road_node_distance_m",
        "network_pagerank",
        "network_component_id",
        "network_component_size",
        "layer_d_alert_flag",
        "propagation_class",
        "ccs_score",
        "delay_minutes_per_vehicle",
        "growth_pct_change",
        "growth_multiplier",
        "layer_b_priority_boost",
        "context_multiplier",
        "validation_uncertainty",
        "resurgence_score",
        "persistence_score",
        "anomaly_score",
        "rop",
        "tvs",
        "vdi",
        "records_total",
        "distinct_days",
        "severity_sum",
        "nearby_sensitive_poi_count",
        "road_class",
        "geometry_source",
        "mappls_address",
        "lat",
        "lon",
        "merged_cluster_ids",
        "physical_hotspot_key",
    ]
    hotspot_summary_cols = [c for c in hotspot_summary_cols if c in hotspot_df.columns]

    network_metric_cols = [
        "cluster_id",
        "cluster_label_display",
        "road_node_id",
        "road_node_distance_m",
        "road_node_degree",
        "road_node_betweenness",
        "road_node_degree_norm",
        "road_node_betweenness_norm",
        "source_pressure_score",
        "source_pressure_norm",
        "spillover_out_score",
        "spillover_in_score",
        "spillover_total_score",
        "influence_asymmetry",
        "neighbor_count",
        "in_neighbor_count",
        "out_neighbor_count",
        "propagation_radius_m",
        "network_pagerank",
        "network_component_id",
        "network_component_size",
        "layer_d_alert_flag",
        "network_vulnerability_score",
        "network_rank",
        "propagation_class",
    ]
    network_metric_cols = [c for c in network_metric_cols if c in hotspot_df.columns]

    # Save files.
    full_output.to_csv(OUT_DIR / "layer_d_full_hotspot_output.csv", index=False)
    hotspot_df[hotspot_summary_cols].to_csv(OUT_DIR / "layer_d_final_ranking.csv", index=False)
    hotspot_df[network_metric_cols].to_csv(OUT_DIR / "layer_d_network_metrics.csv", index=False)
    edge_output.to_csv(OUT_DIR / "layer_d_spillover_edges.csv", index=False)

    # Merge back to upstream file shape for downstream use.
    stage5_with_d = safe_merge_extra_columns(
        base_df.copy(),
        hotspot_df.rename(columns={"cluster_id": "cluster_id"}),
        key="cluster_id",
    )
    stage5_with_d.to_csv(OUT_DIR / "phase5_with_layer_d.csv", index=False)

    # Summary.
    summary = pd.DataFrame(
        [
            {
                "input_source": str(base_src),
                "layer_c_source": str(layer_c_src) if layer_c_src is not None else "",
                "graph_source": graph_source,
                "betweenness_method": bc_method,
                "clusters_scored": int(len(full_output)),
                "distinct_hotspots": int(len(hotspot_df)),
                "spillover_edges": int(len(edge_output)),
                "alerts_flagged": int(hotspot_df["layer_d_alert_flag"].sum()) if "layer_d_alert_flag" in hotspot_df.columns else 0,
                "mean_network_vulnerability_score": float(pd.to_numeric(hotspot_df["network_vulnerability_score"], errors="coerce").mean()) if len(hotspot_df) else np.nan,
                "max_network_vulnerability_score": float(pd.to_numeric(hotspot_df["network_vulnerability_score"], errors="coerce").max()) if len(hotspot_df) else np.nan,
                "mean_spillover_total_score": float(pd.to_numeric(hotspot_df["spillover_total_score"], errors="coerce").mean()) if len(hotspot_df) else np.nan,
                "mean_road_node_distance_m": float(pd.to_numeric(hotspot_df["road_node_distance_m"], errors="coerce").mean()) if len(hotspot_df) else np.nan,
            }
        ]
    )
    summary.to_csv(OUT_DIR / "layer_d_summary.csv", index=False)

    # -------------------------
    # Console output
    # -------------------------
    print("Layer D complete")
    print("Output directory:", OUT_DIR.resolve())
    print("\nSummary:")
    print(summary.to_string(index=False))

    print("\nTop 10 Layer D hotspots:")
    top_cols = [
        "network_rank",
        "cluster_id",
        "cluster_label_display",
        "risk_band",
        "network_vulnerability_score",
        "source_pressure_score",
        "spillover_total_score",
        "spillover_out_score",
        "spillover_in_score",
        "propagation_radius_m",
        "road_node_betweenness",
        "network_pagerank",
        "network_component_size",
        "layer_d_alert_flag",
    ]
    top_cols = [c for c in top_cols if c in hotspot_df.columns]
    print(hotspot_df[top_cols].head(10).to_string(index=False))


if __name__ == "__main__":
    main()

Collapsing duplicate physical hotspots...
Loading road graph...
Road graph source: cache:content\layer_d_outputs_2\cache\bengaluru_drive_graph.graphml
Loading / computing node betweenness...
Betweenness method: cache
Assigning nearest road nodes...
Building hotspot spillover network...
Computing pairwise hotspot distances...
Building spillover edges for 6504 hotspot pairs...
  processed 1000/6504 pairs
  processed 2000/6504 pairs
  processed 3000/6504 pairs
  processed 4000/6504 pairs
  processed 5000/6504 pairs
  processed 6000/6504 pairs
Layer D complete
Output directory: D:\Flipkart_Hackathon\approach1\content\layer_d_outputs_2

Summary:
                                             input_source                                       layer_c_source                                                        graph_source betweenness_method  clusters_scored  distinct_hotspots  spillover_edges  alerts_flagged  mean_network_vulnerability_score  max_network_vulnerability_score  mean_spillover_t

In [28]:
# ============================================================
# Phase 7 — Next-Week Forecasting
# Predicts next-week hotspot risk, forecast band, and escalation probability
# using weekly hotspot history derived from Phases 1–6.
#
# ------------------------------------------------------------
# CORRECTIONS APPLIED IN THIS VERSION
# ------------------------------------------------------------
# 1. ROOT-CAUSE FIX for "weekly_growth_pct always 0.0" and "predictions
#    jumping from ~5 to ~98":
#    prepare_record_history() used to drop every row that didn't have a
#    *known* next-week target before returning. That target is only known
#    up to a hotspot's second-to-last week (the true last/latest week, by
#    definition, has no future week recorded yet). Forecasting code then
#    picked the "latest row per hotspot" out of that already-filtered
#    table, so it was always grabbing a STALE row, not the real present.
#    For a hotspot with only 2 observed weeks (the common case here, since
#    most hotspots have very little history), that stale row is literally
#    the hotspot's first-ever week — which has no prior week to diff
#    against, hence weekly_growth_pct == 0.0 — and the "current" score it
#    forecasts from is up to a week (or more) behind reality, which is what
#    produced unrealistic-looking deltas.
#    Fix: prepare_record_history() now returns the FULL weekly history
#    (including each hotspot's true latest, target-less week).
#    fit_predict_forecast() still drops target-less rows when building the
#    *training* set (model_df) — that part of the original logic was
#    already correct — but now pulls each hotspot's truly latest row from
#    the full, un-filtered history for inference.
#    NOTE: weekly_growth_pct's formula itself (lag-based, via shift(1)) was
#    already implemented correctly; nothing needed to change there.
#
# 2. Production guardrails on top of the fix above: a clip on week-over-week
#    score swings (MAX_DELTA_SCORE) and exponential smoothing toward the
#    current score (FORECAST_SMOOTHING_ALPHA), so a single noisy prediction
#    on a sparse/young hotspot still can't publish a 90+ point jump.
#
# 3. Escalation probabilities now pass through probability calibration
#    (CalibratedClassifierCV) so confidence values are less overconfident.
#    Added an escalation_brier_score metric — AUC/accuracy/F1 are all blind
#    to overconfidence, Brier score isn't — to monitor this going forward.
#    (The escalation_flag label definition itself — next-week raw pressure
#    >= 1.15x current raw pressure, using only past/contemporaneous
#    features — was already correct and uses no future-leaking columns.)
#
# 4. New diagnostics: a weeks-of-history-per-hotspot distribution (makes
#    data sparsity visible instead of silently producing growth == 0 for
#    single-week hotspots), and a warning when most Phase 6 hotspots carry
#    geometry_source == "fallback" (an upstream Phase 5/6 issue this script
#    cannot itself fix, but should surface loudly).
#
# 5. Forecasts are now ranked by a composite forecast_priority_score
#    (predicted score + escalation probability + upside delta) instead of
#    predicted score alone.
#
# 6. Replaced a `groupby(...).apply(per_group_function)` step in the weekly
#    feature engineering with plain vectorized groupby().shift()/.rolling()
#    calls. Whether that kind of apply's sub-frame includes the grouping
#    column is version-dependent — pandas 3.0 always excludes it with no
#    opt-out — so on newer pandas the original helper silently dropped
#    cluster_key and crashed the very next merge. This is now version-safe.
# ============================================================
import ast
import warnings
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# -------------------------
# Optional model backends
# -------------------------
BACKEND = None
XGBRegressor = XGBClassifier = None
LGBMRegressor = LGBMClassifier = None

try:
    from xgboost import XGBRegressor, XGBClassifier  # type: ignore
    BACKEND = "xgboost"
except Exception:
    try:
        from lightgbm import LGBMRegressor, LGBMClassifier  # type: ignore
        BACKEND = "lightgbm"
    except Exception:
        from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
        BACKEND = "sklearn_forest"

from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    # scikit-learn >= 1.4
    from sklearn.metrics import root_mean_squared_error  # type: ignore

    def rmse_metric(y_true, y_pred) -> float:
        return float(root_mean_squared_error(y_true, y_pred))
except Exception:
    def rmse_metric(y_true, y_pred) -> float:
        return float(np.sqrt(mean_squared_error(y_true, y_pred)))

try:
    import joblib
    JOBLIB_OK = True
except Exception:
    import pickle
    JOBLIB_OK = False

# -------------------------
# Config
# -------------------------
PHASE6_DIRS = [
    Path("content/phase6_outputs_2"),
    Path("phase6_outputs_2"),
    Path("/content/phase6_outputs_2"),
]
PHASE5_DIRS = [
    Path("content/phase5_outputs_2"),
    Path("phase5_outputs_2"),
    Path("/content/phase5_outputs_2"),
]
PHASE4_DIRS = [
    Path("content/phase4_outputs_2"),
    Path("phase4_outputs_2"),
    Path("/content/phase4_outputs_2"),
]
PHASE3_DIRS = [
    Path("content/phase3_outputs_2"),
    Path("phase3_outputs_2"),
    Path("/content/phase3_outputs_2"),
]

OUT_DIR = Path("content/phase7_outputs_2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EPS = 1e-9
RANDOM_STATE = 42
VALIDATION_FRACTION_BY_WEEKS = 0.20
ESCALATION_INCREASE_THRESHOLD = 0.15  # 15% week-over-week increase

# --- Production guardrails (see CORRECTIONS #2 above) -----------------
ENABLE_DELTA_CLAMP = True
MAX_DELTA_SCORE = 20.0            # max allowed |next_week_score - current_week_score|
ENABLE_FORECAST_SMOOTHING = True
FORECAST_SMOOTHING_ALPHA = 0.7    # weight on CURRENT score; (1 - alpha) on the model's raw prediction

# --- Probability calibration (see CORRECTIONS #3 above) ----------------
ENABLE_PROBABILITY_CALIBRATION = True

# --- Ranking weights (see CORRECTIONS #5 above) -------------------------
PRIORITY_WEIGHT_SCORE = 0.5
PRIORITY_WEIGHT_ESCALATION = 30.0
PRIORITY_WEIGHT_DELTA = 10.0

# -------------------------
# Severity mapping
# -------------------------
SEVERITY_RULES = {
    5: {
        "DOUBLE PARKING",
        "NEAR ROAD CROSSING",
        "NEAR TRAFFIC LIGHT",
        "NEAR ZEBRA CROSSING",
        "NEAR TRAFFIC LIGHT / ZEBRA CROSSING",
        "NEAR TRAFFIC LIGHT/ZEBRA CROSSING",
    },
    4: {
        "PARKING IN MAIN ROAD",
        "NEAR BUS STOP",
        "NEAR SCHOOL",
        "NEAR HOSPITAL",
        "OPPOSITE ANOTHER VEHICLE",
    },
    3: {"PARKING ON FOOTPATH"},
    2: {"WRONG PARKING", "PARKING OTHER THAN BUS STOP"},
    1: {"NO PARKING", "NO PARKING (GENERIC)"},
}

# -------------------------
# Helpers
# -------------------------
def clean_text(x) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()


def normalize_tag(tag) -> str:
    return clean_text(tag).upper().replace("&", "AND").strip()


def parse_listlike(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    s = str(value).strip()
    if not s:
        return []
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple)):
            return list(parsed)
        return [parsed]
    except Exception:
        if s.startswith("[") and s.endswith("]"):
            s = s[1:-1]
        parts = [p.strip().strip("'").strip('"') for p in s.split(",")]
        return [p for p in parts if p]


def safe_float(x, default=np.nan):
    try:
        if pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default


def standardize_cluster_col(df: pd.DataFrame) -> str:
    for c in ["st_dbscan_cluster_id", "cluster_id", "dbscan_cluster_id"]:
        if c in df.columns:
            return c
    raise ValueError("No cluster id column found.")


def standardize_vehicle_col(df: pd.DataFrame) -> Optional[str]:
    for c in ["canonical_vehicle_number", "vehicle_number", "updated_vehicle_number"]:
        if c in df.columns:
            return c
    return None


def standardize_vehicle_type_col(df: pd.DataFrame) -> Optional[str]:
    for c in ["canonical_vehicle_type", "vehicle_type", "updated_vehicle_type"]:
        if c in df.columns:
            return c
    return None


def normalize_cluster_key(x) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip()
    if not s:
        return ""
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
        return str(f)
    except Exception:
        return s


def load_first_existing(paths: Iterable[Path], filenames: Iterable[str]):
    for d in paths:
        for name in filenames:
            p = d / name
            if p.exists():
                return pd.read_csv(p, low_memory=False), p
    return None, None


def derive_coords(df: pd.DataFrame):
    if df is None or len(df) == 0:
        return df
    df = df.copy()
    if "lat" in df.columns and "lon" in df.columns:
        df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
        df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    elif {"centroid_lat", "centroid_lon"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["centroid_lat"], errors="coerce")
        df["lon"] = pd.to_numeric(df["centroid_lon"], errors="coerce")
    elif {"latitude_mean", "longitude_mean"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["latitude_mean"], errors="coerce")
        df["lon"] = pd.to_numeric(df["longitude_mean"], errors="coerce")
    elif {"latitude", "longitude"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["latitude"], errors="coerce")
        df["lon"] = pd.to_numeric(df["longitude"], errors="coerce")
    else:
        df["lat"] = np.nan
        df["lon"] = np.nan
    return df


def ensure_label_column(df: pd.DataFrame):
    if df is None or len(df) == 0:
        return df
    df = df.copy()
    if "cluster_label" in df.columns:
        df["cluster_label"] = df["cluster_label"].fillna("").astype(str).str.strip()
        return df
    if "hotspot_unit" in df.columns:
        df["cluster_label"] = df["hotspot_unit"].fillna("").astype(str).str.strip()
        return df
    if "dominant_junction_name" in df.columns:
        df["cluster_label"] = df["dominant_junction_name"].fillna("").astype(str).str.strip()
        return df
    if "st_dbscan_cluster_id" in df.columns:
        df["cluster_label"] = "CLUSTER::" + df["st_dbscan_cluster_id"].astype(str)
        return df
    df["cluster_label"] = "UNKNOWN"
    return df


def parse_datetime_ist(df: pd.DataFrame) -> pd.Series:
    if "created_datetime_ist" in df.columns:
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce", utc=True)
        if ts.notna().any():
            return ts.dt.tz_convert("Asia/Kolkata")
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce")
        return ts.dt.tz_localize("Asia/Kolkata", nonexistent="NaT", ambiguous="NaT")

    if "created_datetime_parsed" in df.columns:
        ts = pd.to_datetime(df["created_datetime_parsed"], errors="coerce", utc=True)
        return ts.dt.tz_convert("Asia/Kolkata")

    if "created_datetime" not in df.columns:
        return pd.Series(pd.NaT, index=df.index)

    ts = pd.to_datetime(df["created_datetime"], errors="coerce", utc=True)
    return ts.dt.tz_convert("Asia/Kolkata")


def week_start_monday(series):
    dt = pd.to_datetime(series, errors="coerce", utc=True).dt.tz_convert("Asia/Kolkata")
    week_start = dt.dt.normalize() - pd.to_timedelta(dt.dt.weekday, unit="D")
    return week_start.dt.tz_localize(None)


def severity_from_tags(tags):
    if not tags:
        return 1
    normalized = [normalize_tag(t) for t in tags]
    score = 1
    for sev in sorted(SEVERITY_RULES.keys(), reverse=True):
        vocab = SEVERITY_RULES[sev]
        if any(any(v == tag or v in tag for v in vocab) for tag in normalized):
            score = sev
            break
    return score


def make_hotspot_unit(row):
    junction = clean_text(row.get("junction_name", ""))
    if junction and junction.upper() != "NO JUNCTION":
        return f"JUNCTION::{junction}"
    station = clean_text(row.get("police_station", "UNKNOWN"))
    if not station:
        station = "UNKNOWN"
    return f"POLICE_STATION::{station}"


def add_missing_cols(df: pd.DataFrame, defaults: Dict[str, object]) -> pd.DataFrame:
    df = df.copy()
    for c, default in defaults.items():
        if c not in df.columns:
            df[c] = default
    return df


def build_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_slope(values: np.ndarray) -> float:
    y = np.asarray(values, dtype=float)
    if len(y) < 2 or np.allclose(y, y[0], equal_nan=True):
        return 0.0
    x = np.arange(len(y), dtype=float)
    try:
        coef = np.polyfit(x, y, 1)[0]
        if np.isfinite(coef):
            return float(coef)
    except Exception:
        pass
    return 0.0


def pressure_raw_from_week(df: pd.DataFrame) -> pd.Series:
    rec = pd.to_numeric(df["records_week"], errors="coerce").fillna(0.0).clip(lower=0.0)
    sev_mean = pd.to_numeric(df["severity_mean_week"], errors="coerce").fillna(1.0).clip(lower=1.0, upper=5.0)
    peak_ratio = pd.to_numeric(df["peak_window_ratio_week"], errors="coerce").fillna(0.0).clip(lower=0.0, upper=1.0)
    repeat_ratio = pd.to_numeric(df["repeat_vehicle_ratio_week"], errors="coerce").fillna(0.0).clip(lower=0.0, upper=1.0)
    growth_surge = pd.to_numeric(df["growth_surge_week"], errors="coerce").fillna(0.0).clip(lower=0.0, upper=5.0)
    congestion = np.log1p(rec) * (1.0 + sev_mean / 5.0) * (1.0 + 0.5 * peak_ratio) * (1.0 + repeat_ratio) * (1.0 + growth_surge)
    return congestion.replace([np.inf, -np.inf], np.nan).fillna(0.0)


def score_to_0_100(raw: pd.Series, lo: float, hi: float) -> pd.Series:
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo + EPS:
        return pd.Series(np.full(len(raw), 50.0), index=raw.index)
    return 100.0 * ((pd.to_numeric(raw, errors="coerce").fillna(lo) - lo) / (hi - lo + EPS)).clip(0.0, 1.0)


def band_from_score(score: float) -> str:
    if score >= 80:
        return "Critical"
    if score >= 60:
        return "High"
    if score >= 40:
        return "Moderate"
    return "Watch"


def ensure_bool_int(s):
    if s is None:
        return None
    if s.dtype == bool:
        return s.astype(int)
    if str(s.dtype).startswith("boolean"):
        return s.fillna(False).astype(int)
    return pd.to_numeric(s, errors="coerce").fillna(0).astype(int)

# -------------------------
# Load inputs
# -------------------------
def load_static_phase6():
    df, src = load_first_existing(
        PHASE6_DIRS,
        [
            "phase6_cluster_ccs_full.csv",
            "phase6_stage6_handoff.csv",
            "phase6_weekly_dispatch_priority_table.csv",
        ],
    )
    if df is None:
        raise FileNotFoundError("Could not find Phase 6 outputs.")
    return df, src


def load_record_history():
    df, src = load_first_existing(
        PHASE5_DIRS + PHASE4_DIRS + PHASE3_DIRS,
        [
            "phase5_enriched_records.csv",
            "phase4_merged_with_prior_scores.csv",
            "phase3_clustered_dataset.csv",
        ],
    )
    if df is None:
        raise FileNotFoundError("Could not find Phase 5/4/3 record-level source.")
    return df, src

# -------------------------
# Core feature engineering
# -------------------------
def prepare_static_context(static_df: pd.DataFrame) -> pd.DataFrame:
    df = static_df.copy()
    df = ensure_label_column(df)
    df = derive_coords(df)

    cluster_col = standardize_cluster_col(df)
    df["cluster_key"] = df[cluster_col].map(normalize_cluster_key)

    defaults = {
        "cluster_label": "UNKNOWN",
        "cluster_label_display": "",
        "risk_band": "Watch",
        "ccs_score": 0.0,
        "delay_minutes_per_vehicle": 0.0,
        "records_total": 0.0,
        "distinct_days": 0.0,
        "severity_sum": 0.0,
        "severity_mean": 1.0,
        "growth_pct_change": 0.0,
        "growth_multiplier": 1.0,
        "criticality_factor": 1.0,
        "context_multiplier": 1.0,
        "layer_b_priority_boost": 0.0,
        "layer_b_alert_flag": 0,
        "validation_uncertainty": 0.0,
        "resurgence_score": 0.0,
        "persistence_score": 0.0,
        "anomaly_score": 0.0,
        "rop": 0.0,
        "tvs": 0.0,
        "vdi": 0.0,
        "nearby_sensitive_poi_count": 0.0,
        "road_class": "road",
        "lane_count": 2.0,
        "carriageway_width_m": 7.0,
        "link_length_m": 250.0,
        "junction_degree": 4.0,
        "betweenness_centrality": 0.0,
        "geometry_source": "fallback",
        "mappls_address": "",
        "road_node_id": np.nan,
        "road_node_distance_m": 0.0,
        "road_node_degree": 0.0,
        "road_node_betweenness": 0.0,
        "source_pressure_score": 0.0,
        "source_pressure_norm": 0.0,
        "spillover_out_score": 0.0,
        "spillover_in_score": 0.0,
        "spillover_total_score": 0.0,
        "propagation_radius_m": 0.0,
        "network_pagerank": 0.0,
        "network_component_id": 0,
        "network_component_size": 1,
        "neighbor_count": 0.0,
        "in_neighbor_count": 0.0,
        "out_neighbor_count": 0.0,
        "influence_asymmetry": 0.0,
        "network_vulnerability_score": 0.0,
        "layer_d_alert_flag": 0,
        "dominant_vehicle_type": "UNKNOWN",
    }
    df = add_missing_cols(df, defaults)

    numeric_cols = [
        "ccs_score", "delay_minutes_per_vehicle", "records_total", "distinct_days",
        "severity_sum", "severity_mean", "growth_pct_change", "growth_multiplier",
        "criticality_factor", "context_multiplier", "layer_b_priority_boost",
        "validation_uncertainty", "resurgence_score", "persistence_score",
        "anomaly_score", "rop", "tvs", "vdi", "nearby_sensitive_poi_count",
        "lane_count", "carriageway_width_m", "link_length_m", "junction_degree",
        "betweenness_centrality", "road_node_distance_m", "road_node_degree",
        "road_node_betweenness", "source_pressure_score", "source_pressure_norm",
        "spillover_out_score", "spillover_in_score", "spillover_total_score",
        "propagation_radius_m", "network_pagerank", "network_component_id",
        "network_component_size", "neighbor_count", "in_neighbor_count",
        "out_neighbor_count", "influence_asymmetry", "network_vulnerability_score",
    ]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["layer_b_alert_flag"] = ensure_bool_int(df["layer_b_alert_flag"])
    df["layer_d_alert_flag"] = ensure_bool_int(df["layer_d_alert_flag"])

    for c in ["road_class", "geometry_source", "dominant_vehicle_type"]:
        df[c] = df[c].fillna("UNKNOWN").astype(str).str.strip()

    df = df.sort_values(["ccs_score", "delay_minutes_per_vehicle", "records_total"], ascending=[False, False, False])
    df = df.drop_duplicates(subset=["cluster_key"], keep="first").reset_index(drop=True)
    return df


def prepare_record_history(records_df: pd.DataFrame, static_df: pd.DataFrame) -> Tuple[pd.DataFrame, str]:
    r = records_df.copy()
    cluster_col = standardize_cluster_col(r)
    vehicle_col = standardize_vehicle_col(r)
    vehicle_type_col = standardize_vehicle_type_col(r)

    if "validation_status" in r.columns:
        r["validation_status"] = r["validation_status"].fillna("").astype(str).str.lower()
        approved = r[r["validation_status"].eq("approved")].copy()
        if len(approved) > 0:
            r = approved.copy()

    r = ensure_label_column(r)
    r = derive_coords(r)
    r["created_datetime_ist"] = parse_datetime_ist(r)
    r = r.dropna(subset=["created_datetime_ist"]).copy()

    if "severity_score" not in r.columns:
        if "violation_type" in r.columns:
            r["violation_tags"] = r["violation_type"].apply(parse_listlike)
            r["severity_score"] = r["violation_tags"].apply(severity_from_tags)
        else:
            r["severity_score"] = 1
    r["severity_score"] = pd.to_numeric(r["severity_score"], errors="coerce").fillna(1).clip(lower=1, upper=5)

    if "hotspot_unit" not in r.columns:
        r["hotspot_unit"] = r.apply(make_hotspot_unit, axis=1)

    r["service_date"] = pd.to_datetime(r["created_datetime_ist"], errors="coerce").dt.tz_localize(None).dt.date
    r["week_start"] = week_start_monday(r["created_datetime_ist"])
    r["hour_ist"] = r["created_datetime_ist"].dt.hour
    r["is_peak_window"] = r["hour_ist"].between(8, 12, inclusive="both").astype(int)

    r["cluster_key"] = r[cluster_col].map(normalize_cluster_key)

    agg_spec = {
        "records_week": ("week_start", "size"),
        "active_days_week": ("service_date", "nunique"),
        "severity_sum_week": ("severity_score", "sum"),
        "severity_mean_week": ("severity_score", "mean"),
        "peak_window_records_week": ("is_peak_window", "sum"),
    }
    if vehicle_col and vehicle_col in r.columns:
        r[vehicle_col] = r[vehicle_col].fillna("").astype(str).str.strip()
        r = r[r[vehicle_col].ne("")].copy()
        agg_spec["unique_vehicles_week"] = (vehicle_col, "nunique")
    else:
        r["vehicle_fallback"] = "UNKNOWN"
        agg_spec["unique_vehicles_week"] = ("vehicle_fallback", "nunique")

    if vehicle_type_col and vehicle_type_col in r.columns:
        r[vehicle_type_col] = r[vehicle_type_col].fillna("").astype(str).str.strip()
        agg_spec["unique_vehicle_types_week"] = (vehicle_type_col, "nunique")
        agg_spec["dominant_vehicle_type_week"] = (vehicle_type_col, lambda s: s.mode().iloc[0] if not s.mode().empty else "UNKNOWN")
    else:
        r["vehicle_type_fallback"] = "UNKNOWN"
        agg_spec["unique_vehicle_types_week"] = ("vehicle_type_fallback", "nunique")
        agg_spec["dominant_vehicle_type_week"] = ("vehicle_type_fallback", lambda s: "UNKNOWN")

    weekly = (
        r.groupby(["cluster_key", "week_start"])
        .agg(**agg_spec)
        .reset_index()
        .sort_values(["cluster_key", "week_start"])
        .reset_index(drop=True)
    )

    weekly["records_per_active_day_week"] = weekly["records_week"] / weekly["active_days_week"].clip(lower=1)
    weekly["peak_window_ratio_week"] = weekly["peak_window_records_week"] / weekly["records_week"].clip(lower=1)
    weekly["repeat_vehicle_ratio_week"] = (
        (weekly["records_week"] - weekly["unique_vehicles_week"]) / weekly["records_week"].clip(lower=1)
    ).clip(lower=0.0, upper=1.0)

    weekly["prev_records_week"] = weekly.groupby("cluster_key")["records_week"].shift(1)
    weekly["weekly_growth_pct"] = (
        (weekly["records_week"] - weekly["prev_records_week"]) / (weekly["prev_records_week"].abs() + EPS)
    ).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    weekly["growth_surge_week"] = weekly["weekly_growth_pct"].clip(lower=0.0)

    weekly["weekly_pressure_raw"] = pressure_raw_from_week(weekly)

    # NOTE (robustness fix): the per-hotspot feature block below used to be
    # a `weekly.groupby("cluster_key").apply(enrich_group)` helper. Whether
    # a groupby-apply's sub-frame includes the grouping column itself has
    # changed across pandas versions — as of pandas 3.0 it is *always*
    # excluded, with no way to opt back in — which made that helper
    # silently lose the cluster_key column on newer pandas and crash the
    # very next merge. Computing every per-hotspot feature with plain
    # groupby().shift()/.rolling()/.transform() (the same pattern already
    # used above for weekly_growth_pct) sidesteps that ambiguity entirely
    # and behaves identically on old and new pandas.
    weekly = weekly.sort_values(["cluster_key", "week_start"]).reset_index(drop=True)
    grp = weekly.groupby("cluster_key", sort=False)

    weekly["week_index"] = grp.cumcount()
    weekly["hotspot_age_weeks"] = weekly["week_index"] + 1

    first_pressure_raw = grp["weekly_pressure_raw"].transform("first")
    weekly["lag1_pressure_raw"] = grp["weekly_pressure_raw"].shift(1).fillna(first_pressure_raw)
    weekly["lag2_pressure_raw"] = grp["weekly_pressure_raw"].shift(2).fillna(weekly["lag1_pressure_raw"])

    weekly["rolling_4w_mean_pressure_raw"] = grp["weekly_pressure_raw"].transform(
        lambda s: s.rolling(4, min_periods=1).mean()
    )
    weekly["rolling_4w_std_pressure_raw"] = grp["weekly_pressure_raw"].transform(
        lambda s: s.rolling(4, min_periods=1).std(ddof=0)
    ).fillna(0.0)
    weekly["rolling_4w_mean_records_week"] = grp["records_week"].transform(
        lambda s: s.rolling(4, min_periods=1).mean()
    )
    weekly["rolling_4w_mean_severity_week"] = grp["severity_sum_week"].transform(
        lambda s: s.rolling(4, min_periods=1).mean()
    )
    weekly["rolling_4w_mean_growth_week"] = grp["weekly_growth_pct"].transform(
        lambda s: s.rolling(4, min_periods=1).mean()
    )
    weekly["rolling_4w_trend_slope_records_week"] = grp["records_week"].transform(
        lambda s: s.rolling(4, min_periods=1).apply(build_slope, raw=True)
    )
    weekly["rolling_4w_pressure_acceleration"] = weekly["rolling_4w_mean_pressure_raw"] - (
        grp["rolling_4w_mean_pressure_raw"].shift(1).fillna(weekly["rolling_4w_mean_pressure_raw"])
    )

    static_ctx = static_df.copy()
    static_ctx["cluster_key"] = static_ctx[cluster_col].map(normalize_cluster_key)
    static_cols_to_merge = [
        "cluster_label", "cluster_label_display", "risk_band", "ccs_score", "delay_minutes_per_vehicle",
        "records_total", "distinct_days", "severity_sum", "severity_mean", "growth_pct_change",
        "growth_multiplier", "criticality_factor", "context_multiplier", "layer_b_priority_boost",
        "layer_b_alert_flag", "validation_uncertainty", "resurgence_score", "persistence_score",
        "anomaly_score", "rop", "tvs", "vdi", "nearby_sensitive_poi_count", "road_class",
        "lane_count", "carriageway_width_m", "link_length_m", "junction_degree", "betweenness_centrality",
        "geometry_source", "mappls_address", "road_node_id", "road_node_distance_m", "road_node_degree",
        "road_node_betweenness", "source_pressure_score", "source_pressure_norm", "spillover_out_score",
        "spillover_in_score", "spillover_total_score", "propagation_radius_m", "network_pagerank",
        "network_component_id", "network_component_size", "neighbor_count", "in_neighbor_count",
        "out_neighbor_count", "influence_asymmetry", "network_vulnerability_score", "layer_d_alert_flag",
        "dominant_vehicle_type",
    ]
    static_cols_to_merge = [c for c in static_cols_to_merge if c in static_ctx.columns]
    static_ctx = static_ctx[["cluster_key"] + static_cols_to_merge].drop_duplicates("cluster_key")
    weekly = weekly.merge(static_ctx, on="cluster_key", how="left", suffixes=("", "_static"))

    fill_defaults = {
        "cluster_label": "UNKNOWN",
        "cluster_label_display": "",
        "risk_band": "Watch",
        "ccs_score": 0.0,
        "delay_minutes_per_vehicle": 0.0,
        "records_total": 0.0,
        "distinct_days": 0.0,
        "severity_sum": 0.0,
        "severity_mean": 1.0,
        "growth_pct_change": 0.0,
        "growth_multiplier": 1.0,
        "criticality_factor": 1.0,
        "context_multiplier": 1.0,
        "layer_b_priority_boost": 0.0,
        "layer_b_alert_flag": 0,
        "validation_uncertainty": 0.0,
        "resurgence_score": 0.0,
        "persistence_score": 0.0,
        "anomaly_score": 0.0,
        "rop": 0.0,
        "tvs": 0.0,
        "vdi": 0.0,
        "nearby_sensitive_poi_count": 0.0,
        "road_class": "road",
        "lane_count": 2.0,
        "carriageway_width_m": 7.0,
        "link_length_m": 250.0,
        "junction_degree": 4.0,
        "betweenness_centrality": 0.0,
        "geometry_source": "fallback",
        "mappls_address": "",
        "road_node_id": np.nan,
        "road_node_distance_m": 0.0,
        "road_node_degree": 0.0,
        "road_node_betweenness": 0.0,
        "source_pressure_score": 0.0,
        "source_pressure_norm": 0.0,
        "spillover_out_score": 0.0,
        "spillover_in_score": 0.0,
        "spillover_total_score": 0.0,
        "propagation_radius_m": 0.0,
        "network_pagerank": 0.0,
        "network_component_id": 0,
        "network_component_size": 1,
        "neighbor_count": 0.0,
        "in_neighbor_count": 0.0,
        "out_neighbor_count": 0.0,
        "influence_asymmetry": 0.0,
        "network_vulnerability_score": 0.0,
        "layer_d_alert_flag": 0,
        "dominant_vehicle_type": "UNKNOWN",
    }
    weekly = add_missing_cols(weekly, fill_defaults)

    numeric_cols = [
        "records_week", "active_days_week", "severity_sum_week", "severity_mean_week",
        "peak_window_records_week", "unique_vehicles_week", "unique_vehicle_types_week",
        "records_per_active_day_week", "peak_window_ratio_week", "repeat_vehicle_ratio_week",
        "prev_records_week", "weekly_growth_pct", "growth_surge_week", "weekly_pressure_raw",
        "week_index", "hotspot_age_weeks", "lag1_pressure_raw", "lag2_pressure_raw",
        "rolling_4w_mean_pressure_raw", "rolling_4w_std_pressure_raw",
        "rolling_4w_mean_records_week", "rolling_4w_mean_severity_week",
        "rolling_4w_mean_growth_week", "rolling_4w_trend_slope_records_week",
        "rolling_4w_pressure_acceleration",
        "ccs_score", "delay_minutes_per_vehicle", "records_total", "distinct_days",
        "severity_sum", "severity_mean", "growth_pct_change", "growth_multiplier",
        "criticality_factor", "context_multiplier", "layer_b_priority_boost",
        "validation_uncertainty", "resurgence_score", "persistence_score", "anomaly_score",
        "rop", "tvs", "vdi", "nearby_sensitive_poi_count", "lane_count",
        "carriageway_width_m", "link_length_m", "junction_degree", "betweenness_centrality",
        "road_node_distance_m", "road_node_degree", "road_node_betweenness",
        "source_pressure_score", "source_pressure_norm", "spillover_out_score",
        "spillover_in_score", "spillover_total_score", "propagation_radius_m",
        "network_pagerank", "network_component_id", "network_component_size",
        "neighbor_count", "in_neighbor_count", "out_neighbor_count",
        "influence_asymmetry", "network_vulnerability_score",
    ]
    for c in numeric_cols:
        if c in weekly.columns:
            weekly[c] = pd.to_numeric(weekly[c], errors="coerce")

    weekly["layer_b_alert_flag"] = ensure_bool_int(weekly["layer_b_alert_flag"])
    weekly["layer_d_alert_flag"] = ensure_bool_int(weekly["layer_d_alert_flag"])

    for c in ["road_class", "geometry_source", "dominant_vehicle_type"]:
        weekly[c] = weekly[c].fillna("UNKNOWN").astype(str).str.strip()

    weekly["current_week_risk_raw"] = weekly["weekly_pressure_raw"].copy()
    weekly["target_next_week_risk_raw"] = weekly.groupby("cluster_key")["weekly_pressure_raw"].shift(-1)
    weekly["target_next_week_records_week"] = weekly.groupby("cluster_key")["records_week"].shift(-1)
    weekly["target_next_week_growth_surge_week"] = weekly.groupby("cluster_key")["growth_surge_week"].shift(-1)

    weekly["escalation_flag"] = (
        (weekly["target_next_week_risk_raw"] >= weekly["current_week_risk_raw"] * (1.0 + ESCALATION_INCREASE_THRESHOLD))
    ).astype(int)

    # FIX (see CORRECTIONS #1 at top of file): do NOT drop rows with a
    # missing target_next_week_risk_raw here. Each hotspot's most recent
    # observed week never has a known next-week target by definition — and
    # that row is exactly what downstream forecasting needs to predict
    # FROM. The original code dropped it at this point, which silently
    # forced every "latest week" lookup in fit_predict_forecast() onto a
    # stale, second-to-last week instead of the real present. Rows without
    # a target are still excluded from the *training* set later (see
    # model_df inside fit_predict_forecast) — just not from the history
    # returned here.
    weekly = weekly.reset_index(drop=True)
    return weekly, cluster_col

# -------------------------
# Modeling
# -------------------------
def choose_model_backend():
    return BACKEND or "sklearn_forest"


def build_models(scale_pos_weight: Optional[float] = None):
    backend = choose_model_backend()
    if backend == "xgboost":
        reg = XGBRegressor(
            n_estimators=350, learning_rate=0.05, max_depth=6,
            subsample=0.85, colsample_bytree=0.85, reg_lambda=1.0,
            random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist",
        )
        clf_kwargs = dict(
            n_estimators=350, learning_rate=0.05, max_depth=6,
            subsample=0.85, colsample_bytree=0.85, reg_lambda=1.0,
            random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist",
            eval_metric="logloss",
        )
        if scale_pos_weight is not None and np.isfinite(scale_pos_weight) and scale_pos_weight > 0:
            clf_kwargs["scale_pos_weight"] = float(scale_pos_weight)
        clf = XGBClassifier(**clf_kwargs)
    elif backend == "lightgbm":
        reg = LGBMRegressor(
            n_estimators=450, learning_rate=0.05, num_leaves=31,
            subsample=0.85, colsample_bytree=0.85, random_state=RANDOM_STATE, n_jobs=-1,
        )
        clf = LGBMClassifier(
            n_estimators=450, learning_rate=0.05, num_leaves=31,
            subsample=0.85, colsample_bytree=0.85, random_state=RANDOM_STATE, n_jobs=-1,
            class_weight="balanced" if scale_pos_weight is None else None,
        )
    else:
        from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
        reg = RandomForestRegressor(
            n_estimators=450, random_state=RANDOM_STATE, n_jobs=-1, min_samples_leaf=2,
        )
        clf = RandomForestClassifier(
            n_estimators=450, random_state=RANDOM_STATE, n_jobs=-1, min_samples_leaf=2,
            class_weight="balanced",
        )
    return reg, clf


def build_calibrated_classifier(base_pipe, method: str = "sigmoid", cv: int = 3):
    """Wrap a fitted-or-unfitted classifier Pipeline in probability calibration.

    See CORRECTIONS #3 at the top of this file: AUC/accuracy/F1 near-perfect
    scores are consistent with a genuinely strong signal AND overconfident
    probabilities at the same time. Calibration only rescales predict_proba
    output; it does not change which class wins, so ranking/recommendations
    built off predicted_escalation_flag are unaffected.
    """
    try:
        return CalibratedClassifierCV(estimator=base_pipe, method=method, cv=cv)
    except TypeError:
        # Older scikit-learn versions use `base_estimator` instead of `estimator`.
        return CalibratedClassifierCV(base_estimator=base_pipe, method=method, cv=cv)


def build_preprocessor(num_cols: List[str], cat_cols: List[str]):
    num_pipe = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
    cat_pipe = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", build_ohe())])
    return ColumnTransformer(
        transformers=[("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)],
        remainder="drop",
        sparse_threshold=0.0,
    )


def temporal_split(df: pd.DataFrame, week_col: str = "week_start", frac: float = 0.20):
    weeks = sorted(pd.to_datetime(df[week_col], errors="coerce").dropna().unique())
    if len(weeks) < 3:
        idx = np.arange(len(df))
        rng = np.random.default_rng(RANDOM_STATE)
        rng.shuffle(idx)
        cut = max(1, int(len(idx) * (1 - frac)))
        return df.iloc[idx[:cut]].copy(), df.iloc[idx[cut:]].copy()

    valid_weeks_count = max(1, int(round(len(weeks) * frac)))
    train_weeks = weeks[:-valid_weeks_count]
    valid_weeks = weeks[-valid_weeks_count:]

    train_df = df[df[week_col].isin(train_weeks)].copy()
    valid_df = df[df[week_col].isin(valid_weeks)].copy()

    if len(train_df) == 0 or len(valid_df) == 0:
        cut = max(1, int(len(df) * (1 - frac)))
        train_df = df.iloc[:cut].copy()
        valid_df = df.iloc[cut:].copy()
    return train_df, valid_df


def fit_predict_forecast(weekly_df, static_df, cluster_col):
    numeric_features = [
        "records_week", "active_days_week", "severity_sum_week", "severity_mean_week",
        "peak_window_records_week", "unique_vehicles_week", "unique_vehicle_types_week",
        "records_per_active_day_week", "peak_window_ratio_week", "repeat_vehicle_ratio_week",
        "weekly_growth_pct", "growth_surge_week", "current_week_risk_raw",
        "week_index", "hotspot_age_weeks", "lag1_pressure_raw", "lag2_pressure_raw",
        "rolling_4w_mean_pressure_raw", "rolling_4w_std_pressure_raw",
        "rolling_4w_mean_records_week", "rolling_4w_mean_severity_week",
        "rolling_4w_mean_growth_week", "rolling_4w_trend_slope_records_week",
        "rolling_4w_pressure_acceleration",
        "records_total", "distinct_days", "severity_sum", "severity_mean",
        "growth_pct_change", "growth_multiplier", "criticality_factor",
        "context_multiplier", "layer_b_priority_boost", "validation_uncertainty",
        "resurgence_score", "persistence_score", "anomaly_score", "rop", "tvs", "vdi",
        "nearby_sensitive_poi_count", "lane_count", "carriageway_width_m",
        "link_length_m", "junction_degree", "betweenness_centrality",
        "road_node_distance_m", "road_node_degree", "road_node_betweenness",
        "source_pressure_score", "source_pressure_norm", "spillover_out_score",
        "spillover_in_score", "spillover_total_score", "propagation_radius_m",
        "network_pagerank", "network_component_id", "network_component_size",
        "neighbor_count", "in_neighbor_count", "out_neighbor_count",
        "influence_asymmetry", "network_vulnerability_score",
        "layer_b_alert_flag", "layer_d_alert_flag",
    ]
    categorical_features = ["road_class", "geometry_source", "dominant_vehicle_type"]

    numeric_features = [c for c in numeric_features if c in weekly_df.columns]
    categorical_features = [c for c in categorical_features if c in weekly_df.columns]

    # model_df: rows with a KNOWN next-week target -> used to train/validate.
    # This correctly excludes each hotspot's true latest week (which has no
    # future yet). weekly_df itself (the full, un-filtered history) is kept
    # around below so the truly latest row per hotspot can still be used
    # for inference — see CORRECTIONS #1 at the top of this file.
    model_df = weekly_df.copy().dropna(subset=["target_next_week_risk_raw"]).copy()
    train_df, valid_df = temporal_split(model_df, week_col="week_start", frac=VALIDATION_FRACTION_BY_WEEKS)
    if len(train_df) == 0 or len(valid_df) == 0:
        raise RuntimeError("Not enough temporal history to train the forecast model.")

    X_train = train_df[numeric_features + categorical_features].copy()
    y_train_raw = train_df["target_next_week_risk_raw"].astype(float).copy()

    X_valid = valid_df[numeric_features + categorical_features].copy()
    y_valid_raw = valid_df["target_next_week_risk_raw"].astype(float).copy()

    y_train_cls = train_df["escalation_flag"].astype(int).copy()
    y_valid_cls = valid_df["escalation_flag"].astype(int).copy()

    target_min = float(y_train_raw.min())
    target_max = float(y_train_raw.max())

    def raw_to_score(raw_series: pd.Series) -> pd.Series:
        return score_to_0_100(raw_series, target_min, target_max)

    pos = int(y_train_cls.sum())
    neg = int(len(y_train_cls) - pos)
    scale_pos_weight = (neg / max(pos, 1)) if pos > 0 else 1.0

    reg_model, clf_model = build_models(scale_pos_weight=scale_pos_weight if pos > 0 else None)
    preprocessor = build_preprocessor(numeric_features, categorical_features)

    reg_pipe = Pipeline(steps=[("preprocess", preprocessor), ("model", reg_model)])
    base_clf_pipe = Pipeline(steps=[("preprocess", preprocessor), ("model", clf_model)])

    reg_pipe.fit(X_train, y_train_raw)

    use_classifier = y_train_cls.nunique() >= 2 and len(y_train_cls) >= 20
    clf_pipe = None
    if use_classifier:
        base_clf_pipe.fit(X_train, y_train_cls)
        clf_pipe = base_clf_pipe
        if ENABLE_PROBABILITY_CALIBRATION:
            # See CORRECTIONS #3: rescale overconfident probabilities
            # without changing which hotspots get flagged as escalating.
            min_class_count = int(min(pos, neg))
            if min_class_count >= 4:
                cv_folds = 3 if min_class_count >= 6 else 2
                method = "isotonic" if len(y_train_cls) >= 200 else "sigmoid"
                try:
                    calibrated = build_calibrated_classifier(base_clf_pipe, method=method, cv=cv_folds)
                    calibrated.fit(X_train, y_train_cls)
                    clf_pipe = calibrated
                except Exception:
                    clf_pipe = base_clf_pipe

    valid_pred_raw = pd.Series(reg_pipe.predict(X_valid), index=valid_df.index)
    valid_pred_score = raw_to_score(valid_pred_raw)
    valid_true_score = raw_to_score(y_valid_raw)

    valid_pred_escalation = None
    valid_pred_escalation_prob = None
    if clf_pipe is not None:
        try:
            valid_pred_escalation_prob = pd.Series(clf_pipe.predict_proba(X_valid)[:, 1], index=valid_df.index)
            valid_pred_escalation = (valid_pred_escalation_prob >= 0.5).astype(int)
        except Exception:
            valid_pred_escalation = pd.Series(clf_pipe.predict(X_valid), index=valid_df.index).astype(int)
            valid_pred_escalation_prob = valid_pred_escalation.astype(float)

    rmse = rmse_metric(y_valid_raw, valid_pred_raw) if len(valid_df) else np.nan
    mae = float(mean_absolute_error(y_valid_raw, valid_pred_raw)) if len(valid_df) else np.nan
    r2 = float(r2_score(y_valid_raw, valid_pred_raw)) if len(valid_df) else np.nan

    metrics_rows = [
        {"metric": "backend", "value": choose_model_backend()},
        {"metric": "train_rows", "value": len(train_df)},
        {"metric": "validation_rows", "value": len(valid_df)},
        {"metric": "target_raw_min_train", "value": target_min},
        {"metric": "target_raw_max_train", "value": target_max},
        {"metric": "regression_rmse_raw", "value": rmse},
        {"metric": "regression_mae_raw", "value": mae},
        {"metric": "regression_r2_raw", "value": r2},
    ]

    if use_classifier and valid_pred_escalation_prob is not None and len(valid_df):
        try:
            auc = float(roc_auc_score(y_valid_cls, valid_pred_escalation_prob))
        except Exception:
            auc = np.nan
        try:
            ap = float(average_precision_score(y_valid_cls, valid_pred_escalation_prob))
        except Exception:
            ap = np.nan
        try:
            acc = float(accuracy_score(y_valid_cls, valid_pred_escalation))
        except Exception:
            acc = np.nan
        try:
            f1 = float(f1_score(y_valid_cls, valid_pred_escalation))
        except Exception:
            f1 = np.nan
        try:
            brier = float(brier_score_loss(y_valid_cls, valid_pred_escalation_prob))
        except Exception:
            brier = np.nan
        metrics_rows.extend([
            {"metric": "escalation_auc", "value": auc},
            {"metric": "escalation_average_precision", "value": ap},
            {"metric": "escalation_accuracy", "value": acc},
            {"metric": "escalation_f1", "value": f1},
            {"metric": "escalation_brier_score", "value": brier},
        ])

    metrics_df = pd.DataFrame(metrics_rows)

    importance_df = pd.DataFrame()
    try:
        feature_names = reg_pipe.named_steps["preprocess"].get_feature_names_out()
        model = reg_pipe.named_steps["model"]
        if hasattr(model, "feature_importances_"):
            importance_df = pd.DataFrame({
                "feature": feature_names,
                "importance": model.feature_importances_,
            }).sort_values("importance", ascending=False).reset_index(drop=True)
    except Exception:
        importance_df = pd.DataFrame(columns=["feature", "importance"])

    # FIX (CORRECTIONS #1): pull each hotspot's truly latest observed week
    # from the full, un-filtered weekly_df — NOT from model_df. model_df
    # only contains rows with a known next-week target, so its "latest" row
    # per hotspot is the second-to-last (or older) observed week, not the
    # actual present. We only need *features* here (not the target), so a
    # missing target on these rows is expected and fine.
    latest_rows = (
        weekly_df.sort_values(["cluster_key", "week_start"])
        .groupby("cluster_key", as_index=False)
        .tail(1)
        .copy()
    )

    latest_X = latest_rows[numeric_features + categorical_features].copy()
    latest_pred_raw = pd.Series(reg_pipe.predict(latest_X), index=latest_rows.index)
    latest_pred_score = raw_to_score(latest_pred_raw)

    current_score = raw_to_score(latest_rows["current_week_risk_raw"].astype(float))

    escalation_prob = pd.Series(np.full(len(latest_rows), np.nan), index=latest_rows.index)
    escalation_pred = pd.Series(np.full(len(latest_rows), 0), index=latest_rows.index, dtype=int)
    if clf_pipe is not None:
        try:
            escalation_prob = pd.Series(clf_pipe.predict_proba(latest_X)[:, 1], index=latest_rows.index)
            escalation_pred = (escalation_prob >= 0.5).astype(int)
        except Exception:
            escalation_pred = pd.Series(clf_pipe.predict(latest_X), index=latest_rows.index).astype(int)
            escalation_prob = escalation_pred.astype(float)

    forecast_cols = [
        "cluster_key", "cluster_label", "cluster_label_display", "week_start",
        "current_week_risk_raw", "records_week", "severity_sum_week",
        "severity_mean_week", "weekly_growth_pct", "growth_surge_week",
        "hotspot_age_weeks", "road_class", "geometry_source",
        "dominant_vehicle_type", "layer_b_alert_flag", "layer_d_alert_flag",
        "ccs_score", "delay_minutes_per_vehicle", "criticality_factor",
        "context_multiplier", "layer_b_priority_boost", "validation_uncertainty",
        "resurgence_score", "persistence_score", "anomaly_score", "rop",
        "tvs", "vdi", "network_vulnerability_score", "spillover_total_score",
        "road_node_betweenness", "road_node_degree", "road_node_distance_m",
        "neighbor_count", "in_neighbor_count", "out_neighbor_count",
        "influence_asymmetry", "nearby_sensitive_poi_count",
    ]
    forecast_cols = [c for c in forecast_cols if c in latest_rows.columns]
    forecast = latest_rows[forecast_cols].copy()

    forecast["current_week_ccs_proxy_score"] = current_score.values
    forecast["predicted_next_week_risk_raw"] = latest_pred_raw.values
    forecast["model_predicted_next_week_ccs_score"] = latest_pred_score.values
    forecast["probability_of_escalation"] = escalation_prob.values
    forecast["predicted_escalation_flag"] = escalation_pred.values

    # --- CORRECTIONS #2: stabilize the published score ---------------------
    # Even after fixing which row counts as "latest", a hotspot with very
    # little history can still produce a noisy regression output. Blend
    # toward the current score and cap the swing so the published forecast
    # can't jump by 90+ points off a single prediction.
    stabilized_score = forecast["model_predicted_next_week_ccs_score"].astype(float).copy()

    if ENABLE_FORECAST_SMOOTHING:
        stabilized_score = (
            FORECAST_SMOOTHING_ALPHA * forecast["current_week_ccs_proxy_score"].astype(float)
            + (1.0 - FORECAST_SMOOTHING_ALPHA) * stabilized_score
        )

    if ENABLE_DELTA_CLAMP:
        raw_delta = stabilized_score - forecast["current_week_ccs_proxy_score"].astype(float)
        clamped_delta = raw_delta.clip(lower=-MAX_DELTA_SCORE, upper=MAX_DELTA_SCORE)
        stabilized_score = (forecast["current_week_ccs_proxy_score"].astype(float) + clamped_delta).clip(0.0, 100.0)

    forecast["predicted_next_week_ccs_score"] = stabilized_score
    # Risk band must be derived from the score actually published, not the
    # pre-stabilization model output, or band and score could disagree.
    forecast["future_risk_band"] = forecast["predicted_next_week_ccs_score"].apply(lambda x: band_from_score(float(x)))

    forecast["predicted_delta_score"] = forecast["predicted_next_week_ccs_score"] - forecast["current_week_ccs_proxy_score"]
    forecast["predicted_delta_pct"] = (forecast["predicted_delta_score"] / (forecast["current_week_ccs_proxy_score"].abs() + EPS)) * 100.0
    forecast["forecast_week_start"] = pd.to_datetime(forecast["week_start"], errors="coerce") + pd.Timedelta(days=7)

    coord_lookup = pd.DataFrame(columns=["cluster_key", "lat", "lon"])

    if static_df is not None:
        coord_lookup = static_df.copy()
        coord_lookup["cluster_key"] = coord_lookup[cluster_col].map(normalize_cluster_key)

        if "lat" not in coord_lookup.columns:
            coord_lookup["lat"] = np.nan
        if "lon" not in coord_lookup.columns:
            coord_lookup["lon"] = np.nan

        coord_lookup = coord_lookup[["cluster_key", "lat", "lon"]].drop_duplicates("cluster_key")

    forecast = forecast.merge(coord_lookup, on="cluster_key", how="left")
    forecast["lat"] = pd.to_numeric(forecast["lat"], errors="coerce")
    forecast["lon"] = pd.to_numeric(forecast["lon"], errors="coerce")

    # --- CORRECTIONS #5: rank by a composite priority score, not score alone
    forecast["forecast_priority_score"] = (
        PRIORITY_WEIGHT_SCORE * forecast["predicted_next_week_ccs_score"]
        + PRIORITY_WEIGHT_ESCALATION * forecast["probability_of_escalation"].fillna(0.0)
        + PRIORITY_WEIGHT_DELTA * forecast["predicted_delta_score"].clip(lower=0.0)
    )

    forecast = forecast.sort_values(
        ["forecast_priority_score", "predicted_next_week_ccs_score", "probability_of_escalation"],
        ascending=[False, False, False],
    ).reset_index(drop=True)
    forecast["forecast_rank"] = np.arange(1, len(forecast) + 1)

    forecast["forecast_recommendation"] = np.select(
        [
            forecast["future_risk_band"].eq("Critical"),
            forecast["future_risk_band"].eq("High"),
            forecast["future_risk_band"].eq("Moderate"),
        ],
        [
            "Immediate patrol deployment",
            "Targeted enforcement + tow readiness",
            "Monitor and schedule peak-window checks",
        ],
        default="Routine monitoring",
    )

    return {
        "reg_pipe": reg_pipe,
        "clf_pipe": clf_pipe,
        "metrics_df": metrics_df,
        "importance_df": importance_df,
        "forecast_df": forecast,
        "train_df": train_df,
        "valid_df": valid_df,
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "target_min": target_min,
        "target_max": target_max,
        "cluster_col": "cluster_key",
    }

# -------------------------
# Main
# -------------------------
def main():
    print("Loading Phase 6 static snapshot...")
    static_df, static_src = load_static_phase6()
    print("Static source:", static_src)

    print("Loading record-level history...")
    records_df, records_src = load_record_history()
    print("Record source:", records_src)

    static_df = prepare_static_context(static_df)

    # CORRECTIONS #4: surface upstream Phase 5/6 geometry-join failures
    # instead of silently forecasting on all-fallback road context.
    if "geometry_source" in static_df.columns and len(static_df):
        fallback_share = float(static_df["geometry_source"].astype(str).str.lower().eq("fallback").mean())
        if fallback_share > 0.5:
            print(
                f"\n[WARNING] {fallback_share:.0%} of Phase 6 hotspots have "
                f"geometry_source == 'fallback'. This points to an upstream "
                f"Phase 5/6 road-graph join issue (road_class, lane_count, "
                f"junction_degree, betweenness_centrality not attaching) — "
                f"not something this script can fix. Forecasts below will "
                f"still run on record-level signal, but road-network "
                f"features will contribute little until that join is "
                f"repaired.\n"
                # PATCH: actionable guidance for the upstream fix.
                f"  ACTION: Re-run stage5 with ENABLE_OSMNX_PHASE5=1 and "
                f"check for a saved graph at content/phase5_outputs_2/"
                f"osmnx_graph_cache.graphml.  If that file is absent or "
                f"zero-bytes, the OSMnx download itself is failing — check "
                f"network access and the exception messages printed during "
                f"build_osmnx_graph().  If the file exists but fallback "
                f"share is still high, check the per-point fallback logs "
                f"emitted by road_context_for_points() — they now print the "
                f"exception reason for the first 3 and every 50th failure.\n"
            )

    weekly_df, cluster_col = prepare_record_history(records_df, static_df)

    if len(weekly_df) == 0:
        raise RuntimeError("No weekly rows available after feature engineering.")

    print("Weekly rows built:", len(weekly_df))
    print("Hotspots in weekly history:", weekly_df["cluster_key"].nunique())

    # CORRECTIONS #4: make data sparsity visible. A hotspot with only one
    # observed week will correctly show weekly_growth_pct == 0.0 — there is
    # no prior week to compare against — which is expected behavior, not a
    # bug, given how little history most hotspots currently have.
    weeks_per_cluster = weekly_df.groupby("cluster_key")["week_start"].nunique()
    print("\nWeeks of history per hotspot (describe):")
    print(weeks_per_cluster.describe().to_string())
    single_week_share = float((weeks_per_cluster <= 1).mean())
    print(
        f"Share of hotspots with only ONE observed week: {single_week_share:.1%} "
        f"-- weekly_growth_pct == 0.0 for these is expected, not a bug.\n"
    )

    print("Training model backend:", choose_model_backend())

    artifacts = fit_predict_forecast(weekly_df, static_df, cluster_col)

    reg_pipe = artifacts["reg_pipe"]
    clf_pipe = artifacts["clf_pipe"]
    metrics_df = artifacts["metrics_df"]
    importance_df = artifacts["importance_df"]
    forecast_df = artifacts["forecast_df"]
    train_df = artifacts["train_df"]
    valid_df = artifacts["valid_df"]

    weekly_df.to_csv(OUT_DIR / "phase7_weekly_hotspot_history.csv", index=False)
    train_df.to_csv(OUT_DIR / "phase7_training_rows.csv", index=False)
    valid_df.to_csv(OUT_DIR / "phase7_validation_rows.csv", index=False)
    forecast_df.to_csv(OUT_DIR / "phase7_next_week_forecast.csv", index=False)
    forecast_df.to_csv(OUT_DIR / "phase7_next_week_forecast_ranked.csv", index=False)
    metrics_df.to_csv(OUT_DIR / "phase7_model_metrics.csv", index=False)
    if len(importance_df):
        importance_df.to_csv(OUT_DIR / "phase7_feature_importance.csv", index=False)

    artifact_path = OUT_DIR / ("phase7_model_artifact.joblib" if JOBLIB_OK else "phase7_model_artifact.pkl")
    artifact_obj = {
        "backend": choose_model_backend(),
        "regressor": reg_pipe,
        "classifier": clf_pipe,
        "numeric_features": artifacts["numeric_features"],
        "categorical_features": artifacts["categorical_features"],
        "target_min": artifacts["target_min"],
        "target_max": artifacts["target_max"],
    }
    if JOBLIB_OK:
        joblib.dump(artifact_obj, artifact_path)
    else:
        with open(artifact_path, "wb") as f:
            pickle.dump(artifact_obj, f)

    print("\nPhase 7 complete")
    print("Outputs saved to:", OUT_DIR.resolve())
    print("\nMetrics:")
    print(metrics_df.to_string(index=False))

    print("\nTop 10 next-week forecasts:")
    top_cols = [
        "forecast_rank", "cluster_key", "cluster_label_display", "future_risk_band",
        "forecast_priority_score", "predicted_next_week_ccs_score", "probability_of_escalation",
        "current_week_ccs_proxy_score", "predicted_delta_score",
        "records_week", "weekly_growth_pct", "road_class",
        "geometry_source", "forecast_recommendation",
    ]
    top_cols = [c for c in top_cols if c in forecast_df.columns]
    print(forecast_df[top_cols].head(10).to_string(index=False))

    print("\nEscalation probability distribution across all forecasted hotspots:")
    print(forecast_df["probability_of_escalation"].describe().to_string())

    print("\nTop feature importance:")
    if len(importance_df):
        print(importance_df.head(20).to_string(index=False))


if __name__ == "__main__":
    main()

Loading Phase 6 static snapshot...
Static source: content\phase6_outputs_2\phase6_cluster_ccs_full.csv
Loading record-level history...
Record source: content\phase5_outputs_2\phase5_enriched_records.csv
Weekly rows built: 1608
Hotspots in weekly history: 798

Weeks of history per hotspot (describe):
count    798.000000
mean       2.015038
std        1.534145
min        1.000000
25%        1.000000
50%        2.000000
75%        2.000000
max       13.000000
Share of hotspots with only ONE observed week: 38.2% -- weekly_growth_pct == 0.0 for these is expected, not a bug.

Training model backend: xgboost

Phase 7 complete
Outputs saved to: D:\Flipkart_Hackathon\approach1\content\phase7_outputs_2

Metrics:
                      metric      value
                     backend    xgboost
                  train_rows        719
             validation_rows         91
        target_raw_min_train   0.831777
        target_raw_max_train  63.616456
         regression_rmse_raw   7.660391
        

In [30]:

from __future__ import annotations

import ast
import math
import os
import warnings
from pathlib import Path
from typing import Iterable, Optional, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ============================================================
# Phase 8 — Prescriptive Enforcement Engine
# Inputs: Phase 7 forecasts + Phase 6/5/4/3 context
# Outputs: patrol allocation, tow readiness, time-window
# recommendations, zone ranking, route plan, offender actions.
# ============================================================

PHASE8_OUT_DIR = Path("content/phase8_outputs_2")
PHASE8_OUT_DIR.mkdir(parents=True, exist_ok=True)

PHASE7_DIRS = [Path("content/phase7_outputs_2"), Path("phase7_outputs_2"), Path("/content/phase7_outputs_2")]
PHASE6_DIRS = [Path("content/phase6_outputs_2"), Path("phase6_outputs_2"), Path("/content/phase6_outputs_2")]
PHASE5_DIRS = [Path("content/phase5_outputs_2"), Path("phase5_outputs_2"), Path("/content/phase5_outputs_2")]
PHASE4_DIRS = [Path("content/phase4_outputs_2"), Path("phase4_outputs_2"), Path("/content/phase4_outputs_2")]
PHASE3_DIRS = [Path("content/phase3_outputs_2"), Path("phase3_outputs_2"), Path("/content/phase3_outputs_2")]

EPS = 1e-9
TOTAL_PATROL_UNITS = int(os.environ.get("PHASE8_TOTAL_PATROL_UNITS", "15"))
ROUTE_PLAN_TOP_N = int(os.environ.get("PHASE8_ROUTE_PLAN_TOP_N", "12"))
DEPOT_LAT = float(os.environ.get("PHASE8_DEPOT_LAT", "12.9716"))
DEPOT_LON = float(os.environ.get("PHASE8_DEPOT_LON", "77.5946"))
MAX_PATROLS_PER_HOTSPOT = int(os.environ.get("PHASE8_MAX_PATROLS_PER_HOTSPOT", "3"))
PATROL_ELIGIBILITY_SCORE = float(os.environ.get("PHASE8_PATROL_ELIGIBILITY_SCORE", "40"))
TOW_PRIORITY_THRESHOLD = float(os.environ.get("PHASE8_TOW_PRIORITY_THRESHOLD", "70"))
ESCALATION_THRESHOLD = float(os.environ.get("PHASE8_ESCALATION_THRESHOLD", "0.50"))
HIGH_BETWEENNESS_PERCENTILE = float(os.environ.get("PHASE8_HIGH_BETWEENNESS_PERCENTILE", "0.75"))

SEVERITY_RULES = {
    5: {"DOUBLE PARKING", "NEAR ROAD CROSSING", "NEAR TRAFFIC LIGHT", "NEAR ZEBRA CROSSING", "NEAR TRAFFIC LIGHT / ZEBRA CROSSING", "NEAR TRAFFIC LIGHT/ZEBRA CROSSING"},
    4: {"PARKING IN MAIN ROAD", "NEAR BUS STOP", "NEAR SCHOOL", "NEAR HOSPITAL", "OPPOSITE ANOTHER VEHICLE"},
    3: {"PARKING ON FOOTPATH"},
    2: {"WRONG PARKING", "PARKING OTHER THAN BUS STOP"},
    1: {"NO PARKING", "NO PARKING (GENERIC)"},
}


# ------------------------- helpers -------------------------
def clean_text(x) -> str:
    return "" if pd.isna(x) else str(x).strip()


def normalize_tag(tag) -> str:
    return clean_text(tag).upper().replace("&", "AND").strip()


def parse_listlike(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    s = str(value).strip()
    if not s:
        return []
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple)):
            return list(parsed)
        return [parsed]
    except Exception:
        if s.startswith("[") and s.endswith("]"):
            s = s[1:-1]
        return [p.strip().strip("'").strip('"') for p in s.split(",") if p.strip()]


def safe_float(x, default=np.nan):
    try:
        if pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default


def load_first_existing(base_dirs: Iterable[Path], filenames: Iterable[str]):
    for d in base_dirs:
        for name in filenames:
            p = d / name
            if p.exists():
                return pd.read_csv(p, low_memory=False), p
    return None, None


def standardize_cluster_col(df: pd.DataFrame) -> str:
    for c in ["cluster_key", "st_dbscan_cluster_id", "cluster_id", "dbscan_cluster_id"]:
        if c in df.columns:
            return c
    raise ValueError("No cluster id column found.")


def standardize_vehicle_col(df: pd.DataFrame) -> Optional[str]:
    for c in ["canonical_vehicle_number", "vehicle_number", "updated_vehicle_number"]:
        if c in df.columns:
            return c
    return None


def standardize_vehicle_type_col(df: pd.DataFrame) -> Optional[str]:
    for c in ["canonical_vehicle_type", "vehicle_type", "updated_vehicle_type"]:
        if c in df.columns:
            return c
    return None


def ensure_label_column(df: pd.DataFrame):
    if df is None or len(df) == 0:
        return df
    df = df.copy()
    if "cluster_label" in df.columns:
        df["cluster_label"] = df["cluster_label"].fillna("").astype(str).str.strip()
        return df
    if "hotspot_unit" in df.columns:
        df["cluster_label"] = df["hotspot_unit"].fillna("").astype(str).str.strip()
        return df
    if "dominant_junction_name" in df.columns:
        df["cluster_label"] = df["dominant_junction_name"].fillna("").astype(str).str.strip()
        return df
    if "st_dbscan_cluster_id" in df.columns:
        df["cluster_label"] = "CLUSTER::" + df["st_dbscan_cluster_id"].astype(str)
        return df
    df["cluster_label"] = "UNKNOWN"
    return df


def derive_coords(df: pd.DataFrame):
    if df is None or len(df) == 0:
        return df
    df = df.copy()
    if "lat" in df.columns and "lon" in df.columns:
        df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
        df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    elif {"centroid_lat", "centroid_lon"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["centroid_lat"], errors="coerce")
        df["lon"] = pd.to_numeric(df["centroid_lon"], errors="coerce")
    elif {"latitude_mean", "longitude_mean"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["latitude_mean"], errors="coerce")
        df["lon"] = pd.to_numeric(df["longitude_mean"], errors="coerce")
    elif {"latitude", "longitude"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["latitude"], errors="coerce")
        df["lon"] = pd.to_numeric(df["longitude"], errors="coerce")
    else:
        df["lat"] = np.nan
        df["lon"] = np.nan
    return df


def parse_datetime_ist(df: pd.DataFrame) -> pd.Series:
    if "created_datetime_ist" in df.columns:
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce", utc=True)
        if ts.notna().any():
            return ts.dt.tz_convert("Asia/Kolkata")
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce")
        return ts.dt.tz_localize("Asia/Kolkata", nonexistent="NaT", ambiguous="NaT")
    if "created_datetime_parsed" in df.columns:
        ts = pd.to_datetime(df["created_datetime_parsed"], errors="coerce", utc=True)
        return ts.dt.tz_convert("Asia/Kolkata")
    if "created_datetime" not in df.columns:
        return pd.Series(pd.NaT, index=df.index)
    ts = pd.to_datetime(df["created_datetime"], errors="coerce", utc=True)
    return ts.dt.tz_convert("Asia/Kolkata")


def week_start_monday(series):
    dt = pd.to_datetime(series, errors="coerce", utc=True).dt.tz_convert("Asia/Kolkata")
    week_start = dt.dt.normalize() - pd.to_timedelta(dt.dt.weekday, unit="D")
    return week_start.dt.tz_localize(None)


def severity_from_tags(tags):
    if not tags:
        return 1
    normalized = [normalize_tag(t) for t in tags]
    score = 1
    for sev in sorted(SEVERITY_RULES.keys(), reverse=True):
        vocab = SEVERITY_RULES[sev]
        if any(any(v == tag or v in tag for v in vocab) for tag in normalized):
            score = sev
            break
    return score


def dominant_label(series, exclude=None, default=""):
    s = pd.Series(series).dropna().astype(str).str.strip()
    s = s[s.ne("")]
    if exclude:
        excl = {clean_text(x).lower() for x in exclude}
        s = s[~s.str.lower().isin(excl)]
    if s.empty:
        return default
    m = s.mode()
    return m.iloc[0] if not m.empty else s.iloc[0]


def minmax(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    valid = s.dropna()
    if valid.nunique(dropna=True) <= 1:
        return pd.Series(np.zeros(len(s)), index=s.index, dtype=float)
    mn = valid.min()
    mx = valid.max()
    return (s.fillna(mn) - mn) / (mx - mn + EPS)


def band_from_score(score: float) -> str:
    if score >= 80:
        return "Critical"
    if score >= 60:
        return "High"
    if score >= 40:
        return "Moderate"
    return "Watch"


def band_index(band: str) -> int:
    return {"Watch": 0, "Moderate": 1, "High": 2, "Critical": 3}.get(str(band), 0)


def haversine_m(lat1, lon1, lat2, lon2) -> float:
    if any(pd.isna(v) for v in [lat1, lon1, lat2, lon2]):
        return np.nan
    r = 6371000.0
    p1 = math.radians(float(lat1))
    p2 = math.radians(float(lat2))
    dp = math.radians(float(lat2) - float(lat1))
    dl = math.radians(float(lon2) - float(lon1))
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.atan2(math.sqrt(a), math.sqrt(max(0.0, 1 - a)))


def build_zone_name(df: pd.DataFrame) -> pd.Series:
    zone = pd.Series([""] * len(df), index=df.index, dtype="object")
    for c in ["dominant_police_station", "police_station", "dominant_junction_name", "hotspot_unit", "cluster_label_display", "cluster_label"]:
        if c in df.columns:
            s = df[c].fillna("").astype(str).str.strip()
            s = s.replace({"No Junction": "", "NO JUNCTION": ""})
            zone = zone.mask(zone.eq(""), s)
    return zone.replace({"": "UNASSIGNED"}).astype(str)


def coalesce_from_suffix(df: pd.DataFrame, base_cols: Iterable[str], suffix: str = "_static"):
    out = df.copy()
    for c in base_cols:
        s = f"{c}{suffix}"
        if s in out.columns:
            if c in out.columns:
                out[c] = out[c].combine_first(out[s])
            else:
                out[c] = out[s]
            out.drop(columns=[s], inplace=True)
    return out


def safe_to_int_series(s, default=0):
    return pd.to_numeric(s, errors="coerce").fillna(default).astype(int)


def pressure_raw_from_week(df: pd.DataFrame) -> pd.Series:
    rec = pd.to_numeric(df["records_week"], errors="coerce").fillna(0.0).clip(lower=0.0)
    sev = pd.to_numeric(df["severity_mean_week"], errors="coerce").fillna(1.0).clip(lower=1.0, upper=5.0)
    peak = pd.to_numeric(df["peak_window_ratio_week"], errors="coerce").fillna(0.0).clip(lower=0.0, upper=1.0)
    repeat = pd.to_numeric(df["repeat_vehicle_ratio_week"], errors="coerce").fillna(0.0).clip(lower=0.0, upper=1.0)
    growth = pd.to_numeric(df["growth_surge_week"], errors="coerce").fillna(0.0).clip(lower=0.0, upper=5.0)
    raw = np.log1p(rec) * (1.0 + sev / 5.0) * (1.0 + 0.5 * peak) * (1.0 + repeat) * (1.0 + growth)
    return pd.Series(raw, index=df.index).replace([np.inf, -np.inf], np.nan).fillna(0.0)


def compute_shannon_diversity_from_tags(tag_series: pd.Series) -> float:
    tags = pd.Series(tag_series).dropna().astype(str)
    tags = tags[tags.str.strip().ne("")]
    if tags.empty:
        return 0.0
    probs = tags.value_counts(normalize=True)
    entropy = float(-(probs * np.log(probs + EPS)).sum())
    max_entropy = float(np.log(len(probs) + EPS))
    return 0.0 if max_entropy <= EPS else float(np.clip(entropy / max_entropy, 0.0, 1.0))


def make_hotspot_unit(row):
    junction = clean_text(row.get("junction_name", ""))
    if junction and junction.upper() != "NO JUNCTION":
        return f"JUNCTION::{junction}"
    station = clean_text(row.get("police_station", "UNKNOWN")) or "UNKNOWN"
    return f"POLICE_STATION::{station}"


# ------------------------- load phase inputs -------------------------
def load_phase7_forecast():
    df, src = load_first_existing(PHASE7_DIRS, ["phase7_next_week_forecast_ranked.csv", "phase7_next_week_forecast.csv"])
    if df is None:
        raise FileNotFoundError("Could not find Phase 7 forecast outputs.")
    return df, src


def load_phase6_static():
    df, src = load_first_existing(PHASE6_DIRS, ["phase6_cluster_ccs_full.csv", "phase6_stage6_handoff.csv", "phase6_weekly_dispatch_priority_table.csv"])
    if df is None:
        raise FileNotFoundError("Could not find Phase 6 outputs.")
    return df, src


def load_record_history():
    df, src = load_first_existing(PHASE5_DIRS + PHASE4_DIRS + PHASE3_DIRS, ["phase5_enriched_records.csv", "phase4_merged_with_prior_scores.csv", "phase3_clustered_dataset.csv"])
    if df is None:
        raise FileNotFoundError("Could not find record-level source from Phase 5/4/3.")
    return df, src


def load_chronic_offenders():
    return load_first_existing(PHASE6_DIRS + PHASE5_DIRS, ["phase6_chronic_offender_list.csv", "phase5_chronic_offenders.csv"])


# ------------------------- preparation -------------------------
def prepare_static_context(static_df: pd.DataFrame) -> pd.DataFrame:
    df = static_df.copy()
    df = ensure_label_column(df)
    df = derive_coords(df)
    cluster_col = standardize_cluster_col(df)
    df["cluster_key"] = df[cluster_col].map(normalize_cluster_key)

    defaults = {
        "cluster_label": "UNKNOWN", "cluster_label_display": "", "risk_band": "Watch",
        "ccs_score": 0.0, "delay_minutes_per_vehicle": 0.0, "records_total": 0.0,
        "distinct_days": 0.0, "severity_sum": 0.0, "severity_mean": 1.0,
        "growth_pct_change": 0.0, "growth_multiplier": 1.0, "criticality_factor": 1.0,
        "context_multiplier": 1.0, "layer_b_priority_boost": 0.0, "layer_b_alert_flag": 0,
        "validation_uncertainty": 0.0, "resurgence_score": 0.0, "persistence_score": 0.0,
        "anomaly_score": 0.0, "rop": 0.0, "tvs": 0.0, "vdi": 0.0,
        "nearby_sensitive_poi_count": 0.0, "road_class": "road", "lane_count": 2.0,
        "carriageway_width_m": 7.0, "link_length_m": 250.0, "junction_degree": 4.0,
        "betweenness_centrality": 0.0, "geometry_source": "fallback", "mappls_address": "",
        "road_node_id": np.nan, "road_node_distance_m": 0.0, "road_node_degree": 0.0,
        "road_node_betweenness": 0.0, "source_pressure_score": 0.0, "source_pressure_norm": 0.0,
        "spillover_out_score": 0.0, "spillover_in_score": 0.0, "spillover_total_score": 0.0,
        "propagation_radius_m": 0.0, "network_pagerank": 0.0, "network_component_id": 0,
        "network_component_size": 1, "neighbor_count": 0.0, "in_neighbor_count": 0.0,
        "out_neighbor_count": 0.0, "influence_asymmetry": 0.0, "network_vulnerability_score": 0.0,
        "layer_d_alert_flag": 0, "dominant_vehicle_type": "UNKNOWN",
        "repeat_vehicle_count_2plus": 0.0, "chronic_vehicle_count_5plus": 0.0,
    }
    for c, default in defaults.items():
        if c not in df.columns:
            df[c] = default

    numeric_cols = [
        "ccs_score", "delay_minutes_per_vehicle", "records_total", "distinct_days", "severity_sum", "severity_mean",
        "growth_pct_change", "growth_multiplier", "criticality_factor", "context_multiplier", "layer_b_priority_boost",
        "validation_uncertainty", "resurgence_score", "persistence_score", "anomaly_score", "rop", "tvs", "vdi",
        "nearby_sensitive_poi_count", "lane_count", "carriageway_width_m", "link_length_m", "junction_degree",
        "betweenness_centrality", "road_node_distance_m", "road_node_degree", "road_node_betweenness",
        "source_pressure_score", "source_pressure_norm", "spillover_out_score", "spillover_in_score", "spillover_total_score",
        "propagation_radius_m", "network_pagerank", "network_component_id", "network_component_size",
        "neighbor_count", "in_neighbor_count", "out_neighbor_count", "influence_asymmetry", "network_vulnerability_score",
        "repeat_vehicle_count_2plus", "chronic_vehicle_count_5plus",
    ]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    for c in ["layer_b_alert_flag", "layer_d_alert_flag"]:
        df[c] = safe_to_int_series(df[c], 0)

    for c in ["road_class", "geometry_source", "dominant_vehicle_type"]:
        df[c] = df[c].fillna("UNKNOWN").astype(str).str.strip()

    # Best row per hotspot
    df = df.sort_values(["ccs_score", "delay_minutes_per_vehicle", "records_total"], ascending=[False, False, False])
    df = df.drop_duplicates(subset=["cluster_key"], keep="first").reset_index(drop=True)
    return df


def prepare_record_history(records_df: pd.DataFrame, static_df: pd.DataFrame) -> Tuple[pd.DataFrame, str]:
    r = records_df.copy()
    cluster_col = standardize_cluster_col(r)
    vehicle_col = standardize_vehicle_col(r)
    vehicle_type_col = standardize_vehicle_type_col(r)

    if "validation_status" in r.columns:
        r["validation_status"] = r["validation_status"].fillna("").astype(str).str.lower()
        approved = r[r["validation_status"].eq("approved")].copy()
        if len(approved) > 0:
            r = approved.copy()

    r = ensure_label_column(r)
    r = derive_coords(r)
    r["created_datetime_ist"] = parse_datetime_ist(r)
    r = r.dropna(subset=["created_datetime_ist"]).copy()

    if "severity_score" not in r.columns:
        if "violation_type" in r.columns:
            r["violation_tags"] = r["violation_type"].apply(parse_listlike)
            r["severity_score"] = r["violation_tags"].apply(severity_from_tags)
        else:
            r["severity_score"] = 1
    r["severity_score"] = pd.to_numeric(r["severity_score"], errors="coerce").fillna(1).clip(lower=1, upper=5)

    if "hotspot_unit" not in r.columns:
        r["hotspot_unit"] = r.apply(make_hotspot_unit, axis=1)

    r["service_date"] = pd.to_datetime(r["created_datetime_ist"], errors="coerce").dt.tz_localize(None).dt.date
    r["week_start"] = week_start_monday(r["created_datetime_ist"])
    r["hour_ist"] = r["created_datetime_ist"].dt.hour
    r["day_of_week"] = r["created_datetime_ist"].dt.dayofweek
    r["is_peak_window"] = r["hour_ist"].between(8, 12, inclusive="both").astype(int)

    r["cluster_key"] = r[cluster_col].map(normalize_cluster_key)

    if vehicle_col and vehicle_col in r.columns:
        r[vehicle_col] = r[vehicle_col].fillna("").astype(str).str.strip()
        r = r[r[vehicle_col].ne("")].copy()
    else:
        vehicle_col = None

    if vehicle_type_col and vehicle_type_col in r.columns:
        r[vehicle_type_col] = r[vehicle_type_col].fillna("").astype(str).str.strip()
    else:
        vehicle_type_col = None

    agg_spec = {
        "records_week": ("week_start", "size"),
        "active_days_week": ("service_date", "nunique"),
        "severity_sum_week": ("severity_score", "sum"),
        "severity_mean_week": ("severity_score", "mean"),
        "peak_window_records_week": ("is_peak_window", "sum"),
        "unique_hotspots_week": ("hotspot_unit", "nunique"),
    }
    if vehicle_col:
        agg_spec["unique_vehicles_week"] = (vehicle_col, "nunique")
    else:
        r["vehicle_fallback"] = "UNKNOWN"
        agg_spec["unique_vehicles_week"] = ("vehicle_fallback", "nunique")

    if vehicle_type_col:
        agg_spec["unique_vehicle_types_week"] = (vehicle_type_col, "nunique")
        agg_spec["dominant_vehicle_type_week"] = (vehicle_type_col, lambda s: s.mode().iloc[0] if not s.mode().empty else "UNKNOWN")
    else:
        r["vehicle_type_fallback"] = "UNKNOWN"
        agg_spec["unique_vehicle_types_week"] = ("vehicle_type_fallback", "nunique")
        agg_spec["dominant_vehicle_type_week"] = ("vehicle_type_fallback", lambda s: "UNKNOWN")

    weekly = r.groupby(["cluster_key", "week_start"]).agg(**agg_spec).reset_index().sort_values(["cluster_key", "week_start"]).reset_index(drop=True)
    weekly["records_per_active_day_week"] = weekly["records_week"] / weekly["active_days_week"].clip(lower=1)
    weekly["peak_window_ratio_week"] = weekly["peak_window_records_week"] / weekly["records_week"].clip(lower=1)
    weekly["repeat_vehicle_ratio_week"] = ((weekly["records_week"] - weekly["unique_vehicles_week"]) / weekly["records_week"].clip(lower=1)).clip(0.0, 1.0)
    weekly["prev_records_week"] = weekly.groupby("cluster_key")["records_week"].shift(1)
    weekly["weekly_growth_pct"] = ((weekly["records_week"] - weekly["prev_records_week"]) / (weekly["prev_records_week"].abs() + EPS)).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    weekly["growth_surge_week"] = weekly["weekly_growth_pct"].clip(lower=0.0)
    weekly["weekly_pressure_raw"] = pressure_raw_from_week(weekly)

    def enrich_group(g: pd.DataFrame) -> pd.DataFrame:
        g = g.sort_values("week_start").copy()
        g["week_index"] = np.arange(len(g), dtype=int)
        g["hotspot_age_weeks"] = g["week_index"] + 1
        g["lag1_pressure_raw"] = g["weekly_pressure_raw"].shift(1).fillna(g["weekly_pressure_raw"].iloc[0] if len(g) else 0.0)
        g["lag2_pressure_raw"] = g["weekly_pressure_raw"].shift(2).fillna(g["lag1_pressure_raw"])
        g["rolling_4w_mean_pressure_raw"] = g["weekly_pressure_raw"].rolling(4, min_periods=1).mean()
        g["rolling_4w_std_pressure_raw"] = g["weekly_pressure_raw"].rolling(4, min_periods=1).std(ddof=0).fillna(0.0)
        g["rolling_4w_mean_records_week"] = g["records_week"].rolling(4, min_periods=1).mean()
        g["rolling_4w_mean_severity_week"] = g["severity_sum_week"].rolling(4, min_periods=1).mean()
        g["rolling_4w_mean_growth_week"] = g["weekly_growth_pct"].rolling(4, min_periods=1).mean()
        vals = g["records_week"].to_numpy(dtype=float)
        slopes = []
        for i in range(len(g)):
            start = max(0, i - 3)
            y = vals[start:i+1]
            if len(y) < 2 or np.allclose(y, y[0], equal_nan=True):
                slopes.append(0.0)
            else:
                x = np.arange(len(y), dtype=float)
                try:
                    slopes.append(float(np.polyfit(x, y, 1)[0]))
                except Exception:
                    slopes.append(0.0)
        g["rolling_4w_trend_slope_records_week"] = slopes
        g["rolling_4w_pressure_acceleration"] = g["rolling_4w_mean_pressure_raw"] - g["rolling_4w_mean_pressure_raw"].shift(1).fillna(g["rolling_4w_mean_pressure_raw"])
        return g

    weekly = weekly.groupby("cluster_key", group_keys=False).apply(enrich_group).reset_index(drop=True)

    # Validation uncertainty
    if "validation_status" in r.columns:
        val = r.groupby("cluster_key")["validation_status"].agg(
            validation_total="size",
            validation_approved=lambda s: int((s == "approved").sum()),
            validation_rejected=lambda s: int((s == "rejected").sum()),
            validation_processing=lambda s: int((s == "processing").sum()),
        ).reset_index()
        val["validation_uncertainty"] = 1.0 - (val["validation_approved"] / val["validation_total"].clip(lower=1))
    else:
        val = pd.DataFrame({"cluster_key": weekly["cluster_key"].unique(), "validation_total": 0, "validation_uncertainty": 0.0})

    # Repeat offenders
    if vehicle_col:
        repeat_stats = r.groupby(["cluster_key", vehicle_col]).size().reset_index(name="vehicle_cluster_count")
        repeat_stats["repeat_flag_2plus"] = (repeat_stats["vehicle_cluster_count"] >= 2).astype(int)
        repeat_stats["chronic_flag_5plus"] = (repeat_stats["vehicle_cluster_count"] >= 5).astype(int)
        repeat_agg = repeat_stats.groupby("cluster_key").agg(
            repeat_vehicle_count_2plus=("repeat_flag_2plus", "sum"),
            chronic_vehicle_count_5plus=("chronic_flag_5plus", "sum"),
        ).reset_index()
    else:
        repeat_agg = pd.DataFrame({"cluster_key": weekly["cluster_key"].unique(), "repeat_vehicle_count_2plus": 0, "chronic_vehicle_count_5plus": 0})

    # Diversity (VDI)
    if "violation_tags" in r.columns:
        tag_rows = r[["cluster_key", "violation_tags"]].explode("violation_tags").copy()
        tag_rows["violation_tag"] = tag_rows["violation_tags"].map(normalize_tag)
        tag_rows = tag_rows[tag_rows["violation_tag"].astype(str).str.strip().ne("")]
        diversity = tag_rows.groupby("cluster_key")["violation_tag"].apply(compute_shannon_diversity_from_tags).reset_index(name="vdi")
    else:
        diversity = pd.DataFrame({"cluster_key": weekly["cluster_key"].unique(), "vdi": 0.0})

    # Summary stats per hotspot
    rows = []
    for cid, g in weekly.groupby("cluster_key"):
        counts = g["records_week"].to_numpy(dtype=float)
        if len(counts) < 2:
            tvs = 0.0
            resurgence = 0.0
            persistence = 1.0
        else:
            tvs = float(np.std(counts, ddof=0) / (np.mean(counts) + EPS))
            split = max(1, len(counts) // 2)
            first = float(np.mean(counts[:split]))
            second = float(np.mean(counts[split:])) if len(counts[split:]) else first
            resurgence = max(0.0, (second - first) / (first + EPS))
            persistence = float(np.mean(counts[-3:]) / (np.mean(counts[:3]) + EPS)) if len(counts) >= 3 else 1.0
        recent2 = float(np.mean(counts[-2:])) if len(counts) >= 2 else float(np.mean(counts)) if len(counts) else 0.0
        prev2 = float(np.mean(counts[-4:-2])) if len(counts) >= 4 else float(np.mean(counts[:-2])) if len(counts) > 2 else recent2
        resurgence_score = max(0.0, (recent2 - prev2) / (prev2 + EPS)) if prev2 > 0 else recent2

        hours = r.loc[r["cluster_key"].eq(cid), "hour_ist"]
        days = r.loc[r["cluster_key"].eq(cid), "day_of_week"]
        hour_counts = hours.value_counts().reindex(range(24), fill_value=0).astype(float)
        day_counts = days.value_counts().reindex(range(7), fill_value=0).astype(float)
        peak_hour = int(hour_counts.idxmax()) if hour_counts.sum() > 0 else 8
        rolling3 = hour_counts.rolling(3, min_periods=1).sum()
        start_hour = int(rolling3.idxmax()) if len(rolling3) else 8
        end_hour = min(24, start_hour + 3)
        best_day = int(day_counts.idxmax()) if day_counts.sum() > 0 else 0

        rows.append({
            "cluster_key": cid,
            "weekly_rows": len(g),
            "records_week_mean": float(np.mean(counts)) if len(counts) else 0.0,
            "records_week_std": float(np.std(counts, ddof=0)) if len(counts) else 0.0,
            "weekly_growth_pct_mean": float(g["weekly_growth_pct"].mean()),
            "weekly_growth_pct_max": float(g["weekly_growth_pct"].max()),
            "weekly_pressure_raw_mean": float(g["weekly_pressure_raw"].mean()),
            "weekly_pressure_raw_max": float(g["weekly_pressure_raw"].max()),
            "rolling_4w_mean_pressure_raw_max": float(g["rolling_4w_mean_pressure_raw"].max()),
            "rolling_4w_std_pressure_raw_max": float(g["rolling_4w_std_pressure_raw"].max()),
            "rolling_4w_mean_records_week_max": float(g["rolling_4w_mean_records_week"].max()),
            "rolling_4w_mean_severity_week_max": float(g["rolling_4w_mean_severity_week"].max()),
            "rolling_4w_trend_slope_records_week_max": float(g["rolling_4w_trend_slope_records_week"].max()),
            "rolling_4w_pressure_acceleration_max": float(g["rolling_4w_pressure_acceleration"].max()),
            "peak_hour": peak_hour,
            "best_window_start_hour": start_hour,
            "best_window_end_hour": end_hour,
            "best_day_of_week": best_day,
            "best_day_name": ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"][best_day],
            "peak_hours_top3": ",".join(str(i) for i in hour_counts.sort_values(ascending=False).head(3).index.tolist()),
            "peak_window_violation_ratio": float(hour_counts.iloc[start_hour:end_hour].sum() / (hour_counts.sum() + EPS)),
            "tvs": float(tvs),
            "resurgence_score": float(np.clip(resurgence_score, 0.0, 5.0)),
            "persistence_score": float(np.clip(persistence, 0.0, 5.0)),
        })

    summary = pd.DataFrame(rows).merge(val[["cluster_key", "validation_total", "validation_uncertainty"]], on="cluster_key", how="left")
    summary = summary.merge(repeat_agg, on="cluster_key", how="left")
    summary = summary.merge(diversity, on="cluster_key", how="left")

    static_ctx = static_df.copy()
    static_ctx["cluster_key"] = static_ctx[standardize_cluster_col(static_ctx)].map(normalize_cluster_key)
    static_cols = [
        "cluster_key", "lat", "lon", "cluster_label", "cluster_label_display", "risk_band", "ccs_score",
        "delay_minutes_per_vehicle", "records_total", "distinct_days", "severity_sum", "severity_mean",
        "growth_pct_change", "growth_multiplier", "criticality_factor", "context_multiplier",
        "layer_b_priority_boost", "layer_b_alert_flag", "validation_uncertainty", "resurgence_score",
        "persistence_score", "anomaly_score", "rop", "tvs", "vdi", "nearby_sensitive_poi_count",
        "road_class", "lane_count", "carriageway_width_m", "link_length_m", "junction_degree",
        "betweenness_centrality", "geometry_source", "mappls_address", "road_node_id", "road_node_distance_m",
        "road_node_degree", "road_node_betweenness", "source_pressure_score", "source_pressure_norm",
        "spillover_out_score", "spillover_in_score", "spillover_total_score", "propagation_radius_m",
        "network_pagerank", "network_component_id", "network_component_size", "neighbor_count",
        "in_neighbor_count", "out_neighbor_count", "influence_asymmetry", "network_vulnerability_score",
        "layer_d_alert_flag", "dominant_vehicle_type", "repeat_vehicle_count_2plus", "chronic_vehicle_count_5plus",
        "dominant_police_station", "police_station", "dominant_junction_name", "hotspot_unit",
    ]
    static_cols = [c for c in static_cols if c in static_ctx.columns]
    static_ctx = static_ctx[static_cols].drop_duplicates("cluster_key")
    static_ctx = static_ctx.rename(columns={c: f"static_{c}" for c in static_ctx.columns if c != "cluster_key"})
    summary = summary.merge(static_ctx, on="cluster_key", how="left")
    summary = coalesce_from_suffix(summary, [c for c in summary.columns if c.startswith("static_")], suffix="")  # no-op safe
    return weekly, summary, cluster_col


# ------------------------- prescriptive logic -------------------------
def compute_priority_score(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    components = {
        "probability_of_escalation": 0.22,
        "predicted_next_week_ccs_score": 0.18,
        "current_week_ccs_proxy_score": 0.08,
        "criticality_factor": 0.08,
        "growth_multiplier": 0.07,
        "growth_pct_change": 0.05,
        "layer_b_priority_boost": 0.05,
        "validation_uncertainty": 0.05,
        "rop": 0.08,
        "tvs": 0.06,
        "vdi": 0.05,
        "resurgence_score": 0.05,
        "anomaly_score": 0.04,
        "network_vulnerability_score": 0.04,
        "spillover_total_score": 0.01,
        "nearby_sensitive_poi_count": 0.01,
    }
    for c in components:
        if c not in out.columns:
            out[c] = 0.0
        out[f"{c}_norm"] = minmax(pd.to_numeric(out[c], errors="coerce").fillna(0.0))
    raw = np.zeros(len(out), dtype=float)
    total_w = float(sum(components.values()))
    for c, w in components.items():
        raw += w * out[f"{c}_norm"].to_numpy(dtype=float)
    out["priority_score"] = 100.0 * np.clip(raw / max(total_w, EPS), 0.0, 1.0)
    out["priority_band"] = out["priority_score"].apply(band_from_score)
    out["final_risk_band"] = out.apply(
        lambda row: row["priority_band"] if band_index(row["priority_band"]) >= band_index(row.get("future_risk_band", "Watch")) else row.get("future_risk_band", "Watch"),
        axis=1,
    )
    return out


def compute_time_window(row):
    """
    Returns:
        recommended_time_window,
        recommended_day_name,
        recommended_shift
    """
    start_hour = row.get("peak_start_hour", np.nan)
    end_hour = row.get("peak_end_hour", np.nan)
    # fallback to peak hour
    if pd.isna(start_hour) or pd.isna(end_hour):
        peak_hour = row.get("peak_hour", np.nan)
        if pd.isna(peak_hour):
            peak_hour = 8
        peak_hour = int(round(float(peak_hour)))
        start_hour = max(0, peak_hour - 1)
        end_hour = min(24, peak_hour + 2)
    start_hour = int(start_hour)
    end_hour = int(end_hour)
    time_window = (
        f"{start_hour:02d}:00-{end_hour:02d}:00"
    )
    day_name = row.get("peak_day_name", "All Days")
    # shift classification
    if start_hour < 12:
        shift = "Morning"
    elif start_hour < 17:
        shift = "Afternoon"
    else:
        shift = "Evening"
    return (
        time_window,
        day_name,
        shift
    )


def allocate_patrols(df: pd.DataFrame, total_units: int = TOTAL_PATROL_UNITS) -> pd.DataFrame:
    out = df.copy()
    out["recommended_patrol_count"] = 0
    eligible = out["priority_score"] >= PATROL_ELIGIBILITY_SCORE
    if eligible.sum() == 0:
        eligible = out.sort_values("priority_score", ascending=False).head(min(total_units, len(out))).index
    eligible_idx = list(out.index[eligible]) if not isinstance(eligible, pd.Index) else list(eligible)

    if len(eligible_idx) <= total_units:
        out.loc[eligible_idx, "recommended_patrol_count"] = 1
        remain = total_units - len(eligible_idx)
        if remain > 0:
            weights = pd.to_numeric(out.loc[eligible_idx, "priority_score"], errors="coerce").fillna(0.0)
            weights = weights / max(weights.sum(), EPS)
            extra = weights * remain
            add = np.floor(extra).astype(int)
            out.loc[eligible_idx, "recommended_patrol_count"] += add.values
            leftover = remain - int(add.sum())
            if leftover > 0:
                frac = (extra - add).sort_values(ascending=False)
                for idx in frac.index[:leftover]:
                    out.at[idx, "recommended_patrol_count"] += 1
    else:
        top = out.sort_values("priority_score", ascending=False).head(total_units).index.tolist()
        out.loc[top, "recommended_patrol_count"] = 1

    out["recommended_patrol_count"] = pd.to_numeric(out["recommended_patrol_count"], errors="coerce").fillna(0).astype(int).clip(0, MAX_PATROLS_PER_HOTSPOT)

    critical_mask = out["final_risk_band"].isin(["Critical", "High"]) & (out["recommended_patrol_count"] == 0)
    for idx in out[critical_mask].sort_values("priority_score", ascending=False).index:
        if int(out["recommended_patrol_count"].sum()) >= total_units:
            break
        out.at[idx, "recommended_patrol_count"] = 1
    return out


def compute_tow_readiness(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    road = out["road_class"].fillna("road").astype(str).str.lower()
    arterial = road.isin({"motorway", "trunk", "primary", "secondary", "tertiary", "road"})
    width = pd.to_numeric(out["carriageway_width_m"], errors="coerce").fillna(7.0)
    lanes = pd.to_numeric(out["lane_count"], errors="coerce").fillna(2)
    bet = pd.to_numeric(out["betweenness_centrality"], errors="coerce").fillna(0.0)
    bet_hi = bet >= bet.quantile(HIGH_BETWEENNESS_PERCENTILE) if len(bet) else pd.Series(False, index=out.index)
    chronic = pd.to_numeric(out.get("chronic_vehicle_count_5plus", 0), errors="coerce").fillna(0)
    repeat2 = pd.to_numeric(out.get("repeat_vehicle_count_2plus", 0), errors="coerce").fillna(0)
    esc = pd.to_numeric(out["probability_of_escalation"], errors="coerce").fillna(0.0)
    pscore = pd.to_numeric(out["priority_score"], errors="coerce").fillna(0.0)

    tow = ((pscore >= TOW_PRIORITY_THRESHOLD) | out["final_risk_band"].isin(["Critical", "High"]) | (esc >= ESCALATION_THRESHOLD)) & (arterial | (width <= 7.5) | (lanes <= 2) | bet_hi) & ((chronic >= 1) | (repeat2 >= 3) | (pscore >= 80))
    out["tow_required"] = tow.astype(bool)
    out["tow_reason"] = np.where(out["tow_required"], "High-risk corridor with repeat/chronic pressure", "No tow required")
    return out


def build_enforcement_action(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    def action(row):
        b = str(row.get("final_risk_band", "Watch"))
        tow = bool(row.get("tow_required", False))
        patrols = int(row.get("recommended_patrol_count", 0) or 0)
        if b == "Critical":
            return "Immediate patrol + tow standby + camera watch" if tow else "Immediate patrol deployment"
        if b == "High":
            return "Targeted patrol + tow readiness" if tow else "Targeted patrol + close monitoring"
        if b == "Moderate":
            return "Scheduled patrol + peak-window checks" if patrols >= 2 else "Peak-window checks"
        return "Routine monitoring"

    out["enforcement_action"] = out.apply(action, axis=1)
    out["dispatch_notes"] = out.apply(
        lambda row: " | ".join([
            f"forecast={safe_float(row.get('predicted_next_week_ccs_score', 0.0), 0.0):.2f}",
            f"esc_prob={safe_float(row.get('probability_of_escalation', 0.0), 0.0):.2f}",
            f"priority={safe_float(row.get('priority_score', 0.0), 0.0):.1f}",
            f"patrols={int(row.get('recommended_patrol_count', 0) or 0)}" if int(row.get("recommended_patrol_count", 0) or 0) > 0 else "patrols=0",
            f"tow={'yes' if bool(row.get('tow_required', False)) else 'no'}",
            f"window={clean_text(row.get('recommended_time_window', ''))}",
        ]),
        axis=1,
    )
    return out


def nearest_neighbor_route(df: pd.DataFrame, depot_lat: float = DEPOT_LAT, depot_lon: float = DEPOT_LON, top_n: int = ROUTE_PLAN_TOP_N) -> pd.DataFrame:
    cand = df[df["recommended_patrol_count"] > 0].copy()
    cand = cand.dropna(subset=["lat", "lon"]).copy()
    if len(cand) == 0:
        return pd.DataFrame(columns=["route_id", "stop_sequence", "cluster_key", "cluster_label_display", "lat", "lon", "distance_from_prev_m", "cumulative_distance_m", "priority_score"])

    cand = cand.sort_values(["priority_score", "probability_of_escalation"], ascending=[False, False]).head(top_n).copy().reset_index(drop=True)
    remaining = cand.copy()
    rows = []
    cur_lat, cur_lon = float(depot_lat), float(depot_lon)
    cum = 0.0
    seq = 0
    while len(remaining):
        dists = remaining.apply(lambda r: haversine_m(cur_lat, cur_lon, r["lat"], r["lon"]), axis=1)
        idx = dists.idxmin()
        row = remaining.loc[idx]
        d = float(dists.loc[idx]) if pd.notna(dists.loc[idx]) else 0.0
        cum += d
        seq += 1
        rows.append({
            "route_id": "route_1",
            "stop_sequence": seq,
            "cluster_key": row["cluster_key"],
            "cluster_label_display": row.get("cluster_label_display", row.get("cluster_label", "")),
            "lat": float(row["lat"]),
            "lon": float(row["lon"]),
            "distance_from_prev_m": d,
            "cumulative_distance_m": cum,
            "priority_score": float(row["priority_score"]),
        })
        cur_lat, cur_lon = float(row["lat"]), float(row["lon"])
        remaining = remaining.drop(index=idx)
    return pd.DataFrame(rows)


def build_offender_actions(offenders_df: Optional[pd.DataFrame]) -> pd.DataFrame:
    if offenders_df is None or len(offenders_df) == 0:
        return pd.DataFrame(columns=["vehicle_number", "total_violations", "unique_clusters", "unique_hotspots", "first_seen", "last_seen", "dominant_vehicle_type", "offender_pressure_score", "offender_priority_band", "recommended_action"])

    out = offenders_df.copy()
    for c in ["vehicle_number", "dominant_vehicle_type"]:
        if c in out.columns:
            out[c] = out[c].fillna("").astype(str).str.strip()
    for c in ["total_violations", "unique_clusters", "unique_hotspots"]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0)
    if "first_seen" in out.columns:
        out["first_seen"] = pd.to_datetime(out["first_seen"], errors="coerce")
    if "last_seen" in out.columns:
        out["last_seen"] = pd.to_datetime(out["last_seen"], errors="coerce")

    out["offender_pressure_raw"] = np.log1p(pd.to_numeric(out.get("total_violations", 0), errors="coerce").fillna(0).clip(lower=0)) * (1.0 + minmax(pd.to_numeric(out.get("unique_hotspots", 0), errors="coerce").fillna(0))) * (1.0 + minmax(pd.to_numeric(out.get("unique_clusters", 0), errors="coerce").fillna(0)))
    out["offender_pressure_score"] = 100.0 * minmax(out["offender_pressure_raw"].fillna(0.0))
    out["offender_priority_band"] = out["offender_pressure_score"].apply(band_from_score)
    out["recommended_action"] = np.select(
        [
            out["offender_priority_band"].eq("Critical"),
            out["offender_priority_band"].eq("High"),
            out["offender_priority_band"].eq("Moderate"),
        ],
        [
            "Special watch + field verification",
            "Targeted surveillance + repeat checks",
            "Monitor vehicle pattern",
        ],
        default="Routine tracking",
    )
    return out.sort_values(["offender_pressure_score", "total_violations"], ascending=[False, False]).reset_index(drop=True)


# ------------------------- main -------------------------
def main():
    print("Loading Phase 7 forecast...")
    phase7_df, phase7_src = load_phase7_forecast()
    print("Phase 7 source:", phase7_src)

    print("Loading Phase 6 static context...")
    phase6_df, phase6_src = load_phase6_static()
    print("Phase 6 source:", phase6_src)

    print("Loading record-level history...")
    records_df, records_src = load_record_history()
    print("Record source:", records_src)

    offenders_df, offenders_src = load_chronic_offenders()

    static_df = prepare_static_context(phase6_df)

    forecast = phase7_df.copy()
    forecast = ensure_label_column(forecast)
    forecast = derive_coords(forecast)
    cluster_col = standardize_cluster_col(forecast)
    forecast["cluster_key"] = forecast[cluster_col].map(normalize_cluster_key)

    # If static coords/context exist, merge them as suffix columns and coalesce.
    merge_cols = [
        "cluster_key", "lat", "lon", "road_class", "lane_count", "carriageway_width_m", "link_length_m",
        "junction_degree", "betweenness_centrality", "criticality_factor", "context_multiplier",
        "layer_b_priority_boost", "layer_b_alert_flag", "validation_uncertainty", "resurgence_score",
        "persistence_score", "anomaly_score", "rop", "tvs", "vdi", "nearby_sensitive_poi_count",
        "geometry_source", "mappls_address", "road_node_id", "road_node_distance_m", "road_node_degree",
        "road_node_betweenness", "source_pressure_score", "source_pressure_norm", "spillover_out_score",
        "spillover_in_score", "spillover_total_score", "propagation_radius_m", "network_pagerank",
        "network_component_id", "network_component_size", "neighbor_count", "in_neighbor_count",
        "out_neighbor_count", "influence_asymmetry", "network_vulnerability_score", "layer_d_alert_flag",
        "dominant_vehicle_type", "cluster_label", "cluster_label_display", "risk_band", "ccs_score",
        "delay_minutes_per_vehicle", "records_total", "distinct_days", "severity_sum", "severity_mean",
        "growth_pct_change", "growth_multiplier", "repeat_vehicle_count_2plus", "chronic_vehicle_count_5plus",
        "dominant_police_station", "police_station", "dominant_junction_name", "hotspot_unit",
    ]
    merge_cols = [c for c in merge_cols if c in static_df.columns]
    static_lookup = static_df[merge_cols].drop_duplicates("cluster_key").rename(columns={c: f"static_{c}" for c in merge_cols if c != "cluster_key"})
    forecast = forecast.merge(static_lookup, on="cluster_key", how="left")
    forecast = coalesce_from_suffix(forecast, [c for c in merge_cols if c != "cluster_key"], suffix="_static")

    # Weekly history + novel features
    weekly_hist, hist_summary, _ = prepare_record_history(records_df, static_df)
    print("Weekly rows built:", len(weekly_hist))
    print("Hotspots in weekly history:", weekly_hist["cluster_key"].nunique())

    week_counts = weekly_hist.groupby("cluster_key")["week_start"].nunique()
    print("\nWeeks of history per hotspot (describe):")
    print(week_counts.describe().to_string())
    single_week_share = float((week_counts == 1).mean() * 100.0) if len(week_counts) else 0.0
    print(f"Share of hotspots with only ONE observed week: {single_week_share:.1f}% -- weekly_growth_pct == 0.0 for these is expected, not a bug.")

    # Latest history row per hotspot.
    hist_latest = weekly_hist.sort_values(["cluster_key", "week_start"]).groupby("cluster_key", as_index=False).tail(1).copy()
    hist_cols = [c for c in ["cluster_key", "peak_hour", "best_window_start_hour", "best_window_end_hour", "best_day_of_week", "best_day_name", "peak_hours_top3", "peak_window_violation_ratio", "tvs", "resurgence_score", "persistence_score", "validation_total", "validation_uncertainty", "repeat_vehicle_count_2plus", "chronic_vehicle_count_5plus", "vdi", "weekly_rows", "records_week_mean", "records_week_std", "severity_sum_week_mean", "peak_window_records_week_mean", "unique_vehicles_week_mean", "unique_vehicle_types_week_mean", "weekly_growth_pct_mean", "weekly_growth_pct_max", "weekly_pressure_raw_mean", "weekly_pressure_raw_max", "rolling_4w_mean_pressure_raw_max", "rolling_4w_std_pressure_raw_max", "rolling_4w_mean_records_week_max", "rolling_4w_mean_severity_week_max", "rolling_4w_trend_slope_records_week_max", "rolling_4w_pressure_acceleration_max"] if c in hist_latest.columns]
    hist_latest = hist_latest[hist_cols].drop_duplicates("cluster_key").rename(columns={c: f"hist_{c}" for c in hist_cols if c != "cluster_key"})

    forecast = forecast.merge(hist_latest, on="cluster_key", how="left")
    forecast = coalesce_from_suffix(forecast, [c for c in hist_cols if c != "cluster_key"], suffix="_hist")

    # Use history values as fallbacks.
    for c in ["peak_hour", "best_window_start_hour", "best_window_end_hour", "best_day_name", "peak_hours_top3", "peak_window_violation_ratio", "tvs", "resurgence_score", "persistence_score", "validation_total", "validation_uncertainty", "repeat_vehicle_count_2plus", "chronic_vehicle_count_5plus", "vdi", "weekly_rows", "records_week_mean", "records_week_std", "severity_sum_week_mean", "peak_window_records_week_mean", "unique_vehicles_week_mean", "unique_vehicle_types_week_mean", "weekly_growth_pct_mean", "weekly_growth_pct_max", "weekly_pressure_raw_mean", "weekly_pressure_raw_max", "rolling_4w_mean_pressure_raw_max", "rolling_4w_std_pressure_raw_max", "rolling_4w_mean_records_week_max", "rolling_4w_mean_severity_week_max", "rolling_4w_trend_slope_records_week_max", "rolling_4w_pressure_acceleration_max", "dominant_vehicle_type", "cluster_label", "cluster_label_display", "risk_band", "ccs_score", "delay_minutes_per_vehicle", "records_total", "distinct_days", "severity_sum", "severity_mean", "growth_pct_change", "growth_multiplier", "criticality_factor", "context_multiplier", "layer_b_priority_boost", "layer_b_alert_flag", "geometry_source", "road_class", "lane_count", "carriageway_width_m", "link_length_m", "junction_degree", "betweenness_centrality", "road_node_distance_m", "road_node_degree", "road_node_betweenness", "source_pressure_score", "source_pressure_norm", "spillover_out_score", "spillover_in_score", "spillover_total_score", "propagation_radius_m", "network_pagerank", "network_component_id", "network_component_size", "neighbor_count", "in_neighbor_count", "out_neighbor_count", "influence_asymmetry", "network_vulnerability_score"]:
        if c not in forecast.columns:
            forecast[c] = np.nan

    forecast["zone_name"] = build_zone_name(forecast)

    # Score + actions
    components = {
        "probability_of_escalation": 0.22,
        "predicted_next_week_ccs_score": 0.18,
        "current_week_ccs_proxy_score": 0.08,
        "criticality_factor": 0.08,
        "growth_multiplier": 0.07,
        "growth_pct_change": 0.05,
        "layer_b_priority_boost": 0.05,
        "validation_uncertainty": 0.05,
        "rop": 0.08,
        "tvs": 0.06,
        "vdi": 0.05,
        "resurgence_score": 0.05,
        "anomaly_score": 0.04,
        "network_vulnerability_score": 0.04,
        "spillover_total_score": 0.01,
        "nearby_sensitive_poi_count": 0.01,
    }
    for c in components:
        forecast[c] = pd.to_numeric(forecast.get(c, 0.0), errors="coerce").fillna(0.0)
        forecast[f"{c}_norm"] = minmax(forecast[c])

    raw = np.zeros(len(forecast), dtype=float)
    total_w = float(sum(components.values()))
    for c, w in components.items():
        raw += w * forecast[f"{c}_norm"].to_numpy(dtype=float)
    forecast["priority_score"] = 100.0 * np.clip(raw / max(total_w, EPS), 0.0, 1.0)
    forecast["priority_band"] = forecast["priority_score"].apply(band_from_score)
    forecast["final_risk_band"] = forecast.apply(
        lambda row: row["priority_band"] if band_index(row["priority_band"]) >= band_index(row.get("future_risk_band", "Watch")) else row.get("future_risk_band", "Watch"),
        axis=1,
    )

    # Time windows
    tw = forecast.apply(lambda r: pd.Series(compute_time_window(r), index=["recommended_time_window", "recommended_day_name", "recommended_shift"]), axis=1)
    forecast = pd.concat([forecast, tw], axis=1)

    # Tow + patrol + actions
    road = forecast["road_class"].fillna("road").astype(str).str.lower()
    arterial = road.isin({"motorway", "trunk", "primary", "secondary", "tertiary", "road"})
    width = pd.to_numeric(forecast["carriageway_width_m"], errors="coerce").fillna(7.0)
    lanes = pd.to_numeric(forecast["lane_count"], errors="coerce").fillna(2)
    bet = pd.to_numeric(forecast["betweenness_centrality"], errors="coerce").fillna(0.0)
    bet_hi = bet >= bet.quantile(HIGH_BETWEENNESS_PERCENTILE) if len(bet) else pd.Series(False, index=forecast.index)
    chronic = pd.to_numeric(forecast.get("chronic_vehicle_count_5plus", 0), errors="coerce").fillna(0)
    repeat2 = pd.to_numeric(forecast.get("repeat_vehicle_count_2plus", 0), errors="coerce").fillna(0)
    esc = pd.to_numeric(forecast["probability_of_escalation"], errors="coerce").fillna(0.0)
    pscore = pd.to_numeric(forecast["priority_score"], errors="coerce").fillna(0.0)

    forecast["tow_required"] = (((pscore >= TOW_PRIORITY_THRESHOLD) | forecast["final_risk_band"].isin(["Critical", "High"]) | (esc >= ESCALATION_THRESHOLD)) & (arterial | (width <= 7.5) | (lanes <= 2) | bet_hi) & ((chronic >= 1) | (repeat2 >= 3) | (pscore >= 80))).astype(bool)
    forecast["tow_reason"] = np.where(forecast["tow_required"], "High-risk corridor with repeat/chronic pressure", "No tow required")

    forecast = forecast.sort_values(["priority_score", "probability_of_escalation", "predicted_next_week_ccs_score"], ascending=[False, False, False]).reset_index(drop=True)
    forecast["dispatch_rank"] = np.arange(1, len(forecast) + 1)

    # Patrol allocation
    forecast["recommended_patrol_count"] = 0
    eligible = forecast["priority_score"] >= PATROL_ELIGIBILITY_SCORE
    if eligible.sum() == 0:
        eligible = forecast.index.isin(forecast.head(min(TOTAL_PATROL_UNITS, len(forecast))).index)
    eligible_idx = list(forecast.index[eligible]) if not isinstance(eligible, pd.Index) else list(eligible)
    if len(eligible_idx) <= TOTAL_PATROL_UNITS:
        forecast.loc[eligible_idx, "recommended_patrol_count"] = 1
        remain = TOTAL_PATROL_UNITS - len(eligible_idx)
        if remain > 0:
            w = pd.to_numeric(forecast.loc[eligible_idx, "priority_score"], errors="coerce").fillna(0.0)
            w = w / max(w.sum(), EPS)
            extra = w * remain
            add = np.floor(extra).astype(int)
            forecast.loc[eligible_idx, "recommended_patrol_count"] += add.values
            leftover = remain - int(add.sum())
            if leftover > 0:
                frac = (extra - add).sort_values(ascending=False)
                for idx in frac.index[:leftover]:
                    forecast.at[idx, "recommended_patrol_count"] += 1
    else:
        top = forecast.head(TOTAL_PATROL_UNITS).index.tolist()
        forecast.loc[top, "recommended_patrol_count"] = 1
    forecast["recommended_patrol_count"] = pd.to_numeric(forecast["recommended_patrol_count"], errors="coerce").fillna(0).astype(int).clip(0, MAX_PATROLS_PER_HOTSPOT)

    # Action text
    def action(row):
        b = str(row.get("final_risk_band", "Watch"))
        tow = bool(row.get("tow_required", False))
        patrols = int(row.get("recommended_patrol_count", 0) or 0)
        if b == "Critical":
            return "Immediate patrol + tow standby + camera watch" if tow else "Immediate patrol deployment"
        if b == "High":
            return "Targeted patrol + tow readiness" if tow else "Targeted patrol + close monitoring"
        if b == "Moderate":
            return "Scheduled patrol + peak-window checks" if patrols >= 2 else "Peak-window checks"
        return "Routine monitoring"
    forecast["enforcement_action"] = forecast.apply(action, axis=1)

    forecast["dispatch_notes"] = forecast.apply(
        lambda row: " | ".join([
            f"forecast={safe_float(row.get('predicted_next_week_ccs_score', 0.0), 0.0):.2f}",
            f"esc_prob={safe_float(row.get('probability_of_escalation', 0.0), 0.0):.2f}",
            f"priority={safe_float(row.get('priority_score', 0.0), 0.0):.1f}",
            f"patrols={int(row.get('recommended_patrol_count', 0) or 0)}",
            f"tow={'yes' if bool(row.get('tow_required', False)) else 'no'}",
            f"window={clean_text(row.get('recommended_time_window', ''))}",
        ]),
        axis=1,
    )

    # Zone summary
    zone_summary = forecast.groupby("zone_name").agg(
        zone_hotspots=("cluster_key", "size"),
        zone_priority_total=("priority_score", "sum"),
        zone_priority_mean=("priority_score", "mean"),
        zone_priority_max=("priority_score", "max"),
        zone_ccs_mean=("predicted_next_week_ccs_score", "mean"),
        zone_ccs_max=("predicted_next_week_ccs_score", "max"),
        zone_escalation_mean=("probability_of_escalation", "mean"),
        zone_tow_count=("tow_required", "sum"),
        zone_patrol_units=("recommended_patrol_count", "sum"),
        zone_critical_count=("final_risk_band", lambda s: int((s == "Critical").sum())),
        zone_high_count=("final_risk_band", lambda s: int((s == "High").sum())),
        zone_top_label=("cluster_label_display", dominant_label),
    ).reset_index()
    zone_summary["zone_rank"] = zone_summary["zone_priority_total"].rank(method="dense", ascending=False).astype(int)
    zone_summary = zone_summary.sort_values(["zone_priority_total", "zone_priority_max"], ascending=[False, False]).reset_index(drop=True)

    # Route plan (heuristic, top assigned hotspots)
    route_candidates = forecast[forecast["recommended_patrol_count"] > 0].copy()
    route_candidates = route_candidates.dropna(subset=["lat", "lon"]).copy()
    if len(route_candidates):
        route_candidates = route_candidates.sort_values(["priority_score", "probability_of_escalation"], ascending=[False, False]).head(ROUTE_PLAN_TOP_N).reset_index(drop=True)
        remaining = route_candidates.copy()
        route_rows = []
        cur_lat, cur_lon = float(DEPOT_LAT), float(DEPOT_LON)
        cum = 0.0
        seq = 0
        while len(remaining):
            dists = remaining.apply(lambda r: haversine_m(cur_lat, cur_lon, r["lat"], r["lon"]), axis=1)
            idx = dists.idxmin()
            row = remaining.loc[idx]
            d = float(dists.loc[idx]) if pd.notna(dists.loc[idx]) else 0.0
            cum += d
            seq += 1
            route_rows.append({
                "route_id": "route_1",
                "stop_sequence": seq,
                "cluster_key": row["cluster_key"],
                "cluster_label_display": row.get("cluster_label_display", row.get("cluster_label", "")),
                "lat": float(row["lat"]),
                "lon": float(row["lon"]),
                "distance_from_prev_m": d,
                "cumulative_distance_m": cum,
                "priority_score": float(row["priority_score"]),
            })
            cur_lat, cur_lon = float(row["lat"]), float(row["lon"])
            remaining = remaining.drop(index=idx)
        route_plan = pd.DataFrame(route_rows)
    else:
        route_plan = pd.DataFrame(columns=["route_id", "stop_sequence", "cluster_key", "cluster_label_display", "lat", "lon", "distance_from_prev_m", "cumulative_distance_m", "priority_score"])

    # Offenders
    offender_actions = build_offender_actions(offenders_df)

    # Final tables
    dispatch_cols = [
        "dispatch_rank", "cluster_key", "cluster_label", "cluster_label_display", "zone_name",
        "final_risk_band", "future_risk_band", "priority_band", "priority_score",
        "probability_of_escalation", "predicted_next_week_ccs_score", "current_week_ccs_proxy_score",
        "predicted_delta_score", "predicted_delta_pct", "recommended_patrol_count", "tow_required",
        "tow_reason", "recommended_time_window", "recommended_day_name", "recommended_shift",
        "enforcement_action", "forecast_recommendation", "road_class", "geometry_source",
        "dominant_vehicle_type", "records_week", "weekly_growth_pct", "validation_uncertainty",
        "repeat_vehicle_count_2plus", "chronic_vehicle_count_5plus", "vdi", "tvs",
        "resurgence_score", "persistence_score", "criticality_factor", "context_multiplier",
        "layer_b_priority_boost", "layer_b_alert_flag", "nearby_sensitive_poi_count",
        "lane_count", "carriageway_width_m", "link_length_m", "junction_degree",
        "betweenness_centrality", "road_node_distance_m", "road_node_degree",
        "road_node_betweenness", "source_pressure_score", "source_pressure_norm",
        "spillover_out_score", "spillover_in_score", "spillover_total_score",
        "propagation_radius_m", "network_pagerank", "network_component_id",
        "network_component_size", "neighbor_count", "in_neighbor_count", "out_neighbor_count",
        "influence_asymmetry", "network_vulnerability_score", "dispatch_notes", "lat", "lon",
    ]
    dispatch_cols = [c for c in dispatch_cols if c in forecast.columns]
    dispatch_sheet = forecast[dispatch_cols].copy()

    if "forecast_recommendation" not in dispatch_sheet.columns:
        dispatch_sheet["forecast_recommendation"] = np.select(
            [
                dispatch_sheet["future_risk_band"].eq("Critical"),
                dispatch_sheet["future_risk_band"].eq("High"),
                dispatch_sheet["future_risk_band"].eq("Moderate"),
            ],
            [
                "Immediate patrol deployment",
                "Targeted enforcement + tow readiness",
                "Monitor and schedule peak-window checks",
            ],
            default="Routine monitoring",
        )

    allocation_output = dispatch_sheet[dispatch_sheet["recommended_patrol_count"] > 0].copy().reset_index(drop=True)

    # Save outputs
    dispatch_sheet.to_csv(PHASE8_OUT_DIR / "phase8_dispatch_sheet.csv", index=False)
    allocation_output.to_csv(PHASE8_OUT_DIR / "phase8_allocated_hotspots.csv", index=False)
    zone_summary.to_csv(PHASE8_OUT_DIR / "phase8_zone_summary.csv", index=False)
    route_plan.to_csv(PHASE8_OUT_DIR / "phase8_route_plan.csv", index=False)
    offender_actions.to_csv(PHASE8_OUT_DIR / "phase8_chronic_offender_actions.csv", index=False)

    summary_out = pd.DataFrame([{
        "phase7_source": str(phase7_src),
        "phase6_source": str(phase6_src),
        "records_source": str(records_src),
        "offenders_source": str(offenders_src) if offenders_src else "",
        "hotspots_forecasted": int(len(dispatch_sheet)),
        "hotspots_allocated": int(len(allocation_output)),
        "total_patrol_units_requested": int(TOTAL_PATROL_UNITS),
        "total_patrol_units_allocated": int(dispatch_sheet["recommended_patrol_count"].sum()),
        "tow_required_hotspots": int(dispatch_sheet["tow_required"].sum()),
        "critical_hotspots": int((dispatch_sheet["final_risk_band"] == "Critical").sum()),
        "high_hotspots": int((dispatch_sheet["final_risk_band"] == "High").sum()),
        "zone_count": int(len(zone_summary)),
        "route_stops": int(len(route_plan)),
        "mean_priority_score": float(pd.to_numeric(dispatch_sheet["priority_score"], errors="coerce").mean()) if len(dispatch_sheet) else 0.0,
        "mean_escalation_probability": float(pd.to_numeric(dispatch_sheet["probability_of_escalation"], errors="coerce").mean()) if len(dispatch_sheet) else 0.0,
    }])
    summary_out.to_csv(PHASE8_OUT_DIR / "phase8_summary.csv", index=False)

    # Console output
    print("\nPhase 8 complete")
    print("Outputs saved to:", PHASE8_OUT_DIR.resolve())
    print("\nSummary:")
    print(summary_out.to_string(index=False))

    print("\nTop 10 dispatch priorities:")
    top_cols = [c for c in ["dispatch_rank", "cluster_key", "cluster_label_display", "zone_name", "final_risk_band", "priority_score", "probability_of_escalation", "predicted_next_week_ccs_score", "recommended_patrol_count", "tow_required", "recommended_time_window", "enforcement_action"] if c in dispatch_sheet.columns]
    print(dispatch_sheet[top_cols].head(10).to_string(index=False))

    print("\nTop 10 zones:")
    zone_cols = [c for c in ["zone_rank", "zone_name", "zone_priority_total", "zone_priority_mean", "zone_priority_max", "zone_hotspots", "zone_patrol_units", "zone_tow_count", "zone_critical_count", "zone_high_count"] if c in zone_summary.columns]
    print(zone_summary[zone_cols].head(10).to_string(index=False))

    print("\nRoute plan:")
    if len(route_plan):
        print(route_plan.head(20).to_string(index=False))
    else:
        print("No route plan generated.")


if __name__ == "__main__":
    main()


Loading Phase 7 forecast...
Phase 7 source: content\phase7_outputs_2\phase7_next_week_forecast_ranked.csv
Loading Phase 6 static context...
Phase 6 source: content\phase6_outputs_2\phase6_cluster_ccs_full.csv
Loading record-level history...
Record source: content\phase5_outputs_2\phase5_enriched_records.csv
Weekly rows built: 1608
Hotspots in weekly history: 798

Weeks of history per hotspot (describe):
count    798.000000
mean       2.015038
std        1.534145
min        1.000000
25%        1.000000
50%        2.000000
75%        2.000000
max       13.000000
Share of hotspots with only ONE observed week: 38.2% -- weekly_growth_pct == 0.0 for these is expected, not a bug.

Phase 8 complete
Outputs saved to: D:\Flipkart_Hackathon\approach1\content\phase8_outputs_2

Summary:
                                                phase7_source                                        phase6_source                                       records_source                                          offend

In [ ]:
import math
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore")

# =========================
# Config
# =========================
INPUT_PATHS = [
    Path("content/phase6_outputs_2/phase6_stage6_handoff.csv"),
    Path("content/phase6_outputs_2/phase6_cluster_ccs_full.csv"),
    Path("phase6_outputs_2/phase6_stage6_handoff.csv"),
    Path("phase6_outputs_2/phase6_cluster_ccs_full.csv"),
    Path("/content/phase6_outputs_2/phase6_stage6_handoff.csv"),
    Path("/content/phase6_outputs_2/phase6_cluster_ccs_full.csv"),
    Path("phase6_outputs/phase6_stage6_handoff.csv"),
    Path("phase6_outputs/phase6_cluster_ccs_full.csv"),
    Path("content/phase6_outputs/phase6_stage6_handoff.csv"),
    Path("content/phase6_outputs/phase6_cluster_ccs_full.csv"),
]

AUX_PATHS = [
    Path("content/phase5_outputs_2/phase5_enriched_records.csv"),
    Path("phase5_outputs_2/phase5_enriched_records.csv"),
    Path("/content/phase5_outputs_2/phase5_enriched_records.csv"),
    Path("content/phase4_outputs_2/phase4_merged_with_prior_scores.csv"),
    Path("phase4_outputs_2/phase4_merged_with_prior_scores.csv"),
    Path("/content/phase4_outputs_2/phase4_merged_with_prior_scores.csv"),
    Path("content/phase3_outputs_2/phase3_clustered_dataset.csv"),
    Path("phase3_outputs_2/phase3_clustered_dataset.csv"),
    Path("/content/phase3_outputs_2/phase3_clustered_dataset.csv"),
]

OUT_DIR = Path("content/validation_outputs_2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_N = int(os.environ.get("VALIDATION_TOP_N", "20"))
VALIDATION_CORRIDOR_METERS = float(os.environ.get("VALIDATION_CORRIDOR_METERS", "500"))
REQUEST_TIMEOUT = int(os.environ.get("VALIDATION_REQUEST_TIMEOUT", "25"))

MAPPLS_ACCESS_TOKEN = os.environ.get("MAPPLS_ACCESS_TOKEN", "").strip()
MAPPLS_REGION = os.environ.get("MAPPLS_REGION", "ind").strip().lower()
MAPPLS_PROFILE = os.environ.get("MAPPLS_PROFILE", "driving").strip().lower()

DEFAULT_ROAD_CLASS = "road"
EPS = 1e-9

# Same table Stage 5 uses, so the "free-flow ETA" computed here for the
# synthetic validation corridor is on the same basis as the model's own
# free_flow_time_min — this is what makes the comparison meaningful.
ROAD_CLASS_SPEED_KMH = {
    "motorway": 60.0, "trunk": 55.0, "primary": 45.0, "secondary": 40.0,
    "tertiary": 35.0, "unclassified": 30.0, "residential": 30.0,
    "living_street": 20.0, "service": 25.0, "road": 30.0,
}


# =========================
# Helpers
# =========================
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def safe_float(x, default=np.nan):
    try:
        if pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default

def valid_coords(lat, lon):
    try:
        return (
            pd.notna(lat) and pd.notna(lon)
            and -90 <= float(lat) <= 90
            and -180 <= float(lon) <= 180
        )
    except Exception:
        return False

def road_speed_kmh(road_class):
    return ROAD_CLASS_SPEED_KMH.get(clean_text(road_class).lower(), 30.0)

def load_first_existing(paths):
    for p in paths:
        if p.exists():
            return pd.read_csv(p, low_memory=False), p
    raise FileNotFoundError("Could not find a Stage 6 output file.")

def load_all_existing(paths):
    frames = []
    for p in paths:
        if p.exists():
            try:
                frames.append((p, pd.read_csv(p, low_memory=False)))
            except Exception:
                pass
    return frames

def standardize_cluster_col(df):
    for c in ["st_dbscan_cluster_id", "cluster_id", "dbscan_cluster_id"]:
        if c in df.columns:
            return c
    raise ValueError("No cluster id column found.")

def derive_lat_lon(df):
    df = df.copy()
    if {"lat", "lon"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
        df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    elif {"centroid_lat", "centroid_lon"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["centroid_lat"], errors="coerce")
        df["lon"] = pd.to_numeric(df["centroid_lon"], errors="coerce")
    elif {"latitude_mean", "longitude_mean"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["latitude_mean"], errors="coerce")
        df["lon"] = pd.to_numeric(df["longitude_mean"], errors="coerce")
    elif {"latitude", "longitude"}.issubset(df.columns):
        df["lat"] = pd.to_numeric(df["latitude"], errors="coerce")
        df["lon"] = pd.to_numeric(df["longitude"], errors="coerce")
    else:
        df["lat"] = np.nan
        df["lon"] = np.nan
    return df

def offset_point(lat, lon, meters=500.0, bearing_deg=90.0):
    """Arbitrary nearby point used only to probe live road speed near the
    hotspot (a 'is traffic moving normally here right now' check) — it is
    NOT meant to represent the hotspot's actual congested approach/exit."""
    R = 6371000.0
    bearing = math.radians(bearing_deg)
    lat1 = math.radians(lat)
    lon1 = math.radians(lon)
    d = meters / R
    lat2 = math.asin(
        math.sin(lat1) * math.cos(d) + math.cos(lat1) * math.sin(d) * math.cos(bearing)
    )
    lon2 = lon1 + math.atan2(
        math.sin(bearing) * math.sin(d) * math.cos(lat1),
        math.cos(d) - math.sin(lat1) * math.sin(lat2),
    )
    return math.degrees(lat2), math.degrees(lon2)

def deep_first_value(obj, keys):
    if isinstance(obj, dict):
        for k in keys:
            if k in obj and obj[k] not in (None, "", [], {}):
                return obj[k]
        for v in obj.values():
            found = deep_first_value(v, keys)
            if found not in (None, "", [], {}):
                return found
    elif isinstance(obj, list):
        for item in obj:
            found = deep_first_value(item, keys)
            if found not in (None, "", [], {}):
                return found
    return None

def extract_point_to_point(matrix, src_idx=0, dst_idx=1):
    """This script only ever submits exactly one source + one destination,
    so the Mappls response is a 1xN (N>=2) row-ordered matrix and the value
    we want is unambiguously matrix[src_idx][dst_idx] (per Mappls Distance
    Matrix docs: row-ordered source->target array). No need to scan for a
    'minimum off-diagonal' value — that was only safe by coincidence for a
    2x2 case and would silently pick the wrong cell for larger matrices."""
    if not isinstance(matrix, list) or len(matrix) <= src_idx:
        return None
    row = matrix[src_idx]
    if not isinstance(row, list) or len(row) <= dst_idx:
        return None
    try:
        v = float(row[dst_idx])
        return v if np.isfinite(v) and v >= 0 else None
    except Exception:
        return None

def parse_reverse_geocode_address(payload):
    if not isinstance(payload, dict):
        return ""
    results = payload.get("results")
    if isinstance(results, list) and results:
        first = results[0]
        if isinstance(first, dict):
            addr = first.get("formatted_address") or first.get("address") or ""
            if addr:
                return str(addr).strip()
            parts = []
            for k in [
                "houseNumber", "houseName", "poi", "street", "subSubLocality",
                "subLocality", "locality", "subDistrict", "district", "city",
                "state", "pincode"
            ]:
                v = clean_text(first.get(k, ""))
                if v:
                    parts.append(v)
            if parts:
                return ", ".join(dict.fromkeys(parts))
    addr = deep_first_value(payload, ["formatted_address", "address"])
    return clean_text(addr) if addr is not None else ""

def parse_route_response(payload):
    """Per Mappls Route Driving Directions API: {"code":"Ok","routes":[{"distance":..,
    "duration":..,"legs":[{"distance":..,"duration":..}]}]} — distance is in
    meters, duration is in seconds, both directly on routes[0]."""
    if not isinstance(payload, dict):
        return None, None, "invalid"

    routes = payload.get("routes")
    if isinstance(routes, list) and routes:
        r0 = routes[0]
        d = safe_float(r0.get("distance"), None)
        t = safe_float(r0.get("duration"), None)
        if d is not None or t is not None:
            return d, t, "routes[0]"

        legs = r0.get("legs")
        if isinstance(legs, list) and legs:
            leg0 = legs[0]
            d = safe_float(leg0.get("distance"), None)
            t = safe_float(leg0.get("duration"), None)
            if d is not None or t is not None:
                return d, t, "routes[0].legs[0]"
            summary = leg0.get("summary")
            if isinstance(summary, dict):
                d = safe_float(summary.get("distance"), None)
                t = safe_float(summary.get("duration"), None)
                if d is not None or t is not None:
                    return d, t, "routes[0].legs[0].summary"

    return None, None, "unknown"

def parse_matrix_any(payload):
    """Per Mappls Distance Matrix API: {"results":{"code":"Ok","distances":[[...]],
    "durations":[[...]]}} — row-ordered, source->target. distances in meters,
    durations in seconds."""
    out = {"ok": False, "distance_m": None, "duration_sec": None, "source": "unknown", "raw": payload}
    if not isinstance(payload, dict):
        return out

    results = payload.get("results")
    if isinstance(results, list) and results:
        results = results[0]

    if isinstance(results, dict):
        if "distances" in results or "durations" in results:
            d = extract_point_to_point(results.get("distances"))
            t = extract_point_to_point(results.get("durations"))
            if d is not None or t is not None:
                out.update({"ok": True, "distance_m": d, "duration_sec": t, "source": "results.distances/durations"})
                return out

        stt = results.get("sources_to_targets")
        if isinstance(stt, list):
            dists, times = [], []
            for row in stt:
                if isinstance(row, list):
                    for cell in row:
                        if isinstance(cell, dict):
                            for key, arr in [("distance", dists), ("time", times), ("duration", times)]:
                                if key in cell:
                                    try:
                                        v = float(cell[key])
                                        if np.isfinite(v) and v >= 0:
                                            arr.append(v)
                                    except Exception:
                                        pass
            if dists or times:
                out.update({
                    "ok": True,
                    "distance_m": dists[0] if dists else None,
                    "duration_sec": times[0] if times else None,
                    "source": "results.sources_to_targets",
                })
                return out

    d = deep_first_value(payload, ["distance", "distances", "length"])
    t = deep_first_value(payload, ["duration", "durations", "time"])
    d = extract_point_to_point(d) if isinstance(d, list) else (safe_float(d, None) if d is not None else None)
    t = extract_point_to_point(t) if isinstance(t, list) else (safe_float(t, None) if t is not None else None)

    if d is not None or t is not None:
        out.update({"ok": True, "distance_m": d, "duration_sec": t, "source": "generic"})
    return out

def make_key(series):
    return series.fillna("").astype(str).str.strip().str.lower()

def fill_coords_from_lookup(df, lookup, key):
    if key not in df.columns or key not in lookup.columns:
        return df.copy()
    work = df.copy()
    lk = lookup.copy()
    if "lat" not in work.columns:
        work["lat"] = np.nan
    if "lon" not in work.columns:
        work["lon"] = np.nan
    work["_join_key"] = make_key(work[key])
    lk["_join_key"] = make_key(lk[key])
    lk = lk[lk["_join_key"].ne("")].copy()
    if lk.empty:
        work.drop(columns=["_join_key"], inplace=True, errors="ignore")
        return work
    lk = lk.groupby("_join_key", as_index=False).agg(lat=("lat", "mean"), lon=("lon", "mean"))
    merged = work.merge(lk, on="_join_key", how="left", suffixes=("", "_aux"))
    if "lat_aux" in merged.columns:
        merged["lat"] = merged["lat"].combine_first(merged["lat_aux"])
        merged["lon"] = merged["lon"].combine_first(merged["lon_aux"])
        merged.drop(columns=["lat_aux", "lon_aux"], inplace=True, errors="ignore")
    merged.drop(columns=["_join_key"], inplace=True, errors="ignore")
    return merged

def backfill_coordinates(df, aux_frames, cluster_col):
    work = df.copy()
    if "lat" not in work.columns:
        work["lat"] = np.nan
    if "lon" not in work.columns:
        work["lon"] = np.nan
    join_keys = [cluster_col, "cluster_label", "hotspot_unit", "dominant_junction_name"]
    for _, aux in aux_frames:
        a = derive_lat_lon(aux.copy()).dropna(subset=["lat", "lon"], how="any")
        if a.empty:
            continue
        for key in join_keys:
            if key in work.columns and key in a.columns:
                work = fill_coords_from_lookup(work, a[[key, "lat", "lon"]].copy(), key)
    return derive_lat_lon(work)

def count_status(df, status):
    if df.empty or "validation_status" not in df.columns:
        return 0
    return int((df["validation_status"] == status).sum())


# =========================
# Mappls client
# =========================
class MapplsClient:
    def __init__(self, access_token: str, timeout: int = 25):
        self.access_token = access_token.strip()
        self.timeout = timeout
        self.session = requests.Session()

    def _access_query(self):
        if not self.access_token:
            raise RuntimeError(
                "MAPPLS_ACCESS_TOKEN is missing. The current Mappls REST docs use the static "
                "access_token query parameter for reverse geocode, routing, and distance matrix."
            )
        return {"access_token": self.access_token}

    def reverse_geocode(self, lat: float, lng: float):
        return self.session.get(
            "https://search.mappls.com/search/address/rev-geocode",
            params={"lat": float(lat), "lng": float(lng), **self._access_query()},
            timeout=self.timeout,
        )

    def _request_first_success(self, url_for_resource, resources):
        """Shared GET loop for route()/distance_matrix(). Tries each resource
        in order and returns the requests.Response for the first one that is
        both an HTTP success AND reports an 'Ok' code in the payload (Mappls
        can return HTTP 200 with an internal error code, e.g. 'NoRoute')."""
        last_resp = None
        for resource in resources:
            params = {**self._access_query(), "rtype": 0}
            if MAPPLS_REGION:
                params["region"] = MAPPLS_REGION
            resp = self.session.get(url_for_resource(resource), params=params, timeout=self.timeout)
            last_resp = resp
            if resp.status_code >= 400:
                continue
            try:
                payload = resp.json()
            except Exception:
                continue
            code = None
            if isinstance(payload, dict):
                code = payload.get("code")
                if code is None and isinstance(payload.get("results"), dict):
                    code = payload["results"].get("code")
            if code is not None and str(code).lower() != "ok":
                continue
            return resp
        return last_resp

    def route(self, start_lon, start_lat, end_lon, end_lat, prefer_live_traffic=True):
        """Returns a requests.Response (so callers can do resp.json()),
        mirroring distance_matrix(). Previously this returned a plain dict,
        called a nonexistent self.get_json(), didn't accept
        prefer_live_traffic, and referenced an unset last_error — all fixed
        by sharing _request_first_success with distance_matrix()."""
        resources = []
        if prefer_live_traffic:
            resources.extend(["route_eta", "route_traffic"])
        resources.append("route_adv")
        return self._request_first_success(
            lambda resource: (
                f"https://route.mappls.com/route/direction/{resource}/{MAPPLS_PROFILE}/"
                f"{start_lon},{start_lat};{end_lon},{end_lat}"
            ),
            resources,
        )

    def distance_matrix(self, start_lon, start_lat, end_lon, end_lat, prefer_live_traffic=True):
        resources = []
        if prefer_live_traffic:
            resources.extend(["distance_matrix_eta", "distance_matrix_traffic"])
        resources.append("distance_matrix")
        return self._request_first_success(
            lambda resource: (
                f"https://route.mappls.com/route/dm/{resource}/{MAPPLS_PROFILE}/"
                f"{start_lon},{start_lat};{end_lon},{end_lat}"
            ),
            resources,
        )


# =========================
# Auxiliary source loading
# =========================
def load_aux_frames():
    frames = []
    for path, df in load_all_existing(AUX_PATHS):
        if "st_dbscan_cluster_id" in df.columns or "cluster_id" in df.columns or "dbscan_cluster_id" in df.columns:
            frames.append((path, df))
    return frames


# =========================
# Main
# =========================
def main():
    if not MAPPLS_ACCESS_TOKEN:
        raise RuntimeError("MAPPLS_ACCESS_TOKEN is missing. Put the Mappls static key into this environment variable.")

    client = MapplsClient(MAPPLS_ACCESS_TOKEN, timeout=REQUEST_TIMEOUT)

    df, source = load_first_existing(INPUT_PATHS)
    df = df.copy()
    cluster_col = standardize_cluster_col(df)

    if "cluster_label" not in df.columns:
        df["cluster_label"] = df["hotspot_unit"] if "hotspot_unit" in df.columns else "CLUSTER::" + df[cluster_col].astype(str)
    if "cluster_label_display" not in df.columns:
        df["cluster_label_display"] = df["cluster_label"].astype(str) + " (Cluster " + df[cluster_col].astype(str) + ")"

    for col, default in [("ccs_score", 0.0), ("delay_minutes_per_vehicle", 0.0), ("risk_band", "Watch"),
                          ("recommended_action", ""), ("road_class", DEFAULT_ROAD_CLASS),
                          ("link_length_m", np.nan), ("free_flow_time_min", np.nan)]:
        if col not in df.columns:
            df[col] = default

    df["ccs_score"] = pd.to_numeric(df["ccs_score"], errors="coerce").fillna(0.0)
    df["delay_minutes_per_vehicle"] = pd.to_numeric(df["delay_minutes_per_vehicle"], errors="coerce").fillna(0.0)
    df["link_length_m"] = pd.to_numeric(df["link_length_m"], errors="coerce")
    df["free_flow_time_min"] = pd.to_numeric(df["free_flow_time_min"], errors="coerce")

    df = derive_lat_lon(df)

    aux_frames = load_aux_frames()
    if aux_frames:
        df = backfill_coordinates(df, aux_frames, cluster_col)

    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

    # Dedup by cluster_col (the actual unique key), NOT by cluster_label. Multiple
    # distinct ST-DBSCAN clusters can legitimately fall back to the same
    # "POLICE_STATION::X" label when they have no junction name; that is real
    # data, not a duplicate row, and deduping on the label would silently
    # discard genuinely separate hotspots. cluster_label_display (label +
    # cluster ID) is what disambiguates them for humans reading the output.
    df = df.sort_values(
        ["ccs_score", "delay_minutes_per_vehicle"], ascending=[False, False]
    ).drop_duplicates(subset=[cluster_col], keep="first").reset_index(drop=True)

    validate_df = df.head(TOP_N).copy()

    print("Primary source:", source)
    print("Aux sources loaded:", [str(p) for p, _ in aux_frames])
    print("Top rows available:", len(validate_df))
    print("Coordinate rows available:", int(validate_df[["lat", "lon"]].dropna().shape[0]))
    print(validate_df[["cluster_label_display", "lat", "lon"]].head(10).to_string(index=False))

    rows = []
    for _, row in validate_df.iterrows():
        lat = safe_float(row.get("lat"))
        lon = safe_float(row.get("lon"))
        label = clean_text(row.get("cluster_label", ""))
        label_display = clean_text(row.get("cluster_label_display", "")) or f"{label} (Cluster {row.get(cluster_col)})"
        risk_band = clean_text(row.get("risk_band", "Watch"))
        ccs = safe_float(row.get("ccs_score", 0.0), 0.0)
        model_delay_min = safe_float(row.get("delay_minutes_per_vehicle", 0.0), 0.0)
        road_class = clean_text(row.get("road_class", "")) or DEFAULT_ROAD_CLASS
        free_flow_speed = road_speed_kmh(road_class)

        # Free-flow ETA over the SAME synthetic corridor used for the live
        # route/matrix probe below — comparing live-over-corridor against
        # free-flow-over-the-same-corridor is a same-units congestion ratio,
        # unlike comparing it against the model's incremental delay (a
        # different physical quantity measured over a different link).
        corridor_km = VALIDATION_CORRIDOR_METERS / 1000.0
        corridor_free_flow_eta_min = (corridor_km / free_flow_speed) * 60.0 if free_flow_speed > 0 else np.nan

        # The model's own delay is "extra minutes over this link's free-flow
        # time" — express it as a percentage of that same free-flow time
        # rather than dividing it by an unrelated travel time.
        existing_fft = safe_float(row.get("free_flow_time_min"), np.nan)
        link_length_m = safe_float(row.get("link_length_m"), np.nan)
        if pd.notna(existing_fft) and existing_fft > 0:
            free_flow_time_min_link = existing_fft
        elif pd.notna(link_length_m) and free_flow_speed > 0:
            free_flow_time_min_link = link_length_m * 60.0 / (free_flow_speed * 1000.0)
        else:
            free_flow_time_min_link = np.nan
        delay_pct_of_freeflow = (
            100.0 * model_delay_min / free_flow_time_min_link
            if pd.notna(free_flow_time_min_link) and free_flow_time_min_link > 0 else np.nan
        )

        row_out = {
            "cluster_id": row.get(cluster_col),
            "cluster_label": label,
            "cluster_label_display": label_display,
            "risk_band": risk_band,
            "ccs_score": ccs,
            "delay_minutes_per_vehicle_model": model_delay_min,
            "road_class": road_class,
            "free_flow_time_min_link": free_flow_time_min_link,
            "delay_pct_of_freeflow": delay_pct_of_freeflow,
            "latitude": lat,
            "longitude": lon,
            "reverse_geocode_ok": False,
            "reverse_geocode_address": "",
            "reverse_geocode_error": "",
            "route_eta_ok": False,
            "route_eta_distance_m": np.nan,
            "route_eta_duration_sec": np.nan,
            "route_eta_duration_min": np.nan,
            "route_eta_error": "",
            "route_eta_source": "",
            "distance_matrix_ok": False,
            "distance_matrix_source": "",
            "distance_matrix_distance_m": np.nan,
            "distance_matrix_duration_sec": np.nan,
            "distance_matrix_duration_min": np.nan,
            "distance_matrix_error": "",
            "corridor_free_flow_eta_min": corridor_free_flow_eta_min,
            "corridor_congestion_ratio_route": np.nan,
            "corridor_congestion_ratio_matrix": np.nan,
            "validation_status": "Not Validated",
        }

        if not valid_coords(lat, lon):
            row_out["reverse_geocode_error"] = "invalid_coordinates"
            row_out["route_eta_error"] = "invalid_coordinates"
            row_out["distance_matrix_error"] = "invalid_coordinates"
            rows.append(row_out)
            continue

        # Reverse geocode — informational location sanity-check only. It
        # confirms the coordinates resolve to a real address; it says
        # nothing about congestion, so it does NOT feed validation_status.
        try:
            rev_data = client.reverse_geocode(lat, lon).json()
            addr = parse_reverse_geocode_address(rev_data)
            row_out["reverse_geocode_ok"] = bool(addr)
            row_out["reverse_geocode_address"] = addr
            row_out["reverse_geocode_error"] = "" if addr else "empty_or_unparsed"
        except Exception as e:
            row_out["reverse_geocode_error"] = str(e)

        dest_lat, dest_lon = offset_point(lat, lon, meters=VALIDATION_CORRIDOR_METERS, bearing_deg=90.0)

        # Route ETA
        try:
            route_resp = client.route(lon, lat, dest_lon, dest_lat, prefer_live_traffic=True)
            route_data = route_resp.json()
            d_m, t_s, src = parse_route_response(route_data)
            if d_m is not None or t_s is not None:
                row_out["route_eta_ok"] = True
                row_out["route_eta_distance_m"] = d_m
                row_out["route_eta_duration_sec"] = t_s
                row_out["route_eta_duration_min"] = (t_s / 60.0) if t_s is not None and pd.notna(t_s) else np.nan
                row_out["route_eta_source"] = src
            else:
                row_out["route_eta_error"] = "unparsed_route_payload"
        except Exception as e:
            row_out["route_eta_error"] = str(e)

        # Distance matrix
        try:
            dm_data = client.distance_matrix(lon, lat, dest_lon, dest_lat, prefer_live_traffic=True).json()
            parsed = parse_matrix_any(dm_data)
            if parsed["ok"]:
                row_out["distance_matrix_ok"] = True
                row_out["distance_matrix_source"] = parsed["source"]
                row_out["distance_matrix_distance_m"] = parsed["distance_m"]
                row_out["distance_matrix_duration_sec"] = parsed["duration_sec"]
                row_out["distance_matrix_duration_min"] = (
                    parsed["duration_sec"] / 60.0
                    if parsed["duration_sec"] is not None and pd.notna(parsed["duration_sec"]) else np.nan
                )
            else:
                row_out["distance_matrix_error"] = "unparsed_matrix_payload"
        except Exception as e:
            row_out["distance_matrix_error"] = str(e)

        if pd.notna(corridor_free_flow_eta_min) and corridor_free_flow_eta_min > 0:
            if row_out["route_eta_ok"] and pd.notna(row_out["route_eta_duration_min"]):
                row_out["corridor_congestion_ratio_route"] = row_out["route_eta_duration_min"] / corridor_free_flow_eta_min
            if row_out["distance_matrix_ok"] and pd.notna(row_out["distance_matrix_duration_min"]):
                row_out["corridor_congestion_ratio_matrix"] = row_out["distance_matrix_duration_min"] / corridor_free_flow_eta_min

        # Validation status reflects only the two live road-network checks
        # (route ETA, distance matrix) — reverse geocode is excluded since it
        # validates an address, not congestion.
        congestion_checks = [row_out["route_eta_ok"], row_out["distance_matrix_ok"]]
        n_ok = sum(1 for c in congestion_checks if c)
        row_out["validation_status"] = (
            "Validated" if n_ok == len(congestion_checks)
            else "Partially Validated" if n_ok > 0
            else "Not Validated"
        )

        rows.append(row_out)

    validation_df = pd.DataFrame(rows)

    expected_validation_cols = [
        "cluster_id", "cluster_label", "cluster_label_display", "risk_band", "ccs_score",
        "delay_minutes_per_vehicle_model", "road_class", "free_flow_time_min_link", "delay_pct_of_freeflow",
        "latitude", "longitude",
        "reverse_geocode_ok", "reverse_geocode_address", "reverse_geocode_error",
        "route_eta_ok", "route_eta_distance_m", "route_eta_duration_sec",
        "route_eta_duration_min", "route_eta_error", "route_eta_source",
        "distance_matrix_ok", "distance_matrix_source", "distance_matrix_distance_m",
        "distance_matrix_duration_sec", "distance_matrix_duration_min", "distance_matrix_error",
        "corridor_free_flow_eta_min", "corridor_congestion_ratio_route", "corridor_congestion_ratio_matrix",
        "validation_status",
    ]
    for c in expected_validation_cols:
        if c not in validation_df.columns:
            validation_df[c] = np.nan

    merge_cols = [
        "cluster_id", "validation_status", "reverse_geocode_ok", "reverse_geocode_address",
        "route_eta_ok", "route_eta_duration_min", "distance_matrix_ok", "distance_matrix_duration_min",
        "corridor_congestion_ratio_route", "corridor_congestion_ratio_matrix", "delay_pct_of_freeflow",
    ]
    for c in merge_cols:
        if c not in validation_df.columns:
            validation_df[c] = np.nan

    merged = df.merge(
        validation_df[merge_cols].rename(columns={"cluster_id": cluster_col}),
        on=cluster_col, how="left",
    )

    summary = pd.DataFrame([{
        "input_source": str(source),
        "validated_hotspots": count_status(validation_df, "Validated"),
        "partially_validated_hotspots": count_status(validation_df, "Partially Validated"),
        "not_validated_hotspots": count_status(validation_df, "Not Validated"),
        "mean_route_eta_min": float(validation_df["route_eta_duration_min"].dropna().mean()) if validation_df["route_eta_duration_min"].notna().any() else np.nan,
        "mean_distance_matrix_eta_min": float(validation_df["distance_matrix_duration_min"].dropna().mean()) if validation_df["distance_matrix_duration_min"].notna().any() else np.nan,
        "mean_model_delay_min": float(validation_df["delay_minutes_per_vehicle_model"].dropna().mean()) if validation_df["delay_minutes_per_vehicle_model"].notna().any() else np.nan,
        "mean_corridor_congestion_ratio_route": float(validation_df["corridor_congestion_ratio_route"].dropna().mean()) if validation_df["corridor_congestion_ratio_route"].notna().any() else np.nan,
        "mean_corridor_congestion_ratio_matrix": float(validation_df["corridor_congestion_ratio_matrix"].dropna().mean()) if validation_df["corridor_congestion_ratio_matrix"].notna().any() else np.nan,
        "mean_delay_pct_of_freeflow": float(validation_df["delay_pct_of_freeflow"].dropna().mean()) if validation_df["delay_pct_of_freeflow"].notna().any() else np.nan,
    }])

    validation_df.to_csv(OUT_DIR / "validation_layer_top_hotspots.csv", index=False)
    merged.to_csv(OUT_DIR / "phase6_with_validation.csv", index=False)
    summary.to_csv(OUT_DIR / "validation_summary.csv", index=False)

    print("Validation layer complete")
    print("Input source:", source)
    print("Top hotspots validated:", len(validation_df))
    print("Outputs saved to:", OUT_DIR.resolve())
    print("\nSummary:")
    print(summary.to_string(index=False))

    print("\nTop validation rows:")
    show_cols = [
        "cluster_id", "cluster_label_display", "risk_band", "validation_status",
        "reverse_geocode_address", "route_eta_duration_min", "distance_matrix_duration_min",
        "corridor_congestion_ratio_matrix", "delay_minutes_per_vehicle_model", "delay_pct_of_freeflow",
        "distance_matrix_source",
    ]
    print(validation_df[[c for c in show_cols if c in validation_df.columns]].head(TOP_N).to_string(index=False))


if __name__ == "__main__":
    main()

Primary source: content\phase6_outputs_2\phase6_stage6_handoff.csv
Aux sources loaded: ['content\\phase5_outputs_2\\phase5_enriched_records.csv', 'content\\phase4_outputs_2\\phase4_merged_with_prior_scores.csv', 'content\\phase3_outputs_2\\phase3_clustered_dataset.csv']
Top rows available: 20
Coordinate rows available: 20
                              cluster_label_display       lat       lon
                BTP040 - Elite Junction (Cluster 2) 12.973169 77.577548
         BTP051 - Safina Plaza Junction (Cluster 3) 12.981962 77.607733
       BTP044 - Sagar Theatre Junction (Cluster 37) 12.976579 77.577565
           POLICE_STATION::HSR Layout (Cluster 507) 12.909785 77.644517
          POLICE_STATION::Rajajinagar (Cluster 172) 13.018507 77.531826
          POLICE_STATION::Kodigehalli (Cluster 290) 13.071124 77.587949
         BTP027 - Modi Bridge Junction (Cluster 56) 12.999430 77.550354
              BTP208 - Havanur Circle (Cluster 408) 12.987071 77.537450
BTP189 - New Diagonal Road, 